# Length-masked scaled softmax in Triton

Attention over a padded batch needs a softmax where row $i$ only covers its first $\ell_i$ columns; the rest are padding and must get probability exactly 0. Passing the lengths to the kernel avoids materialising a `-inf` mask tensor in memory.

Write `masked_softmax(x, lengths, scale)`.

## Inputs
- `x`: contiguous `float32` CUDA tensor `(M, N)`, finite entries, $1 \le N \le 8192$.
- `lengths`: `int64` CUDA tensor `(M,)` with $0 \le \ell_i \le N$.
- `scale`: positive Python float (e.g. $1/\sqrt{d}$), applied to the logits before the softmax.

## Output
- `float32` tensor `(M, N)`:
$$y_{ij} = \begin{cases} \dfrac{e^{s x_{ij} - m_i}}{\sum_{k < \ell_i} e^{s x_{ik} - m_i}} & j < \ell_i \\[1ex] 0 & j \ge \ell_i \end{cases}, \qquad m_i = \max_{k < \ell_i} s\,x_{ik}$$
- A row with $\ell_i = 0$ is all zeros (not NaN).

## Requirements
- One program per row, `BLOCK = triton.next_power_of_2(N)`; load the row's length inside the kernel with `tl.load(lengths_ptr + row)`.
- Two masks: `cols < N` (memory safety) and `cols < length` (which entries count). Padding must not affect the max or the sum.

This problem runs in **Colab with a GPU runtime**. Expected outputs come from an equivalent PyTorch reference.

**Runtime → Change runtime type → GPU** before running.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch
import triton
import triton.language as tl

def masked_softmax(x, lengths, scale):
    """x: (M, N) float32 CUDA, lengths: (M,) int64 CUDA. Row i is a softmax over its first lengths[i] columns."""
    raise NotImplementedError


@triton.jit
def _masked_softmax_kernel(x_ptr, len_ptr, y_ptr, N, stride, scale, BLOCK: tl.constexpr):
    pass


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 2.0, 3.0, 4.0], [1.0, 2.0, 3.0, 4.0]], "dtype": "float32"}, {"__nd__": [4, 2], "dtype": "int64"}, 1.0], "e": {"__nd__": [[0.032058604061603546, 0.08714431524276733, 0.23688282072544098, 0.6439142823219299], [0.2689414322376251, 0.7310585975646973, 0.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.0, 0.0, 0.0], [5.0, 1.0, 9.0]], "dtype": "float32"}, {"__nd__": [0, 1], "dtype": "int64"}, 0.5], "e": {"__nd__": [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[-1.1756094694137573, 0.14976119995117188, -0.49557051062583923, -0.5039081573486328, 0.18784762918949127, -0.6771332621574402, 0.89076828956604, -0.14667858183383942, 1.7042090892791748, -1.971096396446228, -1.0819799900054932, -2.568249225616455, -5.009883880615234, -6.697498321533203, 0.18035899102687836, -2.4915268421173096, -2.265411138534546, -1.7421423196792603, 1.2531708478927612, 1.7034257650375366, 2.058051109313965, 2.7743895053863525, 8.947563171386719, 0.8841571807861328, 0.4631015658378601, -6.546604633331299, -5.052181720733643, 6.578852653503418, 8.237359046936035, 1.339036464691162, -0.2408456951379776, 1.6712665557861328, -6.105839252471924, -1.5918214321136475, -1.2162561416625977, 5.206497669219971, -2.9737210273742676, 1.0890623331069946, 2.172307014465332, 3.240946054458618, -1.2870888710021973, -2.0612082481384277, -6.530261993408203, 3.442762851715088, 3.546600103378296, -1.4697952270507812, 0.6933978199958801, -4.188180923461914, -0.6278021931648254, 2.1827096939086914], [-1.450202465057373, 0.18926075100898743, 3.2999274730682373, 1.8013056516647339, -1.8384935855865479, 4.844215393066406, 5.323814868927002, 3.010432243347168, 1.8020071983337402, 2.1473050117492676, 2.6455490589141846, -3.7880165576934814, 3.238617420196533, -2.5214335918426514, 0.6002119779586792, -0.29663971066474915, 1.4241622686386108, 5.275801658630371, 1.598612666130066, -0.3557108938694, 2.0086731910705566, 0.34765923023223877, -4.96127986907959, -5.433441162109375, -1.6000770330429077, 5.972759246826172, -0.8429880738258362, 1.9663116931915283, 3.8320789337158203, -3.9909801483154297, 4.0434746742248535, 2.5529532432556152, 1.603933572769165, 2.8762195110321045, -6.21357536315918, 0.4802527129650116, -4.689764499664307, 0.9884641766548157, -0.5715824365615845, -1.3430638313293457, 1.7027472257614136, -5.366713047027588, 0.2000051736831665, 6.520041465759277, 3.083176374435425, -2.9020142555236816, 1.2446043491363525, -0.5405672788619995, 1.4803835153579712, -3.57521915435791], [1.4133334159851074, 0.00193138443864882, -3.5113813877105713, 2.0958361625671387, 4.113320827484131, 3.461108922958374, -7.921792984008789, 1.0444018840789795, 1.0958276987075806, -0.5824476480484009, -0.5253745317459106, 1.281101107597351, 2.022529363632202, -2.178626775741577, 0.656289279460907, -0.7097569108009338, -2.252643346786499, 1.1177380084991455, -1.7265139818191528, -5.29663610458374, -1.6152801513671875, -5.3320417404174805, 2.7494089603424072, -0.8539976477622986, 2.9980952739715576, -6.104966640472412, 4.848141193389893, 1.4415130615234375, -3.440793991088867, 6.165755748748779, -2.160352945327759, -3.4562346935272217, -0.48872828483581543, 1.2132165431976318, 2.6106619834899902, 7.033133029937744, -2.590837240219116, -5.443063259124756, 0.9298271536827087, -3.851266384124756, -4.044329643249512, -0.17917244136333466, 3.9881348609924316, -2.5831363201141357, -2.2517244815826416, -0.7077205181121826, -1.069008231163025, 3.3347203731536865, 0.03392186015844345, -3.8094284534454346], [7.1926350593566895, -1.9175753593444824, -5.0111918449401855, 1.7707473039627075, 0.126894012093544, 1.8769826889038086, -0.26599687337875366, 0.06376848369836807, 0.5041853785514832, 2.7073612213134766, -6.672126293182373, 0.3895638585090637, 0.017454732209444046, 6.62544059753418, -2.2266974449157715, 3.7516579627990723, 2.84649658203125, -2.056030750274658, 1.8223570585250854, 2.6926398277282715, 3.899653911590576, 2.783432722091675, -0.15336695313453674, 1.6928828954696655, -1.8411686420440674, -2.6933484077453613, -1.3239881992340088, -2.1014723777770996, -2.4663472175598145, 5.013635635375977, -5.327541828155518, 1.1660865545272827, 1.2716022729873657, -3.4562971591949463, 4.177831649780273, -2.940185785293579, 0.6734140515327454, -5.090453624725342, -4.155025482177734, -2.4663732051849365, 0.9069177508354187, 4.719589710235596, -3.6322782039642334, 2.568462371826172, -1.7254072427749634, 2.2495813369750977, 2.1992125511169434, 0.64356529712677, 1.011252522468567, -5.609788417816162], [2.1624512672424316, 3.768672466278076, -0.30411162972450256, 2.5346832275390625, 2.3892223834991455, 3.7672040462493896, -2.3298563957214355, -1.550834059715271, 2.42746639251709, 2.3236401081085205, 4.317752361297607, -0.7006610631942749, 0.08764417469501495, -2.111772060394287, -0.6230275630950928, 4.2574143409729, 0.06134536862373352, 1.6721574068069458, -0.12196460366249084, -0.8783535361289978, -0.007183755282312632, -0.07058069109916687, -2.956589460372925, -2.252070903778076, -0.5711360573768616, 1.7867850065231323, -0.23776213824748993, -4.726049900054932, 8.055580139160156, 0.9463464617729187, -0.9439440369606018, 0.3506219685077667, 0.011148334480822086, 0.1268870085477829, 1.7640817165374756, 1.4536490440368652, 0.5201403498649597, 3.387657642364502, -2.780216932296753, 6.302295207977295, 0.8681197762489319, -2.9240570068359375, -5.831695556640625, 0.9056681990623474, 2.358388662338257, -0.9799613952636719, -0.6115046739578247, -1.1522465944290161, 1.9394599199295044, 0.9494418501853943], [-4.36054801940918, 1.379385232925415, 0.27059394121170044, 1.4966425895690918, -2.5059971809387207, 2.6151585578918457, 4.304239749908447, -6.389331340789795, 2.9858810901641846, -0.05380351468920708, -0.14216746389865875, -0.7054613828659058, -1.01371169090271, 0.808616578578949, -1.0415292978286743, -7.397590637207031, 2.0959155559539795, 0.5211091041564941, 2.0416340827941895, -5.4730658531188965, 1.5156784057617188, -2.0602505207061768, 2.5274908542633057, 0.9200307726860046, 4.520020961761475, 3.53851580619812, 3.3301286697387695, -6.539901256561279, 3.95735502243042, 2.0896048545837402, -0.10547523945569992, 3.82566499710083, 0.7968676686286926, 1.3264617919921875, 1.614248275756836, -1.437477946281433, 0.9421896934509277, -4.167627811431885, 2.585510492324829, -0.13615624606609344, 1.5158113241195679, -2.612306594848633, -2.119745969772339, 0.9868976473808289, 4.465237140655518, -4.4115166664123535, -4.653727054595947, -1.8826743364334106, 3.5247933864593506, 0.43970683217048645]], "dtype": "float32"}, {"__nd__": [50, 1, 0, 25, 49, 13], "dtype": "int64"}, 0.125], "e": {"__nd__": [[0.015954962000250816, 0.01882980950176716, 0.017370523884892464, 0.017352430149912834, 0.01891966722905636, 0.01698073372244835, 0.020657265558838844, 0.018144840374588966, 0.022868197411298752, 0.01444479264318943, 0.016142791137099266, 0.013405832462012768, 0.009879733435809612, 0.008000737987458706, 0.018901966512203217, 0.013535017147660255, 0.013923035003244877, 0.014864164404571056, 0.021614564582705498, 0.022865958511829376, 0.023902365937829018, 0.026141386479139328, 0.056552331894636154, 0.02064020186662674, 0.019581960514187813, 0.008153077214956284, 0.009827635250985622, 0.04205917939543724, 0.05174826458096504, 0.021847808733582497, 0.017932511866092682, 0.02277422323822975, 0.00861488189548254, 0.015146104618906975, 0.01587410271167755, 0.03542909026145935, 0.012743303552269936, 0.02117568999528885, 0.02424618788063526, 0.02771127223968506, 0.015734171494841576, 0.01428299956023693, 0.00816974975168705, 0.028419241309165955, 0.028790516778826714, 0.015378902666270733, 0.02015385963022709, 0.010948455892503262, 0.017085766419768333, 0.024277737364172935], [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.08248056471347809, 0.026411153376102448, 0.01794092357158661, 0.0418807677924633, 0.03410162404179573, 0.04244062677025795, 0.0324673093855381, 0.03383359685540199, 0.0357484333217144, 0.04708259552717209, 0.014577329158782959, 0.035239893943071365, 0.03363829478621483, 0.07683523744344711, 0.025410084053874016, 0.053647805005311966, 0.047908615320920944, 0.0259579885751009, 0.04215182363986969, 0.04699603468179703, 0.05464949831366539, 0.04753243923187256, 0.03292763978242874, 0.041475117206573486, 0.02666460908949375, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.02372773177921772, 0.02900366298854351, 0.017432287335395813, 0.02485784702003002, 0.024409949779510498, 0.02899833954870701, 0.013532659970223904, 0.014916735701262951, 0.024526922032237053, 0.02421066164970398, 0.03106423281133175, 0.01658925786614418, 0.018307184800505638, 0.013906641863286495, 0.01675102673470974, 0.03083081915974617, 0.018247101455926895, 0.022317200899124146, 0.01783374510705471, 0.016224848106503487, 0.018091460689902306, 0.017948659136891365, 0.012512953020632267, 0.013664881698787212, 0.016860034316778183, 0.02263927459716797, 0.01757746748626232, 0.01003001257777214, 0.04956503212451935, 0.02038157545030117, 0.016092367470264435, 0.01891898363828659, 0.01813296601176262, 0.01839720830321312, 0.02257511578500271, 0.021715888753533363, 0.019324149936437607, 0.02765468694269657, 0.012791884131729603, 0.03981027007102966, 0.020183248445391655, 0.012563941068947315, 0.008735333569347858, 0.02027820236980915, 0.024316050112247467, 0.01602007821202278, 0.01677517220377922, 0.01567876525223255, 0.02307547815144062, 0.0], [0.042906731367111206, 0.08792818337678909, 0.076548270881176, 0.08922645449638367, 0.054100725799798965, 0.10261581838130951, 0.12673866748809814, 0.03329578414559364, 0.10748296976089478, 0.07350634783506393, 0.07269890606403351, 0.06775610893964767, 0.06519503146409988, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[13.161988258361816, -2.1495699882507324, -8.802547454833984, -2.2821531295776367, 4.414802074432373, -10.423896789550781, 2.2959392070770264, 0.8304921388626099, -1.4617174863815308, 5.758790016174316, -13.966667175292969, -8.362442016601562, 5.191911220550537, 2.02009654045105, 5.4057817459106445, 8.09057903289795, 19.819860458374023, 8.44705581665039, 13.99714469909668, -0.03688129782676697, 9.089924812316895, 3.656968832015991, -5.381086826324463, -1.5296149253845215, -3.1702985763549805, -6.118169784545898, -5.33003044128418, 19.741531372070312, -9.326504707336426, -1.476709246635437, -23.605022430419922, -7.284421443939209, 1.988882303237915, 10.113293647766113, -13.30636978149414, 0.6220518350601196, -5.97249698638916, 0.9530009627342224, -8.19292163848877, -0.18091382086277008, 3.019934892654419, -0.3396311402320862, 16.656553268432617, 6.0758137702941895, 6.279447078704834, -19.192888259887695, 11.772032737731934, 6.052572727203369, 15.018142700195312, -4.958288669586182, 4.7183427810668945, 24.433181762695312, 2.6242618560791016, 2.4512038230895996, 5.617048263549805, 12.670074462890625, -4.742009162902832, 2.9806432723999023, -13.237831115722656, -17.541309356689453, -8.38158893585205, -8.785453796386719, -17.301715850830078, -13.548843383789062, -9.85288143157959, 2.6732943058013916, 3.2775814533233643, 9.171693801879883, 0.225314661860466, -7.506575107574463, -10.859667778015137, 6.265798091888428, -0.5446686744689941, 6.469882488250732, -7.5085272789001465, -11.418231964111328, -3.350656032562256, 27.912418365478516, 2.2102103233337402, -0.2899245619773865, 1.8598604202270508, -9.583098411560059, 18.974363327026367, -12.368130683898926, -3.6599714756011963, -1.843010425567627, -13.479762077331543, -5.771039009094238, -8.777702331542969, 2.91629958152771, -4.717564582824707, 7.7419114112854, 10.022662162780762, -20.60190200805664, 5.601554870605469, 1.0771576166152954, -6.1886067390441895, 9.8951997756958, 2.3089611530303955, -9.43933391571045, -15.028327941894531, -1.0286000967025757, 5.767258644104004, -8.37636661529541, 20.286527633666992, -19.58633804321289, 3.5146145820617676, -6.518849849700928, 12.568846702575684, 6.184787750244141, 3.4471993446350098, -5.506313323974609, -7.793900012969971, 12.101880073547363, -6.903557777404785, -2.148469924926758, -10.15246868133545, -2.6553237438201904, -23.247730255126953, 20.774864196777344, 12.585531234741211, -13.741106986999512, -12.80722713470459, -0.9904564023017883, 4.443697452545166, 4.01511287689209, -14.1458101272583, 1.800720453262329, -2.6842784881591797, -8.205278396606445, 10.716873168945312, 2.7668251991271973, 5.057294845581055, 1.7959030866622925, -2.103379726409912, 19.951305389404297, -4.069231033325195, -18.98407554626465, -3.27754545211792, -15.18305492401123, 9.524700164794922, 13.477065086364746, -5.327094554901123, 4.577779769897461, -4.1472554206848145, 11.820356369018555, 13.92636489868164, 2.749098539352417, -1.3490264415740967, -8.588027954101562, -20.41417694091797, -7.946934700012207, -2.5081629753112793, -2.0485494136810303, 6.833839416503906, 4.823599338531494, 1.905922532081604, 12.379822731018066, -14.74359130859375, 8.277835845947266, -9.866973876953125, 4.458349704742432, 11.992935180664062, 16.283050537109375, -1.3395715951919556, -3.4690921306610107, 10.173083305358887, 7.420513153076172, 4.88929557800293, -3.8952724933624268, -11.101497650146484, -12.281036376953125, 27.29277229309082, -5.209298133850098, 2.0908076763153076, 14.116668701171875, -10.851993560791016, -6.379504680633545, 11.087737083435059, 5.444151401519775, -14.559537887573242, -7.402124404907227, -2.7849225997924805, -2.715831995010376, 6.86953067779541, -5.002782344818115, -11.527490615844727, -2.8278627395629883, 15.60174560546875, -8.551370620727539, -0.6341481804847717, 4.816810607910156, -1.7637500762939453, 18.263277053833008, -14.201127052307129, -3.156984567642212, 6.9121904373168945, 6.569048881530762, 10.331975936889648, -2.7822399139404297, 7.991807460784912, -10.335114479064941, -6.659657955169678, 7.589632511138916, -5.655186653137207, 10.832042694091797, 0.9617633819580078, 2.3549580574035645, -4.521294593811035, -22.231670379638672, 18.21872329711914, -11.122743606567383, 7.614285945892334, 2.5858888626098633, -4.625058650970459, -6.674744606018066, 7.158378601074219, 4.602610111236572, -3.4826958179473877, 6.014728546142578, 1.0008047819137573, -0.07119197398424149, -1.9654771089553833, -0.2157672494649887, 6.238216876983643, -9.766219139099121, -2.700024127960205, -7.38229513168335, 0.6086840629577637, 5.509860515594482, -13.546616554260254, 14.8157377243042, -17.608768463134766, -14.573079109191895, -17.97663116455078, 9.313002586364746, -4.938816547393799, 18.14333724975586, -2.559980869293213, 10.791009902954102, -11.555163383483887, 2.7058911323547363, -4.620970249176025, -9.319265365600586, 19.525646209716797, 11.879400253295898, -4.953739166259766, -14.949539184570312, -0.993739664554596, -2.7310292720794678, 1.0313904285430908, -3.8582098484039307, -5.083491802215576, 3.202343225479126, 17.68696403503418, 8.132197380065918, -7.9184136390686035, -0.6661643385887146, -2.304682970046997, 7.334987640380859, 1.8834198713302612, 1.4394692182540894, 5.857111930847168, -9.61978530883789, -3.397665500640869, -0.8210075497627258, -10.348711967468262, 9.967475891113281, 10.416752815246582, -2.565319538116455, 4.600034236907959, -15.981379508972168, -1.1917122602462769, -2.1656134128570557, 13.358229637145996, 1.9530166387557983, 10.712736129760742, 10.04163646697998, 4.011412143707275, -2.494986057281494, 0.9838303923606873, -8.159585952758789, 17.6379337310791, -1.5797189474105835, 1.1632386445999146, -4.859236240386963, -2.7396836280822754, 3.163351058959961, 10.453158378601074, 21.37204360961914, -3.537484645843506, -2.757885456085205, -0.9554151296615601, -31.924684524536133, -9.11646842956543, -12.655150413513184, 3.6242499351501465, 7.70128059387207, -6.764045715332031, -0.904556393623352, 3.950983762741089, -19.30223274230957, -12.395447731018066, 5.3207292556762695, -13.977132797241211, 9.374338150024414, 5.894630432128906, -5.278774261474609, -5.31016731262207, -15.090977668762207, 1.9814574718475342, 3.580960750579834, 8.536296844482422, 13.857855796813965, 15.281712532043457, -0.05760500580072403, -1.991956353187561, -9.242365837097168, 19.41974639892578, 9.512919425964355, -3.9954581260681152, -3.6798157691955566, 4.729442596435547, 1.1504645347595215, 2.7314295768737793, -13.411266326904297, -8.989614486694336, 2.9551193714141846, -9.130840301513672, 5.402370929718018, 5.394044876098633, 3.103654384613037, 7.5585150718688965, 0.9408276677131653, -0.44883814454078674, -0.9683939814567566, -9.139944076538086, -0.29240602254867554, 2.8895959854125977, -21.25568962097168, 8.708008766174316, -8.234394073486328, -11.83859634399414, -1.0680692195892334, -1.265534520149231, -0.20072932541370392, -9.603740692138672, 5.41990852355957, -22.255535125732422, -15.076682090759277, -1.4041385650634766, 9.918231010437012, 3.656947374343872, 3.4005589485168457, 19.389244079589844, -14.914843559265137, -5.395521640777588, -1.7212742567062378, 8.8465576171875, -17.5025577545166, -13.373682022094727, 2.3109874725341797, -18.693246841430664, -13.47557544708252, -16.656288146972656, -19.776226043701172, 5.68568229675293, 0.45035383105278015, 8.811066627502441, -2.2886266708374023, 17.489233016967773, -12.561939239501953, -7.93062162399292, -7.185144901275635, 2.42954683303833, 0.3555668592453003, 11.831303596496582, -6.772313117980957, 1.0593467950820923, -6.933228969573975, -4.1737823486328125, 7.5336079597473145, 9.01097583770752, -7.454758167266846, -14.170341491699219, 21.59665870666504, -2.258505344390869, -6.069632530212402, 12.204659461975098, -17.16802978515625, 8.857755661010742, -1.524044394493103, -2.137028932571411, -4.3855881690979, 11.67398738861084, -2.41945219039917, -9.770316123962402, 4.747681617736816, -25.859838485717773, 16.146026611328125, 0.570761501789093, 5.37479305267334, -12.199347496032715, -20.38832664489746, 4.504761695861816, -13.721057891845703, 2.5371809005737305, -12.19194221496582, 8.22819995880127, -4.052770137786865, -10.655179977416992, 0.9036691784858704, -14.857396125793457, 16.55010986328125, 10.536888122558594, -3.8813319206237793, -7.0932793617248535, -11.068055152893066, -9.916292190551758, 18.575096130371094, 13.784893035888672, 17.472951889038086, -19.671009063720703, -1.7071856260299683, -27.48060417175293, -10.81442642211914, 5.975569725036621, -20.74277114868164, 13.339415550231934, 0.9083097577095032, -12.589850425720215, 1.3496036529541016, -11.544288635253906, -3.255857229232788, 12.831385612487793, 18.98856544494629, -11.024370193481445, -5.840024471282959, -2.3580400943756104, -0.22117526829242706, 5.147176265716553, -7.420191764831543, -10.046785354614258, -8.316475868225098, 4.203827381134033, 10.913154602050781, -4.6679368019104, 11.359298706054688, -1.2895281314849854, 14.378268241882324, -14.627060890197754, 2.471867084503174, 10.42628288269043, -3.9196553230285645, -2.957685708999634, -8.208270072937012, 16.8492374420166, -6.594557285308838, 8.304550170898438, 22.517093658447266, 4.335701942443848, 16.81671142578125, 7.483482837677002, 6.621483325958252, -4.198592662811279, -7.740159034729004, 7.999410629272461, -0.42552605271339417, -14.938944816589355, 2.5889532566070557, 3.1775763034820557, 4.038796901702881, -0.07774120569229126, -3.1198041439056396, 0.9122296571731567, 1.943561315536499, 8.730182647705078, -3.9864697456359863, 0.0056024943478405476, 2.3280012607574463, 6.714095115661621, -5.5071611404418945, 4.006435394287109, -7.19072151184082, -9.032703399658203, -4.724198341369629, -5.038190841674805, -5.844463348388672, 1.960623025894165, 5.8545050621032715, -6.748039245605469, 0.11468172073364258, -6.189560890197754, -7.378293991088867, -15.103572845458984, -12.634324073791504, 5.231472492218018, -3.2695391178131104, 4.130257606506348, -3.792585849761963, 17.397926330566406, -2.4671452045440674, 0.7444008588790894, -5.18187141418457, 21.062469482421875, 2.1384105682373047, -5.60494327545166, 2.136298179626465, -14.294225692749023, -0.0512174516916275, -4.268101692199707, 6.9647722244262695, -0.14570091664791107, -10.051069259643555, -16.514081954956055, -8.761244773864746, -0.44588738679885864, 6.4337310791015625, 6.163395881652832, -8.354241371154785, -0.1996953785419464, 6.8099365234375, 3.2873129844665527, -13.486982345581055, -9.017709732055664, -6.191053867340088, -10.786551475524902, -11.919084548950195, -7.502445220947266, 9.488065719604492, 1.980049729347229, 7.756030082702637, 6.6692585945129395, 7.4796013832092285, -15.395220756530762, 0.9699109792709351, -9.887810707092285, -5.044336795806885, 3.154423713684082, 12.317456245422363, 4.319093227386475, -0.09939523786306381, -14.749637603759766, 0.19366227090358734, 13.609841346740723, 2.3965835571289062, 2.6080830097198486, -8.735984802246094, 3.254856824874878, -12.074867248535156, -6.320134162902832, -6.7547430992126465, -2.767139196395874, -8.797809600830078, -2.785956859588623, 5.635469436645508, -3.3353676795959473, 13.513612747192383, -15.59471607208252, 1.8839820623397827, -12.871356010437012, 5.377176761627197, -12.90531063079834, -13.943193435668945, 18.305315017700195, -18.151975631713867, 3.4256863594055176, 3.1883649826049805, 0.09748312085866928, -3.240856647491455, -15.10387897491455, -15.411019325256348, -5.53904390335083, -0.9300959706306458, -14.770008087158203, -5.46279239654541, 18.016796112060547, 6.706137657165527, 11.639212608337402, 5.518589973449707, -15.384299278259277, 1.9996401071548462, 2.4931106567382812, 11.438897132873535, 24.85355567932129, -5.79610013961792, 1.4863964319229126, 7.909490585327148, -9.553257942199707, -10.120027542114258, 12.003312110900879, 4.448358535766602, 6.648266792297363, 6.569711208343506, 7.122049808502197, -8.250195503234863, -5.15674352645874, -9.36945629119873, 11.141763687133789, -9.344314575195312, -4.523674488067627, 6.303658962249756, 6.303751468658447, -2.566694974899292, 3.7984063625335693, 7.107702255249023, -12.919417381286621, 9.13209056854248, 13.642217636108398, 21.05194091796875, 6.641457557678223, -6.5211005210876465, 0.05364192649722099, -6.689523220062256, 4.812647342681885, -4.745944023132324, 15.941490173339844, -18.701215744018555, 0.377008318901062, 4.671269416809082, 0.3796728253364563, 7.042022228240967, 1.102215051651001, -22.84156036376953, -7.695511817932129, 21.767017364501953, -4.624363899230957, 5.172214984893799, 9.239542007446289, 8.112342834472656, 3.2653191089630127, 14.339609146118164, 2.390305757522583, 15.893965721130371, 4.134835720062256, -6.071535110473633, -3.3762762546539307, -2.597280740737915, -10.307345390319824, -2.0357134342193604, -0.48608601093292236, 6.058767795562744, -13.455842018127441, 4.2700700759887695, 8.233503341674805, 7.849461078643799, -15.693925857543945, -0.392381489276886, -8.661552429199219, 35.0371208190918, 8.607354164123535, -5.225204944610596, -9.065762519836426, -7.82639217376709, 2.8848190307617188, 5.682385444641113, -3.677131175994873, -8.347132682800293, -19.217613220214844, 9.531309127807617, 5.71184778213501, 6.230429649353027, -14.864043235778809, -5.910004615783691, -1.6227909326553345, 0.96510249376297, 4.284754753112793, 21.215621948242188, 7.097616672515869, 7.698879718780518, -6.526393413543701, -5.380580902099609, 2.298732280731201, 14.914618492126465, -15.482389450073242, -7.178576469421387, 25.351516723632812, 16.08926010131836, 10.554616928100586, 8.864693641662598, -10.27461051940918, -9.413496971130371, 4.2500152587890625, -2.437391757965088, -6.101715564727783, -6.487560749053955, 12.178786277770996, -4.677134037017822, 2.058648109436035, -7.111804008483887, -8.39887809753418, -3.0941786766052246, -10.136433601379395, 11.704991340637207, -8.866151809692383, -2.9300200939178467, 12.788928031921387, 0.9544746279716492, -5.808541774749756, -19.193950653076172, 8.608859062194824, -11.43600082397461, 2.864952564239502, -7.34470272064209, 8.916991233825684, -5.776272773742676, 2.6486237049102783, 4.205078601837158, -12.792076110839844, 7.933542728424072, -7.687903881072998, 2.952127456665039, -3.680079936981201, -4.033621788024902, -14.41673755645752, -4.119650363922119, -23.091354370117188, -0.0984809473156929, -7.077462196350098, 2.0135531425476074, 12.828474998474121, -0.46486765146255493, 0.6514850854873657, 3.222008466720581, -6.095216751098633, 7.426519870758057, -4.2875590324401855, 2.275327205657959, -11.213029861450195, 2.145138740539551, 2.419306993484497, -5.317385673522949, 18.002304077148438, -3.2344982624053955, 11.945804595947266, 10.937408447265625, 5.945164680480957, -0.07261458039283752, -12.022443771362305, -0.872829258441925, -5.727385997772217, 3.719374179840088, 6.4812469482421875, -9.125505447387695, 21.810171127319336, 15.798384666442871, -9.53215217590332, -2.659752368927002, 0.35586848855018616, 2.8319876194000244, 8.08514404296875, -5.713850021362305, -8.399985313415527, -3.8510241508483887, 8.107540130615234, 11.792920112609863, 11.567264556884766, -9.923542022705078, 14.12065601348877, -3.421163558959961, -8.3600492477417, 18.609844207763672, 4.993492126464844, -3.1508655548095703, 3.1380534172058105, 11.449775695800781, 5.577277660369873, -1.6905291080474854, 11.89269733428955, 18.570518493652344, -11.722559928894043, 5.459750175476074, -1.2582629919052124, -9.314393997192383, 0.8495781421661377, -23.30936050415039, -11.529548645019531, 9.050485610961914, 12.373964309692383, 2.3217689990997314, -11.654739379882812, 4.884278297424316, 3.6531026363372803, -9.334684371948242, -12.761693954467773, -1.7175556421279907, -3.448085308074951, 0.6938351988792419, -7.54813814163208, -10.048238754272461, 2.218672275543213, 5.810170650482178, 4.482822895050049, 5.664947509765625, 10.708839416503906, -1.6099802255630493, -4.571232795715332, -12.593936920166016, -14.140762329101562, 12.844315528869629, 9.928497314453125, -4.410605430603027, 5.756847858428955, 2.273883581161499, -7.9934401512146, -2.9691834449768066, 14.105079650878906, 1.383743166923523, 9.842730522155762, 0.9664629101753235, -7.37325382232666, 5.460691928863525, 0.3599492907524109, 11.511259078979492, -1.8991072177886963, 16.248796463012695, -10.17008113861084, -1.546784520149231, -12.515851974487305, 5.0568742752075195, 18.06166648864746, 6.048027515411377, -12.313427925109863, 13.237098693847656, -6.697940826416016, 15.473082542419434, -15.192273139953613, 7.700493812561035, -14.699713706970215, 2.147545099258423, -5.380163669586182, -11.994272232055664, 8.238374710083008, -1.9428397417068481, -6.334821701049805, -1.4120062589645386, 4.586361885070801, 16.63300323486328, -4.584004878997803, -0.6406176090240479, -2.726362705230713, -10.439269065856934, -7.169246196746826, -11.354171752929688, 11.089414596557617, 1.3014413118362427, -0.06325595825910568, 2.8616392612457275, -5.048418045043945, 10.396781921386719, 13.630223274230957, -16.509424209594727, 10.209205627441406, 18.243783950805664, -5.774768829345703, 5.79795503616333, 3.008200168609619, 13.643677711486816, 2.7964279651641846, -12.182863235473633, 1.5254579782485962, 8.40479564666748, 14.440133094787598, 0.9433978199958801, -13.524648666381836, 6.095663547515869, 12.736310005187988, -8.433113098144531, -10.301602363586426, 2.865935802459717, 3.8618311882019043, 0.4809876084327698, -7.031411170959473, -4.432997703552246, -5.535326957702637, -9.081043243408203, 14.975037574768066, -2.679841995239258, -4.445947647094727, 5.067981243133545, 1.770655632019043, 12.675410270690918, -11.687237739562988, 9.675548553466797, -8.941849708557129, 10.40319538116455, -7.701316833496094, 11.798905372619629, -0.5944989919662476, 10.636123657226562, -11.229362487792969, 10.318824768066406, 18.12971305847168, 8.511232376098633, -1.6484462022781372, -8.949838638305664, 13.106953620910645, 1.506191372871399, -4.763387680053711, -3.737478494644165, 17.223194122314453, 9.952264785766602, 13.332401275634766, 14.898712158203125, -9.825104713439941, -8.640713691711426, -1.2167268991470337, 0.839883029460907, 1.8429296016693115, 0.9362668991088867, 8.361371040344238, -6.160923004150391, 21.83787727355957, -4.974667072296143, -9.83899211883545, -10.5475435256958, -7.899728775024414, -3.9363553524017334, -1.6086050271987915, 5.400448799133301, 6.150442600250244, 11.107196807861328, -4.6422576904296875, 11.772042274475098, -2.8140668869018555, 3.141812562942505, 2.859400749206543, 6.043332099914551, 1.5132807493209839, -8.512460708618164, -3.7749648094177246, -12.53525447845459, 9.807354927062988, -0.5656663179397583, 10.142545700073242, -3.7296369075775146, -3.5262179374694824, 12.626431465148926, 4.952909469604492, -6.3378705978393555, 3.5639114379882812, 1.3407429456710815, -2.061004161834717, -1.9910539388656616, -7.493966102600098, -18.41779327392578, 9.869562149047852, -16.46731185913086, -1.6240389347076416, 3.8300914764404297, -6.579156398773193, 15.247323036193848, -6.700387001037598, 6.0228071212768555, 2.0858607292175293, -2.1198110580444336, -4.3247785568237305, 5.772006511688232, 11.254742622375488, 10.502951622009277, -7.0102996826171875, 20.272069931030273, -10.554305076599121, 14.355734825134277, -10.536873817443848, -3.718177318572998, 5.427170276641846, -13.156198501586914, -9.399532318115234, -2.8109099864959717, -2.1913137435913086, 3.4494950771331787, -3.420605182647705, 7.851444244384766, -8.971172332763672, -9.265382766723633, 15.95148754119873, -13.57507038116455, 16.938180923461914, 3.5827271938323975, -16.253734588623047, -11.194048881530762, 6.1430487632751465, 8.129914283752441, 6.6007981300354, -5.353341579437256, -4.029177665710449, -7.121885299682617, -1.8514273166656494, -4.6815385818481445, 8.37800121307373, 9.70116901397705, 5.289866924285889, 12.488811492919922, -3.602851629257202, -6.011225700378418, 7.170502662658691, -6.088444709777832, 9.814153671264648, 0.3983399271965027], [-4.209802627563477, -4.73199462890625, 3.416468381881714, -7.591332912445068, 1.4037123918533325, 12.167567253112793, -3.3639864921569824, 3.0569913387298584, -16.11773109436035, 7.851768493652344, 34.81153106689453, 10.336402893066406, 18.48920440673828, -10.373201370239258, 5.174068450927734, 1.2861649990081787, 11.623791694641113, 6.811938285827637, -0.9681276082992554, -9.882709503173828, -0.44078174233436584, 9.856220245361328, -8.028642654418945, 26.637449264526367, -5.408676624298096, -8.934288024902344, -3.7056498527526855, -5.468077659606934, 22.88981819152832, 21.767711639404297, -3.7968969345092773, 6.146332263946533, 4.957679271697998, 26.309803009033203, 11.172795295715332, -8.80294132232666, 11.699440956115723, 5.093381404876709, -0.016769912093877792, 9.836868286132812, -9.942237854003906, -1.4696654081344604, 8.707935333251953, -6.2362961769104, 6.803704738616943, 4.50305700302124, -19.474029541015625, 6.273026466369629, -15.009964942932129, 16.278993606567383, -12.748544692993164, -6.959450721740723, -8.736342430114746, -6.787755489349365, 1.850600004196167, -17.264917373657227, -1.448966383934021, 15.390026092529297, -25.57421112060547, -1.3901638984680176, 18.229259490966797, 5.903611660003662, -6.930089473724365, 7.638855934143066, -8.093550682067871, -2.538304090499878, -4.358799457550049, 2.2094998359680176, -6.607306957244873, 11.092405319213867, 9.639047622680664, 14.563559532165527, -0.2983006536960602, -17.00169563293457, -0.4819955825805664, -4.534225940704346, -4.101971626281738, -12.410818099975586, 0.7341127991676331, -2.5551047325134277, -3.400139808654785, 4.87606143951416, 3.7760186195373535, 6.988482475280762, -17.288785934448242, -2.7901434898376465, -5.147119522094727, 3.7132644653320312, -15.325090408325195, -5.637478351593018, -3.901061534881592, -22.183231353759766, -3.0038981437683105, -4.598018169403076, -6.443814754486084, -16.976871490478516, 0.5098186731338501, 30.31667709350586, -4.177706241607666, -4.209212779998779, 6.734036445617676, -15.643122673034668, 3.31982421875, 7.157181739807129, 15.93223762512207, 7.9131340980529785, 9.371193885803223, 5.486856460571289, -18.201478958129883, -1.8501118421554565, -8.371153831481934, -6.887894630432129, 5.823585033416748, 5.418276786804199, -1.9752424955368042, -5.608733177185059, 22.598411560058594, 1.9086294174194336, -10.402155876159668, 7.919763088226318, -2.7287135124206543, -2.021294355392456, 10.330655097961426, 13.92934513092041, -7.424427032470703, -4.380565166473389, 15.001323699951172, -20.51911163330078, -2.502715826034546, 2.6648387908935547, -7.941664218902588, 6.1717987060546875, 1.3093578815460205, 2.0531563758850098, 17.398448944091797, 1.098589539527893, 21.74041175842285, 4.717010974884033, -0.08792699128389359, -0.03439229726791382, -1.1115342378616333, -2.918008804321289, 12.480484008789062, -7.817535877227783, -9.331047058105469, -22.656217575073242, -2.9377663135528564, -3.6816041469573975, 0.7886912226676941, -8.395983695983887, -3.532125234603882, -24.923913955688477, -6.465550422668457, -15.088149070739746, 7.3037800788879395, 5.575615406036377, -2.958340883255005, -15.306659698486328, -17.96122169494629, 10.17723274230957, -12.155776023864746, -23.52437973022461, -8.574723243713379, 13.17270565032959, 27.099918365478516, -13.80544662475586, 0.9297513365745544, 6.299319267272949, 21.563871383666992, 7.203752040863037, 7.061237335205078, 5.160061836242676, 1.249517798423767, -11.094225883483887, 5.082111835479736, 7.390722751617432, 10.353093147277832, -22.599441528320312, 0.7803019881248474, -3.5087568759918213, 10.342881202697754, 16.603538513183594, -10.972099304199219, -12.202263832092285, 8.273941040039062, 7.937314987182617, -0.2563839852809906, -1.8931035995483398, -8.785255432128906, -12.946168899536133, -3.901663303375244, 6.736059188842773, 23.529556274414062, 0.8298091888427734, -6.913093566894531, 3.094658374786377, 15.888533592224121, 5.194986820220947, -1.320496678352356, 5.756155490875244, 15.349090576171875, -6.1083455085754395, -2.990947961807251, 9.388558387756348, -11.7974214553833, 3.4271535873413086, 8.264081001281738, -6.301879405975342, -0.6104530096054077, -19.383481979370117, 12.959817886352539, -3.188589572906494, 11.742527961730957, 14.122635841369629, 8.630526542663574, -6.5694451332092285, 13.115349769592285, 10.01543140411377, 14.857314109802246, 0.07837141305208206, 14.990839004516602, -14.097480773925781, -4.772793769836426, -9.944802284240723, -9.368096351623535, 4.461325645446777, 4.601658821105957, 0.677507758140564, 1.9330432415008545, -7.761096000671387, 8.377917289733887, 9.373719215393066, 4.288332939147949, -3.444018602371216, -3.72446870803833, -1.0031466484069824, -5.8014936447143555, -2.435758590698242, 4.452874660491943, 2.3363571166992188, 0.6036716103553772, -1.3903186321258545, -16.1419620513916, 5.973987102508545, -4.860015392303467, 3.7276644706726074, -26.71174430847168, 3.4960479736328125, -5.651137351989746, 1.8640742301940918, 12.004030227661133, 3.8826088905334473, -29.220617294311523, 5.2640767097473145, -10.418265342712402, 2.4224472045898438, -9.768379211425781, 0.0030213063582777977, -8.277318000793457, 9.474780082702637, -8.634431838989258, -3.1826772689819336, 1.4057807922363281, -11.558950424194336, 4.094341278076172, 0.8651297688484192, 8.035001754760742, 5.289119243621826, -14.333977699279785, 7.629085540771484, 1.9617981910705566, 7.813909530639648, 2.057312488555908, -16.72097396850586, -6.240988731384277, -18.8641357421875, -11.549949645996094, 2.8764708042144775, 3.7827250957489014, -4.548107147216797, 2.0600709915161133, 7.3841352462768555, 1.3433287143707275, -8.350939750671387, -3.17014741897583, -9.505263328552246, 10.395674705505371, -13.414183616638184, 3.932227611541748, -9.43198299407959, -5.3622941970825195, -7.0886006355285645, 1.9561580419540405, 9.01075553894043, 16.14043426513672, 7.3735456466674805, 3.7221288681030273, -6.665132999420166, -7.575367450714111, -2.901097297668457, 1.0158648490905762, -23.56629180908203, 20.376123428344727, 2.4687294960021973, -3.7067465782165527, 7.327007293701172, -12.456343650817871, -1.7558335065841675, 11.663702964782715, 18.470693588256836, 4.668301105499268, -7.048126220703125, 8.612730979919434, -6.364073276519775, 1.2467888593673706, -5.409326076507568, 17.717144012451172, 1.4936593770980835, -8.74720287322998, -5.3854146003723145, 5.660706996917725, 12.417450904846191, -3.4398560523986816, -9.076497077941895, -15.264030456542969, 1.093207597732544, 2.1573472023010254, -0.36116787791252136, 3.4565415382385254, -13.169791221618652, -16.987733840942383, -2.782585382461548, 17.23060417175293, 16.27016258239746, 9.150399208068848, 15.184379577636719, -10.117888450622559, 1.040657639503479, 11.95949649810791, -1.749124526977539, -3.091554641723633, -5.315155506134033, -9.26598072052002, 7.250091552734375, -10.389883995056152, -7.066487789154053, -8.859952926635742, -2.9573869705200195, 4.571497440338135, -0.21098542213439941, -0.9934158325195312, 0.8454781174659729, -4.224798202514648, 9.107723236083984, 7.806039333343506, 21.678255081176758, -5.090890407562256, -5.973092079162598, 26.431039810180664, -2.211355447769165, 2.4663121700286865, 0.8675473928451538, -20.923757553100586, 6.496371269226074, -19.272808074951172, -4.220398902893066, -18.77400016784668, 0.4769013822078705, -11.761763572692871, -1.3381562232971191, -2.888906717300415, -8.517287254333496, 14.132925987243652, 7.364813327789307, -13.71323013305664, -27.20969581604004, 3.298603057861328, -15.846677780151367, -3.4285740852355957, 3.1982288360595703, -1.6806831359863281, -5.964319705963135, -0.595441997051239, -19.5767822265625, -5.671351909637451, 4.258746147155762, 5.614440441131592, -0.07324652373790741, 3.3289260864257812, 2.882805347442627, -5.719520092010498, -9.03078842163086, -14.455288887023926, -12.854595184326172, 6.9451470375061035, 0.5052558183670044, -6.956091403961182, -4.70750617980957, -11.384316444396973, -20.75334930419922, 5.351951599121094, 8.470130920410156, -12.31086254119873, -17.23484230041504, 1.0550005435943604, -8.430124282836914, -5.448381423950195, 2.59116268157959, 7.312810897827148, 13.829492568969727, 8.602890968322754, 3.970601797103882, -0.2657369375228882, 12.945340156555176, -6.689378261566162, 8.907109260559082, 19.86331558227539, 14.84598445892334, 13.383996963500977, -8.766419410705566, 5.582796096801758, 2.4669599533081055, -4.3209147453308105, 2.5010502338409424, -18.36821746826172, -11.666969299316406, 5.745361804962158, -8.772788047790527, 11.424697875976562, -10.894405364990234, 5.8897809982299805, 0.7997955083847046, 3.2542524337768555, 6.4091033935546875, -3.3575868606567383, -15.060458183288574, -3.929464101791382, -26.706684112548828, 16.682945251464844, 12.106035232543945, 6.490060806274414, -2.4990079402923584, -6.7311482429504395, -14.781404495239258, -3.7602498531341553, 8.474300384521484, -7.339310169219971, 5.62373685836792, -8.62686824798584, 3.000859498977661, 12.16033935546875, 9.48831844329834, -29.973230361938477, 12.641457557678223, 6.341307163238525, -8.48283863067627, 7.84503173828125, -3.149794340133667, -2.1984174251556396, 12.526758193969727, 10.076252937316895, 14.078129768371582, 6.356470108032227, 26.81159210205078, 8.67957878112793, -12.226065635681152, 1.285222053527832, -10.338157653808594, -19.14680290222168, -3.252634286880493, -2.5442776679992676, -5.771743297576904, 6.404648780822754, -2.999331474304199, 3.128329277038574, 6.708705902099609, -1.2808490991592407, 8.368572235107422, -9.591548919677734, -18.43832778930664, 17.2869930267334, -14.963321685791016, -3.7357559204101562, 2.5293755531311035, -17.423200607299805, 0.5976980924606323, 8.210330963134766, 13.779698371887207, 1.0480531454086304, -3.6152145862579346, -7.986844539642334, 2.6007978916168213, 0.9698629975318909, -23.692798614501953, -9.00820255279541, 3.551377773284912, 5.352273464202881, 4.350522518157959, 10.95887279510498, -14.161701202392578, 7.011383533477783, -5.708992958068848, -6.670772552490234, -0.9075966477394104, -14.775940895080566, -0.037242330610752106, 5.492737293243408, -18.264619827270508, -15.621297836303711, -17.3962459564209, 3.103822946548462, 2.595366954803467, -11.650923728942871, -27.068614959716797, -1.6118625402450562, 21.288341522216797, -10.437891006469727, -8.017085075378418, -8.519071578979492, -6.363408088684082, 12.781641960144043, -18.29446792602539, -10.329177856445312, -8.298811912536621, -4.608581066131592, 4.419708251953125, -7.8840837478637695, 6.071329116821289, 4.414073944091797, -10.657052993774414, 3.139180898666382, -13.856484413146973, -26.617725372314453, -4.822868824005127, 8.43727970123291, 14.216103553771973, 5.796215534210205, -13.060790061950684, -5.510345935821533, 4.282081604003906, -1.4034141302108765, 28.89346694946289, -25.571130752563477, -11.795138359069824, 2.07196307182312, -28.46464729309082, 24.268373489379883, 1.4159729480743408, 0.08528590202331543, 14.363468170166016, -0.648283064365387, -5.10584831237793, 9.460735321044922, 3.3616132736206055, 11.778703689575195, -12.302144050598145, 15.267704963684082, -3.0707223415374756, 13.651798248291016, -6.445645332336426, 2.5416972637176514, -5.096219062805176, 18.583194732666016, -4.158175468444824, 4.153406620025635, 3.3766515254974365, -21.456741333007812, 8.93347454071045, 1.2471303939819336, 12.519134521484375, -11.439713478088379, 0.5657504796981812, -6.226466178894043, -9.80553150177002, 0.1369447410106659, 17.179725646972656, -2.502836227416992, -15.459427833557129, -18.18805694580078, -9.61284351348877, 12.249009132385254, -6.93751859664917, -4.7841596603393555, 13.740204811096191, -5.342647552490234, 10.001241683959961, 4.818235397338867, -12.404181480407715, -5.648359775543213, 6.502936840057373, 0.08300144225358963, -16.422264099121094, 6.903417587280273, 11.352668762207031, 14.252065658569336, 4.5476908683776855, -12.300036430358887, 2.286053419113159, -10.514822959899902, 10.288237571716309, 1.070828914642334, 12.804262161254883, 0.12727290391921997, -8.940438270568848, 25.171545028686523, 0.31974175572395325, -15.718305587768555, -5.3266191482543945, -13.224533081054688, -12.458541870117188, 0.8172746896743774, -12.096050262451172, -4.644506454467773, -16.133464813232422, 6.570776462554932, 4.767582416534424, -3.3845438957214355, -21.94611930847168, 10.162191390991211, -14.155019760131836, 7.184948921203613, -3.5355453491210938, -7.439305782318115, 20.34853744506836, 3.5927891731262207, 4.837470054626465, 1.6109583377838135, 4.499509811401367, -0.44140133261680603, -16.279329299926758, -4.98366641998291, -12.305855751037598, 16.87117576599121, -11.275087356567383, -1.62967848777771, -5.468898296356201, 1.6752526760101318, -5.595278739929199, 11.063283920288086, -0.20033635199069977, -0.16334471106529236, 6.480921745300293, -11.08917236328125, -5.591463565826416, -18.906381607055664, 0.5558555722236633, 6.637889385223389, 3.0425050258636475, 5.222708225250244, 11.535475730895996, -2.063145875930786, -9.195293426513672, 2.783214807510376, 0.7422696352005005, -0.29001614451408386, 13.08574104309082, 15.123369216918945, 16.8785457611084, 2.224902629852295, 18.648773193359375, 8.63697338104248, 11.896444320678711, -3.2367758750915527, -10.586256980895996, -2.839155435562134, 1.1346516609191895, 14.491090774536133, 8.557648658752441, 2.9942615032196045, -2.2270801067352295, 4.934137344360352, 1.1363354921340942, 4.068481922149658, -9.872392654418945, -6.047849655151367, 11.87168025970459, -14.538540840148926, 2.195559024810791, -18.03771209716797, -8.326028823852539, 8.739354133605957, 5.672519683837891, 4.329522609710693, 4.833929061889648, -5.570999622344971, -21.387516021728516, 0.15164776146411896, 3.782857656478882, -2.212381362915039, -8.434861183166504, -3.3253681659698486, 3.8504445552825928, -26.90358543395996, -13.630101203918457, 6.186203956604004, 2.359438419342041, -2.3035826683044434, 10.359726905822754, -0.4018695056438446, 9.080629348754883, 19.7187442779541, -1.9524683952331543, 8.576289176940918, 4.605611801147461, -18.780895233154297, -10.118728637695312, -6.651350975036621, 14.650444984436035, -7.715304374694824, -1.9100797176361084, -5.0317511558532715, -19.059423446655273, 5.433166027069092, -2.4311928749084473, 10.66711711883545, -1.1683517694473267, 2.536782741546631, 6.626558303833008, 11.483925819396973, 9.29971694946289, -8.149649620056152, 11.389973640441895, -8.057551383972168, 13.35711669921875, 4.021195888519287, -8.639509201049805, -7.741093158721924, 9.258851051330566, -2.573321580886841, 2.0442135334014893, -1.4151562452316284, 15.457191467285156, -7.216605186462402, 2.5140342712402344, -19.310840606689453, 3.580684185028076, -11.609187126159668, -4.260290145874023, 19.76148223876953, 5.741927623748779, -10.212182998657227, -0.8215615153312683, 2.2208120822906494, -6.2106242179870605, -2.998929500579834, -13.314059257507324, 15.634252548217773, -5.005668640136719, 3.8748936653137207, 4.462839126586914, 6.161468982696533, 4.056638240814209, 12.336360931396484, -13.73486614227295, 7.584558010101318, 10.096826553344727, -12.15652084350586, -3.5547895431518555, -4.116298675537109, -1.4072784185409546, 14.58658218383789, 8.152957916259766, 3.724236488342285, 4.553170204162598, -14.269128799438477, 12.690707206726074, -8.188638687133789, -1.82038414478302, 16.750505447387695, -8.585212707519531, -20.636856079101562, 4.982970714569092, 7.247482776641846, -1.5400638580322266, 16.166147232055664, 1.5114109516143799, 2.333434820175171, 8.3395414352417, -12.440292358398438, -6.171966552734375, 3.1134982109069824, 5.646084785461426, 20.437150955200195, -7.371686935424805, 4.731545448303223, -4.903242111206055, -8.948890686035156, 16.767541885375977, 4.112066745758057, -5.83034086227417, 16.28083038330078, -16.642845153808594, 14.960160255432129, -4.29236364364624, -6.703068256378174, -9.695213317871094, 5.432009696960449, 4.2833757400512695, 7.139175891876221, -17.5166015625, 13.51627254486084, 0.11414037644863129, 5.621395587921143, -25.0801944732666, -4.999861717224121, 2.8108460903167725, 13.463171005249023, 3.6137661933898926, -0.8152332901954651, -3.024200201034546, -2.175118923187256, 17.031997680664062, 21.457414627075195, -11.078800201416016, 3.6451802253723145, 19.460384368896484, 5.371390342712402, 0.3666057288646698, -4.217720985412598, 4.011608600616455, 27.36640739440918, 2.4562249183654785, -7.176194190979004, -2.903151035308838, 5.473037242889404, 8.753582000732422, 4.1948323249816895, -1.8769030570983887, -2.2301011085510254, 1.3124223947525024, 0.1362306922674179, 17.66101837158203, 5.520632743835449, 2.5741684436798096, 13.809596061706543, 4.139872074127197, 7.116916656494141, -18.242958068847656, -1.604024887084961, 5.393409252166748, -2.620743751525879, -1.9626575708389282, 12.203753471374512, -10.776006698608398, 4.80108642578125, -1.316472053527832, -17.501876831054688, -3.2368619441986084, -1.6327621936798096, -12.331538200378418, 12.119088172912598, -5.5543646812438965, -5.186846733093262, 5.897091388702393, 12.356741905212402, -14.393589973449707, 1.4673073291778564, 7.590561866760254, -8.192020416259766, 11.975196838378906, 16.377595901489258, 4.309486389160156, 2.196735143661499, 10.78929328918457, -4.571203231811523, -11.230939865112305, -20.976680755615234, -14.714045524597168, -10.413872718811035, 22.724536895751953, -2.605708122253418, -3.6289165019989014, 4.960930824279785, 7.514993667602539, -12.298171043395996, 3.1933038234710693, 11.354021072387695, 3.9800407886505127, -9.4834623336792, 3.03324818611145, 20.833295822143555, 12.574809074401855, 6.443327903747559, 16.21784210205078, 14.35257339477539, -11.748984336853027, 13.436139106750488, 3.517549991607666, -8.518486022949219, -7.012855529785156, -2.943796157836914, 5.196442127227783, -4.337315559387207, 15.958531379699707, -4.263923168182373, 2.8620362281799316, -6.834225654602051, -9.46959114074707, -4.899380207061768, -6.293766975402832, -8.893818855285645, 5.006649971008301, 14.467926025390625, 3.780179977416992, 20.390893936157227, 6.566939830780029, 0.7536779046058655, 12.581832885742188, -12.895293235778809, -19.46105194091797, -0.5155808329582214, 16.91957664489746, -0.6350856423377991, 3.7131693363189697, 11.941222190856934, 20.02591896057129, -2.783086061477661, -13.582535743713379, -12.555591583251953, -0.5110753178596497, -8.05825138092041, 2.912353515625, 4.049725532531738, -0.06276566535234451, 9.368988037109375, -1.6265718936920166, 26.618837356567383, -11.155017852783203, 9.041494369506836, -3.7608418464660645, 4.667431831359863, -17.235563278198242, 11.832432746887207, -3.117199659347534, 9.550069808959961, 17.375123977661133, -13.579949378967285, -2.059311628341675, 8.808387756347656, 10.315552711486816, -11.084503173828125, 3.3645598888397217, 13.316620826721191, 5.010731220245361, 5.35125207901001, 1.8202871084213257, 8.52065372467041, 6.319901466369629, -12.765785217285156, 8.281021118164062, -0.4696652293205261, -18.38576889038086, 0.9162113666534424, -5.208103179931641, 6.930639266967773, -11.282966613769531, -0.6387431025505066, -3.121363639831543, -7.9270920753479, -14.853014945983887, -11.08351993560791, 7.90203332901001, 2.6244711875915527, -8.16252326965332, 4.299481391906738, 11.030649185180664, 6.7455878257751465, 4.113425254821777, -1.4822686910629272, 14.466116905212402, 10.429137229919434, -16.797014236450195, -1.7069839239120483, -14.93913745880127, 20.910776138305664, -17.09319305419922, 16.604568481445312, -5.3350653648376465, -11.924295425415039, 10.509828567504883, -13.494036674499512, -11.475807189941406, -5.110803127288818, 14.035784721374512, 8.246251106262207, 12.294921875, 12.407487869262695, -5.34320068359375, 2.6211414337158203, -0.4450913369655609, 2.6211845874786377, -2.492558717727661, 15.238262176513672, 5.905588150024414, -6.808159828186035, 13.309307098388672, -2.4401819705963135, -7.9989705085754395, -4.386780738830566, 15.106693267822266], [-2.9957077503204346, -5.384618759155273, -12.599713325500488, 2.7217493057250977, 6.341660976409912, -5.090426921844482, -0.29087555408477783, -4.599719524383545, 20.107816696166992, -4.783315658569336, -9.172110557556152, 2.265490770339966, -7.552417278289795, 6.822351455688477, 7.560037136077881, -4.559942722320557, -3.090806722640991, 17.174297332763672, -9.349191665649414, -4.6535491943359375, 18.856037139892578, -4.43423318862915, -9.94950008392334, -13.600598335266113, -0.9418875575065613, -12.2805814743042, -0.29247191548347473, -11.251018524169922, -6.45139217376709, 7.411956310272217, -9.766300201416016, 5.677191734313965, -19.848848342895508, -9.061771392822266, 4.232031345367432, 1.623313069343567, -0.06830526143312454, 4.357647895812988, -11.956084251403809, 4.886192321777344, -5.754968166351318, -16.95886993408203, 9.499759674072266, -0.024064015597105026, -2.8631553649902344, -7.432941436767578, -4.264832019805908, -2.6073648929595947, -21.36237335205078, 2.8275623321533203, 28.88027000427246, 1.497677206993103, 10.044194221496582, 12.482789039611816, 6.703700542449951, 4.514552116394043, 14.157031059265137, 3.883023738861084, 1.2505730390548706, 7.4965949058532715, 12.055985450744629, 7.562462329864502, 0.12759341299533844, -10.803255081176758, 15.308732032775879, -7.577343940734863, -1.6684116125106812, -13.108378410339355, 8.219086647033691, -0.4232262074947357, 0.6963508129119873, -19.016218185424805, -3.950183391571045, 2.328019857406616, 10.982149124145508, -6.237446308135986, 12.003782272338867, -1.0173076391220093, 14.499358177185059, 12.377128601074219, -0.699453592300415, -3.363426923751831, 4.218508720397949, -10.408257484436035, 18.505891799926758, -5.858686447143555, 0.2845095694065094, -10.033442497253418, -0.24042034149169922, -2.8638641834259033, -8.699331283569336, 1.3790713548660278, 7.236713409423828, -2.381101608276367, 6.183207988739014, -1.1641550064086914, 10.504979133605957, 22.977008819580078, -10.254151344299316, 6.536311626434326, -10.833303451538086, -25.16579246520996, 14.767820358276367, 3.072906494140625, 3.9725868701934814, 9.497137069702148, 0.5206758975982666, 2.8830673694610596, -0.37015828490257263, 7.503859519958496, 3.6759960651397705, 9.03058910369873, -6.258272171020508, -19.574920654296875, 4.1014790534973145, -2.4313793182373047, 2.9229648113250732, 1.9981075525283813, -9.139307022094727, -14.669055938720703, 7.954591274261475, 14.19005012512207, 11.656428337097168, 1.8874610662460327, -16.652278900146484, 10.28454303741455, -2.7120046615600586, 7.243282794952393, 1.0978564023971558, -15.050142288208008, 16.773649215698242, 2.684957981109619, 5.477783203125, -3.1970150470733643, -29.108102798461914, -4.313347339630127, -6.359136581420898, -6.674480438232422, 8.836918830871582, -23.26725196838379, 4.298396587371826, 0.7419078350067139, 8.716121673583984, -4.794154644012451, -5.639157772064209, 0.5197665691375732, 15.325501441955566, -6.497645378112793, -0.5140672326087952, -7.895901679992676, 1.956811547279358, 5.100436210632324, -12.29971981048584, 7.03577995300293, -11.559243202209473, -8.287001609802246, 0.988241970539093, -1.1011412143707275, -8.22227954864502, 1.3252924680709839, 1.3586407899856567, -5.819547653198242, -8.740545272827148, -9.53148365020752, -2.30285382270813, -14.981406211853027, 6.878391742706299, 11.803655624389648, -2.7041146755218506, 10.786906242370605, 12.27731704711914, 9.96906566619873, -10.806139945983887, -8.611078262329102, -1.8164230585098267, -1.572242259979248, -6.061215877532959, 7.608367919921875, -3.6126434803009033, 15.07480525970459, 6.044732093811035, 10.882085800170898, 3.029100179672241, -5.011399745941162, -4.569869518280029, -6.3573079109191895, -11.917049407958984, -1.323296070098877, -0.0799715593457222, -7.290388584136963, 3.954464912414551, 2.1388511657714844, 1.4479128122329712, -0.11353481560945511, 18.55937385559082, -2.94936203956604, -0.34913885593414307, 9.4531888961792, 15.076509475708008, 13.46457576751709, -7.497746467590332, 3.8031766414642334, 1.9932959079742432, -9.055753707885742, -5.669941425323486, 23.211219787597656, -9.054488182067871, 1.539528250694275, -20.147443771362305, 0.6106960773468018, 4.877266883850098, 4.264735698699951, -7.741462230682373, -10.866877555847168, -12.191657066345215, 7.958418369293213, -15.189227104187012, 16.302160263061523, 10.32888412475586, -7.487098217010498, -4.420446395874023, -6.354248046875, -0.4515403211116791, -2.1049108505249023, -8.08233642578125, 7.2457499504089355, -6.693319797515869, 12.593034744262695, 3.6677348613739014, -1.8584043979644775, -0.19146251678466797, -12.662734985351562, 2.337658405303955, -2.3489067554473877, 20.774702072143555, 0.380587100982666, 4.2521467208862305, 12.578821182250977, 13.903882026672363, -20.922149658203125, -1.7625460624694824, 7.970303058624268, 2.8603951930999756, 10.261885643005371, -1.3727480173110962, -7.799502372741699, 1.348586916923523, 9.819117546081543, 9.325176239013672, -1.510661244392395, -3.107041120529175, -4.130436420440674, -5.145964622497559, 14.6005859375, -3.2851078510284424, -10.376448631286621, 15.149681091308594, -3.5842700004577637, -0.5502523183822632, -0.871152937412262, 7.514594554901123, 16.47547721862793, 8.797860145568848, 5.193072319030762, 17.76480484008789, 2.479658603668213, 0.029299261048436165, 1.2099045515060425, 1.9913148880004883, -7.530473232269287, -13.015461921691895, -3.074416160583496, 3.6713786125183105, 18.943702697753906, -1.8054327964782715, 5.431970596313477, 2.87593936920166, 27.661224365234375, 0.9377363920211792, 2.246553421020508, 13.299497604370117, 10.41745376586914, 0.44542354345321655, -6.41381311416626, 0.4335266649723053, -20.4960994720459, -3.0807907581329346, -12.963459968566895, -3.266672372817993, -9.709665298461914, 3.8174338340759277, -10.613521575927734, -7.301337242126465, 2.194472551345825, 1.7773584127426147, 3.6842353343963623, -15.497815132141113, -13.205270767211914, -3.9790403842926025, 5.654752731323242, -7.53864860534668, -0.10407041013240814, -4.358151435852051, -3.480438232421875, 5.764069080352783, -15.633411407470703, 8.478528022766113, 18.04184913635254, 14.62816333770752, -1.4974113702774048, 1.3080848455429077, 1.3488341569900513, 0.4419364333152771, 8.804386138916016, -4.5726318359375, 4.848737716674805, -10.69375991821289, 15.461953163146973, -3.0378901958465576, 3.8591244220733643, 10.78735065460205, 7.023619651794434, 1.8065005540847778, 0.3746190667152405, -9.866312980651855, 1.8217324018478394, 4.279757976531982, -11.248861312866211, 16.132354736328125, -4.127652168273926, 0.3918367922306061, -7.293589115142822, -6.863010883331299, -10.076327323913574, 13.715007781982422, -8.195520401000977, 2.2042031288146973, 12.666658401489258, -10.915346145629883, 19.891477584838867, 7.465989589691162, -3.135699510574341, 18.27496910095215, 0.29740583896636963, -5.268177032470703, 16.700939178466797, 7.474209308624268, 0.41161683201789856, -4.46309232711792, -8.204151153564453, 0.04193807393312454, 0.7465377449989319, 11.00375747680664, 1.9683514833450317, -13.484663963317871, 12.998723983764648, 16.856313705444336, -3.36118483543396, 17.15135955810547, 2.645846128463745, 6.2327046394348145, 20.214500427246094, 19.580677032470703, 1.5507817268371582, 11.32422924041748, -1.0904768705368042, 20.295804977416992, 17.497989654541016, -16.201173782348633, -9.112566947937012, -7.810176372528076, -17.70463752746582, 13.887036323547363, 8.187955856323242, -7.2638258934021, 9.244491577148438, 9.210208892822266, 4.496971130371094, -1.3700982332229614, -5.57880973815918, 3.8064420223236084, -4.153122901916504, 3.3280045986175537, 8.394353866577148, -9.098184585571289, -0.9123572707176208, -1.3027416467666626, -0.263391375541687, -10.767013549804688, 7.741578102111816, -13.139371871948242, 9.83853530883789, 5.046448230743408, 11.505629539489746, -5.492541790008545, -4.61786413192749, -1.1091387271881104, -1.2440314292907715, -19.568973541259766, -14.029603004455566, 22.488243103027344, 15.082541465759277, -16.264108657836914, -12.979471206665039, 4.918919563293457, 2.2680280208587646, -3.564649820327759, 4.436802387237549, 10.910125732421875, -9.278620719909668, 5.935089588165283, 20.88262367248535, 2.564523935317993, -20.669435501098633, -2.8413329124450684, 4.629040718078613, -2.459595203399658, 2.7633700370788574, -11.182007789611816, -3.8121509552001953, 7.035104751586914, -0.285996675491333, -5.42176628112793, -0.171004056930542, 7.030567646026611, -29.470317840576172, 4.553481101989746, -9.058499336242676, 4.750457763671875, 8.846627235412598, -1.782904863357544, 0.6142755746841431, -10.261127471923828, 7.4126811027526855, -9.590702056884766, 13.896930694580078, -3.3836700916290283, -7.924079418182373, 6.286834716796875, -7.293225288391113, 0.36197629570961, -5.1826066970825195, 9.071578025817871, -19.419368743896484, 15.321574211120605, -13.136502265930176, 4.769559860229492, 5.425046920776367, 29.53400421142578, 5.138954162597656, 3.1866934299468994, -2.321837902069092, 6.864516258239746, -3.443805694580078, -14.469755172729492, 2.0470988750457764, -1.6835120916366577, -3.4640676975250244, 4.736426830291748, 2.524632692337036, -1.1027874946594238, -11.684420585632324, -5.972536087036133, -3.8828842639923096, 12.131932258605957, 4.861159801483154, -8.74874496459961, 9.851373672485352, 4.810132026672363, 7.329264163970947, -7.028494834899902, 17.258365631103516, 2.6524479389190674, 12.822879791259766, 8.944475173950195, -12.717930793762207, 7.580143451690674, 6.90618371963501, 20.244171142578125, -2.7027597427368164, -15.333433151245117, 10.722135543823242, -10.835291862487793, -4.643906593322754, -0.9926318526268005, -6.908071041107178, 5.0656328201293945, -13.698277473449707, -23.988882064819336, -19.545394897460938, -9.161974906921387, -2.8798153400421143, -2.7686729431152344, -6.817983627319336, 2.4599857330322266, -10.842642784118652, -24.52478790283203, -22.56361198425293, 1.2007887363433838, -7.307255744934082, -6.1079840660095215, 2.5229039192199707, -5.499329566955566, -3.0911691188812256, 11.66226863861084, 3.9764153957366943, -0.274751216173172, 7.807344436645508, -8.306008338928223, 2.810365676879883, -1.6269464492797852, -17.270248413085938, 9.194353103637695, 27.61075782775879, 2.6571552753448486, -8.139890670776367, -0.13887208700180054, -8.025304794311523, -19.834095001220703, 12.26010513305664, -8.968496322631836, 7.0334672927856445, 4.175360202789307, -11.866203308105469, -11.199125289916992, -31.087488174438477, 28.56043243408203, 3.5100200176239014, 1.1240402460098267, -2.9547765254974365, 17.240495681762695, 9.699283599853516, -13.337370872497559, -14.78212833404541, -3.3963513374328613, -0.9239782691001892, 3.8389663696289062, -12.150114059448242, 4.657629489898682, -10.62852954864502, -9.00418758392334, -1.4913320541381836, -5.153491020202637, -5.9683122634887695, 10.364514350891113, 1.1919286251068115, -9.650703430175781, -18.545745849609375, 5.775121212005615, 18.742435455322266, 10.866378784179688, -0.41265150904655457, 5.6558942794799805, -11.081184387207031, 16.47092628479004, 10.588106155395508, -21.31816291809082, -1.88522207736969, 2.5599074363708496, -8.565837860107422, -0.8461638689041138, 6.931264877319336, 7.769627571105957, 14.639154434204102, 10.544121742248535, -0.06390351057052612, -8.286918640136719, -0.5329815149307251, -11.986649513244629, -14.67833423614502, -7.164159297943115, -4.770334720611572, 4.009428024291992, -1.9929131269454956, -1.3170409202575684, 2.537187337875366, 6.983663558959961, -6.098762512207031, -14.860299110412598, 17.689617156982422, 18.431846618652344, -2.0473861694335938, -1.7625738382339478, -7.159472942352295, 9.421346664428711, -15.520479202270508, 12.062108039855957, -10.15017032623291, -8.69546127319336, -6.238057613372803, 4.203536510467529, 7.889712333679199, -11.127325057983398, -14.482206344604492, -13.612200736999512, -3.2597060203552246, -5.7393670082092285, -7.587469100952148, -7.716567516326904, -1.3734910488128662, -6.849612236022949, 11.133238792419434, -5.035739421844482, 5.5218505859375, -9.108415603637695, 6.059872627258301, -3.922807216644287, 7.628695011138916, 2.399724006652832, -2.172111749649048, -4.757380962371826, 7.370397567749023, 0.9792330861091614, 8.744135856628418, -2.570730209350586, 1.7731616497039795, -5.653988361358643, -5.992458820343018, 1.4811903238296509, 14.37837028503418, 15.015088081359863, -15.378026962280273, 14.2720365524292, 15.048656463623047, 4.632762908935547, 12.296804428100586, 6.8357014656066895, -2.0158278942108154, 10.616687774658203, 5.2876996994018555, -4.681375503540039, -20.755935668945312, 1.0446189641952515, 1.2168890237808228, -15.340609550476074, -0.02223767712712288, -9.83202838897705, -3.4391441345214844, -16.496356964111328, -5.704110145568848, -1.604267954826355, -15.914799690246582, -12.951164245605469, 0.19786037504673004, 0.016439540311694145, 7.534369468688965, 2.1655070781707764, -10.043270111083984, -16.24165153503418, 5.966582775115967, -6.654691696166992, 2.8662617206573486, 5.223850727081299, -9.970683097839355, 20.692163467407227, 15.291866302490234, -1.4652661085128784, -2.1123554706573486, -8.934028625488281, 14.406911849975586, 12.279929161071777, -3.53228759765625, -6.015587329864502, -6.922907829284668, 3.327200412750244, -7.220727443695068, 11.452561378479004, -15.037692070007324, -3.702695846557617, -11.36574649810791, 0.5984615683555603, -9.125943183898926, -14.164939880371094, 4.408589839935303, 1.3163052797317505, 12.827731132507324, 1.6260790824890137, 1.9376380443572998, 5.585878372192383, 16.08379554748535, 3.6656010150909424, 2.1841554641723633, 8.085951805114746, -6.099054336547852, 6.22279691696167, -0.5841708183288574, -5.073805332183838, -10.173866271972656, -11.43093204498291, 2.900031328201294, -18.71094512939453, 4.732007026672363, 12.349479675292969, -3.20697021484375, -9.792025566101074, 2.868536949157715, -13.920494079589844, -2.2847414016723633, -7.87775182723999, 3.475909471511841, 1.9574397802352905, -19.409818649291992, 3.4577410221099854, 7.068060398101807, 10.19528579711914, 14.087593078613281, 4.579209327697754, 5.457372188568115, -4.794214248657227, -5.215535640716553, -2.632152557373047, -8.949420928955078, 5.809023857116699, 13.116511344909668, 13.352584838867188, -14.339811325073242, 0.1900745928287506, 6.607906818389893, -10.886051177978516, 3.741837978363037, 12.39735221862793, -0.6658235788345337, -0.06927285343408585, -4.789013862609863, -1.1039493083953857, 2.8255977630615234, 10.666022300720215, 0.8659865856170654, 3.8099799156188965, 4.293739318847656, -9.112092018127441, 2.0363993644714355, -1.1025506258010864, -18.0977725982666, 6.065206050872803, -8.457159042358398, -18.778308868408203, -26.61100959777832, 6.1928839683532715, -11.459893226623535, 4.0494513511657715, 6.588447570800781, 12.036897659301758, -7.259747505187988, -6.20278263092041, 1.6007157564163208, -15.806185722351074, -14.765698432922363, -6.386941909790039, 0.5556288957595825, -15.259259223937988, 2.388611078262329, -5.585410118103027, -19.235065460205078, -14.474365234375, -7.568589687347412, -2.259376287460327, -6.174682140350342, 18.690444946289062, 3.4007203578948975, -10.95943546295166, 8.492087364196777, -7.858700752258301, 4.361907958984375, -8.888757705688477, -21.419586181640625, 26.93889808654785, 2.655834197998047, -1.6663471460342407, -2.5655694007873535, -1.0672274827957153, -1.6979793310165405, 12.457840919494629, 6.417232990264893, 6.245619773864746, 1.7932087182998657, 8.849486351013184, -1.6470717191696167, -8.622406959533691, -0.42541027069091797, -1.62226140499115, -18.781450271606445, 3.209059476852417, 3.028890609741211, -14.822651863098145, -8.44834041595459, -4.030528545379639, 0.7225615978240967, -11.550440788269043, 0.7140751481056213, 3.0350701808929443, 21.736595153808594, 9.112687110900879, -8.733098983764648, 5.404176235198975, 9.463189125061035, -2.3919715881347656, -1.251529574394226, -22.293582916259766, -1.3932099342346191, 13.873822212219238, -8.450420379638672, -1.1872581243515015, -6.457184314727783, -2.737891674041748, -0.9944677352905273, 3.9830031394958496, -10.690914154052734, -4.0153326988220215, 9.103541374206543, -7.642340660095215, 14.603290557861328, 12.34934139251709, -11.803386688232422, 13.839374542236328, -9.099996566772461, 4.510626792907715, 8.098894119262695, -3.9731953144073486, -0.6671606302261353, 13.676375389099121, -8.284523010253906, 2.0345208644866943, 5.439685344696045, 19.789777755737305, 7.8528642654418945, 3.5982840061187744, 6.778570652008057, -1.7992624044418335, 1.2500054836273193, -4.969361782073975, 3.096442937850952, -22.9747371673584, -11.319557189941406, -7.392139434814453, -3.042053699493408, 6.118992328643799, 8.32214641571045, -18.837900161743164, -1.0033048391342163, 17.62844467163086, 0.9416666626930237, -2.8625330924987793, -16.721895217895508, -14.770362854003906, 12.670673370361328, -9.23887825012207, -0.7270803451538086, -6.68740177154541, 3.087972640991211, 11.468764305114746, 9.842706680297852, -6.885339736938477, -1.8791643381118774, -7.153536319732666, 1.867559790611267, 0.02807757630944252, 0.1700706034898758, -17.442163467407227, -7.148311138153076, 4.315077304840088, -2.066950798034668, 4.470046043395996, 0.35858774185180664, -14.13940143585205, -5.052924633026123, 5.03021764755249, 23.76909828186035, -20.467771530151367, 2.791027307510376, -14.648759841918945, 1.3346730470657349, 8.52794075012207, 16.20578956604004, -1.696373701095581, -5.279301166534424, 0.1233297809958458, 5.088116645812988, 15.79605770111084, 22.18912696838379, 8.143216133117676, -15.281004905700684, 4.8239054679870605, -12.816271781921387, 1.977593183517456, 5.967407703399658, 10.455172538757324, -7.109891891479492, -4.721116542816162, 5.123891353607178, -10.112491607666016, 4.935854434967041, -1.9208611249923706, -5.837616443634033, 7.587049961090088, -5.437536239624023, 15.481515884399414, -2.59934139251709, -3.5831425189971924, 5.3549394607543945, 8.738520622253418, 15.938084602355957, 4.118178844451904, 11.667375564575195, -0.6797109246253967, 1.7603042125701904, -1.3983533382415771, 6.352845668792725, 0.42447227239608765, 8.289024353027344, -5.748318195343018, 16.17327880859375, 15.709161758422852, 8.510390281677246, 4.317133903503418, -16.130001068115234, -1.3066884279251099, -2.526456117630005, 4.3236846923828125, -6.03507661819458, -5.001835346221924, 8.812857627868652, 4.418532848358154, -5.047784805297852, 4.979686260223389, 11.62049674987793, -0.9853558540344238, -15.475152969360352, 9.307343482971191, -6.932787895202637, 14.5077543258667, -8.277243614196777, 7.8016037940979, 3.980272054672241, -5.878018379211426, -6.365302085876465, -2.371514320373535, 2.2579150199890137, -0.6005537509918213, -23.999553680419922, 7.484797954559326, 2.8910789489746094, 19.195846557617188, 3.875863552093506, 1.349717378616333, -8.970417022705078, -18.48172378540039, 0.7890499830245972, 3.3811869621276855, 0.4803444743156433, -1.8971225023269653, -1.0718488693237305, 19.26656150817871, 3.4101126194000244, -18.649580001831055, -1.1916964054107666, -9.917405128479004, -9.790902137756348, 0.7327250242233276, 0.49581190943717957, -11.557441711425781, -15.006295204162598, 9.875128746032715, -10.202735900878906, 18.970184326171875, 12.711819648742676, 1.1640400886535645, 8.654468536376953, -14.876861572265625, -1.0835075378417969, -5.97218656539917, 9.296298027038574, 14.775371551513672, 3.100881338119507, -12.395546913146973, -19.874828338623047, -20.25422477722168, -5.114616870880127, -19.516836166381836, 11.166866302490234, 19.45012092590332, 6.800434112548828, 1.1100035905838013, -9.941478729248047, -4.397515296936035, 2.940448045730591, 5.879504680633545, -4.060762882232666, -25.041112899780273, 4.4049391746521, 15.641806602478027, -3.365821599960327, -9.924969673156738, 16.90253448486328, 9.139875411987305, 4.1529412269592285, -1.050917148590088, 3.8561952114105225, -2.259873390197754, -2.5257959365844727]], "dtype": "float32"}, {"__nd__": [1000, 777, 3], "dtype": "int64"}, 1.0], "e": {"__nd__": [[3.1561034341542893e-10, 7.0701072483708e-17, 9.121674751321985e-20, 6.192212523334391e-17, 5.015295972033047e-14, 1.802733516308487e-20, 6.026790615095432e-15, 1.3920361801325826e-15, 1.4065544571974193e-16, 1.9230168672723646e-13, 5.215859448028966e-22, 1.4164784273495536e-19, 1.0909146036606338e-13, 4.573926246163439e-15, 1.3510565821488185e-13, 1.9800047018625477e-12, 2.458263281823747e-07, 2.8280175510403405e-12, 7.275369751091887e-10, 5.847295545605418e-16, 5.378690548507592e-12, 2.35057301043156e-14, 2.7925185277920155e-18, 1.3142229690374743e-16, 2.5475880736683795e-17, 1.3362416235885743e-18, 2.93879694052074e-18, 2.2730580440111225e-07, 5.401605399127484e-20, 1.3856249348152556e-16, 3.399706080168374e-26, 4.1628267577279737e-19, 4.433359858017167e-15, 1.496649158772012e-11, 1.009460644573206e-21, 1.1301238519626595e-15, 1.545787876046606e-18, 1.573458990731429e-15, 1.6781536537094654e-19, 5.062936683700831e-16, 1.243121054093346e-14, 4.319887618633837e-16, 1.039492936172337e-08, 2.6403697479283894e-13, 3.2366936821709436e-13, 2.8028955686687616e-24, 7.861424011768037e-11, 2.579712431085429e-13, 2.019616252013634e-09, 4.262015018954973e-18, 6.793954369357577e-14, 2.4783828848740086e-05, 8.369024332051389e-15, 7.039092395853255e-15, 1.668881233294478e-13, 1.9298154740887696e-10, 5.2910756774464675e-18, 1.1952239667282517e-14, 1.081073866689502e-21, 1.4617682503888544e-23, 1.3896152199690111e-19, 9.278937941639655e-20, 1.8575164162554415e-23, 7.921080838988422e-22, 3.1909526104998734e-20, 8.789604989483356e-15, 1.6084513617385145e-14, 5.836982373136346e-12, 7.600204818959775e-16, 3.333559582079781e-19, 1.165946373972368e-20, 3.192816020250455e-13, 3.519052642416174e-16, 3.9156750216394476e-13, 3.3270582009590557e-19, 6.6695609140861134e-21, 2.1271639320431976e-17, 0.0008038622909225523, 5.531647764899037e-15, 4.540040481350294e-16, 3.896722557800679e-15, 4.179127450297582e-20, 1.0554402507523264e-07, 2.5796540209196817e-21, 1.56123028342769e-17, 9.606454102950706e-17, 8.487623323069365e-22, 1.8907843334247366e-18, 9.351143054297292e-20, 1.1207408021378026e-14, 5.4220076150656486e-18, 1.3971459622910687e-12, 1.3669708594232777e-11, 6.849833560948629e-25, 1.6432238599331517e-13, 1.781459577632747e-15, 1.2453591862897966e-18, 1.203380711051194e-11, 6.105784484239894e-15, 4.82527150785771e-20, 1.8040661989891056e-22, 2.1689873418667362e-16, 1.9393713794179507e-13, 1.39689117818521e-19, 3.9201202639560506e-07, 1.8911841713863238e-24, 2.0386848066373912e-14, 8.9510061944491e-19, 1.7440267285895317e-10, 2.944364587212367e-13, 1.9057766974771295e-14, 2.463830866768557e-18, 2.5010627831274516e-19, 1.0933321609174484e-10, 6.092496955146405e-19, 7.077888701829209e-17, 2.3648915998306333e-20, 4.263633976849862e-17, 4.859721831682854e-26, 6.38824076304445e-07, 1.773369090463106e-10, 6.5355997610922e-22, 1.6628929388117506e-21, 2.2533186038398644e-16, 5.1623286944019534e-14, 3.362898338705317e-14, 4.360387380406582e-22, 3.672952546762091e-15, 4.141951738276371e-17, 1.657544707241648e-19, 2.7368517174775064e-11, 9.651377157052828e-15, 9.535152208821687e-14, 3.655301227174258e-15, 7.404336523189996e-17, 2.803588188271533e-07, 1.0368783424607332e-17, 3.45376754384193e-24, 2.2885081574996258e-17, 1.5454515934022218e-22, 8.308011396895854e-12, 4.3250125703053754e-10, 2.9474374624043156e-18, 5.903055279830871e-14, 9.590521446300078e-18, 8.250645305851734e-11, 6.778222427783476e-10, 9.481797772880728e-15, 1.5743367286230213e-16, 1.1304184335194613e-19, 8.2643361057958995e-25, 2.1461619752873408e-19, 4.939591415049415e-17, 7.821655097923409e-17, 5.634704400118984e-13, 7.548052894518209e-14, 4.080411816808141e-15, 1.4436471462708766e-10, 2.3983469257637923e-22, 2.3877607790295796e-12, 3.146299397706215e-20, 5.238525406644698e-14, 9.804781986311184e-11, 7.155028569627575e-09, 1.589292418795776e-16, 1.8895779732193893e-17, 1.5888625484739016e-11, 1.0131186254508373e-12, 8.060583044132777e-14, 1.2338916712652443e-17, 9.154891707470958e-21, 2.8144014568033738e-21, 0.0004325863264966756, 3.315911476915809e-18, 4.909062878557602e-15, 8.199053103119525e-10, 1.174928553760671e-20, 1.0289367881947647e-18, 3.965659398086352e-11, 1.4039035775917363e-13, 2.8830051430118153e-22, 3.7005868576999193e-19, 3.745379845922523e-17, 4.0132990141170187e-17, 5.839445680472233e-13, 4.0765392053343515e-18, 5.979251227929336e-21, 3.5879567874418414e-17, 3.6201319719708636e-09, 1.1726254535219503e-19, 3.217846402883565e-16, 7.496984939315068e-14, 1.0398853425107104e-16, 5.1833772829468217e-08, 4.1257344563702223e-22, 2.5817334276100785e-17, 6.093945335329531e-13, 4.323884186894361e-13, 1.8624837783143278e-11, 3.755441009149128e-17, 1.7937844135409997e-12, 1.9701041428532155e-20, 7.775345246573021e-19, 1.1997973144509122e-12, 2.1230293524694125e-18, 3.070921641978863e-11, 1.5873069206278527e-15, 6.393190810056277e-15, 6.597798947285781e-18, 1.3423958509889285e-25, 4.957507471203826e-08, 8.962438929113833e-21, 1.2297440626571343e-12, 8.053962804023627e-15, 5.947506359233849e-18, 7.658921643307713e-19, 7.795010296962612e-13, 6.051465616585192e-14, 1.864046712740695e-17, 2.483909616619179e-13, 1.6505032017895488e-15, 5.650073452957911e-16, 8.499169620908032e-17, 4.889515687475578e-16, 3.105957602656667e-13, 3.479824113683851e-20, 4.077244707678798e-17, 3.7746991386573735e-19, 1.1151171809943919e-15, 1.4992511626708144e-13, 7.938739711232848e-22, 1.6495501631297316e-09, 1.3664112174822816e-23, 2.844228779878985e-22, 9.458465130566183e-24, 6.722920831148116e-12, 4.3458187547642516e-18, 4.597519875915168e-08, 4.690150799994864e-17, 2.947463453861765e-11, 5.816057540360573e-21, 9.080838633671907e-15, 5.971872204947517e-18, 5.440850918666577e-20, 1.8316933392270585e-07, 8.752464725203524e-11, 4.2814492138566e-18, 1.9519559092659336e-22, 2.245932741237603e-16, 3.952768914740593e-17, 1.7017647887244318e-15, 1.2804809649060684e-17, 3.760451159401301e-18, 1.4918747254276735e-14, 2.9128845113746138e-08, 2.0641478977456718e-12, 2.2082540294746924e-19, 3.116455235608245e-16, 6.054262619962188e-17, 9.300729674269204e-13, 3.989617084642532e-15, 2.559327837820761e-15, 2.1216989482594067e-13, 4.028586394926613e-20, 2.029481015453953e-17, 2.669397863492326e-16, 1.9434970383451377e-20, 1.2935766173505225e-11, 2.02726550824206e-11, 4.6651785481741334e-17, 6.035897828641015e-14, 6.955805644246514e-23, 1.8425458732530532e-16, 6.957583553539497e-17, 3.840411877398253e-10, 4.277172064741158e-15, 2.7255525961167315e-11, 1.3931558164037572e-11, 3.350476092314064e-14, 5.005110737008515e-17, 1.6227233798558048e-15, 1.735038864434737e-19, 2.7735095997627468e-08, 1.249997489905305e-16, 1.941603226530671e-15, 4.705793695276916e-18, 3.918707933221375e-17, 1.43482282126536e-14, 2.1024291682603113e-11, 1.1607347687458969e-06, 1.7646650980455115e-17, 3.8480253498787185e-17, 2.333677678337064e-16, 8.284342516798623e-30, 6.664083669805434e-20, 1.936022484080505e-21, 2.2749094207258173e-14, 1.3415165246027927e-12, 7.004621261790386e-19, 2.4554354881313174e-16, 3.154008155451145e-14, 2.5125760201009302e-24, 2.5101393455161165e-21, 1.2408970335650582e-13, 5.161557040103938e-22, 7.148183520794271e-12, 2.202814075894996e-13, 3.093355637976273e-18, 2.997754238680457e-18, 1.6945097013429908e-22, 4.400564859881849e-15, 2.1785316402949707e-14, 3.091996493911897e-12, 6.329402002513973e-10, 2.6286683940668354e-09, 5.727364679999273e-16, 8.277071626437544e-17, 5.875757927822524e-20, 1.647635201607045e-07, 8.210711624490052e-12, 1.116264106618366e-17, 1.5305541249750385e-17, 6.869785502180287e-14, 1.916958802930307e-15, 9.315736116538575e-15, 9.08936187773314e-22, 7.565408745811623e-20, 1.1651032210140046e-14, 6.568993442278781e-20, 1.3464563123309625e-13, 1.3352921470356077e-13, 1.3516751872508573e-14, 1.1630376566332257e-12, 1.554420972329823e-15, 3.872972653932175e-16, 2.303584715303487e-16, 6.509462287569106e-20, 4.528788707437205e-16, 1.0912090831350762e-14, 3.5624080872023835e-25, 3.671237792934878e-12, 1.6099798833152883e-19, 4.380617257626206e-21, 2.085046773840529e-16, 1.7114243789246995e-16, 4.963599306624807e-16, 4.0937449627410637e-20, 1.370278131414271e-13, 1.3107391093814422e-25, 1.7189075914196662e-22, 1.4899193075172895e-16, 1.2314176588146264e-11, 2.350522696674493e-14, 1.8189315952412723e-14, 1.5981372314399778e-07, 2.0208687784621238e-22, 2.7524984957809344e-18, 1.085006787610947e-16, 4.216805290363501e-12, 1.519525963742059e-23, 9.437480515408654e-22, 6.118169376478173e-15, 4.6195367232466785e-24, 8.523231873242318e-22, 3.541916675771408e-23, 1.5641072545092668e-24, 1.787445517119346e-13, 9.518281811259576e-16, 4.069770995901445e-12, 6.15225639102563e-17, 2.3902854806578944e-08, 2.1251592698453156e-21, 2.1814595814825685e-19, 4.597308027362397e-19, 6.888285802891152e-15, 8.657512107183179e-16, 8.341462937044852e-11, 6.946949712495306e-19, 1.7500112560044447e-15, 5.914380670833244e-19, 9.339458895500433e-18, 1.1344275128652437e-12, 4.970378275626075e-12, 3.510848168792506e-19, 4.254722428269231e-22, 1.4530551197822206e-06, 6.3403896878002e-17, 1.402698864964733e-18, 1.211681865953551e-10, 2.123204295817335e-23, 4.264290309752283e-12, 1.321564362410005e-16, 7.15933224319669e-17, 7.556759771176163e-18, 7.127228668357688e-11, 5.39781043147888e-17, 3.465596607147939e-20, 6.99623329105184e-14, 3.566048418353506e-27, 6.238822791715393e-09, 1.0736208251501669e-15, 1.3098312853180444e-13, 3.0539580916878807e-21, 8.480756643876824e-25, 5.4873860582402637e-14, 6.6679547175524365e-22, 7.671071724612687e-15, 3.0766574948158556e-21, 2.2721353878235595e-12, 1.0540875042326539e-17, 1.430495455193662e-20, 1.4977210103570937e-15, 2.1403618942881473e-22, 9.345311191566452e-09, 2.2860447909667236e-11, 1.251213246882411e-17, 5.039649063779723e-19, 9.46623069624984e-21, 2.9948936801011866e-20, 7.080014086113806e-08, 5.884036591297104e-10, 2.3516840030879393e-08, 1.7376475817574793e-24, 1.1004012805550489e-16, 7.051761676494754e-28, 1.2199068226856686e-20, 2.3885225936860766e-13, 5.9497856693651105e-25, 3.768833300998864e-10, 1.5046874328317867e-15, 2.066663706488465e-21, 2.3393639693306597e-15, 5.879650685021638e-21, 2.3386839408403574e-17, 2.2676308053526384e-10, 1.0705367259333798e-07, 9.888928067200107e-21, 1.7647449623336535e-18, 5.739691788708682e-17, 4.863144375453999e-16, 1.0431880917904576e-13, 3.634327364188949e-19, 2.6285058215569004e-20, 1.4831080841846414e-19, 4.0613593353189606e-14, 3.330390824229923e-11, 5.6978785532066774e-18, 5.2029915098561474e-11, 1.6708497802155444e-16, 1.0650621495855717e-09, 2.694762736186908e-22, 7.186056423850139e-15, 2.0466779312999783e-11, 1.2041696656491726e-17, 3.151126014675038e-17, 1.652593255793694e-19, 1.2603849519621235e-08, 8.298364827047823e-19, 2.4524078514470693e-12, 3.6477233607001835e-06, 4.6338695819945e-14, 1.220049217209862e-08, 1.0789658012530512e-12, 4.556653990760162e-13, 9.110595390799874e-18, 2.63914972964764e-19, 1.8074748512134087e-12, 3.964320392733223e-16, 1.9727456126710005e-22, 8.078681766523349e-15, 1.4553794638622747e-14, 3.443495895328816e-14, 5.613190462060906e-16, 2.6795301642516104e-17, 1.51059718170478e-15, 4.23692063557116e-15, 3.753552590274323e-12, 1.1263427233558959e-17, 6.101062910833662e-16, 6.223152757543239e-15, 4.998812236541028e-13, 2.46174285610732e-18, 3.333843075735421e-14, 4.571741942579914e-19, 7.24634708897402e-20, 5.386158020907907e-18, 3.93472074367336e-18, 1.75692893272564e-18, 4.309829843539021e-15, 2.1161752092411362e-13, 7.117642567798686e-19, 6.804214606502611e-16, 1.244171458327747e-18, 3.789832407964031e-19, 1.6733008288073256e-22, 1.97676557211336e-21, 1.1349376841975067e-13, 2.30690432345056e-17, 3.7732925779136034e-14, 1.3673298007043385e-17, 2.1817037065829936e-08, 5.146415354072989e-17, 1.2772078410006982e-15, 3.4081148182552574e-18, 8.517000082974846e-07, 5.148400084617304e-15, 2.2324227477086316e-18, 5.13753604003582e-15, 3.758971438298607e-22, 5.764065558812618e-16, 8.49883411645158e-18, 6.422949737176775e-13, 5.244393375617568e-16, 2.6172695745993905e-20, 4.083172444416545e-23, 9.50631308992429e-20, 3.8844179219341525e-16, 3.7766461507583726e-13, 2.8820478989444176e-13, 1.428142190974434e-19, 4.968734443867536e-16, 5.501615330839871e-13, 1.624180424755848e-14, 8.426560692730955e-22, 7.35581474952261e-20, 1.2423153684307546e-18, 1.2543899952508296e-20, 4.041845599313827e-21, 3.347355403733517e-19, 8.009160175348473e-12, 4.394374319586841e-15, 1.4170116902773033e-12, 4.779633177957876e-13, 1.074785985037685e-12, 1.2500084169208071e-22, 1.6002924651048532e-15, 3.081419033996221e-20, 3.910612151130196e-18, 1.4220707402444466e-14, 1.3563621059642372e-10, 4.557542426677878e-14, 5.492948841242238e-16, 2.3838896928348602e-22, 7.363407229434178e-16, 4.939140207937953e-10, 6.664927026348456e-15, 8.234712011671169e-15, 9.749500958840612e-20, 1.572312023417552e-14, 3.458789896214779e-21, 1.0918752032451229e-18, 7.070086403419364e-19, 3.8125809915030663e-17, 9.164994458495329e-20, 3.741507978911285e-17, 1.6999089310666682e-13, 2.1599345118068334e-17, 4.486005733550513e-10, 1.0239369742469317e-22, 3.991860874919808e-15, 1.5596008802773045e-21, 1.3129571757065917e-13, 1.5075342429971921e-21, 5.339743679197812e-22, 5.4059206888723565e-08, 7.937239960434278e-24, 1.86521567776491e-14, 1.4711661251200214e-14, 6.688192268552453e-16, 2.3740300301592383e-17, 1.6727886371987837e-22, 1.2304152577367794e-22, 2.3844936994719116e-18, 2.393518703176008e-16, 2.3358199171498023e-22, 2.5734265076675453e-18, 4.0510492738121684e-08, 4.95919223655189e-13, 6.883640879529196e-11, 1.5123961653353002e-13, 1.263735227760377e-22, 4.481310393161233e-15, 7.340346863324299e-15, 5.634070640581079e-11, 3.7734142097178847e-05, 1.843987831040468e-18, 2.682292457806615e-15, 1.6520397246380658e-12, 4.3057138721168984e-20, 2.4428693668763817e-20, 9.907055037450263e-11, 5.186447110542072e-14, 4.680345737709257e-13, 4.326748920084611e-13, 7.516909187113563e-13, 1.5847398886399105e-19, 3.494838501626848e-18, 5.174509747417013e-20, 4.185803786915798e-11, 5.306255062072275e-20, 6.582115602871775e-18, 3.316016352564871e-13, 3.3163231817796845e-13, 4.6587662440656223e-17, 2.70769226538934e-14, 7.409830127952377e-13, 1.4864171055184732e-21, 5.610336414191286e-12, 5.101668532070391e-10, 8.427798547927523e-07, 4.648584577167381e-13, 8.930882958097215e-19, 6.401308562064526e-16, 7.546565700704636e-19, 7.465837843778633e-14, 5.2702969004591354e-18, 5.084796583787465e-09, 4.582870468372006e-24, 8.845145786867768e-16, 6.48155032249853e-14, 8.868745183569456e-16, 6.938790362599756e-13, 1.82666233789592e-15, 7.294721898987884e-26, 2.759650299357068e-19, 1.7229310742550297e-06, 5.9516397807547764e-18, 1.0696379491772351e-13, 6.246755283184635e-12, 2.0235694630360568e-12, 1.5888483088336187e-14, 1.0246734571950356e-09, 6.623217006443286e-15, 4.848796919532106e-09, 3.790606608981839e-14, 1.4000326550553215e-18, 2.0733576529939708e-17, 4.518431413551306e-17, 2.0255788813871136e-20, 7.922700834574927e-17, 3.731366479903589e-16, 2.595743715458343e-13, 8.693095714241748e-22, 4.3395056428460754e-14, 2.2842175199933035e-12, 1.555786532589365e-12, 9.272289381540774e-23, 4.0979178702455193e-16, 1.0502869272655207e-19, 0.9986208081245422, 3.3196995499751303e-12, 3.263583204185092e-18, 7.010705950677537e-20, 2.421103886202896e-19, 1.0860088090587031e-14, 1.7815623650808965e-13, 1.5346685213418773e-17, 1.4383306004286439e-19, 2.7344437413400187e-24, 8.363101010322449e-12, 1.8348321993710837e-13, 3.0818647330302373e-13, 2.1261818355268737e-22, 1.645470067496876e-18, 1.1973006585808588e-16, 1.5926159113830059e-15, 4.4036998982262265e-14, 9.926586699293694e-07, 7.335473374558976e-13, 1.338299588336811e-12, 8.883737799084756e-19, 2.7939317658685624e-18, 6.04364741777874e-15, 1.8209955765513541e-09, 1.145660799386303e-22, 4.627604551272917e-19, 6.208635750226676e-05, 5.8945310854596755e-09, 2.3269349988530585e-11, 4.293978801001019e-12, 2.092983115317675e-20, 4.9515661546406455e-20, 4.253344435011831e-14, 5.3018396133214975e-17, 1.3584103741977252e-18, 9.235502558427525e-19, 1.1807339828084906e-10, 5.645714062237246e-18, 4.753701365921639e-15, 4.947150625756746e-19, 1.365796424983576e-19, 2.7490816608246616e-17, 2.403118491681807e-20, 7.351661640564444e-11, 8.559562227357015e-20, 3.2395211819104144e-17, 2.1733678745583518e-10, 1.5757794374904321e-15, 1.8211878382263625e-18, 2.799919393688589e-24, 3.32469923987333e-12, 6.552097228135749e-21, 1.064646553425655e-14, 3.919300132501417e-19, 4.524519647830516e-12, 1.8809142075606007e-18, 8.575412379979372e-15, 4.0664442435079176e-14, 1.6882793662659766e-21, 1.6922565779231569e-12, 2.780725568895212e-19, 1.1616225932271473e-14, 1.5301497990916226e-17, 1.0744660085215715e-17, 3.3255447512437677e-22, 9.858956444324561e-18, 5.68230561615569e-26, 5.497973334806232e-16, 5.119995701616413e-19, 4.544095016310564e-15, 2.2610401051448292e-10, 3.8113859471284987e-16, 1.1638815030425324e-15, 1.521503260296271e-14, 1.367267100413907e-18, 1.0192224668414962e-12, 8.33506807159772e-18, 5.903837853570839e-15, 8.188707593504942e-21, 5.183155964025516e-15, 6.818110817277028e-15, 2.9761931556288028e-18, 3.992764519011871e-08, 2.3891732256332466e-17, 9.353397223410553e-11, 3.4121531988784426e-11, 2.3169923553563454e-13, 5.642041463035572e-16, 3.6449483470982585e-21, 2.5345883347720677e-16, 1.9751507588457044e-18, 2.5019352570070617e-14, 3.960428447864761e-13, 6.604131816206093e-20, 1.7989095795201138e-06, 4.4068033666633255e-09, 4.397555153917293e-20, 4.244793780346109e-17, 8.660123615638682e-16, 1.0301227610449906e-14, 1.9692724013975482e-12, 2.0020680431587925e-18, 1.3642850109080768e-19, 1.2897152783923654e-17, 2.0138740935288224e-12, 8.027355863360341e-11, 6.405773134154913e-11, 2.973259675908657e-20, 8.23181023346109e-10, 1.982348098714559e-17, 1.4198718065655816e-19, 7.330355344947748e-08, 8.945784651243072e-14, 2.5975795611686343e-17, 1.3989804526957049e-14, 5.69569565123107e-11, 1.6038113491347733e-13, 1.1188836718130357e-16, 8.869623785434655e-11, 7.047678707294835e-08, 4.919594089157694e-21, 1.4259744101664806e-13, 1.7239144091275562e-16, 5.467420218435722e-20, 1.418859807747995e-15, 4.5692586241486107e-26, 5.96695862932433e-21, 5.170688096439724e-12, 1.4352144472873363e-10, 6.184489091632869e-15, 5.2648179985860566e-21, 8.020242591800023e-14, 2.3415028122257715e-14, 5.357602298596504e-20, 1.7403601399335106e-21, 1.0890490538283712e-16, 1.9296917661529033e-17, 1.2142306237349857e-15, 3.19784664391762e-19, 2.6246883507182255e-20, 5.5786551288563354e-15, 2.0244050847554457e-13, 5.368310505328433e-14, 1.7507647892198164e-13, 2.7149525683167752e-11, 1.2127375693467724e-16, 6.276408260035959e-18, 2.0582355923077306e-21, 4.382453776300658e-22, 2.2971413660144435e-10, 1.2441248552208517e-11, 7.370056007936206e-18, 1.9192857210210273e-13, 5.895320937286197e-15, 2.0486389320309874e-19, 3.1151029607428434e-17, 8.104582005508121e-10, 2.4206075579830295e-15, 1.141868023746273e-11, 1.5947839980906217e-15, 3.8089821561325165e-19, 1.4273180077087333e-13, 8.695535945581017e-16, 6.056875834214992e-11, 9.082399450695901e-17, 6.914089745180263e-09, 2.323604785947867e-20, 1.291850843667018e-16, 2.2253941326514575e-21, 9.531142693662564e-14, 4.236961004266959e-08, 2.5680138896443105e-13, 2.724699212368253e-21, 3.4022898387497946e-10, 7.483308097335409e-19, 3.183074026580357e-09, 1.5312707773344792e-22, 1.340461487468747e-12, 2.5059235718644196e-22, 5.195643770766886e-15, 2.795097676941956e-18, 3.749092324923245e-21, 2.2953717919438343e-12, 8.693762867394397e-17, 1.0759553885735039e-18, 1.478243090688287e-16, 5.953934177280185e-14, 1.0152988672018637e-08, 6.196755316540471e-18, 3.1970959485604067e-16, 3.9712580557923786e-17, 1.7752333152327933e-20, 4.670983453558847e-19, 7.110795935707448e-21, 3.972317266787151e-11, 2.2293648826684272e-15, 5.6950911365221085e-16, 1.0611249292441505e-14, 3.894684374845181e-18, 1.9871809059379686e-11, 5.040842743220253e-10, 4.10223511041903e-23, 1.647305035434865e-11, 5.0833154574547734e-08, 1.8837450264119103e-18, 1.9998262215051993e-13, 1.2286186641902262e-14, 5.109122569457725e-10, 9.941355498413549e-15, 3.104717618002478e-21, 2.7891403701468245e-15, 2.71099527375529e-12, 1.1330328897329878e-09, 1.5584211913023498e-15, 8.115066405638828e-22, 2.6933042926967354e-13, 2.0619661533771705e-10, 1.3198297398500628e-19, 2.0372451976396657e-20, 1.0656939096649349e-14, 2.88499015259e-14, 9.814374883466525e-16, 5.361289973164842e-19, 7.206857408982557e-18, 2.3933732765525152e-18, 6.904391269979452e-20, 1.934410187587332e-09, 4.160368418306497e-17, 7.114130462315362e-18, 9.637595083968054e-14, 3.564169375895642e-15, 1.9401401318841494e-10, 5.0964705880601765e-21, 9.660725344695642e-12, 7.935538465716718e-20, 1.999966511845308e-11, 2.743676924740786e-19, 8.075545787633587e-11, 3.3479946034159643e-16, 2.5245396051420954e-11, 8.056050926147451e-21, 1.8381502916997583e-11, 4.535307240871589e-08, 3.0154604941518004e-12, 1.1669742320911947e-16, 7.872394211940805e-20, 2.987101677121018e-10, 2.7359172666643373e-15, 5.1791606221098805e-18, 1.4447945206073783e-17, 1.831937090912561e-08, 1.2740487484452778e-11, 3.7424902066263144e-10, 1.7922593409380738e-09, 3.2808290152280276e-20, 1.0724032165292598e-19, 1.7970269844120972e-16, 1.4051702730127078e-15, 3.831303238668914e-15, 1.5473477178246575e-15, 2.5957907698326288e-12, 1.2803169763496298e-18, 1.8494473579266923e-06, 4.1927783473230595e-18, 3.23558191260395e-20, 1.5930610024847672e-20, 2.249902831810679e-19, 1.1842270028250092e-17, 1.2144064889506594e-16, 1.3438707612001277e-13, 2.844956800522602e-13, 4.0435856052400965e-11, 5.846089880231785e-18, 7.861498951822199e-11, 3.6377987206873466e-17, 1.4042493364408055e-14, 1.0587522205523017e-14, 2.5559841244394976e-13, 2.755382295550478e-15, 1.2191515195447024e-19, 1.391637082030638e-17, 2.1826319645980203e-21, 1.10217997922879e-11, 3.4459311997554667e-16, 1.5410757739364733e-11, 1.4561685849022274e-17, 1.7846595423760208e-17, 1.8474040353044785e-10, 8.590009129352816e-14, 1.072679960142947e-18, 2.1417040030010694e-14, 2.3187270703619927e-15, 7.724842540774693e-17, 8.284544706963593e-17, 3.375858496678446e-19, 6.084520844001757e-24, 1.1729209617450387e-11, 4.2786789980010893e-23, 1.195807365964629e-16, 2.7948594095517434e-14, 8.427156331434447e-19, 2.5398065872650477e-09, 7.465025337846455e-19, 2.504057345590477e-13, 4.884838159782603e-15, 7.283668077175251e-17, 8.030543183447548e-18, 1.9486011925123198e-13, 4.686460941538684e-11, 2.2097660506981143e-11, 5.475678195135386e-19, 3.863852100494114e-07, 1.5823259106561908e-20, 1.0413310214119065e-09, 1.6101495522760897e-20, 1.4729515832225608e-17, 1.380264988675578e-13, 1.1730268259505267e-21, 5.021197960111418e-20, 3.649300832541019e-17, 6.78104931418639e-17, 1.910156874253971e-14, 1.9834553626825224e-17, 1.5588749908979072e-12, 7.706225905341119e-20, 5.742060337674666e-20, 5.135886382845456e-09, 7.7160355686645625e-22, 1.377624592180382e-08, 2.1823832685127255e-14, 5.297428801588522e-23, 8.345622301677831e-21, 2.8239991725284574e-13, 2.0594407255936087e-12, 4.4633667963846213e-13, 2.871082488011077e-18, 1.0792517446735583e-17, 4.897527543821807e-19, 9.525937003869285e-17, 5.620901952583105e-18, 2.639320186015315e-12, 9.911435387699452e-12, 1.2031849583489518e-13, 1.6098829214161725e-10, 1.6530036562847333e-17, 1.4870659107711603e-18, 7.890092659085246e-13, 1.3765577862589495e-18, 1.1096988779268901e-11, 9.03585419615089e-16], [1.1132122237483517e-17, 6.603784012607908e-18, 2.2836281002584953e-14, 3.7844086880850975e-19, 3.0513819823749927e-15, 1.4427109507053615e-10, 2.5936492952061498e-17, 1.594066709415063e-14, 7.499465860930765e-23, 1.926867086027295e-12, 0.9847542643547058, 2.3116111455556698e-11, 8.028445819263652e-08, 2.3434080068002692e-20, 1.324162811735044e-13, 2.7129787674197436e-15, 8.375690419004655e-11, 6.811758188757233e-13, 2.8472101961535008e-16, 3.8270648413108086e-20, 4.824395794490667e-16, 1.4301252890258631e-11, 2.443862726781527e-19, 0.00027756791678257287, 3.3567084383172307e-18, 9.8800701260624e-20, 1.8430144876136768e-17, 3.1631229459711393e-18, 6.543253675772576e-06, 2.130438815584057e-06, 1.6822891586915607e-17, 3.500984766847959e-13, 1.0665096193337997e-13, 0.0002000203967327252, 5.335257236005475e-11, 1.1266866857341108e-19, 9.03388752693246e-11, 1.2215169197318798e-13, 7.372061259177157e-16, 1.402715617271344e-11, 3.605894193855175e-20, 1.7242667218940547e-16, 4.5360755082657356e-12, 1.4671787310865839e-18, 6.755903887538384e-13, 6.769004844489612e-14, 2.6146193197110205e-24, 3.973862254882943e-13, 2.2705292447948973e-22, 8.805400852907042e-09, 2.178960744784641e-21, 7.119032748215658e-19, 1.204277907402204e-19, 8.452540953470051e-19, 4.77064837761383e-15, 2.381297653672485e-23, 1.7603292819723046e-16, 3.6197238539870114e-09, 5.863180559477982e-27, 1.8669449219965094e-16, 6.190686718809957e-08, 2.746489830268012e-13, 7.331154944498756e-19, 1.557345615313399e-12, 2.290274932569978e-19, 5.922438485334307e-17, 9.591123633786016e-18, 6.830391947979268e-15, 1.0124068244238113e-18, 4.923142796209312e-11, 1.1509519899643994e-11, 1.5839649591953275e-09, 5.563162049218107e-16, 3.0983423975761674e-23, 4.629605745192964e-16, 8.047902395782586e-18, 1.2399620189085259e-17, 3.0543739034264663e-21, 1.5620436335804012e-15, 5.823769073146532e-17, 2.5015551417075585e-17, 9.829209522043206e-14, 3.271719646504717e-14, 8.127021191434158e-13, 2.3251324927147405e-23, 4.6039235077088665e-17, 4.3602001168940985e-18, 3.0727150153713606e-14, 1.6567989946298055e-22, 2.6702153247326814e-18, 1.5158719999744673e-17, 1.741073333497918e-25, 3.7178823771277905e-17, 7.550541854511602e-18, 1.1922266872085248e-18, 3.1762187461395857e-23, 1.248197915469304e-15, 0.010996072553098202, 1.1495218957627956e-17, 1.11386908787278e-17, 6.301252922225054e-13, 1.2054526694697105e-22, 2.07325815044547e-14, 9.620476073765594e-13, 6.225223003752944e-09, 2.0488138095797748e-12, 8.805009672763209e-12, 1.8104373794396167e-13, 9.334050721760156e-24, 1.178633111214558e-16, 1.7351063313534484e-19, 7.647111055124105e-19, 2.5352620393673253e-13, 1.6904399158679945e-13, 1.040004257995571e-16, 2.7480848668275045e-18, 4.88919613417238e-06, 5.055676234914323e-15, 2.2765286771295097e-20, 2.0624404961644416e-12, 4.8956098910077255e-17, 9.931961585391144e-17, 2.298362541952592e-11, 8.400589113222168e-10, 4.471820075547157e-19, 9.384621302584603e-18, 2.4539410503621184e-09, 9.194647166894025e-25, 6.137003180530152e-17, 1.0769531798196074e-14, 2.665944436133705e-19, 3.591287152642647e-13, 2.776635834504746e-15, 5.841797772382145e-15, 2.6972573508032838e-08, 2.2489686131996808e-15, 2.07306493393844e-06, 8.383856895397349e-14, 6.865715551220377e-16, 7.243286310810494e-16, 2.466827816574092e-16, 4.051323183387101e-17, 1.9727741662478593e-10, 3.018278657629429e-19, 6.644308237051735e-20, 1.0849301404522564e-25, 3.97206472232574e-17, 1.8878681908932422e-17, 1.649666968512372e-15, 1.6925543511861307e-19, 2.19224716438336e-17, 1.1234499411202864e-26, 1.166592463423083e-18, 2.0997720345975226e-22, 1.113944114901344e-12, 1.9784770610015157e-13, 3.891176384585405e-17, 1.6876180862572352e-22, 1.1868956633074725e-23, 1.971460321381624e-11, 3.9417183750004804e-21, 4.5536927961148864e-26, 1.4155232922359962e-19, 3.941898196746507e-10, 0.0004407750675454736, 7.572560324013351e-22, 1.8995812867426748e-15, 4.079731886520721e-13, 1.7375703009747667e-06, 1.007910118214217e-12, 8.740342434185311e-13, 1.305745062855218e-13, 2.615355467391607e-15, 1.1394911346548762e-20, 1.2078281896778925e-13, 1.215128258430509e-12, 2.350516135896097e-11, 1.1483103070656235e-25, 1.6358854248231893e-15, 2.2440796250549126e-17, 2.326635065164062e-11, 1.2181399000610327e-08, 1.2875077543497813e-20, 3.762670526990046e-21, 2.9389958348946532e-12, 2.0989598947807053e-12, 5.801307479921022e-16, 1.1290354940347765e-16, 1.1467904057934171e-19, 1.788225543846131e-21, 1.5149601160671888e-17, 6.314011271289777e-13, 1.2405906090862118e-05, 1.7189117001921743e-15, 7.456819706169929e-19, 1.655255691898356e-14, 5.959015059175954e-09, 1.3521539302726454e-13, 2.0016475128071767e-16, 2.3699466868143393e-13, 3.4745404331459895e-09, 1.6674441204693472e-18, 3.766342587878087e-17, 8.959238732841879e-12, 5.640492090417244e-21, 2.3081600378834638e-14, 2.9101595265534907e-12, 1.3740437775822477e-18, 4.0715113534170704e-16, 2.8624161100438184e-24, 3.186026720225499e-10, 3.090901641253012e-17, 9.43163810940284e-11, 1.0191892885202947e-09, 4.198189972742794e-12, 1.051473323983772e-18, 3.7221686843835755e-10, 1.6769441738007096e-11, 2.12481721106883e-09, 8.107898368093873e-16, 2.428346634886225e-09, 5.65475641214497e-22, 6.339777739583034e-18, 3.596659174680418e-20, 6.402645482389807e-20, 6.492338134114761e-14, 7.470455189780706e-14, 1.4760800597085752e-15, 5.180623759029699e-15, 3.1935290196140346e-19, 3.261033587262374e-12, 8.827273113853895e-12, 5.4609898010983884e-14, 2.394163275396534e-17, 1.808655390201694e-17, 2.7492293952820635e-16, 2.266289383142931e-18, 6.561989388669686e-17, 6.437702476137785e-14, 7.754235960472956e-15, 1.371018328095888e-15, 1.8666561367010548e-16, 7.319930585233326e-23, 2.94673898997011e-13, 5.810241113254656e-18, 3.117282162110914e-14, 1.879787937000623e-27, 2.4727851263470733e-14, 2.6339908109622997e-18, 4.8353642358829e-15, 1.2250564451754542e-10, 3.639717176197495e-14, 1.5293932466412995e-28, 1.4488769613327929e-13, 2.2401488860485154e-20, 8.451377110348188e-15, 4.290608132415002e-20, 7.519416934033028e-16, 1.9058051320174812e-19, 9.766000508282247e-12, 1.3334782199571878e-19, 3.109229978393717e-17, 3.0577000011285625e-15, 7.159515581317114e-21, 4.4980248096061076e-14, 1.7807097417161903e-15, 2.314349449930586e-12, 1.4856185400792532e-13, 4.463798588492152e-22, 1.542203864612901e-12, 5.3317547656106e-15, 1.8552815533867673e-12, 5.866127523242604e-15, 4.102468928791337e-23, 1.4603100300750967e-18, 4.8115162797239796e-24, 7.22424731101079e-21, 1.3307835431233036e-14, 3.293735049243393e-14, 7.936960104846361e-18, 5.882331263523579e-15, 1.2071499399038399e-12, 2.872580950506135e-15, 1.7705368034345674e-19, 3.148433376345055e-17, 5.581984087039463e-20, 2.4527664621576406e-11, 1.1198661332581814e-21, 3.82487078576417e-14, 6.006394303469082e-20, 3.516067678457715e-18, 6.256506191100356e-19, 5.301767258178956e-15, 6.140354284062122e-12, 7.666083767787768e-09, 1.1944341999844954e-12, 3.1000738407544956e-14, 9.55523888445224e-19, 3.845313479599494e-19, 4.120419896059371e-17, 2.070410891544922e-15, 4.366782373532874e-26, 5.297976031215512e-07, 8.851718762538461e-15, 1.8409943471216998e-17, 1.1401207668931823e-12, 2.9184396136710907e-21, 1.2951594337683322e-16, 8.716735666602915e-11, 7.881199337589351e-08, 7.985266907341998e-14, 6.514929305183228e-19, 4.124141566447248e-12, 1.291189851911333e-18, 2.608227896898699e-15, 3.3545290241983066e-18, 3.7096242522238754e-08, 3.338566597317432e-15, 1.1912697167879834e-19, 3.435707495538495e-18, 2.1541991281071038e-13, 1.852262093704482e-10, 2.4041498269318867e-17, 8.570367274108489e-20, 1.7611154363813354e-22, 2.236897334910123e-15, 6.483298598501663e-15, 5.224187934056409e-16, 2.3769987443199997e-14, 1.4298956886650436e-21, 3.141904243395558e-23, 4.638852367126776e-17, 2.2804908184070882e-08, 8.72798189277546e-09, 7.0605721790018006e-12, 2.9468925077935637e-09, 3.025024703265711e-20, 2.122383774397262e-15, 1.171697183721676e-10, 1.303877785075743e-16, 3.405860916522173e-17, 3.685779083964301e-18, 7.091003862406269e-20, 1.0557151940843168e-12, 2.3046381161133942e-20, 6.39639639135591e-19, 1.0642492836410183e-19, 3.894889763791278e-17, 7.248500031534905e-14, 6.070748131047023e-16, 2.776112103445547e-16, 1.74605741208578e-15, 1.0966435479247308e-17, 6.765594594976276e-12, 1.8407374149237432e-12, 1.948132876350428e-06, 4.612394250771716e-18, 1.9089347440156814e-18, 0.00022580147197004408, 8.212824839132796e-17, 8.830347274246288e-15, 1.785019868868294e-15, 6.134788342781411e-25, 4.968327724057253e-13, 3.1974062878763737e-24, 1.1014785840412257e-17, 5.2653511139412485e-24, 1.2077795149295785e-15, 5.845248776489312e-21, 1.9666095978973085e-16, 4.1709576536361556e-17, 1.4992055358928061e-19, 1.029730967161413e-09, 1.1840493863157775e-12, 8.304086061336431e-22, 1.1424870652857742e-27, 2.0297247227147458e-14, 9.834387486390575e-23, 2.4314269348114353e-17, 1.8358836044278515e-14, 1.3962416987157002e-16, 1.9257544277961815e-18, 4.13308985458907e-16, 2.359301923938298e-24, 2.581280380788583e-18, 5.3017835212115436e-14, 2.0568022653967566e-13, 6.967250978794526e-16, 2.0922147478050213e-14, 1.3392402353653958e-14, 2.4598918326915795e-18, 8.971198319436372e-20, 3.9538465876146124e-22, 1.9597120401731645e-21, 7.782355489205361e-13, 1.2425155949426486e-15, 7.142987898755193e-19, 6.767497117902566e-18, 8.525624557787613e-21, 7.274563667890609e-25, 1.5819584411966986e-13, 3.5760463600736925e-12, 3.3754552865195046e-21, 2.4540031767743606e-23, 2.1530442495054994e-15, 1.635744879185696e-19, 3.2260422324698322e-18, 1.0004599213420291e-14, 1.1240495296702124e-12, 7.60228824336906e-10, 4.083758938649984e-12, 3.974499833523e-14, 5.74730118859968e-16, 3.1402325184615165e-10, 9.32635490703152e-19, 5.5358005505989905e-12, 3.172490892211499e-07, 2.1008794703902822e-09, 4.869317171696252e-10, 1.168596179210022e-19, 1.9927351327213294e-13, 8.836069828837938e-15, 9.961450739664781e-18, 9.142486538606023e-15, 7.900535840236155e-24, 6.426457459597537e-21, 2.344503985483165e-13, 1.161177403091187e-19, 6.86365200786021e-11, 1.3915278166415986e-20, 2.708765557777465e-13, 1.6680873940163113e-15, 1.9416720903092828e-14, 4.553132502103929e-13, 2.6103009372452098e-17, 2.158729100516589e-22, 1.473423076171716e-17, 1.8893241791202565e-27, 1.3188126146701507e-08, 1.3566141265908271e-10, 4.9370739701321e-13, 6.159800939956603e-17, 8.944817849491437e-19, 2.853577223367961e-22, 1.7450837477129113e-17, 3.5909877502127152e-12, 4.869115957732142e-19, 2.0760122950141263e-13, 1.343602393666954e-19, 1.50705372524906e-14, 1.432320789751529e-10, 9.899115381573065e-12, 7.205489525026365e-29, 2.3173267471587877e-10, 4.254678478022439e-13, 1.55175089183553e-19, 1.913929952024329e-12, 3.21317018544468e-17, 8.319772674891002e-17, 2.066207760442751e-10, 1.7821041517485092e-11, 9.748237772555512e-10, 4.319683445577066e-13, 0.0003303684061393142, 4.40925464542663e-12, 3.6741694715889745e-21, 2.7104217867102197e-15, 2.426985560410803e-20, 3.626779728714097e-24, 2.8991515799681145e-17, 5.887166180526066e-17, 2.33472513714674e-18, 4.532894784352315e-13, 3.7348994675614764e-17, 1.711938459102256e-14, 6.143642994511922e-13, 2.0826022896943124e-16, 3.2307010799231817e-12, 5.1205339506259403e-20, 7.365598216353113e-24, 2.4127798425865876e-08, 2.378942714543783e-22, 1.7883553853529077e-17, 9.405153149615476e-15, 2.0326946554011644e-23, 1.3628529304843566e-15, 2.7578679357570524e-12, 7.233008636475802e-10, 2.1381379519414814e-15, 2.0174569526337597e-17, 2.5481766643729993e-19, 1.010146081907061e-14, 1.9773255705317003e-15, 3.8478693619297014e-26, 9.176125853847849e-20, 2.6134596806007097e-14, 1.5824677451672237e-13, 5.811388696652653e-14, 4.307745132825502e-11, 5.303020878773453e-22, 8.315286393473231e-13, 2.485924240544389e-18, 9.501503163909265e-19, 3.0248777391202544e-16, 2.869210553243269e-22, 7.222671646456692e-16, 1.8211156866364264e-13, 8.762911482075618e-24, 1.2320506610566112e-22, 2.0882303053565426e-23, 1.6704951698721764e-14, 1.004674926694156e-14, 6.530405547482539e-21, 1.315594078326171e-27, 1.4957155275055711e-16, 1.319110765507503e-06, 2.196613013316657e-20, 2.472271727931778e-19, 1.4965328377854649e-19, 1.2920490857726225e-18, 2.6660518237520137e-10, 8.505219812193406e-24, 2.448877573868615e-20, 1.8652789688690941e-19, 7.471206134781028e-18, 6.227689127195554e-14, 2.8239560956587645e-19, 3.248005434487028e-13, 6.192699212584016e-14, 1.764301121773882e-20, 1.7306170602485182e-14, 7.195770568292015e-22, 2.0650986051336297e-27, 6.030130535489627e-18, 3.460477564221076e-12, 1.1190445237119206e-09, 2.4668141878139704e-13, 1.5945677394355138e-21, 3.0322094131205875e-18, 5.426957711343605e-14, 1.84237064331209e-16, 0.002649386180564761, 5.881268893515667e-27, 5.6533844757457074e-21, 5.952702337264993e-15, 3.2571127404245203e-28, 2.5971241484512575e-05, 3.089023914792737e-15, 8.16415459068136e-16, 1.296722285104579e-09, 3.920362823842073e-16, 4.543915690171207e-18, 9.62979782720419e-12, 2.1617332828850234e-14, 9.779081017180502e-11, 3.405012655082462e-21, 3.2029638941111216e-09, 3.477556961551329e-17, 6.364622717747181e-10, 1.190046239113835e-18, 9.521757399200397e-15, 4.5878815804993194e-18, 8.819642260959881e-08, 1.1721936755819553e-17, 4.771705220622119e-14, 2.1944877081421683e-14, 3.600208576147344e-25, 5.683694400543393e-12, 2.6091189755592104e-15, 2.0505155906569428e-10, 8.066174195128423e-21, 1.3200012160454063e-15, 1.4816721761894317e-18, 4.134126886270137e-20, 8.596988956123899e-16, 2.1673651318110387e-08, 6.136264673679265e-17, 1.4485311328448096e-22, 9.460176958730513e-24, 5.012646980529151e-20, 1.5651252516235559e-10, 7.276892930291043e-19, 6.268128595694609e-18, 6.952918796265806e-10, 3.585830023368906e-18, 1.6533168931132103e-11, 9.276946750938833e-14, 3.074712125958257e-21, 2.641317149647682e-18, 5.001054908734814e-13, 8.145525159797685e-16, 5.530614191905681e-23, 7.464284724166548e-13, 6.386654399781477e-11, 1.1600200799932736e-09, 7.077976036220027e-14, 3.412196767170099e-21, 7.373818217268e-15, 2.0339600627616084e-20, 2.202910423521054e-11, 2.1873944001319767e-15, 2.727045811390383e-10, 8.514241248715442e-16, 9.819492070889962e-20, 6.408186163753271e-05, 1.0321293397453886e-15, 1.1181463407541527e-22, 3.643768235013958e-18, 1.3537244913472532e-21, 2.9120313873045055e-21, 1.6975005072305985e-15, 4.184312919541881e-21, 7.207563821225677e-18, 7.382395116063622e-23, 5.352097346041085e-13, 8.818744616934798e-14, 2.5408750066891442e-17, 2.2069552323773664e-25, 1.942028829415854e-11, 5.338571368783781e-22, 9.891353144539372e-13, 2.1847623378926125e-17, 4.405777458453041e-19, 5.153823394721258e-07, 2.723958855515001e-14, 9.457112691698896e-14, 3.7540699275053196e-15, 6.745036522587747e-14, 4.821407885769228e-16, 6.380418073042999e-23, 5.134441583834039e-18, 3.392397746844245e-21, 1.59195412408053e-08, 9.509634252395433e-21, 1.469303981418998e-16, 3.1605282871847136e-18, 4.003363582343522e-15, 2.7853088215730033e-18, 4.781841589363012e-11, 6.135741498485437e-16, 6.36696296424014e-16, 4.892159268833629e-13, 1.1452641121604148e-20, 2.7959554632371734e-18, 4.6124835178930974e-24, 1.3070043425027363e-15, 5.723619278183434e-13, 1.5711410850712676e-14, 1.3901618571565688e-13, 7.667706747316316e-11, 9.524872256984806e-17, 7.610388630323754e-20, 1.2122909182857734e-14, 1.5748371133366117e-15, 5.609441282478122e-16, 3.6135755498989397e-10, 2.7724762485803467e-09, 1.6037303041116502e-08, 6.936413367921194e-15, 9.417406232614667e-08, 4.225342298269652e-12, 1.1000999416976853e-10, 2.9454938774780304e-17, 1.893734208123117e-20, 4.38371677899092e-17, 2.3315513609416597e-15, 1.4732375319681523e-09, 3.90311741260696e-12, 1.4971429315464163e-14, 8.084691253525881e-17, 1.0416951453989451e-13, 2.3354805350257355e-15, 4.383200006836778e-14, 3.8667523854897346e-20, 1.7714312700200728e-18, 1.0731916050277235e-10, 3.638010662535638e-22, 6.735831307330166e-15, 1.0994948448326873e-23, 1.8151965432002468e-19, 4.680856396932498e-12, 2.1797968703489434e-13, 5.690622804791282e-14, 9.423684705842095e-14, 2.8537611191890635e-18, 3.8582629741345375e-25, 8.724324477875822e-16, 3.2941717794309974e-14, 8.204403478752516e-17, 1.6280148763613879e-19, 2.695770862962354e-17, 3.524510869788722e-14, 1.551648474558412e-27, 9.023900306551727e-22, 3.643394993254473e-13, 7.935296029212141e-15, 7.48925819359714e-17, 2.3661607395641937e-11, 5.015824181778955e-16, 6.584749238924825e-12, 2.745451865848736e-07, 1.0639611952452299e-16, 3.9765560636007e-12, 7.500044422320551e-14, 5.229171191814645e-24, 3.0224840309389514e-20, 9.68784110652332e-19, 1.7277441699548035e-09, 3.343165733930936e-19, 1.1100305893274522e-16, 4.893394949481492e-18, 3.957943536725878e-24, 1.7157976428539001e-13, 6.592018030138831e-17, 3.2176747377654635e-11, 2.330576214261133e-16, 9.475077413477213e-15, 5.659130932963996e-13, 7.282452418877483e-11, 8.197619666416855e-12, 2.1653303350196087e-19, 6.629407439673329e-11, 2.374225234444043e-19, 4.740172143691268e-10, 4.180759810069358e-14, 1.3267248072435752e-19, 3.258051692332585e-19, 7.869369218760358e-12, 5.718639049502472e-17, 5.789788255354837e-15, 1.8208638150368133e-16, 3.8711944760905226e-09, 5.504783061975947e-19, 9.26196731207982e-15, 3.0780843706440013e-24, 2.691184101467122e-14, 6.808730417077109e-21, 1.0584041462592619e-17, 2.8653303729697654e-07, 2.336466252677444e-13, 2.752816688898129e-20, 3.2966466720600206e-16, 6.9080974800109565e-15, 1.5053317130575563e-18, 3.7364009658093827e-17, 1.2377973297354163e-21, 4.621060423204426e-09, 5.022706026690376e-18, 3.611744082521011e-14, 6.502171847819205e-14, 3.5543811817417836e-13, 4.3315929998660047e-14, 1.707990693322614e-10, 8.126348757414171e-22, 1.475039708431769e-12, 1.8191481307439084e-11, 3.938783257074981e-21, 2.1431202460593455e-17, 1.2223237158348127e-17, 1.8352649633269125e-16, 1.6208550057683624e-09, 2.6040940237503918e-12, 3.106614629173193e-14, 7.116865012894366e-14, 4.762863456943873e-22, 2.4343113635971747e-10, 2.082530718925787e-19, 1.214197245342908e-16, 1.4109905244197307e-08, 1.4007528191761255e-19, 8.173335083921867e-25, 1.0938271772717445e-13, 1.0529646815929383e-12, 1.6070552290952907e-16, 7.865757822855812e-09, 3.3983605588882443e-15, 7.731609169352952e-15, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.9159212112426758, 0.08401701599359512, 6.178620242280886e-05, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[1000.0, 999.0, -1000.0, 5000.0]], "dtype": "float32"}, {"__nd__": [3], "dtype": "int64"}, 1.0], "e": {"__nd__": [[0.7310585975646973, 0.2689414322376251, 0.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[0.6408113241195679, -0.7422678470611572, 0.9908414483070374, -0.830876350402832, -0.6163143515586853, -0.06488239020109177, 0.1956513226032257, -0.0712125226855278, -1.190629243850708, -0.18629302084445953, -1.5499991178512573, 1.5515836477279663, -1.5134544372558594, -0.4971221387386322, 1.8701512813568115, 0.2858133316040039, 0.1169222816824913, 0.7998162508010864, 0.025532396510243416, 0.6050108671188354, -0.6117571592330933, -0.7000864148139954, 0.3629465401172638, 0.9442941546440125, 0.028800074011087418, 0.8947988152503967, -1.1754004955291748, 0.9798157811164856, -0.8085470199584961, -0.9542477130889893, -0.5825952291488647, -0.013204162940382957, 1.1451867818832397, 0.5261544585227966, 0.09142809361219406, -1.1446659564971924, -0.6231424808502197, 2.139493942260742, 0.23879441618919373, 0.6338517665863037, 0.3228476941585541, -0.7454987168312073, -1.6870143413543701, -0.47711533308029175, -0.2921617329120636, -0.7704811692237854, -2.502479314804077, -2.0578505992889404, 1.2132328748703003, -1.4516270160675049, -0.7840376496315002, 0.3424729108810425, -0.7600362300872803, -0.3241058886051178, 0.44701874256134033, -1.4037872552871704, -0.3024364709854126, 2.489130735397339, 1.527458667755127, -0.9950571060180664, -1.0445570945739746, 0.2598831057548523, -1.1655795574188232, -0.724038302898407, -2.6034886837005615, 0.22317464649677277, -1.3967384099960327, -1.7335562705993652, -0.8297799229621887, 0.719476044178009, -0.5618408918380737, -0.20798754692077637, 1.5431703329086304, 0.4044014513492584, 1.6567636728286743, -1.4963079690933228, -0.1016959697008133, -0.4662212133407593, 1.2942084074020386, -1.2484495639801025, 0.38899800181388855, 0.3775566518306732, -0.1270313262939453, 0.19184985756874084, 1.1810057163238525, -1.3054254055023193, -2.9583141803741455, 1.2237083911895752, -1.6612176895141602, 1.1915990114212036, 0.7807748913764954, -0.5163268446922302, -0.5699069499969482, 0.4268609285354614, -0.5564287900924683, -1.6076551675796509, 0.27239304780960083, 0.5223296284675598, 0.23288007080554962, 0.3068579435348511, -0.30719178915023804, -1.1944949626922607, 0.3740614354610443, 0.35821276903152466, 0.6930440664291382, -0.44307640194892883, 0.07315631210803986, -0.49059075117111206, 0.8931739926338196, -0.5498921275138855, 0.19920770823955536, -0.44843748211860657, -0.44610461592674255, 0.1615850180387497, 0.8324604034423828, -1.2344938516616821, -0.7735029458999634, 0.6181135177612305, -0.5405920147895813, -1.225908637046814, -0.6890875101089478, -0.14634589850902557, -0.9383406639099121, 0.45894742012023926, -0.1495591700077057, 0.17369608581066132, 0.03607790172100067, -1.4330925941467285, 1.301230549812317, 0.6829060316085815, -1.9512019157409668, 2.093045949935913, 1.5030876398086548, 0.6405590772628784, 1.5695921182632446, 1.6440486907958984, -0.08507265150547028, -0.21229113638401031, -0.9452081322669983, 0.24898381531238556, 0.4627828299999237, -0.01578366383910179, 0.5475114583969116, -0.25218266248703003, 0.7995489239692688, 1.328877329826355, 0.18907131254673004, 1.2633450031280518, -0.10538466274738312, -0.6671819686889648, 0.01692793518304825, 0.38036632537841797, 1.129266619682312, -1.8414472341537476, 0.835107684135437, 0.19949844479560852, -1.0092434883117676, -0.8476724028587341, -1.1798549890518188, 0.4479225277900696, 1.8401763439178467, -0.35851314663887024, -0.6312440633773804, 0.8843576312065125, 1.7466305494308472, 0.49688920378685, 0.7252688407897949, 0.10839645564556122, 0.6706629395484924, -1.5747096538543701, -1.0408930778503418, -0.4093606770038605, 1.052704930305481, 0.39143046736717224, -2.2687084674835205, -1.0235552787780762, -0.522533118724823, -0.9980680346488953, -0.7399958372116089, 0.30827367305755615, -0.49986645579338074, 1.5580071210861206, -1.475872278213501, -1.4499834775924683, -1.6796141862869263, 0.35992178320884705, -0.07156190276145935, 0.8885384202003479, 0.8181633949279785, 0.7259188890457153, -1.0588552951812744, -0.13999836146831512, -2.324787139892578, 0.4099074602127075, 0.22657270729541779, -0.03933611512184143, -0.3311086893081665, 0.31234586238861084, -0.14491786062717438, 0.20482227206230164, -0.4105605185031891, -1.063605546951294, 1.0185381174087524, -0.4885980188846588, -0.41707363724708557, -0.7660820484161377, -0.39240771532058716, -1.3431222438812256, 0.6910009384155273, -0.2757083475589752, 2.098762035369873, -0.33343228697776794, -2.852001667022705, 1.5242371559143066, -1.7087839841842651, -1.6359703540802002, 1.292326807975769, -0.5019415020942688, -0.47048842906951904, -0.4265006482601166, -0.3409465253353119, 0.12447351962327957, 0.2536652088165283, 0.6213862299919128, 0.9521132111549377, 0.32306575775146484, 0.33707278966903687, -1.186456561088562, -0.5065203905105591, -0.5780823230743408, 1.208510160446167, -0.8507249355316162, -1.5716861486434937, -0.12123100459575653, 0.051572151482105255, 1.8548437356948853, 0.2679750919342041, 1.9508724212646484, 0.8704022169113159, 1.0048810243606567, -0.8545471429824829, -0.07566771656274796, 0.3374970257282257, 1.6166144609451294, 0.020996766164898872, 1.197433352470398, 0.05636851117014885, -0.1593925654888153, -1.4444841146469116, -1.4030567407608032, -2.0553414821624756, 3.634521961212158, 1.7493599653244019, -0.5719161033630371, -0.5866250991821289, 0.5062732696533203, -0.9447136521339417, 0.9731257557868958, -0.2668178379535675, 2.1843643188476562, -0.4736536741256714, -0.8367214202880859, 1.1915642023086548, 0.1453319638967514, -0.8562633991241455, -0.9035381078720093, -0.5253397822380066, -0.04502887278795242, 0.41903847455978394, -0.7446267008781433, -0.23864363133907318, 1.0907944440841675, 1.5496553182601929, 0.24101200699806213, -0.341675728559494, -0.7312409281730652, 0.5384076833724976, -0.3606599271297455, 1.6705639362335205, -0.6366685032844543, 1.3199470043182373, 1.9154624938964844, 0.9265245199203491, 0.2442755103111267, -0.7996565699577332, 1.4773107767105103, -0.3266555666923523, -0.23084688186645508, 0.7029897570610046, -0.26144951581954956, 0.9862760305404663, -0.23894862830638885, -0.4850632846355438, 0.72658371925354, -0.3030763566493988, -0.33742502331733704, -0.11619768291711807, 1.8925695419311523, -0.5961137413978577, 0.9367058873176575, 0.06080559268593788, 0.013803971000015736, -0.8657928109169006, 0.49027490615844727, -0.5909649729728699, -1.7183386087417603, -0.2869967818260193, -0.4515342116355896, 1.4616434574127197, -0.9534783363342285, 0.4580879211425781, -0.31350472569465637, -0.2997798025608063, 0.0018716823542490602, 1.4579198360443115, -0.5117847323417664, -0.23347841203212738, -0.5506601929664612, -1.9043899774551392, -1.2500709295272827, 0.3253440856933594, 1.8718801736831665, -0.15815657377243042, 0.5621760487556458, 0.5207651257514954, -0.6184403896331787, -2.220843553543091, -0.9944713711738586, -0.5667549967765808, -1.1361594200134277, 1.6215720176696777, -1.4737136363983154, 1.3131005764007568, -0.1291903704404831, -0.2140653282403946, -0.9203965067863464, 1.6652381420135498, 0.5487126111984253, 0.776078462600708, 0.2640479505062103, 1.3906973600387573, 0.18094389140605927, -1.0205191373825073, -0.9848859310150146, 0.4336593449115753, 1.8521355390548706, 0.14965081214904785, -1.3707191944122314, 0.2723471224308014, -0.44912612438201904, -1.3185570240020752, 1.0568056106567383, 1.0449910163879395, -1.5268540382385254, 1.7167285680770874, 0.5500350594520569, -1.5195223093032837, -0.842226505279541, 1.6743509769439697, 1.0233502388000488, -0.12485974282026291, -0.863021731376648, -2.6802592277526855, 0.8509407043457031, 1.466941475868225, -0.07982899248600006, 0.13712361454963684, 0.31393930315971375, -0.4539966583251953, -1.1145458221435547, -0.6158246397972107, -1.7235209941864014, -1.164435625076294, 1.1632068157196045, 0.285553902387619, 2.2158470153808594, -1.3531396389007568, -1.273658037185669, 0.8220307230949402, -2.5978662967681885, -0.5146810412406921, 1.1577658653259277, -0.375502347946167, 1.1592813730239868, 0.8272325396537781, 0.7506284117698669, -0.33681508898735046, -0.03969183936715126, -2.5739805698394775, -0.335042268037796, 1.2982512712478638, 0.708911657333374, 0.8403085470199585, -0.8281130790710449, 1.658732533454895, 0.5156517028808594, -1.1174949407577515, 0.35452449321746826, 1.0802074670791626, -1.416542649269104, -0.15784895420074463, -0.6627429127693176, -0.41513583064079285, -0.46057698130607605, -0.08460111916065216, -0.582223117351532, 1.612166166305542, 1.1836470365524292, -2.399662733078003, -0.4427226781845093, -0.8278494477272034, 2.1778149604797363, 0.8867887854576111, 0.5209991335868835, 0.9836509823799133, -0.04436885192990303, -1.2211925983428955, 0.20188142359256744, 0.7157285213470459, 0.7130178809165955, 0.5892999172210693, -0.5667497515678406, -1.0465903282165527, -1.1034554243087769, 1.3539507389068604, 1.4922891855239868, 0.38080814480781555, -0.036456938832998276, -2.0298874378204346, -0.9288419485092163, 1.7795718908309937, -0.379045695066452, 0.14318202435970306, -0.9836234450340271, 1.0484014749526978, 2.5881340503692627, 0.2608242332935333, -1.0333349704742432, 0.6503220796585083, -1.7020574808120728, 1.3563648462295532, 0.2113683521747589, -1.4436817169189453, -0.9592816233634949, -0.056210827082395554, -0.7232746481895447, 0.4656352996826172, -1.0614573955535889, -1.4573689699172974, -0.8077172636985779, -0.27809613943099976, -0.17143434286117554, 0.7664027214050293, 1.8897322416305542, 0.007383633404970169, -1.4431548118591309, 0.6394892930984497, 1.030920147895813, -2.412276268005371, -0.6893506050109863, -0.7543338537216187, 0.4405724108219147, -0.871846079826355, -2.7205536365509033, 0.41291242837905884, -0.2176128774881363, -0.2270883321762085, -0.19250670075416565, 0.3031370937824249, 1.928017020225525, 0.24978134036064148, -1.3616005182266235, -0.48692813515663147, 0.10724841803312302, 0.6601556539535522, 0.2121574431657791, 0.5202662944793701, 0.5747647881507874, -0.17424069344997406, 0.5608837008476257, 0.21616949141025543, 0.987210750579834, -0.3441481292247772, 0.7820969223976135, 0.5929514765739441, -0.5607060194015503, -1.5103034973144531, -0.10144016146659851, 0.15692266821861267, 0.2102234661579132, 0.45988667011260986, 0.07394512742757797, 0.6442137360572815, -0.0804404765367508, -1.0065304040908813, -1.8320667743682861, 1.4019869565963745, 0.1470945030450821, -0.536910355091095, -0.1692049205303192, -0.04783845320343971, 0.569166362285614, 1.8838456869125366, 0.3625626564025879, -0.45608216524124146, -1.008242130279541, 1.6703213453292847, -0.5037621259689331, 0.48270657658576965, -0.04675118625164032, -0.6481566429138184, 0.006560394540429115, -0.41539543867111206, -0.6226456761360168, -1.1080114841461182, 0.02309553138911724, 1.6591861248016357, 0.404651939868927, 1.8222121000289917, -1.892554521560669, -3.6610021591186523, 0.8605291843414307, -1.0097287893295288, 0.1973317414522171, 1.0002192258834839, -0.4794553518295288, 0.6731191277503967, 1.2153466939926147, -0.9306575059890747, -1.065333604812622, 0.8095091581344604, -0.1461859941482544, 0.1613723188638687, -0.13863018155097961, -0.44661766290664673, -0.8491207361221313, -0.360723078250885, 0.13181161880493164, 0.23532362282276154, 0.799894392490387, -1.2100766897201538, 0.30951279401779175, 1.7281889915466309, 0.36029472947120667, 1.3321118354797363, 0.682462215423584, -0.5510064363479614, 0.39753055572509766, 0.4469901919364929, 0.021390845999121666, 1.4410042762756348, -1.8079856634140015, 1.3102830648422241, -0.9625065922737122, -4.598666191101074, -1.0213454961776733, 0.7587115168571472, -3.0906736850738525, -0.059909917414188385, 1.3041434288024902, -0.32579970359802246, -0.6107059121131897, 0.4945172667503357, -0.28420761227607727, 0.42996564507484436, -0.3557915985584259, 0.2374238818883896, -0.8664041757583618, 0.6909321546554565, -0.6896083354949951, -1.992700219154358, -0.9912928342819214, 1.213986873626709, 0.16680403053760529, -0.5997515916824341, -1.1852723360061646, 0.0919821709394455, -0.33363187313079834, -2.168199062347412, -0.35176563262939453, -0.14006313681602478, -0.2184983789920807, -0.12018869072198868, -1.3839409351348877, 0.14918957650661469, 0.7732085585594177, -0.4377617835998535, -0.37759295105934143, -1.053863525390625, -0.18385758996009827, -1.0417964458465576, 2.4558513164520264, -1.0499776601791382, 0.5786271095275879, 1.4312751293182373, 0.8628330230712891, -0.6102303862571716, -0.830198347568512, -1.396782398223877, -0.6950221061706543, -0.4595915377140045, 1.1365457773208618, 0.44174128770828247, -0.311888724565506, -0.396649032831192, 0.040603987872600555, 0.3646741807460785, 0.7158342003822327, 0.29824721813201904, -0.0016242461279034615, 1.059463620185852, 0.3658434748649597, 0.4736732244491577, 0.6948378086090088, -0.2891102433204651, 0.8155685067176819, -1.8454045057296753, -1.5350757837295532, -1.57233726978302, 2.2134625911712646, 0.704025387763977, -1.214529037475586, -0.5550841689109802, -0.30880793929100037, -1.568847417831421, -0.06322009116411209, -0.31199386715888977, -1.1508129835128784, -1.8054555654525757, -1.2139254808425903, 0.6034376621246338, 0.9432971477508545, -0.9761945605278015, -1.490526556968689, -0.2703942060470581, -1.174001693725586, -1.433605670928955, -1.9347704648971558, 0.6704729795455933, -1.5255650281906128, -1.755900263786316, -0.31686240434646606, -0.7829933166503906, -0.8445948362350464, 0.6567805409431458, -0.7474249601364136, -0.3176269829273224, 1.6282579898834229, 1.0012716054916382, -0.6923941373825073, -0.2366340458393097, -1.0923280715942383, 1.394981026649475, -0.14466610550880432, -0.47888457775115967, -1.073574185371399, 0.13398239016532898, 0.44228440523147583, -0.21798129379749298, 1.3280125856399536, -1.4120558500289917, -1.044418215751648, 0.42547500133514404, 1.0665186643600464, -0.9275263547897339, 0.1450163871049881, 0.957226574420929, 0.06344905495643616, -0.5953462719917297, -0.22178249061107635, -1.090806007385254, 0.9718329906463623, 0.0845753401517868, -1.1573410034179688, -0.4244311451911926, 0.2923118472099304, 0.6913474798202515, 0.2739430069923401, 0.21930190920829773, 0.7828473448753357, 0.5342902541160583, 0.2651525139808655, 0.8457714319229126, 0.16556164622306824, -0.8129425644874573, 0.15283839404582977, 0.308492511510849, -1.0205899477005005, 0.27887341380119324, -1.1642452478408813, 0.5230237245559692, -0.06341548264026642, 0.4814895987510681, 0.37517955899238586, 0.1799234300851822, -2.5252091884613037, 0.07544754445552826, -0.29076218605041504, -1.2496694326400757, 0.008975004777312279, -0.7826648354530334, 0.5497918725013733, -0.9786728024482727, -0.17168550193309784, -1.7133392095565796, 0.34949633479118347, -0.1587621122598648, -0.3010556995868683, -0.9519636034965515, 0.3305278718471527, 0.7643860578536987, -0.1062297523021698, -0.3884395956993103, -0.47910192608833313, 0.42905616760253906, -0.03185907006263733, 0.4329012632369995, -0.7129424810409546, 0.2151670604944229, 1.9139446020126343, 0.8367985486984253, 2.0459649562835693, -0.024967743083834648, -1.1621148586273193, 1.1838761568069458, -0.6899986267089844, 1.1723765134811401, -0.5039446949958801, 0.2889772057533264, 0.26177075505256653, 1.2753766775131226, -0.2652559280395508, -0.6138553023338318, -1.4229496717453003, 0.05511678382754326, 1.3355374336242676, -0.0084761967882514, -1.35173761844635, 0.04774148017168045, -0.4823334813117981, 0.43551477789878845, -0.41938069462776184, -0.300523579120636, -2.2969048023223877, 0.8810016512870789, -2.511924982070923, 0.4720414876937866, 1.339178442955017, 0.3446117043495178, 0.331200510263443, -0.674135684967041, 0.07661835104227066, 0.7061376571655273, -0.5069082379341125, -0.33882641792297363, 0.5915716290473938, -0.5570428967475891, 1.0629373788833618, -0.7640408873558044, 0.015543265268206596, 0.5071596503257751, -0.34680652618408203, -0.28378215432167053, -1.7760440111160278, -0.1464638113975525, 1.634121060371399, 0.618545413017273, -1.51287841796875, 0.27731189131736755, -1.2057210206985474, -0.22951184213161469, -0.9361466765403748, 0.38242989778518677, 0.9848707914352417, -0.6725485920906067, 0.9389303922653198, 0.3186395764350891, -0.05995410308241844, 0.1318604052066803, 1.2082620859146118, 1.8599762916564941, -0.7640765309333801, 2.350769519805908, -0.4593043029308319, -0.4318195879459381, 1.0892858505249023, 1.3914895057678223, -2.0122416019439697, 2.7049615383148193, 0.5862863063812256, -0.11383183300495148, -0.9390043616294861, 0.04648208990693092, -0.5028765797615051, -0.7415640950202942, -0.7946413159370422, -0.3853878974914551, 0.4275975227355957, -0.34799349308013916, -0.3930436372756958, -0.9272568821907043, -0.34479132294654846, -0.8499575257301331, -0.036001745611429214, 1.0454233884811401, 0.6854406595230103, -0.6717641353607178, 1.202876329421997, -0.24205198884010315, 0.3321932852268219, 1.40848708152771, -0.7970633506774902, 0.0768776386976242, -0.4654896855354309, -0.0897536352276802, -0.43170544505119324, -0.8297355771064758, 0.521943986415863, -2.914647340774536, 0.1444246917963028, 1.8192768096923828, 0.8498771786689758, -0.4999026656150818, 0.2720462381839752, 0.3476965129375458, -0.6856631636619568, 0.6045880317687988, 1.1679236888885498, -0.7240183353424072, 0.6572932600975037, -1.6163510084152222, -0.26952552795410156, -0.006275450345128775, -2.417617082595825, 1.1244028806686401, -0.7780059576034546, -0.48256170749664307, 1.085983395576477, -0.5643057227134705, -2.6019251346588135, -1.0779869556427002, 0.4114849865436554, 0.22340157628059387, -0.7167366147041321, 1.3910678625106812, -0.19025909900665283, -0.07777535170316696, 1.2977440357208252, 0.14275473356246948, 0.49763330817222595, -1.0405769348144531, -0.928264319896698, -0.9005389213562012, -0.31945130228996277, 0.44281309843063354, 2.748868465423584, -0.9701833724975586, -0.8436281085014343, -0.70096355676651, -0.25802701711654663, -0.16288259625434875, -0.3676351308822632, -0.9409403800964355, 0.27025455236434937, -0.40722447633743286, -0.27460557222366333, -1.8153612613677979, -0.70162034034729, 0.3316948711872101, -0.5623865723609924, 0.35287046432495117, -0.013864491134881973, 0.5755730867385864, 0.29807978868484497, -1.3326361179351807, 0.13984067738056183, -0.7025431990623474, 1.2677011489868164, 0.20744547247886658, -1.3950002193450928, -0.7692956924438477, -0.6611445546150208, 1.4946810007095337, 0.08306062966585159, -0.2132752239704132, -0.9487835168838501, 0.22949109971523285, -0.39944323897361755, 1.2576045989990234, 0.5600330233573914, -1.9926190376281738, -0.2747935950756073, -0.6624923944473267, 0.0464228056371212, 0.8002655506134033, -0.33602532744407654, 0.15150289237499237, 0.12468619644641876, -1.1694166660308838, 0.7532597780227661, 0.912617027759552, 1.004002571105957, -0.6776746511459351, -1.0387675762176514, 1.6044386625289917, 0.010821353644132614, -0.45369383692741394, -1.1428194046020508, -0.32304877042770386, 1.0378491878509521, -1.220840334892273, -1.0758048295974731, 0.2851286232471466, -1.2195631265640259, -0.03361544758081436, 0.06488344073295593, 0.5114688277244568, -1.4073528051376343, 2.288396120071411, -0.806768000125885, 0.11211777478456497, 1.1485216617584229, -0.3780294954776764, -0.7433260083198547, 0.9116134643554688, 0.15356262028217316, 1.931091547012329, 0.005330080632120371, 0.4989026188850403, 0.09768124669790268, -0.662040114402771, 0.2049606889486313, 0.14600065350532532, 2.1703250408172607, -0.806935727596283, 1.3713263273239136, 2.2672231197357178, -0.35459333658218384, 0.5002750754356384, -0.3180426061153412, 0.15626855194568634, -2.557250499725342, 0.22736692428588867, 0.8877067565917969, -1.3943144083023071, -1.4326376914978027, 1.1295796632766724, -0.5899564027786255, 0.34103116393089294, -1.945338487625122, 0.3937099575996399, 1.3852137327194214, 0.6647937297821045, 0.5856081247329712, -0.23235532641410828, -1.0952945947647095, 0.7248914241790771, -0.2667142450809479, 0.5726855397224426, -1.6528184413909912, 0.9081425666809082, -1.440090537071228, 0.020706184208393097, -0.7003697752952576, 0.5677334666252136, -0.26690083742141724, 0.1244882345199585, 0.42176926136016846, -0.2543256878852844, -0.7726377248764038, -0.35764631628990173, -0.02097497135400772, 0.22099998593330383, -1.2249174118041992, -1.5909863710403442, -0.008499142713844776, 0.3769401013851166, 0.7038093209266663, 0.006518181879073381, -0.6420595645904541, -0.28853750228881836, -0.7638885974884033, -1.1795177459716797, -0.633733868598938, -0.1234084740281105, 0.22432841360569, 1.5787016153335571, -1.1057066917419434, -0.46423736214637756, -0.6545534729957581, 2.0617847442626953, -0.49666839838027954, -0.7033034563064575, -0.7935632467269897, -1.3724884986877441, 0.12291834503412247, -1.8707811832427979, -0.2580185532569885, -0.9201852083206177, -0.550882875919342, 0.2161729782819748, -0.7708476781845093, 1.013101577758789, -1.0278757810592651, 0.04182838648557663, 0.17571385204792023, 0.2778572142124176, -0.03173764795064926, 1.2228186130523682, 0.0647934302687645, -1.9031988382339478, -1.4141227006912231, -0.38952988386154175, 2.290677309036255, 2.221151828765869, -1.2298996448516846, 0.3068309426307678, -1.4938337802886963, -0.9048708081245422, 1.7717597484588623, 0.9227891564369202, -0.6417421698570251, 0.003578099189326167, 0.4474521279335022, -0.08379300683736801, -1.8268287181854248, -1.341071605682373, 0.5037260055541992, 0.6055041551589966, 0.12064079195261002, -0.8697184920310974, 1.5448036193847656, 1.4375362396240234, -0.8660937547683716, -0.3695564270019531, -0.12519174814224243, -0.5447174906730652, 0.2756989598274231, -0.33727172017097473, -1.2660932540893555, -1.1917593479156494, -0.6232476830482483, 1.0591989755630493, -0.8708344101905823, -0.2290227711200714, 0.1672246754169464, -0.020840728655457497, -1.1454758644104004, 0.2323717325925827, -0.08777488023042679, 1.1950935125350952, 1.1044995784759521, -0.24192604422569275, 0.07382127642631531, -0.2866124212741852, 0.6552284359931946, 0.15631107985973358, 0.8406168222427368, -0.8389080762863159, -0.4680022895336151, 0.9814416766166687, 0.26719534397125244, 1.36506986618042, -0.8214091658592224, 0.5525626540184021, 0.702519416809082, -1.7843549251556396, -1.3440817594528198, -0.5326922535896301, -0.46362265944480896, -1.1922892332077026, -0.16991442441940308, -1.5367518663406372, 1.2327768802642822, -0.521526038646698, 0.6683172583580017, 0.7748464345932007, 0.1667608916759491, 1.123671531677246, -0.15517215430736542, 1.4484457969665527, -1.8266887664794922, 2.513439893722534, 1.2710564136505127, 1.1373056173324585, 0.5084694027900696, 0.8306035995483398, -0.4537840485572815, 0.21921315789222717, -1.347198724746704, 0.7213590145111084, 0.491915225982666, -0.644544780254364, -0.623896062374115, -0.20015379786491394, -0.11883620917797089, 0.12562739849090576, -1.0737375020980835, 0.8188255429267883, 1.0314234495162964, 1.0992320775985718, -0.45941323041915894, -0.35665157437324524, 0.4402833580970764, 0.16701309382915497, 1.0947699546813965, 0.12862788140773773, -0.3230486810207367, -0.4702833294868469, 0.11365329474210739, -1.0158584117889404, -0.8529396653175354, -0.8006249070167542, 0.8581766486167908, -0.34272846579551697, -1.0342024564743042, 2.3311753273010254, -1.6794941425323486, 0.8171043992042542, 1.0480053424835205, -0.17007745802402496, 2.249824285507202, 0.9059614539146423, -1.4663580656051636, -0.41872021555900574, -0.09338083863258362, 0.8097059726715088, 1.0308133363723755, 1.4128144979476929, 0.48541611433029175, -1.2700402736663818, -0.06921514868736267, 2.980795383453369, -2.0666940212249756, -1.0558714866638184, -2.116448163986206, -0.03574220836162567, -1.4725102186203003, 0.8732068538665771, 0.24648058414459229, -0.21699152886867523, 0.4968964755535126, -0.05844101309776306, -2.74672532081604, 1.1186445951461792, 0.3392537236213684, -0.7467344999313354, -0.8588111996650696, 1.330101728439331, -0.08776586502790451, 1.3667755126953125, 0.8401714563369751, 0.34692543745040894, -0.7480343580245972, -0.07704733312129974, -0.39499831199645996, -0.18716038763523102, -0.6502753496170044, 0.999816358089447, 1.6883094310760498, 0.39997148513793945, 1.5266941785812378, -1.0693303346633911, 0.13946405053138733, -0.6310951709747314, 1.9814517498016357, -0.016249580308794975, -0.11232684552669525, -1.5278440713882446, 0.1907074749469757, -1.0670769214630127, -0.013748488388955593, 0.2216029316186905, 0.2372078150510788, -1.5163218975067139, 3.074852466583252, 0.2336425632238388, -0.19603528082370758, 0.5003117918968201, -0.008450211957097054, -0.8382675051689148, 2.7248432636260986, -0.42409250140190125, 1.2499967813491821, -1.4467554092407227, -0.43432509899139404, 0.13115720450878143, 0.5453499555587769, 0.7433138489723206, 0.5255808234214783, -0.5066070556640625, -1.1868382692337036, 0.31685954332351685, 0.27933624386787415, -0.413074254989624, -0.3645993173122406, -1.4269070625305176, 0.363951712846756, -0.7515851855278015, -1.3395423889160156, 0.5231143832206726, 0.06789565086364746, -0.37895259261131287, 0.2597631812095642, 0.962320864200592, -0.2505105435848236, -0.2354871779680252, 0.5471649169921875, -0.020480819046497345, -0.501065731048584, 1.351950764656067, 0.4844764173030853, 0.049555547535419464, -0.24064573645591736, 1.8641337156295776, -0.5994781851768494, 2.206284523010254, 1.146605372428894, 1.1176143884658813, -1.2338614463806152, -0.34059178829193115, -1.243123173713684, -0.9237463474273682, -0.22530587017536163, 0.24293971061706543, 0.7034789323806763, -0.8807909488677979, -1.3588210344314575, 1.4340476989746094, -0.04072254151105881, 0.24854178726673126, -0.14305835962295532, -0.017193490639328957, -0.30883070826530457, 1.1860220432281494, -0.1801750361919403, -0.7872465252876282, -0.5821248888969421, 1.0527452230453491, -1.8716425895690918, 1.4856619834899902, -1.1151618957519531, 1.3947449922561646, 1.4368833303451538, -0.8144821524620056, 2.2663896083831787, -0.2910258173942566, -0.12002953141927719, 0.3487730622291565, 0.4327358603477478, -0.4086223840713501, 1.0963811874389648, -0.2538594901561737, 1.1792364120483398, 3.4262571334838867, 0.6490810513496399, -0.19038605690002441, 1.2955024242401123, -2.003466844558716, -0.5118945837020874, 0.06868978589773178, -1.13230562210083, 1.730002760887146, -1.793725609779358, -1.4982819557189941, -1.831783413887024, -0.17636610567569733, 0.16270670294761658, 0.20441928505897522, -0.3674830198287964, -0.8200730681419373, 0.5909976363182068, 1.129768967628479, -0.4716935157775879, 0.8728640079498291, 1.0422024726867676, 0.7123662233352661, 0.21247132122516632, -0.22719182074069977, 0.5054605603218079, 1.023471474647522, -0.8758044242858887, -0.8436759114265442, 0.2457249015569687, -0.7384061813354492, -0.028355438262224197, 1.8965203762054443, 0.13789457082748413, -0.5447661876678467, -0.29387393593788147, 0.16943268477916718, -0.011362900026142597, 0.321475088596344, 1.2925586700439453, -0.4694080054759979, -1.018696904182434, 0.2598218619823456, 1.5513933897018433, -2.2865958213806152, 0.6100538969039917, -0.42057308554649353, -1.8924471139907837, 0.0045250109396874905, 0.5679022669792175, -0.900684118270874, -0.9133682250976562, 0.19924677908420563, -1.217900276184082, -0.6055959463119507, 0.7049638032913208, -0.4353558421134949, -0.3320731222629547, 0.1377956122159958, -2.2046215534210205, -0.9584337472915649, 0.012625297531485558, -0.7232679724693298, -0.6143227815628052, -0.4397749602794647, -0.5640150904655457, 2.859001636505127, -1.4718022346496582, -1.8675928115844727, -2.566089391708374, -0.27626174688339233, 0.6209820508956909, -0.8697860836982727, 1.0637434720993042, -0.4798625707626343, -0.5966175198554993, -0.879112958908081, -0.013747898861765862, -0.07321786880493164, 0.7410784363746643, -0.4885837733745575, 1.2857635021209717, 0.6394650340080261, 0.14254775643348694, -0.07297016680240631, -0.11227177828550339, 2.5365986824035645, 1.2224223613739014, -0.46520617604255676, -1.5800518989562988, -0.680954098701477, -0.5476484298706055, -0.9906507730484009, -0.4123895764350891, 0.34242939949035645, 0.8644256591796875, 0.5278596878051758, 0.4040001928806305, 0.34029966592788696, -0.6118547320365906, 0.2885178029537201, 0.13369815051555634, 2.3640923500061035, -1.4330154657363892, -0.7148380875587463, -0.048534050583839417, 0.16582298278808594, -0.06275219470262527, -1.34918212890625, 0.11654242128133774, 0.439752995967865, 1.979108452796936, -1.2199583053588867, 0.5945283770561218, -1.721606969833374, 0.5211154222488403, 0.4981198012828827, -1.0924580097198486, -1.6907163858413696, 0.9378798604011536, -1.9048889875411987, -0.19921721518039703, -1.0363441705703735, 1.092194676399231, 1.029552698135376, -0.2523626983165741, -0.2573983371257782, 0.8154056072235107, 1.4649149179458618, 0.7594128847122192, 0.25543397665023804, -1.9719730615615845, 0.5638391971588135, -1.1047165393829346, 1.0561466217041016, -0.011150198988616467, -0.9041858315467834, -0.4967028498649597, 1.5422660112380981, -0.5208092927932739, 1.0612999200820923, 0.4784705340862274, -1.9023851156234741, 0.24954959750175476, -0.7614595293998718, 0.3187750279903412, 0.9041125178337097, -0.5062941908836365, 0.6875689625740051, -0.001512703369371593, -0.9103131294250488, 0.23410165309906006, -2.14491605758667, -0.11787965893745422, -1.9904121160507202, -0.2623711824417114, 1.1124516725540161, -0.1860884428024292, -1.0278325080871582, -0.49935436248779297, 1.4180525541305542, 0.0793951079249382, 0.5491331219673157, 0.4237613081932068, 0.20871300995349884, 1.890120506286621, 1.0706099271774292, 0.7473176121711731, 0.7607035636901855, -1.012893795967102, 1.230733036994934, 1.7576709985733032, 0.1839400827884674, 0.9619238376617432, 1.1372836828231812, -0.907672643661499, 0.12055917084217072, -0.567684531211853, 1.7981541156768799, -1.3346762657165527, 0.36128273606300354, 0.21345259249210358, -1.3236355781555176, -1.8031233549118042, 0.1272386759519577, -0.6077695488929749, 0.8654484748840332, 0.4130662679672241, -0.14814384281635284, -1.4781523942947388, 1.3986756801605225, 0.3822358250617981, 0.15256930887699127, 0.8530008792877197, -0.8369429111480713, -1.4254306554794312, 0.2594415247440338, -0.9662483930587769, 0.5745334029197693, -0.5289570689201355, 0.7147156000137329, -1.6031162738800049, -1.1370166540145874, -0.7166813611984253, -0.821793258190155, 0.606839120388031, 1.2922286987304688, 0.48405322432518005, -1.3524088859558105, 0.10615140944719315, -1.1666332483291626, -0.9826793074607849, -0.4998546838760376, 0.30958741903305054, 0.24865859746932983, 1.7566031217575073, 0.6404687166213989, -0.4608787000179291, 1.0996534824371338, 0.4288197159767151, -0.8875879645347595, 1.1173585653305054, 0.5180737376213074, -0.41335493326187134, -0.09910408407449722, 1.642268419265747, 0.09306074678897858, 1.0273157358169556, 0.8282613754272461, 0.535460352897644, -1.7926198244094849, 0.7174176573753357, 1.5499861240386963, -1.1453663110733032, 0.16357946395874023, -0.133935809135437, 0.6674094200134277, 0.5308176875114441, -1.61698317527771, 0.7977994680404663, -2.011899709701538, 0.9691998362541199, 1.815293312072754, -1.4862287044525146, -0.15215030312538147, -1.6391412019729614, 0.8229176998138428, -0.5905281901359558, 0.13045912981033325, 1.815140724182129, 0.8821167945861816, -0.0013145774137228727, 0.31825658679008484, 0.4050033986568451, 0.7136316895484924, -0.6512830853462219, -1.785631537437439, 0.14131061732769012, 2.221858024597168, -0.043087735772132874, -0.24013902246952057, -0.9878873229026794, -0.10206560790538788, 0.3199711740016937, -0.018036101013422012, -0.22474125027656555, -0.17233584821224213, -0.47913336753845215, 0.2653788924217224, -1.7952853441238403, 0.43755629658699036, -1.5559173822402954, 0.07046569883823395, 0.6479395627975464, -1.1279480457305908, -0.17082847654819489, -0.3343343138694763, 0.22239619493484497, 0.9073435068130493, 0.5072088837623596, 1.7866888046264648, 0.479164719581604, -0.6099860072135925, 0.8900035619735718, 1.1921919584274292, 0.019180605188012123, 1.0421884059906006, -0.3875049650669098, -0.24665294587612152, 0.6574724316596985, -0.40259549021720886, 0.9091134667396545, -0.14371326565742493, 0.1025739386677742, -0.6834922432899475, -1.0881519317626953, -0.8567678332328796, -0.2525816559791565, 0.9201692342758179, -1.011362075805664, -1.1284806728363037, -2.063929796218872, -0.08558597415685654, 1.4584641456604004, -0.47180449962615967, -0.7074305415153503, -1.0225077867507935, -0.7940316200256348, 0.1480957418680191, -1.3760960102081299, 0.2813257575035095, 0.5789597034454346, 2.378560781478882, -0.7099911570549011, -1.6818186044692993, 0.9147871136665344, 0.0023504835553467274, -0.34804829955101013, -0.2414775788784027, 0.22503265738487244, 0.12754300236701965, 0.8665270209312439, -0.42838090658187866, -0.15271233022212982, 0.986176073551178, 0.5331951975822449, -1.4462755918502808, 0.24744844436645508, 1.4829645156860352, -1.116655945777893, -0.9961509704589844, -0.8674880266189575, 0.1815505176782608, -0.8024411201477051, -0.4609079360961914, -0.5665805339813232, 0.9162946343421936, 0.10194088518619537, -0.15287892520427704, -0.9241216778755188, 0.275549054145813, -1.4598649740219116, -2.88643479347229, -0.14860717952251434, -0.08598301559686661, 0.8723462820053101, -0.3017841875553131, -0.930774450302124, -0.5693789124488831, -0.4388333559036255, 0.14206573367118835, 0.18154864013195038, -1.0102466344833374, 2.045395612716675, -0.748855710029602, 0.3006483018398285, -0.5383829474449158, -2.330043315887451, 0.19662074744701385, -0.43052709102630615, -0.23470038175582886, 1.8175982236862183, 0.5459185242652893, 0.8046362996101379, 1.0995705127716064, 0.39549607038497925, -0.8426755666732788, 3.4357998371124268, 0.757008969783783, -2.148391008377075, -0.3687276840209961, 0.04426334798336029, -3.45103120803833, 0.7780188918113708, -0.5068977475166321, -1.3243578672409058, -0.767135739326477, -0.9396801590919495, 0.3416120409965515, -0.5951454639434814, 0.518369197845459, 1.7223644256591797, 0.7906590104103088, 0.41710466146469116, 0.4716424345970154, 0.7202439904212952, 0.3386678695678711, -0.6166828870773315, -0.4149455428123474, 0.15535521507263184, 1.5306445360183716, 0.7247570753097534, 0.3095126152038574, 0.6575478911399841, -0.6070581674575806, 0.26054707169532776, -1.6338545083999634, -0.09775285422801971, -0.4470033645629883, 0.5227292776107788, 0.12034410238265991, 1.6085917949676514, -0.9549601674079895, 0.48866623640060425, -0.9589986801147461, 0.7147934436798096, -0.9154250025749207, -1.6935234069824219, -0.647850751876831, -0.001444576308131218, 0.9006539583206177, -1.393542766571045, -1.1307034492492676, -0.6071545481681824, -0.050590068101882935, -0.17152541875839233, 1.0130321979522705, 0.9831655025482178, 1.1813921928405762, 0.5430881977081299, 0.8634909987449646, -1.8355742692947388, 1.44379460811615, -1.1107237339019775, 0.5565292239189148, -1.9417341947555542, -0.370013564825058, 1.880079984664917, 0.8406714797019958, 0.2170027643442154, -1.9032591581344604, -1.112685203552246, -0.18256686627864838, 0.5614945888519287, 0.2525584399700165, -0.800624668598175, 0.5865594148635864, 1.8058292865753174, 1.1338587999343872, -0.30491212010383606, -1.4376956224441528, 1.4229153394699097, -0.2101169377565384, -0.8255647420883179, -0.6306529641151428, -1.0333576202392578, 0.39184820652008057, 0.25541627407073975, -0.0613035224378109, -0.8620127439498901, 0.2668580412864685, -0.788031816482544, -0.29239559173583984, -0.7778818011283875, -0.7258706092834473, 0.257589191198349, -1.6411861181259155, 1.7011696100234985, 1.8952562808990479, 0.5797985196113586, 1.6051162481307983, 2.061605930328369, 0.2092779278755188, 1.27656888961792, 1.3375662565231323, -0.044169697910547256, -1.027785062789917, -1.103368878364563, 1.1574491262435913, -1.2749412059783936, 0.08321437239646912, -0.06213311851024628, -1.1336050033569336, -0.9225677847862244, -1.0353401899337769, 1.5295480489730835, -0.4066164791584015, 0.27414843440055847, 0.30767232179641724, 0.07627499848604202, 0.7705844640731812, -0.8340686559677124, 0.2881498336791992, 0.6753426194190979, 0.8724090456962585, -0.9791238903999329, -0.015878545120358467, -0.4867705702781677, -0.4896395206451416, -0.1916572004556656, -1.1506024599075317, 0.06624376773834229, 1.1265531778335571, 0.40724849700927734, -0.8304917216300964, 0.1723938286304474, 0.8364977240562439, -1.6033464670181274, -1.558205485343933, -0.3018973767757416, -0.924127995967865, 1.716850996017456, 0.04622415825724602, -0.03482941910624504, -0.20091073215007782, -0.655393660068512, -0.24703049659729004, 1.7251455783843994, -0.352769136428833, 1.2713104486465454, 0.22159302234649658, -0.18235601484775543, -0.3282822072505951, -1.71857488155365, -2.2413511276245117, 1.9837875366210938, 0.7074536085128784, 2.1968324184417725, 0.3433990776538849, 1.0175399780273438, 0.061344679445028305, 1.9141179323196411, -0.8813291788101196, 1.586909294128418, -2.0383920669555664, 0.1279904991388321, -0.11504583060741425, -0.4932818114757538, -1.5012681484222412, -0.7295899391174316, 0.24647505581378937, 1.7613165378570557, -0.1006365641951561, 0.9310559630393982, -1.70261812210083, -0.04859880357980728, -0.0008290962432511151, 1.1798797845840454, -0.056755151599645615, 0.940754234790802, 1.3169611692428589, 0.3656204044818878, 0.7803671956062317, -0.5319333672523499, -0.752718985080719, 1.7447891235351562, -1.1540796756744385, -0.15274040400981903, 1.1162049770355225, 0.49836012721061707, -1.2149713039398193, 0.1236552819609642, 0.5440442562103271, 0.11648676544427872, 0.00920561421662569, 1.0034836530685425, 0.3748728930950165, 0.015710581094026566, 0.5570496916770935, -2.063342571258545, -0.6365491151809692, 1.2119210958480835, -0.23334901034832, 0.527262270450592, 0.7977021932601929, -2.380267858505249, -1.3541353940963745, 0.1585952490568161, 0.3690493106842041, -0.5523478984832764, 1.8534533977508545, 0.6429600715637207, 0.15274474024772644, -0.7072947025299072, -2.957628011703491, -0.4147816002368927, -0.9165685772895813, -1.0325669050216675, -0.7021565437316895, 1.1410447359085083, -0.985413670539856, -0.30919578671455383, -0.32100623846054077, -1.109580397605896, -0.5624433755874634, -0.013685224577784538, -0.07116059958934784, -1.6251180171966553, 1.839376449584961, 1.7480655908584595, -0.09536890685558319, -0.09014086425304413, -0.7800549268722534, -1.188462495803833, 0.4381263554096222, -0.6959388852119446, -0.506367564201355, -0.04936535283923149, 0.007931079715490341, 0.2850368916988373, -0.08584149181842804, 1.8459359407424927, -1.2751493453979492, -0.8170285820960999, 1.092698574066162, -0.2427978217601776, 0.3014969229698181, -0.04951712116599083, -0.5329967141151428, -0.554173469543457, -1.12369704246521, -0.11217673122882843, 1.8162791728973389, 1.185560941696167, 2.1862306594848633, 0.8630320429801941, 0.05262521281838417, 0.8440412878990173, 0.10034900903701782, -1.0132309198379517, -0.8362410664558411, 1.2828381061553955, -0.4490987956523895, -1.3286975622177124, -0.2730100452899933, 1.0740147829055786, 0.9572066068649292, 0.15829065442085266, -1.3927457332611084, 0.006659106817096472, -0.4696444869041443, 2.5093915462493896, -0.15938960015773773, -0.027371317148208618, -0.9899601340293884, -1.631800651550293, -0.04669266939163208, 0.11724687367677689, 1.5715357065200806, -0.40551966428756714, -0.11963798850774765, 1.648408055305481, 0.6342961192131042, 0.7400029301643372, 0.7489358186721802, -2.0438132286071777, 0.47134771943092346, 2.4474635124206543, -0.5562734603881836, -0.9274534583091736, 1.1099493503570557, 1.5850557088851929, 0.23512126505374908, -2.137730598449707, -1.2320107221603394, -0.9271930456161499, -0.7852389216423035, -1.0654081106185913, -0.2933028042316437, -0.5540568232536316, -0.13044655323028564, 0.12412521243095398, -0.09523341059684753, -0.8485729694366455, 0.5043599605560303, -0.8639698028564453, -0.356219083070755, -0.42706361413002014, 0.700061559677124, -0.14472121000289917, 0.250307559967041, -1.2082467079162598, 0.5511112213134766, 0.005277118179947138, -0.13992759585380554, 0.9770381450653076, -0.9001479744911194, 0.49505406618118286, -1.8551890850067139, -0.9201778173446655, 1.1734246015548706, -0.12424413114786148, -1.3596028089523315, 1.219287395477295, 0.19818781316280365, 0.2782093584537506, -1.5368468761444092, 0.7300000786781311, 0.5313047766685486, -0.3452538251876831, -0.05610661581158638, -0.7559771537780762, 0.05005413293838501, -1.2895927429199219, 0.4549095034599304, 0.19440419971942902, -1.0139573812484741, -0.01804293505847454, -0.8601729869842529, -0.1249619573354721, -1.37147057056427, -0.4062013328075409, 1.7616902589797974, 0.5712494254112244, 1.1800345182418823, 1.8775410652160645, -1.0028141736984253, -0.00773283326998353, -0.8435264229774475, 1.5921703577041626, 1.2814375162124634, 1.66997230052948, -1.3104660511016846, 1.2271956205368042, 0.011740673333406448, -0.6956509947776794, 0.26819276809692383, -1.423231601715088, 0.6559585928916931, -1.1232178211212158, -0.6195013523101807, 0.6190264821052551, 0.7916384339332581, 0.7523564100265503, 0.07288716733455658, -0.3277736306190491, 0.13391681015491486, -0.29349932074546814, 0.8868511915206909, 0.3832024931907654, 0.024947961792349815, 0.7387635707855225, -1.1461501121520996, -0.9213330745697021, 0.9609839916229248, 0.5229353904724121, 0.016161290928721428, 0.4116140604019165, -0.27832460403442383, -0.5764358043670654, 0.2532782256603241, 0.9393364191055298, 1.1576632261276245, 0.4000144302845001, -0.040205977857112885, -1.2386780977249146, 0.23580507934093475, -0.42341017723083496, 0.741311252117157, -0.7905306816101074, -0.2965445816516876, -1.035475730895996, 0.8521978855133057, 0.5759726762771606, -1.78643798828125, -0.7699057459831238, 0.02853572927415371, 1.267845869064331, 1.8736683130264282, 1.0913711786270142, -0.1988307237625122, 0.5178206562995911, -0.7835773229598999, 1.5171574354171753, -1.2267189025878906, -1.70357346534729, 0.3290702998638153, 0.9926040172576904, 1.2819963693618774, 0.44277942180633545, 0.6307216286659241, -0.4175269901752472, -0.39493173360824585, -0.9494919180870056, 0.2673049867153168, -1.0777723789215088, -0.8719984889030457, -1.1232138872146606, 0.42572328448295593, 0.3838476538658142, 0.13343702256679535, 0.9584464430809021, -1.5831270217895508, 0.08584224432706833, 1.1879527568817139, 1.1926095485687256, -0.9888437390327454, 0.2884124517440796, -0.4735294580459595, -0.19651401042938232, -0.6415870189666748, -0.7599951028823853, 0.33718493580818176, 1.63100004196167, -0.3859546482563019, -0.10240954905748367, -1.6342560052871704, -0.12206873297691345, 0.0038296696729958057, -0.12055224180221558, 0.7574641704559326, 1.2519809007644653, -0.9704483151435852, -0.8800711631774902, -0.9820786118507385, -0.42949193716049194, -0.965724527835846, 0.2248358279466629, -1.396179437637329, -1.4891619682312012, -1.374584436416626, -1.2001688480377197, 0.2414470911026001, 0.40675440430641174, 0.24202458560466766, 0.3425527513027191, -0.004044450353831053, 1.1195875406265259, -0.28584712743759155, 0.493476003408432, 1.6182101964950562, -0.8671779036521912, -0.8557332158088684, -0.16352057456970215, 1.7997987270355225, 2.1132121086120605, -0.596428632736206, 1.8534488677978516, -0.23245546221733093, -1.0913417339324951, 0.2854207158088684, -0.15071599185466766, -0.19410265982151031, -1.2269301414489746, -0.907383918762207, 0.29776960611343384, 0.5976645946502686, -0.3477056324481964, -1.2631038427352905, -0.11291193962097168, -0.044763870537281036, -0.9952284097671509, 0.2001473605632782, -0.35671061277389526, 0.8840718269348145, -0.6856219172477722, 0.6481201648712158, 1.1130353212356567, -1.2920947074890137, -0.4330480098724365, -1.4305731058120728, -0.13710784912109375, -0.16786593198776245, -0.937508225440979, 0.7899238467216492, 0.15910285711288452, 0.0041404003277421, -0.3326328694820404, -1.1023781299591064, 1.1358966827392578, -0.44636374711990356, 0.7151446342468262, -2.7978568077087402, -0.10619669407606125, -2.041191577911377, -0.5889464020729065, 0.8310307264328003, -0.041476089507341385, -1.1337932348251343, 0.9829028248786926, -1.1868091821670532, -1.5155726671218872, -0.15396231412887573, -1.6882388591766357, 2.8029584884643555, 1.5418862104415894, -0.893053412437439, 1.3903615474700928, 0.5727935433387756, -1.3558279275894165, 1.0296430587768555, -0.21508419513702393, 0.30726566910743713, 0.5212951898574829, 0.6216266751289368, -1.028110384941101, 1.017354130744934, -0.2007054090499878, 2.185250759124756, -0.440525621175766, -1.2065513134002686, 0.20279811322689056, -1.787602424621582, -2.429701566696167, -0.10450182110071182, 0.8142560124397278, 0.3820565640926361, -0.8023179173469543, -0.4573417901992798, -1.0948816537857056, -0.3185490667819977, -0.43767115473747253, 0.7883719801902771, -0.321959912776947, 1.89803946018219, -0.8556769490242004, 0.3885011374950409, 0.70124751329422, 0.4148043990135193, 0.8048850297927856, 2.3818328380584717, 1.054271936416626, 0.838691234588623, 1.0368435382843018, -0.8205195665359497, -1.3745412826538086, -0.4704127311706543, -1.1020989418029785, 0.5859166383743286, -2.450282335281372, 0.21353250741958618, 0.5250588059425354, -0.8913817405700684, 0.9269710779190063, 0.002718805568292737, 0.531932532787323, -0.6201717853546143, 0.7627265453338623, 2.125822067260742, 0.22925116121768951, 0.4809817373752594, 0.8614042401313782, 0.3783724308013916, -1.0265663862228394, 0.5877135992050171, -2.0792202949523926, 0.5725321769714355, -0.8693835139274597, 0.6706176400184631, 1.4128254652023315, -1.1440746784210205, -0.05093056336045265, -0.16701044142246246, -0.3708173930644989, -0.3513759672641754, -1.2796508073806763, 1.6541632413864136, -0.5404185056686401, -1.5922876596450806, 0.7183670401573181, -0.9408547282218933, -0.19384516775608063, -0.00231853686273098, -1.6100788116455078, 0.5273700952529907, 0.39284414052963257, 1.2148642539978027, -1.556410789489746, -0.00869310088455677, 0.27204927802085876, 0.07211828231811523, 1.15620756149292, 0.03730563446879387, -0.19503933191299438, 0.0012989856768399477, -0.24554382264614105, 0.44262316823005676, -0.7395826578140259, -0.3984018564224243, 0.317837119102478, 1.0886350870132446, 0.1508847177028656, 1.0612752437591553, -0.6566133499145508, -0.22868455946445465, 0.8041995167732239, -0.6312975883483887, 1.3188743591308594, -2.4480812549591064, 0.6633622050285339, -0.7068333029747009, 0.19303612411022186, 0.000982048106379807, 0.6201435327529907, -1.5264198780059814, -0.38314345479011536, 1.2814897298812866, 0.43158164620399475, -0.4875645339488983, 0.508128821849823, -0.5568087697029114, -2.359614610671997, -0.7024033069610596, 0.200429305434227, -1.5883030891418457, -1.3003041744232178, 0.2817057967185974, 1.305383324623108, 0.07125097513198853, -1.2840825319290161, 0.22242119908332825, 0.005843899678438902, -0.19334067404270172, 0.013298949226737022, -0.9092909097671509, -0.9142038822174072, -1.2182750701904297, 1.711769938468933, 1.08684241771698, -1.21898353099823, -0.6641855835914612, -1.006978988647461, 0.682327151298523, 0.6456800699234009, 1.3404074907302856, 0.2882330119609833, -0.48059138655662537, 0.21192598342895508, 0.5343666672706604, -0.42689526081085205, -0.7006077766418457, -1.0571657419204712, 1.1356889009475708, -0.030808884650468826, 1.5639244318008423, 0.9230182766914368, 0.14400216937065125, 0.02092822454869747, 0.2076263129711151, -0.90926593542099, -0.8148735165596008, 0.27847447991371155, -1.2372533082962036, 1.618643045425415, 1.2572723627090454, -0.44319799542427063, -0.09350185096263885, 0.313342422246933, -0.7813791036605835, -0.7499611973762512, -0.26586008071899414, 1.5261144638061523, 0.17935121059417725, 0.9934622645378113, -2.2392044067382812, 0.741489052772522, 0.4848971962928772, 0.25713297724723816, -1.4952882528305054, 0.5181103348731995, 0.06535649299621582, 0.17171809077262878, -1.5794713497161865, 1.8798456192016602, -0.23630014061927795, -1.0532546043395996, -0.12106576561927795, 0.40011468529701233, -0.18013659119606018, -1.0984433889389038, 0.4083483815193176, 1.3872485160827637, -0.5336718559265137, -0.006415360141545534, 0.3481636643409729, 0.2758953869342804, -0.8146525621414185, 0.813001275062561, -1.437828540802002, -0.11539075523614883, 0.5491613149642944, 1.246964931488037, -2.016542673110962, 1.207265853881836, 1.1113309860229492, -1.1931612491607666, 1.7896147966384888, -0.28472572565078735, -0.5107109546661377, 0.7698436975479126, -0.9676224589347839, 1.0845000743865967, -2.246483564376831, -0.28880882263183594, -2.019249439239502, 1.2105005979537964, 0.38003623485565186, -2.2114386558532715, -0.554222822189331, 1.6661641597747803, -0.13000167906284332, -3.372758388519287, 0.6730567812919617, -1.9230948686599731, -0.2683393657207489, 2.321411371231079, 0.15268108248710632, 0.7290394306182861, 1.234478235244751, -1.2000468969345093, -0.582444965839386, -0.3695269227027893, 0.9510456919670105, -0.5377417802810669, -0.5226870179176331, -0.2545894980430603, -0.8875200152397156, 0.026675302535295486, -0.8174302577972412, 0.09943225979804993, 0.5580984950065613, -0.8726682066917419, -0.591275155544281, 0.22512269020080566, -1.1820193529129028, 1.261625051498413, -0.954695999622345, 0.15520301461219788, 0.014680864289402962, -0.5150651931762695, 0.6128585934638977, 0.03283001855015755, -1.9617840051651, -0.5603898763656616, 1.1253714561462402, -1.2550424337387085, 2.9280474185943604, 1.8950343132019043, -0.7268345952033997, 1.5845937728881836, -1.331652045249939, -0.6478529572486877, 1.2716257572174072, 2.1878068447113037, 0.9800153374671936, 1.1610444784164429, -0.43191370368003845, -0.22960148751735687, -0.6526399850845337, -0.45184582471847534, 0.4343859553337097, 1.0638024806976318, 2.9084861278533936, -2.0744705200195312, -0.9672943353652954, 0.5297022461891174, 0.9035978317260742, 0.2700670659542084, -0.7215161919593811, -1.3877583742141724, 1.6355998516082764, -0.23227442800998688, -0.2231234908103943, 0.16820703446865082, 1.0222822427749634, 1.8875269889831543, 1.541736125946045, 0.8918545246124268, -0.05722961574792862, 0.6780610680580139, 0.24538268148899078, -0.9159471988677979, -0.09406466037034988, -0.8683180809020996, -1.45610773563385, 1.1187069416046143, 0.011418404057621956, -1.4971369504928589, 0.22877882421016693, -0.29120826721191406, -0.8523573279380798, 0.16023564338684082, 2.3108181953430176, 0.6385683417320251, -0.06617067754268646, -0.11748231202363968, -0.1401257961988449, -0.7231418490409851, -0.8737169504165649, 1.2271151542663574, -0.9586894512176514, 0.4792693257331848, -1.31174898147583, 0.7974314093589783, -1.314808964729309, 0.32264047861099243, 1.9429851770401, -1.0155633687973022, 0.6531996726989746, 0.5150860548019409, -0.5353460311889648, -0.23453359305858612, 0.9001347422599792, -1.261268973350525, -1.5928575992584229, -0.617358386516571, -1.1516700983047485, -0.48080769181251526, -1.6280641555786133, 0.12684611976146698, 0.04754410684108734, 0.6913101077079773, -0.6232641339302063, 1.0438435077667236, 0.970606803894043, 0.8344343900680542, -0.3043069541454315, -0.169820636510849, 2.1965842247009277, -0.8210753202438354, -0.9588474631309509, 0.482116162776947, -0.7229352593421936, -1.0003479719161987, -0.6085909008979797, 0.9148354530334473, -1.701645016670227, -0.7683964967727661, 1.7144060134887695, -0.7265276908874512, -1.402812123298645, -1.2627776861190796, -0.23844224214553833, 0.3842519223690033, -0.9002770185470581, -1.5602293014526367, -0.8159458041191101, 0.15526454150676727, 0.06566696614027023, 1.5903315544128418, 0.10085700452327728, -0.1139805018901825, 1.9825164079666138, 1.0419821739196777, 0.18563948571681976, -0.8595008254051208, 0.6060554385185242, -1.0609333515167236, -0.2678995430469513, -0.2430853396654129, 0.48468464612960815, 1.2801783084869385, -1.4235057830810547, 2.2060446739196777, -0.42642050981521606, 1.719185471534729, -1.5184330940246582, 0.43934178352355957, -1.0843172073364258, -1.6535388231277466, -0.37701669335365295, -0.7564431428909302, -1.859924554824829, -1.0010595321655273, 0.5937818288803101, 0.502609372138977, -0.48196107149124146, -0.5375211834907532, 0.6436174511909485, -0.7163543701171875, -1.2403513193130493, 0.4317050278186798, -0.5395145416259766, 0.09707825630903244, 0.8130254149436951, 2.9165823459625244, 0.11990229040384293, -0.8622093796730042, 1.1988253593444824, -1.1386387348175049, 0.3030184507369995, 0.8780689239501953, -0.5467124581336975, 0.23574769496917725, -0.22982466220855713, -1.090110182762146, 0.6103875637054443, 2.0856494903564453, 1.3501887321472168, -0.6778687238693237, 0.2782684564590454, -1.4540338516235352, 0.6924939751625061, -0.49304360151290894, -1.0593442916870117, -1.1359539031982422, 0.1948956400156021, 0.0777197778224945, -1.3986644744873047, 1.3444290161132812, -1.940793514251709, -0.35370922088623047, 0.38655561208724976, 0.5007667541503906, -1.8014705181121826, -1.1047219038009644, 0.440315842628479, 1.075194001197815, -1.203948736190796, -0.43145981431007385, 0.9658833146095276, 0.27347278594970703, -0.3759027421474457, -0.43145430088043213, 0.9235730767250061, 0.642116129398346, 0.07673689723014832, 0.8848524689674377, -0.6411334872245789, 0.1082429438829422, 1.0528781414031982, 0.5677623748779297, 0.3726078271865845, -1.568914771080017, 0.6147811412811279, -0.8322176337242126, 0.6007034182548523, -1.8590342998504639, 0.8924976587295532, -0.5747882723808289, -2.4371516704559326, 1.2194865942001343, -1.16049325466156, -1.0062518119812012, -0.8384101390838623, -0.3006713390350342, 1.6216607093811035, 0.5337608456611633, 1.0165497064590454, 1.6383295059204102, -0.9601989388465881, -0.8584228157997131, -0.8083728551864624, 0.904147744178772, 0.30132967233657837, -0.37747159600257874, 1.3748778104782104, 0.10986515879631042, 0.5054667592048645, 0.2637336254119873, -0.15146119892597198, -0.43813762068748474, -0.15398620069026947, -0.9506987929344177, 1.0878310203552246, 0.6948156952857971, 0.7365480661392212, 0.46020373702049255, -0.03632189705967903, -2.7906694412231445, 0.7332934737205505, 0.42897534370422363, -0.7758181691169739, 0.023404119536280632, 0.11567201465368271, 1.332519292831421, 0.037824515253305435, -0.7432951927185059, 1.4786343574523926, -1.4418529272079468, -0.4525550603866577, -2.2804436683654785, 2.6910810470581055, 0.09077247977256775, -1.5458465814590454, -1.288875699043274, 0.5326646566390991, -0.37982502579689026, -1.0863596200942993, 1.2836837768554688, 0.05870453640818596, -0.013185695745050907, 2.8213815689086914, 1.6748358011245728, -1.1166040897369385, -0.3267519474029541, -0.330784410238266, -0.35359787940979004, -1.0080658197402954, 1.3134397268295288, 0.6405602693557739, 0.09042699635028839, -1.3820322751998901, 0.14166781306266785, -0.5824622511863708, 1.8495277166366577, 0.2091326266527176, -0.03801409527659416, 0.6334282755851746, -0.1407856047153473, -0.6299782395362854, 3.519019603729248, -0.5231081247329712, 1.1147520542144775, -1.768186330795288, 0.6297136545181274, 1.2132606506347656, 0.5337967276573181, 1.0483229160308838, -1.7503650188446045, -0.5838891267776489, -1.0397436618804932, 0.7174125909805298, 0.4249212443828583, 0.5705680847167969, -0.6863306760787964, 0.48772382736206055, -0.048637278378009796, -0.1984362006187439, -1.0606938600540161, 0.3663651645183563, 0.6211077570915222, 0.4002281129360199, -0.9254290461540222, 0.09637580811977386, 2.2652130126953125, -0.689357340335846, 0.6888686418533325, 0.10458925366401672, -1.1129730939865112, -1.454857349395752, 0.8402941823005676, 0.16459865868091583, -0.5733655691146851, 2.41833758354187, -0.4757263958454132, 0.5089128017425537, -0.4682369828224182, -0.43255770206451416, 0.2084084153175354, -1.0759029388427734, 1.3167262077331543, -1.4988071918487549, 1.0365405082702637, -0.30609774589538574, -0.1896413117647171, 0.6200416684150696, 0.7918803095817566, -0.9510443806648254, -1.2339249849319458, 0.8072625994682312, 0.5580225586891174, 0.9375270009040833, 1.744858741760254, 0.7831770777702332, -1.3275200128555298, 0.15996012091636658, -0.014092721045017242, -0.03299037739634514, 1.2761837244033813, 0.000816380837932229, 1.3722175359725952, -0.49826866388320923, 1.1999390125274658, -1.469948172569275, 0.13514921069145203, -0.13307788968086243, -1.3208394050598145, -0.9197512269020081, 0.35907888412475586, 1.001291036605835, 0.5593755841255188, 0.400173544883728, 0.7044087052345276, -0.4618382453918457, 0.05637519061565399, -0.6129557490348816, 0.08659039437770844, -0.5024048686027527, -0.17692774534225464, -2.7317616939544678, -1.3922852277755737, 1.0022237300872803, -0.4446372091770172, -0.6497172713279724, -0.402417927980423, 0.8230209946632385, 1.3262544870376587, 1.1526172161102295, 1.2083135843276978, -0.8615527153015137, 0.22388608753681183, -2.041388988494873, -0.6185427308082581, 0.19839416444301605, 0.995041012763977, 1.4116977453231812, -2.097496509552002, -0.04125696048140526, 1.983445167541504, -0.8769308924674988, -0.6690859794616699, 0.05848664417862892, 1.8804991245269775, -2.0201306343078613, -0.43440425395965576, 1.9074252843856812, -0.15048788487911224, -1.1097840070724487, -1.0701781511306763, 1.3167471885681152, 0.32288286089897156, 1.0557894706726074, 1.2005648612976074, 0.6208527088165283, 0.5742992758750916, 0.13009321689605713, 0.9978134036064148, -1.202511191368103, 0.8523699641227722, -0.9349998831748962, -0.4636506140232086, 0.7364397644996643, -0.48891469836235046, -2.4474198818206787, -0.7298283576965332, -0.5208048224449158, -0.08453736454248428, -0.7073764204978943, 1.1661829948425293, -1.035216212272644, -0.18901725113391876, -1.1216421127319336, 0.5941147208213806, 0.7314325571060181, 0.7635579109191895, 0.8052311539649963, 0.10897013545036316, 1.0696061849594116, -0.4769834280014038, -1.0747114419937134, -0.5841809511184692, 0.8365650773048401, 0.8001341819763184, -0.3321782648563385, -0.10390037298202515, -1.3164042234420776, -0.4369768798351288, -0.7345765829086304, 0.597567617893219, -0.4821360111236572, 2.781705141067505, 0.9802209734916687, -0.9714092016220093, -0.1812707632780075, -2.304748058319092, -0.6419422030448914, 0.28163957595825195, -1.8169550895690918, -0.06056174635887146, -0.5986606478691101, -0.6665008068084717, 0.292970210313797, -2.2890195846557617, -0.7719935774803162, -0.33216825127601624, 1.320447325706482, -1.3647570610046387, 1.7057236433029175, 0.6314383745193481, -0.48667311668395996, 0.5932717323303223, -0.16283921897411346, -0.6964395642280579, 1.3938268423080444, -0.8191204071044922, 1.5228657722473145, -0.22412064671516418, 0.17107249796390533, -0.24521324038505554, 0.759488046169281, -0.5671080946922302, 0.3459334075450897, 0.27512338757514954, -0.42663902044296265, 0.6430546045303345, -0.7023198008537292, 0.7712964415550232, 0.9870524406433105, 1.7409307956695557, -0.007177003193646669, 0.04166579991579056, -0.28206154704093933, -0.7604578137397766, -1.5014582872390747, 1.3572520017623901, -0.9962917566299438, -0.6967204213142395, 0.2450413554906845, 2.161177158355713, -1.6801351308822632, -1.1940430402755737, 0.2647227346897125, 0.4400574862957001, -0.011205094866454601, -0.498374879360199, -0.41093361377716064, -0.21227309107780457, 0.8593742251396179, 1.127461314201355, -0.25262463092803955, 0.4530576467514038, 1.5375125408172607, 0.3988547623157501, -1.0320172309875488, 0.4540104866027832, -0.492463618516922, 0.023331547155976295, 0.1646461933851242, 0.7646828889846802, 0.05199480056762695, 1.194075584411621, 0.6301826238632202, 1.0632396936416626, -0.20663033425807953, 1.5727797746658325, -0.5231010913848877, 0.018592607229948044, -1.6475850343704224, -0.24375638365745544, 1.595812201499939, -1.298998475074768, 0.012990949675440788, -0.0944330245256424, 0.9032242894172668, 2.608743906021118, -0.2351720631122589, 0.030300457030534744, -0.9846010804176331, 0.9869045615196228, 0.925834059715271, 0.22921034693717957, 1.0532199144363403, 0.0358719527721405, 0.563366711139679, -0.03431671857833862, -0.2783164978027344, 1.7556092739105225, -1.8377196788787842, -0.9770065546035767, -0.5904071927070618, 0.19223721325397491, 0.10145458579063416, 1.1730754375457764, 0.3389512598514557, 0.4505942165851593, 1.4597371816635132, 0.13402697443962097, 0.9863045811653137, 1.4650230407714844, -0.4881121814250946, 1.3419172763824463, 1.7986276149749756, 2.501438617706299, 0.011824374087154865, 0.7590633034706116, 0.40710583329200745, 0.8587868809700012, -0.9417259097099304, -0.8145061135292053, -0.5934074521064758, -0.34582507610321045, 0.014076764695346355, -2.3305859565734863, 1.4384777545928955, -0.15971295535564423, -1.1391884088516235, -0.44370198249816895, -2.1537933349609375, 0.6929095983505249, 1.6944971084594727, -0.09795552492141724, 0.6491609215736389, -0.029394976794719696, 0.2852524220943451, 0.9418817758560181, 0.3891286551952362, -1.178471565246582, 0.03706783801317215, 0.44837769865989685, 1.8790513277053833, -1.493241548538208, 1.6386991739273071, -0.6426312327384949, 1.0910077095031738, 1.5563100576400757, -1.0908362865447998, -1.1473050117492676, -0.7921255230903625, -0.46248358488082886, 1.8844879865646362, -0.9867674708366394, 0.848133385181427, -2.8452839851379395, -1.5034797191619873, -1.3345999717712402, 1.165981650352478, -1.0776170492172241, 0.3860202729701996, 0.03639603778719902, 0.14598636329174042, -1.3720860481262207, -1.0347427129745483, -0.9463664889335632, -0.5173694491386414, 0.6272968053817749, 0.4279247522354126, -0.2442304790019989, 0.7450410723686218, -0.9975786209106445, -0.6967374682426453, -1.4069275856018066, 0.4519924223423004, -0.4455913007259369, -0.44100016355514526, -2.315052032470703, 0.0022784778848290443, 0.31813469529151917, 0.42144855856895447, -0.4279222786426544, 0.31672874093055725, 0.9325882792472839, 0.54802405834198, 0.251471608877182, 2.3535685539245605, 0.4436147212982178, 1.3851733207702637, -0.7642256617546082, 2.066293478012085, 0.258070707321167, 0.3342912495136261, -1.5816454887390137, -0.3272288143634796, -0.4449833035469055, -1.4415993690490723, 0.6378898024559021, 0.1536180078983307, 0.2880817651748657, 1.384317398071289, -0.5794193744659424, -0.5324670672416687, 0.2781864404678345, 2.21557354927063, 0.9726707935333252, -0.7699333429336548, -1.4007776975631714, -0.4540776312351227, 0.675172746181488, -1.0672310590744019, 0.2163527011871338, -0.9563264846801758, -0.2869819402694702, 0.4461486339569092, 0.07036348432302475, -1.48264479637146, 0.2078634798526764, -1.232465386390686, 0.48570600152015686, 0.6219756007194519, -2.1936473846435547, -1.403640627861023, 0.5746128559112549, -0.32096603512763977, 0.5229098200798035, 0.9302114844322205, -0.04117891564965248, 0.05490643531084061, 0.4768439531326294, -0.08414003998041153, -1.3597989082336426, -0.5646629929542542, -0.43610161542892456, -1.3474222421646118, -0.4153367578983307, -0.36290645599365234, 0.32901397347450256, 0.4317272901535034, -0.6163654327392578, -0.8252192735671997, -0.6927877068519592, 0.9926925301551819, -2.0839779376983643, 0.7874794006347656, -0.5153539180755615, 0.28193211555480957, -1.0140247344970703, 0.7339779138565063, -0.32340705394744873, 0.24043849110603333, -0.07848256826400757, -0.6645810008049011, 1.5237009525299072, 0.42662692070007324, 0.34984350204467773, 1.2110122442245483, -2.648958206176758, -1.0634708404541016, 0.16167573630809784, -0.0933976024389267, 1.233536958694458, 0.48012086749076843, -0.7206827998161316, 0.6418213248252869, -0.43548980355262756, -0.24034029245376587, 0.7350330948829651, 1.1896436214447021, -0.29998284578323364, 0.8646398782730103, -0.7407271862030029, 1.0689493417739868, 0.4837930500507355, 0.1519586592912674, 1.132457971572876, 0.8413416147232056, 0.6331867575645447, -0.2760346531867981, -0.30498722195625305, -0.07614891976118088, 1.2076677083969116, -0.8310472965240479, -0.3067944049835205, -0.10825619846582413, 1.0044105052947998, -0.26302605867385864, -0.9739022850990295, 0.28846943378448486, -0.6300064921379089, 0.23752154409885406, 0.563256025314331, 1.4756896495819092, -0.6500412225723267, -0.15308509767055511, -0.29730138182640076, 0.9593591094017029, -0.06003469601273537, -0.0973925068974495, 0.3464675843715668, 2.2287278175354004, 0.07093863189220428, -1.0807843208312988, -0.10914858430624008, -0.898245632648468, -0.8259711861610413, 0.13218368589878082, 0.41972115635871887, -0.17796671390533447, 0.021324167028069496, -0.22535312175750732, -0.7137205004692078, 0.6979994177818298, 1.125065803527832, -0.22734203934669495, -0.34833279252052307, 1.2181077003479004, 1.5978410243988037, -0.2329297810792923, -0.29318082332611084, 0.28078800439834595, -0.49603354930877686, 0.43666383624076843, -1.713881492614746, 1.2647769451141357, 0.8123753070831299, 1.151665449142456, 0.8625551462173462, 0.48215895891189575, -0.3980739414691925, 0.6036639213562012, 0.7103027701377869, -0.2037765085697174, -0.3511134386062622, 1.0160833597183228, -0.7279155254364014, -1.9197672605514526, 0.1601833552122116, 0.4814356565475464, -0.04090233892202377, 1.1728569269180298, 0.33597567677497864, 1.4357765913009644, 0.08444881439208984, 0.9253830909729004, -0.9132445454597473, -0.2907266616821289, 1.6276904344558716, 1.7262465953826904, -1.030633568763733, -0.43735992908477783, -0.15028539299964905, -1.816007375717163, -0.22665071487426758, -0.5743013024330139, -0.926762044429779, -0.49989041686058044, 0.4562699496746063, 1.5481435060501099, -0.21184979379177094, -0.41390588879585266, -0.3861171305179596, 0.906214714050293, -0.5532273054122925, 0.5318363308906555, -0.2490643858909607, 0.6252666115760803, 0.07583504170179367, -1.307518482208252, -0.5528546571731567, -1.5361756086349487, -0.14125989377498627, 0.6162722706794739, -0.06120598316192627, -0.8662825226783752, -0.20907758176326752, 0.5538501739501953, 0.6466779708862305, 0.8120577335357666, 0.27682197093963623, 0.35522788763046265, -0.42764753103256226, 2.071887254714966, 1.6916512250900269, 0.9407273530960083, 0.3255036771297455, 0.2719775140285492, -2.0265002250671387, -0.5304660797119141, 0.09507398307323456, 0.49727964401245117, 0.8129928708076477, -0.9071404933929443, 1.0047917366027832, 1.0925838947296143, 0.7328118681907654, 0.2166881561279297, 1.3307881355285645, -0.05609806999564171, -0.7091297507286072, 0.02632948011159897, 0.23990045487880707, 1.565645694732666, -0.20101642608642578, -0.22350434958934784, 0.31984126567840576, 0.9650785326957703, -1.0718826055526733, 0.660323977470398, -0.7383471131324768, 0.4811747074127197, -0.4814532399177551, -0.6857790350914001, 0.8312795758247375, -0.8677144646644592, 0.5419247150421143, 0.5209270715713501, 0.6211862564086914, -0.11708338558673859, -0.509434163570404, 0.5888004302978516, 2.1559972763061523, 0.553165853023529, 2.063152313232422, 1.1495994329452515, 1.1611586809158325, 1.1813656091690063, 1.494335412979126, 0.016673503443598747, -1.36509108543396, 0.1553332358598709, 1.140466570854187, -0.07535750418901443, 0.29272109270095825, 1.8030691146850586, 1.0876975059509277, 0.07737378031015396, -0.5389467477798462, -0.2110074758529663, -0.35499995946884155, -0.6563894748687744, 0.32354962825775146, 0.18326514959335327, -0.44660913944244385, -0.688569188117981, 0.27106863260269165, -0.9970083236694336, -0.7163470983505249, -1.1312756538391113, 0.6659457683563232, 0.6898220777511597, -1.116060733795166, -2.0294718742370605, 0.3179655969142914, -2.215418815612793, -1.5414037704467773, -1.573382019996643, 1.3601127862930298, 0.9573590755462646, -0.8664364218711853, -0.259155809879303, 0.5745289921760559, -1.2058789730072021, -0.4209899306297302, 1.7286063432693481, 0.328043133020401, -0.4569518268108368, 2.4873597621917725, 0.9308249950408936, -0.5945518016815186, -0.960963249206543, 1.4778697490692139, -0.12395716458559036, 0.16091206669807434, 1.1525042057037354, -0.6847623586654663, 0.4487573206424713, 0.6263877749443054, 0.39795389771461487, -0.29763132333755493, -0.5845639705657959, -0.840875506401062, 0.9277262091636658, 1.9058935642242432, 0.08087025582790375, -0.9776037335395813, -0.1533786952495575, 0.7776812314987183, 0.8066099882125854, -1.5688267946243286, 0.11492808908224106, -0.9784498810768127, -0.27264395356178284, 0.1335749626159668, 1.8049342632293701, -0.037895187735557556, -0.6386721730232239, 0.9503228068351746, 0.9181994199752808, -0.45074018836021423, 0.26915693283081055, -0.5706648230552673, -1.401339054107666, -0.8982962369918823, -0.2064226120710373, -1.2131776809692383, -0.27911049127578735, -0.8901671767234802, -0.028001591563224792, 1.024325966835022, 0.02076725848019123, -1.431992530822754, -0.8648161888122559, -0.29457491636276245, -0.2133161723613739, 0.7314485907554626, -0.7642419934272766, -2.0829110145568848, 0.4629635512828827, 0.6972543001174927, 0.275202214717865, -2.7508397102355957, 0.0030468865297734737, -0.5732952356338501, -0.1317032277584076, 0.692803680896759, 1.2049559354782104, -0.13565242290496826, 0.6341475248336792, 0.7966241836547852, -0.48551496863365173, -1.9229295253753662, 1.1656280755996704, -0.9944067001342773, 0.5199429988861084, 0.19545133411884308, 1.4006929397583008, -1.7599726915359497, -0.2088165432214737, -0.585597574710846, -0.4904639422893524, 0.17979714274406433, 2.11145281791687, -0.602489709854126, -0.2515680491924286, 1.0000176429748535, 0.1997593641281128, 0.1817910373210907, -1.6700620651245117, 0.5806530714035034, 1.0851061344146729, -1.3836604356765747, 0.6673576831817627, -1.9319199323654175, 0.6444560885429382, -0.5620695948600769, -0.5179203152656555, -1.8636723756790161, -0.5363264679908752, 1.1041927337646484, -0.47569340467453003, -0.6065810322761536, -1.2338647842407227, 0.0670342892408371, 0.5496248602867126, -0.8234353065490723, -2.292022466659546, -0.03946109488606453, 1.7384188175201416, 0.4449164569377899, -1.1100538969039917, 1.0789920091629028, -0.13818788528442383, -0.9349908828735352, -0.5075360536575317, -2.7395615577697754, -0.30555328726768494, -0.18938016891479492, -2.0155928134918213, -1.1371259689331055, -1.1235531568527222, -0.5335692167282104, 0.45969218015670776, -0.9004743099212646, -1.650246500968933, 0.4717610478401184, -0.28215470910072327, 0.9718403816223145, 0.5851534008979797, -0.24774695932865143, 0.9985848069190979, -0.37602362036705017, -0.012832753360271454, -0.8500390648841858, 0.8360859751701355, 0.44866085052490234, 0.042532432824373245, -0.36702853441238403, -0.4845298230648041, -0.680238664150238, -1.3879635334014893, 0.9217761158943176, 1.2626352310180664, 0.19997654855251312, 1.1366949081420898, -0.2999158501625061, -1.9505116939544678, -1.5499905347824097, 1.671203851699829, -0.72202068567276, -0.31971579790115356, 0.3326583802700043, -1.4487942457199097, 0.31836727261543274, -0.85582035779953, 1.429050326347351, 0.41982975602149963, 0.9594643115997314, 0.304372102022171, -2.0938024520874023, -0.24861449003219604, -0.775538444519043, 0.1866353154182434, -0.9053589105606079, 0.19693568348884583, -0.7380226254463196, 0.4093954861164093, -1.8798691034317017, -0.4326043128967285, 0.5044907927513123, 0.2997150719165802, -1.5583914518356323, 1.4117815494537354, 1.6202471256256104, 0.646241307258606, -0.8278992176055908, -0.26139336824417114, -0.40052029490470886, -0.8374799489974976, -2.0724775791168213, -2.650270700454712, 1.0324229001998901, 0.05693446472287178, -0.6044018268585205, 1.3291877508163452, -0.42678219079971313, -0.8626207113265991, 0.9722487330436707, -1.056646704673767, 1.248995065689087, -0.3813158869743347, 2.3000125885009766, -1.0631695985794067, 0.23036722838878632, -1.6249858140945435, 0.3845638036727905, 1.0235422849655151, -0.39134669303894043, 0.6603253483772278, -0.6744968891143799, 0.7271225452423096, 1.1320909261703491, -0.14792655408382416, -2.372770309448242, -0.3399498760700226, 0.6107253432273865, 1.5394374132156372, 0.4949342906475067, -1.5334370136260986, -1.0217671394348145, 0.15843771398067474, 1.7815498113632202, 1.807043433189392, -0.5519382953643799, 0.18840733170509338, -0.0561131089925766, -0.9073333740234375, -0.28332510590553284, -0.43208983540534973, 0.49585407972335815, 1.9050344228744507, -0.2779540717601776, -0.15634988248348236, 0.07610564678907394, -0.7676794528961182, -0.39012855291366577, 0.014880470000207424, -0.049878720194101334, -0.6203601360321045, 0.44936081767082214, 1.5213998556137085, -2.1042821407318115, -0.7590531706809998, 1.4039306640625, -2.0449938774108887, 1.7348880767822266, -0.9529335498809814, 0.02321367710828781, -1.8480496406555176, 0.9980862140655518, 0.3931078612804413, 0.13708457350730896, 0.059827279299497604, -1.8240958452224731, 1.0538209676742554, 1.1392499208450317, 0.9752081036567688, 0.4434143006801605, 1.5383479595184326, -0.5467154383659363, -1.1115312576293945, -0.7698817253112793, -0.16079743206501007, -0.392537385225296, 0.42757323384284973, 1.549088954925537, -0.45407453179359436, -0.8603792786598206, -0.551028847694397, -0.4309362471103668, -0.004272010177373886, 0.24131985008716583, -1.0586224794387817, -0.050726816058158875, -0.17653745412826538, 0.027320653200149536, -0.46938690543174744, 0.36430972814559937, 1.1373603343963623, 0.6011812090873718, 1.3856079578399658, 0.6663672924041748, 0.4179823100566864, 0.4952617585659027, -1.4908689260482788, 0.914351224899292, -0.05015654116868973, 0.2375979870557785, 1.2811319828033447, 0.5730245113372803, 0.3835667669773102, -1.133887767791748, 0.12372658401727676, 0.3024868369102478, 0.21620000898838043, -0.12224256992340088, -1.2956608533859253, -1.1739596128463745, 0.00532703148201108, -0.500042200088501, -1.4593585729599, 0.5375168919563293, 0.129227876663208, 0.6816684603691101, 1.0919549465179443, -1.521590232849121, 1.5520988702774048, 0.21866005659103394, 1.6649365425109863, 0.4118182361125946, -0.6576460003852844, 0.9955188035964966, 0.6404143571853638, -1.1216155290603638, 1.0496636629104614, -0.20369645953178406, -1.360824465751648, -1.7595505714416504, -0.5939791798591614, 0.16551633179187775, 0.3646986782550812, 1.5652790069580078, 0.23499582707881927, 0.6391760110855103, 0.9986787438392639, -0.6802953481674194, -0.3944038152694702, -1.1203628778457642, -0.9648464322090149, -0.7026767730712891, 0.19281093776226044, -1.297469139099121, 0.959064781665802, 0.024730827659368515, -1.6405483484268188, 1.7744989395141602, -0.3982473611831665, -0.6332688331604004, 0.44820207357406616, -0.16354872286319733, 1.0193147659301758, -0.8100160360336304, -0.17205344140529633, -1.3494161367416382, 1.5641251802444458, 0.8809316754341125, -1.710852861404419, 0.35274550318717957, -0.4368290901184082, -0.04649988189339638, 0.9792469143867493, 0.27696889638900757, -1.273512601852417, 0.2035403996706009, -1.50200355052948, -1.0135000944137573, 1.0705851316452026, 0.19434548914432526, 0.6435374617576599, -0.13190574944019318, 0.0819454938173294, -0.49006155133247375, -0.46340274810791016, 0.46482113003730774, 1.1414122581481934, 1.7587037086486816, -1.1022027730941772, -0.48479005694389343, 2.603031635284424, -0.8638299107551575, 0.24751050770282745, 0.8613334894180298, 1.437154769897461, -0.5343621969223022, -0.9480506777763367, -0.9816429615020752, -0.8132348656654358, -1.4970707893371582, -1.945112705230713, 1.339261770248413, 1.210647702217102, -1.9580601453781128, -1.3479880094528198, -1.9132620096206665, -0.12801401317119598, -0.9868226051330566, 0.5840278267860413, -1.4627807140350342, -0.5452459454536438, 1.0574455261230469, 1.2960925102233887, -0.8949776887893677, -0.12653499841690063, -0.7032237648963928, -0.657923698425293, -1.735557198524475, -0.06743951886892319, -1.497667670249939, 0.3765307068824768, -1.2745190858840942, -0.39075642824172974, 2.9151968955993652, -0.2949712872505188, 0.8940567970275879, 0.7821384072303772, -0.11724291741847992, 0.585599958896637, -0.4606892168521881, -0.5952423810958862, 1.2392016649246216, -1.5816437005996704, -0.8813830018043518, 0.2725687325000763, 0.38877320289611816, 0.31376272439956665, 0.742953360080719, 1.839583396911621, 0.7008188962936401, -0.3691498041152954, 2.4501230716705322, -0.8961715698242188, 0.40457746386528015, -2.117830514907837, 1.5781387090682983, -0.7852020263671875, -0.3701295852661133, -2.056690216064453, 0.37150833010673523, -0.13302113115787506, -0.631992757320404, -1.5358847379684448, -0.3536691963672638, 2.384587049484253, -0.12530255317687988, 0.1419583112001419, -2.0633864402770996, -1.0415854454040527, 1.2595332860946655, -1.449562907218933, -1.8028481006622314, 0.7436813116073608, -0.21182560920715332, -0.33435022830963135, 1.3593426942825317, 0.5046217441558838, 0.34530311822891235, 0.0828157439827919, 0.7606117725372314, -1.800313115119934, 2.102870225906372, -0.37604790925979614, -0.18925856053829193, -0.5494900941848755, -1.045453429222107, 0.3565886914730072, 0.5414136052131653, 0.07997364550828934, 2.8746917247772217, 0.135919451713562, -1.0089586973190308, 0.704679548740387, -0.4732016921043396, 1.0474878549575806, 1.0482605695724487, 0.44313666224479675, 1.3421999216079712, 0.5013774633407593, -1.186012864112854, -1.0096787214279175, 0.9416527152061462, -0.16169899702072144, 0.45804351568222046, 0.9744611382484436, -0.6836128830909729, 2.042903423309326, -1.658335566520691, 0.35449084639549255, -0.051148802042007446, 1.7306122779846191, -0.2706336975097656, -1.1760836839675903, -0.5813342928886414, 1.4442412853240967, 0.007612918969243765, 0.015189006924629211, -0.6599775552749634, 2.2449376583099365, -1.3745452165603638, -1.1015604734420776, 0.16210417449474335, 0.6720491647720337, -1.4946632385253906, -0.14638124406337738, -1.6935449838638306, -0.8994650840759277, 1.681506633758545, 0.14112047851085663, 0.6236857175827026, 0.4627886712551117, -0.8724312782287598, -1.5393601655960083, -0.2770090103149414, -1.0115866661071777, -0.4581385850906372, 1.1538203954696655, -1.585342288017273, 0.3639511466026306, 0.003968924283981323, -0.19877876341342926, -1.7948613166809082, 2.1828653812408447, 0.5447545647621155, 0.32213863730430603, 0.9121981263160706, -0.4312366843223572, 0.015561146661639214, 1.4637339115142822, -1.1242074966430664, 1.0322346687316895, 0.5203407406806946, 0.15078434348106384, -1.308957576751709, -1.8517522811889648, 0.5621482133865356, 0.8080975413322449, 0.25441041588783264, -0.25334617495536804, -0.5609287619590759, 1.255157470703125, 0.5539760589599609, 1.0883769989013672, 0.5742380619049072, 0.25459712743759155, 0.35743916034698486, 0.7477977275848389, 0.2957805097103119, -1.32254958152771, -0.9163117408752441, -2.666926145553589, -0.4395236074924469, -0.8559443950653076, -1.0793670415878296, 2.054708957672119, 0.4522773325443268, -0.5319678783416748, 2.167060375213623, -0.10892895609140396, -0.28540492057800293, -0.838139533996582, -1.0194473266601562, 1.902050495147705, 1.1868921518325806, -0.4442780911922455, -0.1961938738822937, -0.7801343202590942, -0.8015827536582947, 1.4643287658691406, -0.6114899516105652, 1.461440920829773, 0.2477898895740509, 0.8015943169593811, 0.7524527311325073, 0.2214508205652237, 1.2024815082550049, -0.7776282429695129, 0.6622717976570129, -1.9248278141021729, -0.6841800808906555, 0.3249434232711792, 0.2638311982154846, -0.13314767181873322, -0.258281946182251, 0.2577281594276428, 0.8994514346122742, 0.7763340473175049, 0.32199814915657043, -0.3838397264480591, 0.5283017754554749, 0.030821599066257477, -0.031569018959999084, -2.175719738006592, 0.4096355140209198, -1.8882112503051758, -0.09532535076141357, 0.7976050972938538, 1.8842257261276245, -0.7625377178192139, -0.06175669655203819, -0.15944607555866241, 1.1975533962249756, 0.15254172682762146, -0.4667389988899231, 0.2450937032699585, -0.9066776633262634, -0.8159541487693787, -1.5438586473464966, 0.04744343087077141, -0.08823882043361664, -1.4440306425094604, -0.9255744814872742, -0.5571444034576416, -0.17369672656059265, -0.4465082287788391, 1.4021775722503662, 0.9909810423851013, 0.29807159304618835, 2.5807080268859863, 1.4356215000152588, 1.1342660188674927, -0.27114686369895935, 1.2548954486846924, -0.4607226550579071, 0.7144309282302856, -0.41511353850364685, 1.1529468297958374, -1.1968224048614502, 0.7133768200874329, 0.7163721323013306, -0.17303626239299774, -0.030267706140875816, 0.9256796836853027, -0.060271866619586945, 0.9326136112213135, 0.3916511535644531, 0.9214160442352295, 1.9985369443893433, 0.916809618473053, 2.985398054122925, -0.5263642072677612, -0.5940828919410706, -0.5562890768051147, -0.40746721625328064, -0.3521879315376282, 0.3932066261768341, -0.6413694024085999, 0.030310895293951035, -0.0517093800008297, 0.7388430833816528, -2.0388100147247314, 0.4757881462574005, 1.2141778469085693, 0.9858134984970093, 0.3840593993663788, 1.7526060342788696, 1.7787259817123413, 0.3614877164363861, -0.14017905294895172, -0.04295964166522026, -2.4790327548980713, -0.8125677108764648, 0.5369092226028442, -0.28202441334724426, 0.08331260830163956, 1.4948294162750244, 1.2303637266159058, 0.3857366740703583, 1.1365324258804321, -0.917205274105072, 1.021666169166565, -0.7876598834991455, -0.4628978669643402, 0.235952690243721, 0.29971328377723694, -1.299898624420166, -1.314412236213684, 1.7984647750854492, 0.07705764472484589, 0.32788440585136414, -1.5297220945358276, -1.5411388874053955, -2.1278152465820312, -0.32220929861068726, 1.0713776350021362, 0.9268421530723572, -0.22665780782699585, 1.7145092487335205, -0.054390061646699905, -0.062090665102005005, -0.5283660292625427, 0.8096177577972412, -0.5881982445716858, 0.19673770666122437, 0.7490777373313904, 0.8136048316955566, -0.7129188179969788, -0.3846401572227478, -1.4658007621765137, 0.06740930676460266, -1.5098956823349, 0.07541278004646301, -1.9890133142471313, 0.35250720381736755, -0.8097066283226013, -0.8751264214515686, -0.6893475651741028, -0.6656259298324585, -0.45431479811668396, 1.0012233257293701, -0.6688104271888733, 1.281714677810669, 1.9140992164611816, 0.09848260879516602, -0.3548794984817505, -0.05439579486846924, -1.6129430532455444, -0.7042503952980042, 1.3034616708755493, -0.1366100311279297, -1.271607756614685, 1.7581273317337036, 0.6207975149154663, -1.4364343881607056, 1.1990545988082886, -0.11028047651052475, 0.07985921949148178, 0.0021814920473843813, 0.8756901025772095, 0.3604549467563629, -1.1867700815200806, 0.35528409481048584, -0.9045620560646057, 0.7999669909477234, 0.6214408278465271, 0.4281466603279114, -0.7506521344184875, 1.9518051147460938, -0.8589341640472412, -1.1994422674179077, 0.9906169176101685, -0.42814841866493225, 0.6944595575332642, -1.3876219987869263, 0.17086873948574066, -0.033419832587242126, 1.0767247676849365, -0.16040650010108948, 0.7830087542533875, 0.8676506280899048, -0.19027923047542572, 0.3262145221233368, -0.8173438310623169, -0.49147406220436096, -0.5342035889625549, -0.4358437657356262, -0.6936798691749573, 1.5499539375305176, 0.594700813293457, 0.7493723630905151, 0.014252208173274994, 1.6400601863861084, 1.271674633026123, -0.9360315799713135, 1.1924145221710205, 0.2523040771484375, 0.6443008780479431, 0.7796615958213806, 1.2712517976760864, -0.7704755663871765, 1.8580694198608398, 0.8144005537033081, 0.8459084033966064, -0.20504160225391388, 0.4741431474685669, -0.9660727381706238, 0.6621384024620056, -2.418069839477539, -0.03128727525472641, -0.16411472856998444, 0.01961364410817623, -1.226317048072815, 0.3741679787635803, 0.22433029115200043, -0.40447425842285156, 0.7243932485580444, 0.730369508266449, -0.49813151359558105, -0.2486322969198227, 0.14968061447143555, 0.3345947861671448, 1.2953704595565796, -0.4406391680240631, 0.5627800226211548, 1.1828629970550537, 0.034449826925992966, -0.7998260855674744, 1.812673807144165, 0.018566317856311798, -0.6778966784477234, -0.9562363028526306, -0.6588467359542847, -1.1836727857589722, 0.43046122789382935, -0.5549027323722839, -1.045324444770813, -0.5326694250106812, -0.025653941556811333, -0.34259599447250366, -1.2000916004180908, -1.01645028591156, -0.4653080999851227, -0.8246905207633972, -0.08165257424116135, 1.7956925630569458, -0.01190925668925047, -0.7073917984962463, -1.3283965587615967, -0.8154569864273071, -0.9391874074935913, -1.2443463802337646, 0.3229897618293762, -0.5622124075889587, -0.9738085269927979, -0.3425832986831665, 1.5687915086746216, 0.08133551478385925, -0.32639020681381226, -1.2475322484970093, -0.6912301778793335, 0.2659415006637573, 0.6896475553512573, 1.090735673904419, 1.3874843120574951, 1.215057611465454, -1.1500664949417114, -0.4442671835422516, -0.621882975101471, -0.6537392735481262, 1.0310285091400146, -0.761269211769104, 0.7941293120384216, -0.31322577595710754, -1.001663088798523, -0.054292500019073486, 1.708227515220642, -2.8403096199035645, 0.4056200683116913, 1.4502067565917969, -1.103634238243103, 3.2214672565460205, 1.1276029348373413, -0.1966591775417328, 0.39173850417137146, -0.12321966886520386, -0.5363573431968689, 0.8663075566291809, -2.446822166442871, -0.38323262333869934, 0.45813867449760437, -0.9640880823135376, -0.277368426322937, 0.5569900870323181, 0.49699458479881287, 1.446302890777588, -0.32936176657676697, -0.15128949284553528], [1.3928203582763672, -1.1089251041412354, -0.5686270594596863, -2.1145153045654297, -0.466720849275589, 1.4017726182937622, 0.6128081679344177, 0.2065032571554184, -0.380558580160141, 1.7443829774856567, -0.022604120895266533, 0.6284791827201843, -0.6442247629165649, 0.5347661972045898, -0.028323151171207428, -0.48733553290367126, 1.0316427946090698, -2.2497949600219727, 1.288870096206665, -0.02946753427386284, 1.321837306022644, 1.0989257097244263, 1.1275713443756104, 0.24383966624736786, 0.05354750156402588, 2.1039695739746094, -0.6777381300926208, -1.4838486909866333, -1.286258339881897, 1.5428146123886108, -0.7673362493515015, -2.9436144828796387, -0.4201807975769043, -1.3767160177230835, 0.7904669642448425, -0.42826786637306213, 0.23974521458148956, 0.32876166701316833, 1.3196072578430176, 0.935881495475769, -0.1013713851571083, -1.3534613847732544, 0.4491941034793854, -1.9157053232192993, -0.292996883392334, -1.0625922679901123, 0.3097507655620575, -0.6674374938011169, 1.0600614547729492, 1.6568371057510376, 0.6551925539970398, -1.373772382736206, 1.905653953552246, 0.01661992073059082, 0.450885146856308, -0.0030755058396607637, 0.7526631951332092, 0.357645183801651, 0.43353429436683655, 0.23201286792755127, 0.5909913778305054, -0.7036609053611755, -1.8083356618881226, 1.2423652410507202, -0.06238899007439613, -0.7420807480812073, -0.592089831829071, 0.48133522272109985, 0.07427466660737991, -0.38675516843795776, -0.6366393566131592, 0.3611830472946167, -0.2967541217803955, -0.2924352288246155, -1.208954930305481, 0.6642473936080933, 1.4104235172271729, -0.164035364985466, -0.7935827374458313, -0.6751241087913513, 0.37353062629699707, 0.8284645080566406, -1.6051692962646484, -0.6906665563583374, -0.10080177336931229, -1.0611268281936646, -0.5021436810493469, -0.18938608467578888, 0.4144585132598877, 0.06260903179645538, -0.9630432724952698, -0.26558157801628113, -0.007043708581477404, -1.141866683959961, 0.8053169846534729, -1.3626415729522705, 0.7525354623794556, 0.4739697575569153, -0.5876343846321106, -0.47815316915512085, -0.4446549415588379, 0.11541657149791718, 1.1511443853378296, 0.5428881049156189, 0.006986815482378006, -0.6093108654022217, 0.0388963408768177, -1.8774083852767944, 0.5178014636039734, 0.39349818229675293, -0.5906811356544495, 0.9519823789596558, -1.5583871603012085, 1.4447271823883057, -0.40866318345069885, -1.1785005331039429, 0.9791934490203857, -0.2525927424430847, -0.6874591112136841, -2.1751151084899902, -1.699851393699646, 0.7310526371002197, 0.6522362232208252, 0.595161497592926, -0.7165746688842773, -0.06268226355314255, -0.4849282205104828, -1.7137038707733154, -1.0557864904403687, 0.21328367292881012, 0.5623390078544617, 1.0150091648101807, -1.003265142440796, 0.22461125254631042, 1.0793744325637817, -1.0963815450668335, -0.2878023087978363, 1.1190415620803833, -1.4392452239990234, 1.2154651880264282, -1.3372265100479126, -0.43091505765914917, -0.32317498326301575, 0.42442449927330017, 0.4801289737224579, -0.5497750639915466, -0.07923627644777298, -0.4181359112262726, 0.8339585065841675, -1.550376296043396, 1.164727807044983, 0.792188286781311, -0.3348316252231598, -1.7775132656097412, -2.3045780658721924, 1.5370901823043823, -0.2195408195257187, -0.8829357624053955, -0.9092453122138977, -0.6243921518325806, 0.35595110058784485, 0.6238660216331482, -1.1277084350585938, 0.12057676166296005, -0.20176629722118378, 0.7274878025054932, 0.40799808502197266, 0.8550272583961487, 1.4021376371383667, -2.0502142906188965, -0.2542833089828491, -0.807727038860321, 1.426021933555603, -0.24424783885478973, -0.2572716474533081, -0.5787192583084106, -1.3627489805221558, 0.7956148982048035, 0.7401366829872131, 0.35126665234565735, 2.143183708190918, 0.44848188757896423, 0.4959658086299896, -0.4325316250324249, -1.0576626062393188, -0.5475504398345947, 0.06463874131441116, 0.9458070993423462, -0.05286667123436928, -0.9042468070983887, 1.181776523590088, -0.8383952975273132, 0.5289129614830017, -0.8826834559440613, -0.2793003022670746, 2.3524467945098877, -0.7073642611503601, -0.8849723935127258, 2.0830650329589844, -1.0375486612319946, -1.3012349605560303, 1.1282254457473755, 0.07924214005470276, 0.8254382014274597, -0.0460054986178875, 0.9613374471664429, -1.0238665342330933, 1.5971975326538086, 2.2501771450042725, 0.7759760618209839, 0.9553797245025635, -1.8030364513397217, 1.2615243196487427, -1.4123482704162598, 0.5644890069961548, -2.6565847396850586, 0.13274012506008148, -1.1764209270477295, -1.2384016513824463, -0.25810757279396057, 0.8097429275512695, -1.0583659410476685, -0.2815265357494354, 0.8212922215461731, -0.7124578952789307, -0.3406522274017334, 0.7651623487472534, -0.4166571795940399, -2.0642271041870117, 1.289427399635315, 0.8593220710754395, 0.2997967600822449, 0.09445846080780029, -1.0089432001113892, 0.5425953269004822, 1.1087048053741455, 0.034202516078948975, -0.8422409296035767, -2.054069995880127, 1.003441333770752, 0.0007302626618184149, -1.3976235389709473, -0.2634804844856262, -0.6584084630012512, -0.4402160048484802, 0.0870451033115387, 0.13882242143154144, -1.169045090675354, -0.04532315954566002, 2.044170618057251, 0.056423384696245193, 0.9568349123001099, -1.051576852798462, -0.040767569094896317, -0.3312225937843323, -3.3706440925598145, -0.8752090930938721, 0.6511475443840027, -0.47424817085266113, -1.1315507888793945, 0.9435864090919495, 0.00555283622816205, 0.6006433367729187, -0.2792963981628418, 0.8348084688186646, -0.5414187908172607, -0.5468675494194031, 0.8579673767089844, -0.14214108884334564, 0.6038210988044739, -0.6345512270927429, -0.8905003070831299, 0.9719308018684387, -0.6988771557807922, -1.5074793100357056, -2.339002847671509, 2.0887980461120605, -1.4592481851577759, -0.33854636549949646, 0.3737609088420868, -1.302162528038025, -0.11797428131103516, 1.2573575973510742, 0.41492822766304016, 1.206165075302124, 1.6644556522369385, 0.7019827365875244, 0.4367232024669647, -1.1934162378311157, -0.9886393547058105, 0.17824052274227142, -0.2817153036594391, -0.9825402498245239, 0.17440807819366455, -0.09926729649305344, -1.2350350618362427, -0.6899531483650208, 0.7903713583946228, 0.18380308151245117, -0.07239194214344025, -0.04602089524269104, -0.41393619775772095, -0.8565402626991272, -0.04408755525946617, 0.4949703812599182, -0.08467037230730057, -0.9058724641799927, -0.48454591631889343, -1.5305815935134888, 1.032193899154663, 1.2893288135528564, -0.16434688866138458, -0.40248146653175354, -0.21053242683410645, -1.51457941532135, -0.41419899463653564, 0.5237937569618225, 1.4290024042129517, 0.09596778452396393, 2.3215231895446777, 0.22763389348983765, -0.5698842406272888, -1.6240724325180054, -0.008536809124052525, 1.642485499382019, -0.09942902624607086, -1.1062037944793701, -0.8335996866226196, 0.1496364325284958, -0.6620279550552368, 0.649359941482544, 0.5095447301864624, 0.19188204407691956, -0.04304378479719162, -0.2796013057231903, 0.3483566641807556, 1.543349266052246, -0.9494850039482117, -1.2045162916183472, 0.029842708259820938, 0.6029930710792542, -1.1313241720199585, 0.3056699335575104, 0.08613158762454987, -0.448988676071167, -0.8086671829223633, -0.18214944005012512, 1.6412609815597534, -0.04556318372488022, -0.4622519910335541, -0.5687558054924011, 1.147261142730713, 1.0790245532989502, -0.5010548830032349, 1.8205562829971313, 0.48675546050071716, 0.41165778040885925, 0.690643310546875, 1.390735387802124, 0.33300724625587463, 0.34309548139572144, -0.8766341209411621, 0.19634796679019928, -1.864466905593872, -0.11954814195632935, -1.8908230066299438, -0.7893198132514954, 0.2412596195936203, -0.5338147282600403, 0.1721952259540558, -0.5827680826187134, 0.7101683020591736, -0.8108978271484375, -1.2920535802841187, 1.4811644554138184, 1.146105408668518, -0.7073349952697754, -0.08573649078607559, -0.699022650718689, 0.5479503273963928, 0.7714822292327881, 0.7308144569396973, 1.167749285697937, -0.11638478934764862, 0.7204226851463318, -0.300868421792984, -1.930041790008545, 0.2596187889575958, -0.15902309119701385, -2.403447151184082, 0.4503937065601349, 0.1222793236374855, 2.000767230987549, -0.37896373867988586, 0.6352748870849609, -0.37694716453552246, 2.001396417617798, -1.213935136795044, -1.0887746810913086, 1.5973920822143555, -2.085265874862671, 0.32385367155075073, 0.2960156798362732, -2.524592876434326, -0.621134877204895, 0.5096610188484192, -0.8177466988563538, -0.8687257766723633, 0.8007709383964539, 1.3270132541656494, -0.15199677646160126, -1.883117914199829, -1.161816120147705, -0.9617284536361694, 0.31685593724250793, 1.6866676807403564, -0.9825361371040344, 1.0004980564117432, 1.8081419467926025, -0.4387561082839966, -2.3079259395599365, 1.1502560377120972, 0.04068216308951378, 1.891436219215393, 0.198141947388649, 0.1801266372203827, 1.1838173866271973, 1.6893244981765747, 0.3818497657775879, 0.10299474000930786, 0.6596903800964355, -0.3745480477809906, -2.1879730224609375, 0.540520191192627, -0.6318603754043579, -0.8039407730102539, 0.6273589134216309, -0.16255295276641846, -0.3205343782901764, 0.4986928105354309, -0.019970141351222992, -1.266474723815918, 0.6545209288597107, 0.6660580635070801, -0.8441848158836365, 1.3222907781600952, -0.6992098093032837, -0.42690688371658325, 1.5830857753753662, -0.9614672064781189, -1.0457309484481812, 0.03306883946061134, 0.7652972340583801, -1.2714627981185913, -0.04891246557235718, -0.21500219404697418, 0.4923382103443146, -0.08555103093385696, 0.4292370676994324, -0.4835466146469116, 0.2835227847099304, 0.3552927076816559, 0.2033972293138504, 0.12960462272167206, -0.14676494896411896, 0.4004899561405182, -0.04698903113603592, -1.7940151691436768, -1.2266638278961182, 1.0498054027557373, -1.018949270248413, 0.47051456570625305, -0.5422718524932861, -0.6907294392585754, 0.7695980668067932, 1.0015881061553955, 1.6797665357589722, -0.7445968389511108, -2.348137378692627, 0.08475613594055176, 0.45354363322257996, 0.4420000910758972, 1.6935807466506958, 0.8233693838119507, 1.3448097705841064, 0.2883436381816864, 1.1789708137512207, 0.84825199842453, -0.0026751956902444363, 0.0896107405424118, -0.8975548148155212, 0.013195747509598732, 1.111402988433838, 0.38344115018844604, 0.7004343867301941, -0.9334136843681335, -0.7289330363273621, -1.0334099531173706, 0.17348453402519226, -0.3596031069755554, 0.038490068167448044, -0.3685411810874939, -0.29452720284461975, 0.6357696056365967, -0.6016753911972046, -0.8189144730567932, -2.5229508876800537, 1.1095225811004639, 0.3763269782066345, -0.15821653604507446, -0.33138421177864075, 0.7915201187133789, -0.530419111251831, 0.3127193748950958, 0.1830773800611496, 0.9133973121643066, -1.074273705482483, -0.748945415019989, -0.017773272469639778, 0.734480619430542, -0.34743019938468933, 0.40455809235572815, -0.022894050925970078, -1.1122634410858154, 0.40362390875816345, 0.5908936858177185, 1.3947428464889526, 1.984978437423706, -0.9942999482154846, 1.5011100769042969, -1.5312985181808472, 0.21495036780834198, -1.0477217435836792, -1.3518624305725098, 0.8420316576957703, 1.1283527612686157, -0.8082902431488037, -0.6219173073768616, 0.19352714717388153, -1.3056747913360596, 0.3551579713821411, -0.5344306826591492, -0.8536030650138855, 0.49621447920799255, -0.8084882497787476, -1.0269283056259155, -0.4552026093006134, 0.6281968355178833, -0.5812105536460876, 0.22593097388744354, 1.4709622859954834, 2.256239414215088, -0.4552345871925354, 0.6085721254348755, -0.30357253551483154, 0.2970001995563507, 0.5864840745925903, 0.7000753283500671, 0.19638444483280182, 0.5199942588806152, 0.643220841884613, -0.2043183445930481, -2.2536838054656982, -1.9556301832199097, -0.16794909536838531, -0.2324417531490326, 0.9036152958869934, 0.030368929728865623, -0.11574672162532806, 1.5514090061187744, -0.23409467935562134, -1.204060435295105, 1.6952396631240845, 0.5257942080497742, -0.12449268996715546, -2.3662383556365967, -0.3583757281303406, -0.5664952993392944, -1.56879460811615, -0.6106319427490234, -0.9161505699157715, -0.6647101640701294, -0.46408912539482117, -1.714860439300537, -1.035355806350708, -0.9892608523368835, -0.6745150685310364, -0.7882606983184814, -0.05068790540099144, -0.21210868656635284, -0.9207887649536133, -0.4422729015350342, -0.43125399947166443, -0.49124404788017273, -0.4138886034488678, 0.19839254021644592, 0.6748449802398682, -0.2005683332681656, -0.8391098380088806, -0.3804674744606018, 0.6976993083953857, -0.36499980092048645, -1.0517035722732544, 0.1983947902917862, -0.7766110897064209, -0.8386161923408508, 0.03941541910171509, -0.015188971534371376, -0.5638420581817627, 0.9056649208068848, 0.4930787682533264, 1.3376106023788452, 0.18106485903263092, -0.41855260729789734, 1.0526845455169678, 2.2495381832122803, 0.3374257981777191, -0.27570009231567383, 0.2952759861946106, -0.31213489174842834, -0.18589164316654205, -0.5987166166305542, 1.9658135175704956, -0.09879567474126816, -0.3050881326198578, -0.9326506853103638, 0.19879809021949768, 0.38080450892448425, -1.409551739692688, 0.3174695074558258, 0.5500055551528931, -0.03026418574154377, 0.996453583240509, 1.4020112752914429, 0.12232668697834015, -0.737971842288971, 0.2717652916908264, -0.38313770294189453, -1.6034120321273804, -0.7784236669540405, -0.36607545614242554, -0.724539041519165, -0.17913274466991425, 0.3157181441783905, 0.31035086512565613, -0.6202482581138611, 1.392800211906433, -1.1004705429077148, 0.23352433741092682, -0.036756400018930435, 1.3508799076080322, 0.9655044674873352, -0.1096440926194191, 0.7720638513565063, -1.4151952266693115, 0.2545595169067383, -0.3272836208343506, -1.389262318611145, 0.9840672016143799, -0.8351442813873291, -0.2667345702648163, -0.26013636589050293, -0.5456348061561584, -0.8020915389060974, 0.11989854276180267, -0.2959987223148346, 0.9837164282798767, -1.76088547706604, -0.7838977575302124, -0.4009588360786438, -0.1374417096376419, -1.0063544511795044, -1.536240577697754, 1.3449389934539795, 0.0865791067481041, -1.0933152437210083, -0.1773642599582672, -1.1598913669586182, -0.05900944396853447, 0.6178770065307617, 0.5084743499755859, -1.2010812759399414, -1.0076148509979248, 1.584290623664856, 1.4110050201416016, 0.6647961735725403, 0.3527330756187439, 1.2372238636016846, 0.46030548214912415, 0.07448186725378036, 0.826980471611023, -0.4937475919723511, 0.7433974742889404, 0.2967921793460846, -0.8494513630867004, 1.154618501663208, -1.8842865228652954, -1.4892077445983887, -1.142106056213379, 1.0988048315048218, -0.2952142655849457, 1.103578805923462, -0.4428260326385498, 1.0583337545394897, -1.239762783050537, -0.972652792930603, -0.0011292269919067621, 0.8173965215682983, -1.618517279624939, 0.5232242941856384, 0.35054731369018555, 0.3314400315284729, 0.08229909837245941, 0.699152946472168, 1.0798852443695068, 0.16596543788909912, 1.282223105430603, -0.9105972647666931, -0.2804933786392212, 0.5021724104881287, 0.8916985988616943, 0.40195098519325256, 0.2624324858188629, 0.991226315498352, -0.023266557604074478, 0.11275535821914673, 0.5657612681388855, -0.7624721527099609, 2.380733013153076, -0.7085229158401489, -0.24050867557525635, -0.3762762248516083, 0.7379671335220337, 1.0336602926254272, -0.3038429617881775, 0.1559811234474182, 0.7385093569755554, -0.12508495151996613, 0.39275068044662476, 0.9912294149398804, -0.2783101499080658, 0.30205637216567993, -0.5137941241264343, -0.08882050961256027, -0.9237061142921448, 2.5459353923797607, -0.37679800391197205, 0.8894781470298767, 0.3713776767253876, 0.9482027292251587, 0.8089902400970459, -0.5121176838874817, 2.3616743087768555, 0.7850537896156311, -0.8470922112464905, -2.364751100540161, -0.6348307728767395, 0.9362419843673706, 1.1479213237762451, -0.962742805480957, -0.039415404200553894, -0.86965411901474, 1.6071234941482544, 0.9330528974533081, -1.551082968711853, -0.5901818871498108, 0.027516653761267662, 0.023154618218541145, 1.8334293365478516, 1.077549934387207, 0.25645679235458374, -0.6449037790298462, 1.0464587211608887, 0.5607965588569641, -0.00026826257817447186, 1.1668325662612915, -0.13584116101264954, -1.8935827016830444, -2.580007791519165, 0.4688872694969177, -0.6677538156509399, 0.45301032066345215, 2.0990874767303467, 0.3243353068828583, 1.0288050174713135, -0.4386255741119385, -0.36298638582229614, -0.9940611720085144, 0.5449784398078918, 1.3387629985809326, -0.28652817010879517, 0.8059818148612976, 1.00579833984375, -0.842161238193512, 1.406382441520691, -1.3110765218734741, 0.825957715511322, -0.7442483901977539, -1.0679020881652832, -1.2964595556259155, 0.620129406452179, -0.28901341557502747, 0.2519862651824951, 1.319061040878296, 0.8812025189399719, 1.4859631061553955, 0.7353339195251465, -0.021077726036310196, -0.8207196593284607, 0.22596469521522522, -0.07985159009695053, 0.5187405347824097, 0.31147390604019165, 0.13383758068084717, 1.3946384191513062, -0.047562066465616226, 0.23459617793560028, -0.190359428524971, 0.9635285139083862, 0.23662960529327393, -1.8360041379928589, -0.09346985816955566, 1.1364628076553345, -1.8958797454833984, 0.3477385938167572, 0.7782436609268188, 1.7301467657089233, -1.791549801826477, -0.2553662955760956, -0.03371363505721092, 1.5648229122161865, -0.2237159162759781, -0.9118062853813171, 0.6310233473777771, -0.33736535906791687, -0.12554098665714264, 1.730122685432434, 0.49460819363594055, 1.6274685859680176, -0.44324225187301636, 1.260831356048584, 2.0826847553253174, 2.678398609161377, -0.3202095627784729, 0.2971775531768799, -0.029760226607322693, -0.030751371756196022, -0.4556489884853363, -0.8253191113471985, -0.41709083318710327, 0.2629142105579376, 0.2465386539697647, -0.35687556862831116, -0.11079058796167374, 0.0230332612991333, 0.7026028037071228, 0.4845712184906006, -0.6686608195304871, -1.0518962144851685, 0.7223962545394897, -0.5985541343688965, 0.3240824043750763, 0.6784050464630127, -0.5129678249359131, 0.2055765688419342, -1.3874826431274414, -0.8934003114700317, 0.11053714901208878, -1.3233203887939453, 0.8673121333122253, -0.3903055191040039, -0.45001187920570374, 0.7045591473579407, 0.08474364876747131, -1.2098830938339233, -1.003766417503357, 0.10946379601955414, 0.6457671523094177, 0.1642368584871292, 0.22541989386081696, -0.7077717781066895, -0.8270480632781982, 0.627572238445282, -0.40422487258911133, -1.8795620203018188, -0.5258380770683289, -0.599193811416626, -1.4840590953826904, 0.09664104878902435, -0.7933971881866455, -0.39564377069473267, 0.00730115408077836, 1.8181487321853638, 1.6459323167800903, -0.37951335310935974, -0.4075090289115906, 0.28655511140823364, -0.5874919295310974, 0.216652974486351, -0.544041633605957, 0.09908242523670197, 0.08652670681476593, -0.736091673374176, 0.3292900323867798, -1.0000190734863281, -0.783225417137146, -0.31208932399749756, -0.18807080388069153, -1.0140163898468018, -0.3791823983192444, -0.14608339965343475, -0.5218775272369385, -0.8130258321762085, 0.9863100051879883, -0.2402283251285553, 0.8235068917274475, 0.251132071018219, 1.6752265691757202, 0.11501971632242203, 0.0367119126021862, 1.4197564125061035, -1.1664972305297852, 0.9718436598777771, 0.09159830212593079, 0.31625697016716003, -0.3477042019367218, 0.1075083389878273, 0.13637909293174744, 2.2593305110931396, -1.4053305387496948, 0.9516546726226807, 0.7170209288597107, 0.2676410377025604, -0.9528734087944031, -0.4627534747123718, -1.1917623281478882, -0.5263386368751526, -0.42701712250709534, 1.698553204536438, -1.0545313358306885, -0.6310056447982788, -0.6706433296203613, -0.3015913665294647, 0.13048535585403442, -0.02947266586124897, 0.8699570298194885, 0.15344394743442535, -0.09466876834630966, 0.6143743991851807, 1.706146478652954, 1.8261529207229614, -0.09506063908338547, 0.49028801918029785, -1.5729745626449585, 0.9705231189727783, 2.6063477993011475, -1.0038255453109741, -1.3771125078201294, -0.17131739854812622, -1.1402827501296997, 0.5642721056938171, 1.732630729675293, -0.6509910225868225, 0.7550092339515686, 0.6599875688552856, 0.8342055678367615, 1.7414242029190063, 0.448903352022171, 0.10162607580423355, 0.6495723724365234, -0.7786716222763062, -0.5014814138412476, 0.780328631401062, -0.239231139421463, -0.18949627876281738, -0.5643760561943054, 0.3468645513057709, -1.4094265699386597, 0.974380612373352, 0.8178674578666687, -2.449683904647827, 1.2864965200424194, 0.7610278129577637, -1.1801930665969849, -0.9617146253585815, -0.8995698094367981, -1.0164711475372314, 0.2959979176521301, -0.34252047538757324, -1.1413648128509521, 1.1138279438018799, 0.3835025727748871, 0.984981119632721, -0.2772909700870514, 2.462549924850464, 0.4300946593284607, 0.24460238218307495, -0.09655322134494781, 0.017630839720368385, 0.08261680603027344, -0.9294241666793823, 0.326643705368042, 0.4760412573814392, -0.47463375329971313, -0.5417779088020325, 1.4865798950195312, 0.632842481136322, -0.12122132629156113, -0.6665050983428955, 0.020901555195450783, -1.4194059371948242, -0.4956294298171997, -0.5322160720825195, -0.3224068582057953, 0.0559304915368557, 0.8465984463691711, 0.8815299272537231, -0.050009585916996, 0.33395373821258545, -0.23861755430698395, -0.6939102411270142, -0.42626821994781494, 0.2770093083381653, 0.6100378036499023, 1.3860018253326416, 2.0543222427368164, -0.1756111979484558, -2.087207794189453, 0.48795127868652344, -0.23126699030399323, 0.6789728999137878, -1.3517944812774658, 0.9344682097434998, 0.8478837609291077, -1.5359385013580322, 0.3735557198524475, -0.8856093883514404, -1.7834336757659912, -0.4105905592441559, -1.747575283050537, 1.907209873199463, -0.23513568937778473, 2.33963680267334, 0.8869215846061707, 0.36639028787612915, -0.8645871877670288, -0.9247777462005615, 1.7577570676803589, -0.6491032242774963, -1.009714126586914, 1.1095141172409058, 0.605464518070221, 0.7547919154167175, 0.6938614249229431, -0.09053441137075424, 1.4729688167572021, -0.3808172643184662, -0.7896063923835754, -1.0588511228561401, -0.5576599836349487, -0.9577577114105225, 1.5729540586471558, -1.2388079166412354, 1.514007806777954, -2.48337984085083, 2.1730294227600098, -0.9050953984260559, 1.3957583904266357, 0.7982031106948853, 1.6217865943908691, -1.5116426944732666, 1.0914167165756226, 1.6449432373046875, 2.2364556789398193, 0.39073219895362854, 0.6094242930412292, -0.7434514164924622, -0.16521118581295013, -0.6128568649291992, -0.4585379958152771, -0.6702803373336792, -1.7753936052322388, 0.05957714468240738, -1.2503365278244019, -1.3642022609710693, -0.35413721203804016, -1.1792843341827393, -2.3065290451049805, 0.717054545879364, 0.34522178769111633, -0.08120159804821014, 0.6204944252967834, -1.6790118217468262, -0.38752636313438416, 0.7871190309524536, -0.368384450674057, 1.6566780805587769, -0.6037507653236389, 0.02385476790368557, 0.32653146982192993, -0.1012623980641365, -0.6077359318733215, 1.8776172399520874, 1.263759970664978, 0.7459642887115479, -0.4289526343345642, -0.5631564855575562, -0.09439632296562195, 0.9914104342460632, 1.557178258895874, 0.945875346660614, 0.6548940539360046, -0.20104363560676575, -0.6422181725502014, -2.5511558055877686, -0.9770264029502869, 0.3429624140262604, -1.3652485609054565, -0.19407418370246887, -0.7908037304878235, -1.1322535276412964, -0.2663041055202484, 0.7243740558624268, 2.5007095336914062, 1.1242731809616089, 1.1041183471679688, -2.2905263900756836, -0.8621494770050049, 0.6281895041465759, 0.6938243508338928, -0.6435743570327759, -1.1777383089065552, 0.6918277740478516, 0.3550325930118561, -0.8091123700141907, 1.0735565423965454, -0.29800039529800415, -0.22417010366916656, 0.6006070971488953, 0.7810566425323486, -0.487912118434906, -1.3381354808807373, -0.7259361147880554, 2.9850497245788574, 0.41565966606140137, 0.41867610812187195, -0.45146846771240234, -1.177765130996704, -1.030153751373291, -0.025644483044743538, -1.2263274192810059, 0.9699175953865051, -0.40402406454086304, 1.2915266752243042, 0.9866968393325806, 0.5770012736320496, 1.1902776956558228, 1.4423089027404785, -0.6410146355628967, -0.3779691457748413, 0.09188662469387054, -0.8498510122299194, -1.6666371822357178, 0.4146307706832886, 1.3748698234558105, 2.591993570327759, 0.31953033804893494, 0.0952993780374527, 1.6778147220611572, 0.5250644683837891, 0.9628657102584839, -0.19224506616592407, 0.620153546333313, -1.0907537937164307, -2.0443668365478516, -0.49533602595329285, 0.9554140567779541, -0.4043884575366974, 0.7281844019889832, 0.7755547761917114, -1.778617024421692, -0.39292386174201965, -0.2856089472770691, -0.2979533076286316, 1.1497564315795898, 0.07989229261875153, 1.6972297430038452, -0.9051415920257568, 0.36300110816955566, -0.9235963225364685, -0.12009217590093613, -2.7468512058258057, -0.593917965888977, -0.09972752630710602, -2.2607200145721436, -0.2558687627315521, -0.39891353249549866, 0.4698750674724579, 0.9449451565742493, -0.3626421391963959, 0.21717360615730286, 0.30503708124160767, 0.8533610105514526, 0.8276460766792297, 0.8628711700439453, -0.20989342033863068, -0.298927366733551, -1.9065937995910645, -0.2810179889202118, 0.28235018253326416, -2.5678980350494385, -0.3221533000469208, -0.23919221758842468, 0.24347075819969177, -1.2170155048370361, 1.6016104221343994, 0.2618529498577118, 0.9014810919761658, -0.2810802161693573, 0.8024705052375793, 0.41940999031066895, 0.2620186507701874, -0.7860487103462219, 1.2876750230789185, -1.1320816278457642, 1.373694658279419, -0.0635194182395935, 1.3537254333496094, 0.21881003677845, 0.012677870690822601, 0.576240062713623, -0.1626758873462677, 0.040847647935152054, -0.35335302352905273, 0.5551767945289612, 0.6767261624336243, -1.0009279251098633, -0.9654373526573181, 1.2783793210983276, 0.6300861239433289, -1.2389720678329468, -1.5193040370941162, -1.0382136106491089, 1.3661432266235352, -0.7982763648033142, 0.691537618637085, 0.048776645213365555, 0.23324283957481384, -0.08140505850315094, -0.7469168305397034, -0.33888915181159973, 1.0707809925079346, 0.038837820291519165, -0.6015764474868774, 0.39098867774009705, 1.6087135076522827, -0.6537609100341797, -0.892469048500061, 0.24180543422698975, 0.9696805477142334, 0.022410167381167412, -1.4158530235290527, 1.1709330081939697, -0.5497398972511292, -1.687612533569336, -1.6939270496368408, -0.8541639447212219, -1.0412029027938843, 0.18901769816875458, -0.2731765806674957, -1.4262245893478394, -0.21617186069488525, -0.6763061881065369, -0.6728712320327759, 0.08409256488084793, -2.2465195655822754, 2.6253814697265625, 1.2442671060562134, -1.1362780332565308, 0.17726659774780273, -0.5708593130111694, 1.6065555810928345, 0.0790150985121727, 0.1784621924161911, -1.0344007015228271, -2.105135679244995, 0.8784369230270386, -0.7940391898155212, 0.8497642278671265, 0.5133011937141418, -0.25358280539512634, -0.5065116882324219, 0.39168888330459595, 0.9744012355804443, 1.1366804838180542, 0.6182512044906616, -1.1061888933181763, -0.5066353678703308, 0.511640191078186, 1.6538952589035034, -0.09582440555095673, 1.1146434545516968, 0.22625422477722168, 0.3408249020576477, 0.734650731086731, 1.0709306001663208, 0.3196832835674286, 0.37767985463142395, -2.5750765800476074, -0.45376238226890564, -1.3099385499954224, 0.30198368430137634, 0.3611913025379181, 0.20518150925636292, -1.170332431793213, -0.06481098383665085, -1.784202218055725, 0.18264947831630707, -0.9721345901489258, -0.2595013976097107, 0.830777108669281, 0.40111014246940613, -2.588592290878296, -0.9094270467758179, -0.42611002922058105, -1.5825037956237793, 0.37424537539482117, 0.6200582981109619, -0.20179370045661926, 0.3724942207336426, 0.42900991439819336, 0.7394924163818359, -0.5943939685821533, -0.010801422409713268, -0.5847877860069275, 0.5059937238693237, -0.8894568085670471, 1.1356327533721924, -0.16256055235862732, -1.5459493398666382, -1.1504061222076416, 0.0512266531586647, 0.5900366306304932, 0.19554272294044495, 0.12860769033432007, 0.06337031722068787, -0.45982423424720764, -0.4903903603553772, -2.0026185512542725, 0.2424612194299698, -0.1677514761686325, 0.7822744250297546, 0.3761190176010132, 0.7821599245071411, 0.46241119503974915, 0.4047712981700897, 0.7537029981613159, -0.9801200032234192, -0.41845691204071045, 0.6086838245391846, 0.09709425270557404, -0.3771454989910126, -2.3615198135375977, 1.3471391201019287, -1.0661574602127075, 2.337956666946411, -0.30423837900161743, 1.438673734664917, 1.113863468170166, -0.40559422969818115, 0.47758960723876953, -0.6490942239761353, 2.279595136642456, -0.12704923748970032, -0.8120216131210327, -0.4043349325656891, -0.005305512808263302, -1.028422474861145, -0.14916878938674927, 0.211939737200737, 0.5381478071212769, -0.963530421257019, -0.006845003925263882, -1.283809781074524, -0.957659125328064, 0.35136520862579346, -1.5580403804779053, 0.9772464036941528, 0.4989669620990753, -0.6442904472351074, 0.10987263172864914, -2.053029775619507, -0.10841336846351624, 0.8846830725669861, -0.23898598551750183, 0.41277167201042175, -0.9190600514411926, 0.6066358685493469, -0.6239583492279053, 1.5530818700790405, -0.454154372215271, -0.7898033261299133, 1.4783403873443604, -0.1737234741449356, -1.3100372552871704, -0.6394885778427124, -0.10372480005025864, 1.8515772819519043, -1.1173545122146606, 0.5212246775627136, 0.639325737953186, 0.05387486517429352, -0.20824189484119415, -0.1466509848833084, -1.2698241472244263, 0.18192365765571594, -0.11623859405517578, 0.4972311556339264, -0.0013146017445251346, -0.8500986695289612, 1.2823941707611084, -0.2261253446340561, 0.040577519685029984, -2.2069125175476074, -1.1815882921218872, 0.5302966833114624, -0.018498923629522324, -0.7192723155021667, -0.045564424246549606, -1.5351042747497559, 0.8772040009498596, 0.707434892654419, -0.1163322702050209, 0.22682617604732513, 1.1623419523239136, -0.08424434065818787, -2.0439889430999756, 0.1551906168460846, 0.17013615369796753, -0.06207510828971863, -1.0457324981689453, 1.160510540008545, -0.773111879825592, -0.05729770287871361, -0.3475486934185028, -0.5420067310333252, 1.5696011781692505, -1.3304921388626099, 1.6220242977142334, -1.3599821329116821, -0.6646778583526611, 0.3284544050693512, 0.7637644410133362, 0.8361359238624573, -0.5519499182701111, -1.0650988817214966, 0.5999447107315063, 1.5191394090652466, 0.0358002670109272, 0.19334174692630768, 0.5809622406959534, 0.9293459057807922, 0.36125630140304565, 1.1595447063446045, 0.5587234497070312, -0.6382521986961365, 0.5301445126533508, -0.9487946033477783, 0.003896453185006976, 0.8446880578994751, 0.05540119484066963, 1.2982916831970215, -0.9212205410003662, 0.7105875611305237, 1.125907063484192, 0.7019826769828796, -0.6048434376716614, 0.1702806055545807, 1.3773928880691528, -1.9139479398727417, -0.6759134531021118, 0.2894590198993683, 0.36965182423591614, 1.1263689994812012, 1.1753082275390625, 0.2732311189174652, -0.5908045172691345, 0.19904811680316925, -0.1885768622159958, 0.12334183603525162, 0.6119657754898071, -0.26864808797836304, -1.1636240482330322, -0.46633121371269226, 0.513896644115448, 0.16085754334926605, -0.8390610814094543, 0.7942498922348022, -0.030124280601739883, 0.5208727717399597, 0.7499967217445374, -0.1624928116798401, 0.06461094319820404, -0.6943868398666382, -0.4808664917945862, -0.3729200065135956, -0.2780965566635132, 1.9723575115203857, -1.357970952987671, 0.37987715005874634, 1.0453540086746216, 0.8101243376731873, 0.05973716825246811, 0.11971917748451233, -0.36197376251220703, -0.18192198872566223, 0.1738058626651764, 0.05362195149064064, -0.27219438552856445, 0.978240430355072, 2.2087442874908447, -0.23541127145290375, 0.9101328253746033, 0.1652502417564392, -0.149179145693779, -1.4442499876022339, -1.2348629236221313, -0.6074492335319519, -0.258543998003006, -1.1660813093185425, 0.32130730152130127, -0.4343752861022949, -1.2574305534362793, 0.5377230048179626, 0.4963268041610718, 0.10679629445075989, 0.8889887928962708, -1.3872783184051514, -0.18180833756923676, 0.41603967547416687, 0.12481086701154709, -1.1895251274108887, 0.5461125373840332, 0.9279362559318542, 0.8276479244232178, 0.09550829231739044, -0.06874991953372955, -2.1098031997680664, 1.4287843704223633, -0.5763536095619202, 0.15426941215991974, -0.7192510962486267, 0.9259252548217773, 2.1008760929107666, 1.1530840396881104, 0.8592627048492432, -0.22993704676628113, -0.04662803187966347, -0.020453302189707756, 1.4402047395706177, -0.5542828440666199, 0.8077365159988403, 0.23295271396636963, -0.18052737414836884, 1.7641551494598389, 1.109951138496399, 0.6350437998771667, 1.0038028955459595, 0.9167075157165527, 0.7151287794113159, 0.13044095039367676, -0.17222057282924652, 0.8215352296829224, 2.8762543201446533, 0.33492064476013184, -2.9003827571868896, 0.0792057067155838, 2.232707977294922, 0.6110210418701172, -1.3836616277694702, 1.2080528736114502, 0.18138588964939117, 0.468360960483551, -0.2845299541950226, 0.2154349535703659, -1.0904829502105713, -2.091488838195801, 0.6428307890892029, -2.214590311050415, -1.161598563194275, -0.012333330698311329, 0.3421030640602112, -0.2247380018234253, 1.615769624710083, -0.5840451717376709, 0.2600319981575012, -1.634495496749878, -0.7319014072418213, -0.3760378956794739, 1.7621136903762817, -1.0538485050201416, 0.5057330131530762, 1.0824390649795532, 0.6794155240058899, 1.6773113012313843, -0.24859219789505005, 0.42964890599250793, 0.3518729507923126, 0.5789531469345093, -0.12353784590959549, 1.0666744709014893, 1.2949026823043823, 0.04104836657643318, 0.021374564617872238, 0.4370093047618866, 1.9940661191940308, -0.36987441778182983, -0.1671266108751297, -1.0909420251846313, 0.39531028270721436, -0.16321700811386108, -0.7550451159477234, -0.2357480674982071, -1.0696717500686646, 1.1366115808486938, 1.1797878742218018, 0.5012452602386475, 0.5395339727401733, -0.0467376746237278, 0.49820569157600403, -0.8608074188232422, -1.2307751178741455, 1.2087100744247437, -0.33162927627563477, -0.0017972745699808002, 0.9652242660522461, 0.42207804322242737, -1.7535594701766968, 1.6203919649124146, -0.7121985554695129, -1.7562893629074097, 0.9167816638946533, -0.5291470289230347, 0.3251613676548004, 0.048267439007759094, 0.030981536954641342, 0.1757044643163681, -0.4962799847126007, 0.5017970204353333, -1.9945380687713623, 0.41057226061820984, -0.048889242112636566, -0.1978432983160019, 0.26684045791625977, 0.5629154443740845, 1.5109754800796509, -1.8345280885696411, 0.6097601056098938, -2.7289764881134033, -0.19586274027824402, 1.1168022155761719, 1.1629365682601929, -1.3232293128967285, 1.2718017101287842, 0.08473532646894455, -1.1074326038360596, 0.9205760359764099, -0.4183105230331421, -1.3976224660873413, -0.024021025747060776, 0.8772097826004028, 0.6332495808601379, 0.336884468793869, -0.3509323000907898, 0.5508706569671631, -0.5783323645591736, -0.6544725298881531, -1.199007511138916, 0.032547932118177414, 1.6912760734558105, 0.766798734664917, 0.021908622235059738, 0.10634259879589081, -0.5609952807426453, -0.446108341217041, -0.4826025366783142, 0.9231647849082947, 0.07291330397129059, 0.6424732208251953, -0.06890633702278137, 0.9120935201644897, 1.347144365310669, 0.01353311538696289, 0.23441967368125916, 0.8676819205284119, 0.3137088716030121, 2.106426477432251, -2.6062076091766357, 1.246097445487976, 0.28291815519332886, -1.186060905456543, 1.667271614074707, 0.9679415225982666, 1.0742666721343994, 1.6821608543395996, 0.6524534821510315, -0.3451652526855469, -0.4525406062602997, 0.07647894322872162, -0.8852904438972473, 1.8964006900787354, 0.6793094277381897, 1.4546139240264893, 1.8903520107269287, 0.5432411432266235, 0.15435181558132172, -1.096397042274475, -0.000740916992072016, -0.2976172864437103, 1.8273009061813354, 1.354691743850708, 0.10596407204866409, 0.042441852390766144, -0.7037440538406372, 2.6526379585266113, 0.5070152282714844, 0.4274864196777344, -0.7383350729942322, 1.107007622718811, -0.07285904884338379, 0.7340807914733887, 0.44512733817100525, 1.3709758520126343, 0.8944787979125977, 0.5376700758934021, -1.472637414932251, 1.2134486436843872, -0.34767863154411316, -0.007095058914273977, 0.9664795994758606, -0.11894220858812332, 0.4700563848018646, 0.7136383652687073, -1.3801831007003784, -1.4620764255523682, 0.5106008648872375, 0.01675698347389698, -2.1840736865997314, -0.3256649971008301, 0.5388071537017822, -0.00735534867271781, 1.2053312063217163, -1.1542103290557861, -0.16698895394802094, 1.161040186882019, 0.545434832572937, 0.7538142800331116, -1.6392613649368286, -1.0487200021743774, 1.1477713584899902, -0.5742090344429016, -0.4403833746910095, 0.9524579644203186, 0.4711403250694275, -0.8943789005279541, -0.03074302338063717, 0.21746434271335602, 0.2807435691356659, -1.8707777261734009, -0.10873375833034515, 0.5785123705863953, 1.3309717178344727, 1.3694031238555908, 0.628219723701477, 0.6322956085205078, -0.18357744812965393, 0.1766284555196762, 0.788745641708374, -1.5528851747512817, -0.7722246050834656, 0.8997703790664673, -1.5639619827270508, 1.1093806028366089, -0.2649976313114166, -0.49821847677230835, 1.2110047340393066, 0.6609592437744141, 2.04343318939209, -0.8384518623352051, -1.4698337316513062, 0.13827002048492432, -0.5412341356277466, -0.4425794184207916, 0.8567937016487122, -0.220255509018898, 0.814369261264801, -0.6546012163162231, 0.13051408529281616, 3.5281427699374035e-05, 0.508707582950592, -0.6672084927558899, -0.906388521194458, -0.5085992813110352, 0.08841812610626221, -0.48373815417289734, 0.1664040982723236, 0.7280212044715881, -1.490114688873291, -0.7194721102714539, -0.4923102855682373, -1.8177393674850464, -0.24344636499881744, 0.1305835098028183, 0.9138498306274414, 0.3683759570121765, 0.5413923859596252, -1.0987871885299683, -0.7304894328117371, 0.3365495204925537, 1.6783167123794556, 0.37519896030426025, -0.34693774580955505, 1.3871721029281616, 2.1356918811798096, -0.9773813486099243, -1.0088337659835815, 0.606472373008728, -1.4000210762023926, 1.0125778913497925, -0.2554033100605011, 0.006856359075754881, 0.15674111247062683, -1.0378074645996094, 0.3090120851993561, -0.4706672728061676, -0.9214503169059753, -0.4328960180282593, -0.5668811798095703, -0.2606147527694702, -1.1112197637557983, 0.8820363283157349, -1.5467990636825562, 1.3787034749984741, -0.16821196675300598, -1.6187437772750854, -1.0938334465026855, -0.32316887378692627, -1.1615701913833618, 0.747880756855011, -1.3265604972839355, -1.7820850610733032, -0.22640520334243774, -0.45055249333381653, 0.1933651864528656, 2.13272762298584, -1.9225590229034424, -0.5022999048233032, -0.2748993933200836, 1.4475483894348145, -1.7601443529129028, 0.9686664342880249, 1.8359227180480957, -0.4834970533847809, -0.26101619005203247, 0.09392987936735153, -0.2788119316101074, -1.0456544160842896, 0.8241460919380188, -0.49341875314712524, -1.5192606449127197, 0.6327123045921326, 1.4471259117126465, -1.6644703149795532, -2.0513339042663574, 0.17226450145244598, -0.4195229411125183, -1.4955146312713623, 1.887277603149414, -0.8822612166404724, 0.6810815930366516, 2.3747291564941406, -0.10264796763658524, 0.8095504641532898, 1.6991639137268066, -1.4615989923477173, 0.34145089983940125, 0.207522913813591, -0.02867356687784195, 1.3572895526885986, 0.1685812771320343, -0.04997309297323227, 0.376023530960083, 0.7349697351455688, -0.2809814214706421, 0.8234118223190308, -0.9660901427268982, -0.4448789954185486, -0.8657909035682678, 0.08926130831241608, 0.5441761016845703, -0.36767399311065674, -0.21783432364463806, -0.15591752529144287, 0.09734217077493668, -1.2978923320770264, 0.22948838770389557, 0.7467263340950012, 0.03575510159134865, -1.6283574104309082, 0.45410454273223877, -0.7652865052223206, -0.21366845071315765, -0.6335320472717285, 0.24991518259048462, -2.355384349822998, -0.3801969885826111, -1.4042705297470093, 1.6897605657577515, -0.13939325511455536, -1.8028175830841064, 0.9958759546279907, -0.5226625204086304, -1.0967298746109009, 1.112072467803955, 0.02564360946416855, 0.3676522374153137, -1.4327502250671387, 0.6014537811279297, 0.8942683935165405, 0.886126697063446, -0.9769355654716492, 0.8578854203224182, 2.490065336227417, 0.02363070845603943, 1.2424628734588623, 1.5149478912353516, 0.5098373293876648, -0.00739460252225399, 1.1325575113296509, -0.1765950322151184, 0.8067454099655151, -0.4523347020149231, 1.0596096515655518, -0.24466636776924133, -0.08378314971923828, -0.2715797424316406, -0.7708179354667664, 0.49973341822624207, 0.13334055244922638, -0.7399204969406128, 0.3377031981945038, 1.0703364610671997, 0.32980626821517944, -0.41267862915992737, 2.251044988632202, -1.0597037076950073, 0.7187903523445129, -0.6216340661048889, 1.1035068035125732, -1.321391224861145, 2.665062189102173, -1.039333701133728, 1.2890903949737549, -0.37770238518714905, -0.03676449880003929, -0.1741909384727478, -0.6815499067306519, 0.9997824430465698, 3.6729016304016113, -0.137417271733284, -1.0151220560073853, -2.1097519397735596, 0.3914929926395416, 1.3013360500335693, -1.1181551218032837, 0.8750261664390564, -0.02621608041226864, -0.5872063040733337, -0.09612258523702621, 0.8685142397880554, 0.7750850915908813, 1.2051711082458496, -0.8253688216209412, 0.5344870090484619, -1.56107759475708, 0.2863011956214905, 0.5939759612083435, 0.4857065975666046, -0.6128327250480652, -0.4762400686740875, -1.175463080406189, -0.5079532861709595, -1.695326805114746, -0.595707356929779, 0.2769911289215088, 2.360245943069458, 0.08013438433408737, 0.1331796497106552, -1.2175654172897339, 1.0985171794891357, 0.9009111523628235, 0.2897285521030426, -0.7001771926879883, -0.4480486512184143, -1.0656245946884155, -2.5824577808380127, -1.0087594985961914, 1.2459931373596191, -0.565155565738678, -0.03253655135631561, 0.3795830011367798, -1.2544224262237549, 1.1436028480529785, -0.21995463967323303, 1.052323341369629, 1.2450051307678223, 1.0494486093521118, 1.8660355806350708, 0.217601016163826, 1.6836696863174438, 0.21864525973796844, 0.18970316648483276, -0.9409782290458679, 1.128098726272583, -0.7110011577606201, -1.006125807762146, 1.6452300548553467, -0.5560023188591003, -0.5438081622123718, -1.3083434104919434, -0.8551904559135437, -0.12229883670806885, 1.27871835231781, -1.3235069513320923, -0.04286162555217743, 0.4379936754703522, -0.38183096051216125, -0.1673130840063095, 0.3992502987384796, -0.7245261669158936, -0.571757435798645, -0.40118709206581116, -0.6517333388328552, 0.8938224911689758, -1.502821445465088, -0.9195721745491028, 0.1383359730243683, -0.4547722041606903, 0.040268562734127045, -0.8753424882888794, 0.6539029479026794, -1.1861037015914917, 0.5535740852355957, -1.719407320022583, -0.7714697122573853, -0.8140661120414734, 0.001108356285840273, -0.5470112562179565, 0.3160351514816284, 0.02114543318748474, 0.6776748895645142, 0.11151530593633652, 1.7072277069091797, -1.0266592502593994, 1.572403073310852, 1.609660267829895, -1.024195671081543, -0.8462005853652954, 2.3753981590270996, -0.8436659574508667, -1.549582600593567, -1.2414263486862183, 0.1528819501399994, 0.10122257471084595, -2.081322193145752, 0.3032465875148773, 0.9398054480552673, 1.1956024169921875, 0.5534347891807556, -0.285749614238739, 0.6409146189689636, -0.6060488820075989, 0.6682468056678772, 2.880615234375, -1.8522577285766602, 1.3481132984161377, -0.03328055515885353, -0.7679782509803772, 1.7046558856964111, -0.34613826870918274, -0.9480781555175781, 1.1645981073379517, 0.894264280796051, 1.3071776628494263, 1.5217536687850952, -0.6951116323471069, 0.22713205218315125, 0.4424692392349243, -0.6278089880943298, -1.0141547918319702, 1.1873149871826172, 0.47930529713630676, 1.6094945669174194, -1.1379584074020386, 1.5862902402877808, -0.23223812878131866, -0.17930300533771515, -0.5637533664703369, 1.1680972576141357, -0.273688405752182, 0.31055912375450134, -2.612689733505249, -1.538187026977539, -0.47292521595954895, 1.0395742654800415, 0.5030556321144104, 1.4848350286483765, 0.5104214549064636, 0.10616037249565125, 0.6374850869178772, 0.04221397638320923, -0.6965646147727966, 1.4814677238464355, -0.39149773120880127, -0.32078054547309875, -0.2281445562839508, -1.0768169164657593, 1.3886101245880127, 1.5163780450820923, -1.4458978176116943, -1.2958165407180786, -0.2830732762813568, -2.224000930786133, 0.24755117297172546, 0.33464285731315613, -0.22664405405521393, 0.5367629528045654, 0.5167449712753296, -0.7434871792793274, -0.15029248595237732, -0.534401535987854, -0.2800185978412628, -0.9651049971580505, -0.6038867831230164, 0.31614530086517334, -1.8904589414596558, 0.5288209319114685, 0.5001121759414673, -0.1624603420495987, 1.7336875200271606, -0.013489091768860817, -0.6900830864906311, 1.2038514614105225, -1.0492589473724365, -0.3190222978591919, -0.7190567255020142, -1.484325647354126, 2.098804235458374, 0.21548767387866974, -0.18760664761066437, 0.19955918192863464, 0.5996284484863281, -0.681636393070221, -0.8782709240913391, 0.0656675174832344, 1.2016857862472534, 0.39619654417037964, -0.1685950756072998, 1.4580466747283936, 0.42915573716163635, 0.7080570459365845, -0.07692442089319229, 1.7674323320388794, -0.17409098148345947, -0.8154168725013733, 0.0724971741437912, 2.2229650020599365, -0.932458221912384, 1.1200121641159058, 1.712296724319458, 1.1524626016616821, 1.7569397687911987, 0.15229220688343048, 1.8497625589370728, 0.09063980728387833, -0.3377591073513031, -0.2916814088821411, -0.9368879199028015, 1.3998079299926758, 0.5405740141868591, 0.03169350326061249, -0.03634852543473244, 0.3910501301288605, -0.46225112676620483, 0.6724438071250916, 0.5904347896575928, 0.6035146117210388, 0.9179046154022217, 1.5556946992874146, -0.2981718182563782, -0.3195224404335022, -2.545874834060669, 0.2994876205921173, 0.4219816029071808, -0.7026874423027039, 0.14739464223384857, 1.4138193130493164, -0.6153184175491333, -0.9654080271720886, 1.2270640134811401, -0.4053443372249603, 1.847001314163208, 0.8182591795921326, 0.1087985709309578, -0.04301411285996437, 0.7082913517951965, -0.9720585942268372, -0.7398479580879211, -1.2221571207046509, -0.27245616912841797, -0.012708376161754131, -0.16471275687217712, -0.08923113346099854, 0.458453506231308, -1.0949524641036987, 0.6454233527183533, -1.0151335000991821, 0.10006707161664963, 0.6190526485443115, -0.4099624454975128, -1.9314132928848267, -0.6430890560150146, -2.8049585819244385, -0.5847241878509521, 1.6139017343521118, -0.8186933994293213, -0.8180292844772339, 0.29143550992012024, -1.4781972169876099, -2.1735057830810547, -1.3899732828140259, -1.408571481704712, -0.47072911262512207, 0.5429496169090271, 0.09445394575595856, -0.009136407636106014, -0.05952221900224686, 0.03434443473815918, -0.19192002713680267, 0.9715786576271057, 1.2372426986694336, 0.03035658597946167, 0.5625606775283813, -1.343584656715393, 0.9859438538551331, 1.104298710823059, 0.5823128819465637, -0.043853759765625, -0.1997450739145279, -1.0943419933319092, -1.3698570728302002, -1.061237096786499, -0.34451839327812195, -1.6631001234054565, 1.795849323272705, -0.08688586950302124, 1.7332741022109985, 0.006151439622044563, -0.09658244252204895, -0.3009742796421051, -0.3628992438316345, -1.4650980234146118, -0.851700484752655, 0.15161816775798798, 0.3148422837257385, 0.018950333818793297, -1.2101716995239258, -2.0254974365234375, 1.2616996765136719, -0.8019678592681885, 0.8485089540481567, -1.329663634300232, -0.27476969361305237, 0.49105364084243774, 0.019503064453601837, 1.1124752759933472, 0.32495421171188354, 0.4765434265136719, -0.09974775463342667, 0.24491766095161438, -1.2766565084457397, -0.06500808894634247, -1.1514222621917725, -0.7401502728462219, -0.28685665130615234, 0.31722569465637207, 0.03060145303606987, 0.39131346344947815, -1.1582520008087158, 1.1029345989227295, -0.7343477010726929, -1.5290781259536743, 1.7584362030029297, 0.34563353657722473, -0.7070123553276062, 0.02113042026758194, 0.19182057678699493, 0.11666575074195862, -1.311405897140503, -0.06772846728563309, 0.5675070881843567, 1.5177525281906128, -0.18270568549633026, -0.3774254322052002, 0.6837977766990662, -0.42609041929244995, -0.23191076517105103, 0.18666808307170868, -0.46805232763290405, -1.7311581373214722, -0.25115835666656494, 0.5045846104621887, 0.9907859563827515, -0.3454635739326477, -0.13270904123783112, -1.1519511938095093, -0.8041810393333435, 0.6865206360816956, -0.7324920892715454, 2.4307587146759033, 2.4649252891540527, -1.310420036315918, 0.35005444288253784, 1.5956904888153076, 0.09986842423677444, -1.0324790477752686, -0.36040446162223816, 0.7964271903038025, 0.2151179015636444, 0.16864463686943054, -0.03999787196516991, 0.2537012994289398, -2.1146316528320312, -0.1198454424738884, -0.5120725631713867, 0.9940509796142578, 2.7070887088775635, -1.246030330657959, -0.3056158423423767, 0.3864118456840515, 1.2833179235458374, 1.0226472616195679, -0.07506633549928665, -1.0312819480895996, 0.9958034753799438, -0.08319360017776489, -0.604860782623291, 0.48173797130584717, 0.8088334798812866, 1.5951932668685913, -0.927850604057312, 0.12007635086774826, 1.7892248630523682, 0.2417154312133789, 0.24042648077011108, 0.2174069881439209, 0.42412349581718445, 0.8375867605209351, -0.14867429435253143, -0.3426409959793091, 1.2162272930145264, 0.31023454666137695, -0.14142616093158722, -0.2259398251771927, -0.11413592100143433, -0.0957871824502945, -0.4897300601005554, -0.023849185556173325, 0.07015185058116913, 0.48254314064979553, -1.2190754413604736, 1.3421934843063354, -1.592635154724121, -0.2786911427974701, 1.0514601469039917, -0.5773292183876038, -0.1257985681295395, 1.44612455368042, -0.470615953207016, -1.7851147651672363, -1.3235883712768555, -0.04450419172644615, 1.3739291429519653, 0.5158275365829468, -0.7278342843055725, -0.20701459050178528, -0.6428208947181702, 2.0562257766723633, 1.2400790452957153, 0.3982285261154175, -0.34721988439559937, -0.5040727853775024, -1.7062674760818481, -0.22501341998577118, 1.6299220323562622, -1.248173713684082, 1.655091643333435, 0.9872010350227356, 1.1746444702148438, 1.6116636991500854, -0.616566002368927, -0.290073037147522, -0.21559461951255798, -1.1840083599090576, -0.3285082280635834, 0.6359196901321411, -3.019474744796753, -0.6341651082038879, -1.1980109214782715, -0.502499520778656, 0.8731013536453247, -2.8679022789001465, -1.5464245080947876, 1.1119229793548584, -0.7608980536460876, 2.411320924758911, -0.7026065587997437, 0.7859331965446472, 0.19376029074192047, -0.31403011083602905, -2.369530200958252, -1.1991767883300781, -1.3554843664169312, -0.7384332418441772, 0.6756752133369446, -0.1808425784111023, -0.2807244658470154, -0.16685990989208221, 1.0966852903366089, 0.3547503352165222, -1.3711446523666382, -0.58857661485672, 0.15398012101650238, -0.5983206629753113, 0.6692944765090942, -0.26793956756591797, -1.8018465042114258, 0.2689513564109802, -1.2289063930511475, -0.0867738202214241, -0.10768311470746994, 0.025449197739362717, 0.77797931432724, 0.10591162741184235, 0.8880118727684021, 0.2542524039745331, 0.11970774084329605, 0.8638383746147156, -0.04930988699197769, -0.37327539920806885, -0.4556635320186615, 0.43473973870277405, 1.062954068183899, 0.05370732396841049, 0.16971930861473083, 0.08088505268096924, -1.4406452178955078, 1.1890524625778198, -0.6529458165168762, -3.137216329574585, -0.46488985419273376, 0.7260956168174744, 0.6787742376327515, 0.6219494938850403, -0.8142244815826416, -0.6955735087394714, -0.023558977991342545, -0.9033581614494324, 1.4480922222137451, -0.7604867815971375, 0.11764056235551834, -0.04052136838436127, -1.2873685359954834, 0.5745333433151245, -2.2504067420959473, 1.3286054134368896, -1.0278806686401367, 1.0009641647338867, 0.6069590449333191, 0.3884316384792328, 0.22939881682395935, -0.8295376896858215, -1.4527145624160767, 1.229353904724121, 0.4637187123298645, -0.17919416725635529, 0.3814206123352051, -0.0025630639865994453, 0.10729843378067017, -0.987639307975769, 0.4782193601131439, -0.2681334316730499, -0.32248249650001526, 1.3828155994415283, -0.10641380399465561, 0.4823158383369446, 0.3660864233970642, -0.37648457288742065, 0.873041033744812, 0.12782022356987, 0.20571473240852356, 1.5080965757369995, 0.03637555614113808, 0.36175286769866943, -0.7898017764091492, -1.6053000688552856, -0.3718855381011963, -0.30978894233703613, 0.12085137516260147, -1.3189761638641357, 1.0380370616912842, -0.11117620766162872, -1.0705915689468384, 0.5575910210609436, 0.6535746455192566, 0.2095113843679428, -2.4831950664520264, -0.3884223401546478, 1.1631149053573608, 0.3719641864299774, 1.743018388748169, -0.9794517159461975, -0.21981695294380188, 0.37236735224723816, -1.5051658153533936, 1.5853468179702759, -0.07688786089420319, 0.17678482830524445, -1.1612858772277832, 0.6594412922859192, -1.4358512163162231, -1.1969115734100342, 0.5516269207000732, 0.7804403901100159, 0.6238122582435608, -0.41769978404045105, 0.2523072361946106, -0.7033896446228027, -1.3987069129943848, -1.8016774654388428, -0.05583738908171654, 1.438856601715088, 0.020482197403907776, 0.6120308637619019, 1.7721108198165894, -1.4384574890136719, 1.2415688037872314, 0.9472941160202026, 1.6662817001342773, 1.6018257141113281, -0.43795835971832275, 0.1688869744539261, 0.3647315800189972, -1.0939427614212036, -1.3380839824676514, -0.6149978637695312, 0.2985164225101471, 0.153771311044693, -0.21820607781410217, 1.1401400566101074, 1.098729133605957, -0.06379228085279465, 1.3142659664154053, 0.4382798969745636, 0.3523353040218353, -1.065768837928772, 1.0896562337875366, -0.99208003282547, -0.1809079945087433, 0.6582435965538025, 0.0002048723545158282, -0.645748496055603, -0.823784351348877, 1.136540174484253, -0.058102939277887344, -1.2704083919525146, -0.08530282229185104, -1.1529383659362793, -0.2532571256160736, 1.6177382469177246, -0.07774762064218521, -0.7032201886177063, -0.25655800104141235, 0.2559163272380829, -1.2087476253509521, -0.17344538867473602, 0.540763795375824, 0.24728161096572876, -1.6278235912322998, -1.6955492496490479, 0.5081257224082947, -3.098491907119751, 1.7130016088485718, -0.05575873330235481, -0.7323856949806213, 0.6099594235420227, 0.3963744044303894, 1.0865211486816406, -0.017167525365948677, -0.023913562297821045, -0.34991687536239624, 0.14647264778614044, 1.6905806064605713, 1.4319437742233276, 0.5057082772254944, 1.5313594341278076, -0.5375935435295105, 0.0470060259103775, -0.17469072341918945, 0.4129446744918823, -2.7075631618499756, -0.20862826704978943, 0.3383690416812897, 0.8809875845909119, 0.2673836350440979, -0.1347230225801468, 0.22799357771873474, -1.004020094871521, -0.4242437779903412, 0.4129737317562103, 0.04908552020788193, 0.731065034866333, -1.4472097158432007, -1.1938552856445312, 1.124650478363037, 1.1211555004119873, -0.7269195318222046, -0.10531707108020782, 0.15966694056987762, -0.2502894699573517, -1.4925577640533447, 0.04618046432733536, 0.04163679480552673, 1.1356333494186401, -2.003849506378174, 1.0746277570724487, -2.3296778202056885, 0.8931424021720886, 0.11332781612873077, -2.0539493560791016, 0.048821937292814255, 0.6103904843330383, 0.6469173431396484, 0.04437797889113426, 1.2001633644104004, -0.23645582795143127, -0.8665130734443665, 0.1797494739294052, -1.5034278631210327, -0.06384900957345963, 0.5198206901550293, -1.3484306335449219, 0.12970615923404694, -0.3421769142150879, 0.048302698880434036, -0.40323606133461, -0.013059532269835472, 0.3490729331970215, 0.5993431806564331, 1.613043189048767, -1.9153715372085571, -1.4576361179351807, -0.3201353847980499, 0.03731556609272957, 0.07785741239786148, -0.9720866680145264, -1.7602126598358154, 1.3274495601654053, 1.089524507522583, -0.25654834508895874, 1.2527328729629517, -0.5155013799667358, 0.17484644055366516, -0.7099625468254089, 1.031505823135376, -0.452779084444046, -1.4346176385879517, -1.49337899684906, -0.19662337005138397, -1.0361289978027344, -0.1865655928850174, 0.3243447542190552, 0.010715467855334282, 1.4647746086120605, 0.8150302767753601, -2.0770015716552734, -0.03713138401508331, 0.3897066116333008, -1.0070182085037231, 0.08420920372009277, -0.5595254898071289, 1.5103973150253296, -0.15465208888053894, 0.47608861327171326, 1.3754422664642334, 1.2073057889938354, -1.5225685834884644, 0.5964793562889099, -0.5834898352622986, 1.6623462438583374, 1.7125742435455322, 0.894422173500061, 0.48996636271476746, -0.3509819805622101, 2.1438939571380615, 2.3263492584228516, 0.32614728808403015, 0.8943504691123962, 0.9341400265693665, -1.1884641647338867, -1.3426380157470703, 0.6946704387664795, 1.5709912776947021, -1.2579891681671143, -0.5687536597251892, 0.8267149329185486, 0.22372573614120483, -0.8478546142578125, -0.12776023149490356, -0.10773833096027374, 0.4685121774673462, -0.18026107549667358, -0.08655758947134018, -1.8690141439437866, -0.7689586281776428, -1.4597653150558472, 1.002600908279419, 0.5136246681213379, -1.0244107246398926, 0.05099295452237129, 0.6348612308502197, 0.8050007224082947, 0.17837651073932648, -1.598807454109192, -1.534881591796875, -1.1823949813842773, 1.0339465141296387, 1.8966341018676758, 0.025219418108463287, -0.5111547708511353, 0.43882936239242554, 2.225551128387451, 0.3243805766105652, 0.1689148098230362, -0.741247296333313, -1.9330921173095703, -1.8304857015609741, 1.4558629989624023, 1.4843720197677612, -0.8292440176010132, -0.34535571932792664, 0.7233356833457947, 0.09173616021871567, -1.3466179370880127, -0.7142748236656189, -0.41486018896102905, 0.6336077451705933, 0.21771340072155, 0.8926344513893127, -0.21758870780467987, 0.3557300567626953, -0.8136159777641296, 1.320479154586792, 0.23885491490364075, -0.802646815776825, 0.9323887228965759, -0.08525092154741287, -1.2035610675811768, -0.5769782066345215, -0.710006594657898, 1.84033203125, 1.1240689754486084, 0.38146770000457764, 0.5338622331619263, -1.299680233001709, 0.480159729719162, 0.25185009837150574, 0.28688523173332214, 0.7655288577079773, -0.0845097228884697, 0.18515898287296295, 0.17847292125225067, 0.878391444683075, 2.592350959777832, -0.819090723991394, -0.4602276682853699, 0.48205724358558655, 1.121781826019287, -0.057678647339344025, 0.08288801461458206, 0.9165295362472534, -1.501731514930725, 0.05729915201663971, -1.0742311477661133, 0.9897600412368774, 2.9361181259155273, -0.8106008172035217, -1.046845555305481, -0.29647204279899597, -1.4597851037979126, -0.5471169352531433, -0.28657734394073486, -0.8230015635490417, 0.7705869674682617, -0.39141494035720825, -0.08262539654970169, 0.6277594566345215, 0.19446754455566406, -1.7314280271530151, 1.4622411727905273, -1.457749843597412, 0.9150553941726685, -0.532421886920929, -0.7441490292549133, -0.9318979978561401, 1.6921722888946533, 1.582066535949707, 0.8569250702857971, -1.3337706327438354, 0.12657690048217773, -0.36991679668426514, -0.7983108758926392, -0.5191489458084106, -0.010268635116517544, 0.42510202527046204, -0.6691691875457764, 0.9093791842460632, 0.05746991187334061, -1.940138339996338, 0.024639876559376717, -0.3857426941394806, 0.6503459215164185, -1.0419889688491821, -0.49288320541381836, 2.274829387664795, 1.1166976690292358, -0.2696325480937958, -0.6079724431037903, 1.7721080780029297, -0.782841145992279, -0.4433089792728424, 1.4316610097885132, -0.5184628367424011, 0.8607351183891296, 0.2851933538913727, 1.6981605291366577, 0.1685228794813156, 0.8736175298690796, -2.523387908935547, 0.8189143538475037, -0.5371997356414795, 0.03165857493877411, -1.5173776149749756, -0.20476703345775604, -1.5769027471542358, -1.064022183418274, -1.0834811925888062, 0.9592816233634949, 1.0546261072158813, -0.7831087112426758, -1.8522807359695435, 1.833372950553894, -0.2692483365535736, -0.9928693175315857, 1.1901943683624268, -0.2721623480319977, -1.0291004180908203, -0.3438674211502075, 0.6820979118347168, -0.5405786037445068, -1.0937410593032837, 0.7477661967277527, 1.0020195245742798, -1.941766619682312, 0.5654708743095398, -0.19437658786773682, -1.7227636575698853, -0.6475908756256104, 0.765831470489502, -0.4286428689956665, 0.3579843044281006, -1.9991461038589478, -0.7706862688064575, 0.7143704295158386, 0.2567901611328125, -0.24649201333522797, -0.06541123986244202, -0.8881609439849854, 0.15858346223831177, -0.17222610116004944, 0.1612396538257599, -0.30254244804382324, 0.7775685787200928, 1.5879921913146973, -0.3470304310321808, 1.9224638938903809, 0.10356763750314713, -1.32328200340271, -0.5586869716644287, 0.34557774662971497, -1.030112862586975, -0.9342789053916931, -0.948810875415802, 0.028290104120969772, -2.0172276496887207, 0.16355769336223602, -0.5056277513504028, 0.2526977062225342, 0.2047346830368042, -0.6666006445884705, 2.2931909561157227, -0.03996067866683006, -0.6388669610023499, 0.05146284028887749, 0.13264791667461395, 0.8987274765968323, 1.023596167564392, 2.976762294769287, -0.03025803156197071, 0.2457212507724762, 0.6690837144851685, 0.7304574847221375, 0.12669968605041504, -0.16909755766391754, -1.0957541465759277, 0.8654098510742188, -0.30550795793533325, -0.535591185092926, 0.3747936487197876, 1.3672807216644287, 0.501893162727356, -2.2355737686157227, -1.0358167886734009, -0.9378079175949097, -0.6057659983634949, 0.6271786093711853, 0.7170186638832092, 0.7216516137123108, 0.7698862552642822, 0.10667973011732101, 0.1664266586303711, -0.6981831192970276, 0.4116598069667816, 0.6816626191139221, 1.3686493635177612, -0.05379726365208626, 0.9171262383460999, 2.0867185592651367, 0.08391284197568893, 0.7582014203071594, 0.4899533987045288, -1.5765215158462524, -1.1322036981582642, -1.0861735343933105, -0.08525819331407547, -1.095007300376892, -1.1389068365097046, -0.23206499218940735, -1.2879722118377686, 1.098273754119873, 0.8865388631820679, 1.247675895690918, 0.17922794818878174, 1.2576128244400024, -1.2939742803573608, -0.1768873780965805, -0.5898751616477966, -0.3785025477409363, 1.1359138488769531, -0.4553658962249756, 0.9354010820388794, 1.0120458602905273, 0.775386393070221, -1.1637896299362183, -0.2533360421657562, -0.9874160885810852, 0.38359493017196655, -0.9512411952018738, -0.6023403406143188, 0.12041366845369339, -0.09847727417945862, -1.2150758504867554, 0.523783802986145, 1.3612641096115112, 0.10864757746458054, -1.5759541988372803, 0.45853543281555176, 0.9465855360031128, 1.2424026727676392, -0.6662697196006775, -1.4130527973175049, -0.40343019366264343, 1.4467052221298218, 0.07603957504034042, 1.224551796913147, 0.2803522050380707, -1.0971283912658691, 0.35760053992271423, -0.17808017134666443, -0.018365487456321716, -0.6433379650115967, -0.5729802250862122, -0.9857281446456909, -0.7668794393539429, 0.220060795545578, -0.4210362434387207, 0.08686216920614243, -1.434130311012268, -0.48284849524497986, -1.0450607538223267, -1.702032208442688, 0.5757233500480652, -0.5772421956062317, 0.4884365499019623, 0.005668565630912781, -1.793837308883667, -0.3453691899776459, -0.24546442925930023, 0.4348530173301697, 0.4917606711387634, -0.37847408652305603, 0.7062116861343384, 0.8906152248382568, 1.6876773834228516, 1.7800873517990112, 0.9881210327148438, -1.3600653409957886, 0.2512918710708618, 1.8811439275741577, -0.7065674066543579, -0.24695657193660736, 0.6946361660957336, -1.741495132446289, 0.39252549409866333, -0.6407681107521057, 1.9813772439956665, -0.05472339689731598, 0.7283979058265686, 1.3437998294830322, -0.36186036467552185, 0.9219562411308289, 0.18365298211574554, 0.04400939866900444, 0.1808076649904251, 1.5982948541641235, 2.442887544631958, 1.3699852228164673, -1.4695340394973755, 1.4436198472976685, -1.539819598197937, -0.7900468707084656, 0.3316803574562073, 0.5396322011947632, 0.4404478073120117, 0.8900952339172363, -0.7723318934440613, 1.5519561767578125, 1.194494605064392, 0.9876572489738464, -1.25104820728302, -1.470982551574707, 0.6084056496620178, 2.1952834129333496, -0.6678028702735901, -0.3248213231563568, -0.7304811477661133, -1.3971141576766968, 1.7316606044769287, -0.997414231300354, 2.1895315647125244, 1.1580255031585693, -1.70986008644104, 0.35069549083709717, -1.7012337446212769, 0.12683679163455963, 0.9011533856391907, -0.2215212881565094, -2.5765035152435303, -1.0029845237731934, 0.16045448184013367, 0.004467700608074665, 0.6590482592582703, -2.150852680206299, -1.0698295831680298, -0.46715155243873596, -0.8271596431732178, -0.4119405448436737, 1.7535603046417236, 1.2910852432250977, -1.677207589149475, 0.6028192639350891, 1.2787339687347412, -1.8028380870819092, 0.48104360699653625, -2.32037353515625, 0.8985234498977661, 1.0788450241088867, -1.7186440229415894, -0.9927147030830383, 1.1957894563674927, 1.3806626796722412, 0.3948056995868683, 0.11387476325035095, -1.051614761352539, -0.3985273838043213, -0.20864684879779816, 0.9110297560691833, 1.1754783391952515, -0.7933520674705505, -0.37865176796913147, -0.846027672290802, 0.5121067762374878, -0.20539870858192444, -1.0197529792785645, -1.5587888956069946, -0.016129881143569946, -0.0739373192191124, 0.000785582757089287, -1.3044801950454712, 0.8117046356201172, 0.9096587896347046, -0.19543036818504333, -1.1246949434280396, -0.5757842659950256, -0.9587568640708923, -2.0708982944488525, 0.3422680199146271, -0.4018622636795044, 0.3503943085670471, 1.1108394861221313, -0.36119571328163147, 0.7466336488723755, 0.6606119871139526, -0.26089194416999817, -0.9905322790145874, 1.155558466911316, -1.4671272039413452, -0.5382730960845947, 1.5134536027908325, 0.3047976493835449, 0.9409609436988831, 0.12811186909675598, -0.8344550728797913, 0.014525298960506916, 0.6341775059700012, 0.46864521503448486, 0.0017219483852386475, -0.8449415564537048, 0.5026370882987976, -0.0895729809999466, -0.2345522791147232, -0.7831602692604065, 0.41704243421554565, -1.1061512231826782, -1.5033247470855713, 0.885895848274231, -1.307276725769043, 1.42908775806427, 0.6661574244499207, 0.3282090723514557, 0.5051879286766052, 0.5684125423431396, -1.6262766122817993, -0.4522814452648163, -0.6218503713607788, 0.5949018001556396, 0.5109416842460632, -0.1353493630886078, 0.2978799641132355, -0.9527286291122437, -0.3217645585536957, 0.7676171064376831, -0.016141464933753014, -1.4908877611160278, 0.9672164916992188, 0.6074514389038086, 1.4216148853302002, -0.8123098611831665, -0.8950693607330322, 0.3965109586715698, 1.9465796947479248, 0.2296871393918991, 1.5117295980453491, -0.47898727655410767, 0.5051866769790649, -1.3790074586868286, -2.036288261413574, 0.6999777555465698, -0.08983509242534637, -2.635361909866333, -0.34187424182891846, 0.3909038007259369, 0.21544577181339264, 0.670395016670227, 0.24032379686832428, 2.206265926361084, -0.024852434173226357, 1.3843200206756592, -0.6817842721939087, -0.5070288777351379, -2.4855284690856934, 1.3897756338119507, -1.159902811050415, 1.977147102355957, 0.35162246227264404, 0.30540984869003296, -0.40031468868255615, 0.3013114631175995, 0.8576141595840454, -0.9110713601112366, 0.40877076983451843, -1.0858405828475952, -0.6168681383132935, 1.855257272720337, -0.3611413836479187, 0.004609718918800354, -1.914581537246704, 1.037846326828003, 0.7293850779533386, -0.24646642804145813, 0.8095522522926331, 1.2595174312591553, -0.04553225636482239, -0.0013399863382801414, 1.7185848951339722, -0.11303239315748215, 0.6316115260124207, -1.2975369691848755, 0.21355503797531128, -0.8157727122306824, -2.316955327987671, -0.9878710508346558, 0.04907108098268509, 1.1209179162979126, 1.217362642288208, 1.076900601387024, 0.07774533331394196, -0.01579894870519638, -1.0823496580123901, -0.8533012866973877, 1.926603078842163, 0.10200704634189606, 0.37792858481407166, 2.0040814876556396, -0.49820658564567566, -1.3509583473205566, -1.314256191253662, -1.992376446723938, 0.5052849650382996, 0.1157255470752716, 0.8815374970436096, -0.19053854048252106, 0.8942995667457581, 0.7565547227859497, 0.2855411171913147, -0.04195097088813782, 0.7509956359863281, -0.23481109738349915, -0.5828516483306885, -0.6673039197921753, -0.14037911593914032, 0.6258096098899841, 0.11666087061166763, -0.46770986914634705, 0.10462592542171478, -0.16034622490406036, 0.35641390085220337, -1.9164365530014038, 2.1661410331726074, -0.3807629346847534, -0.8661578297615051, -1.6883413791656494, 0.6080784201622009, 0.7108156681060791, -1.553869366645813, 0.27341553568840027, 0.5263627767562866, -2.1134753227233887, -0.027230901643633842, 0.26860329508781433, 0.27296507358551025, 0.5433645844459534, -1.268746018409729, -0.0953197255730629, -0.10404402017593384, 0.5011266469955444, 1.2558284997940063, -0.4247775971889496, -0.3498721420764923, -0.8222959637641907, -0.3702131509780884, -1.993237853050232, -0.8808827996253967, -0.9350168704986572, -1.937108039855957, -0.838067889213562, 0.07205981016159058, 0.7867241501808167, -1.6886433362960815, 0.9389370083808899, 0.21445299685001373, -1.4763438701629639, 0.21416699886322021, 1.1364871263504028, 0.7575664520263672, 0.44694602489471436, -0.9124276041984558, -0.3131539225578308, -1.0102967023849487, -0.9570778012275696, 0.16049471497535706, -0.8779717087745667, 0.3069268465042114, -0.33018219470977783, -0.29403403401374817, -0.9778866171836853, 2.2622885704040527, -0.4087357521057129, 1.05336332321167, 0.5390723347663879, -0.9660137295722961, 0.007940622046589851, 0.3191269338130951, 2.403995990753174, 1.7664297819137573, -0.20827141404151917, -0.24348603188991547, -0.9294469952583313, -0.7829566597938538, 0.2575119733810425, 2.444467782974243, 0.539463222026825, 0.16628724336624146, 0.765393078327179, -0.6500362157821655, 0.5688334703445435, 1.1367648839950562, 0.05549008399248123, 1.2789517641067505, 0.3119727373123169, 2.478337526321411, -0.5892170071601868, 0.41620346903800964, -0.4684125781059265, 0.7171820402145386, 1.1151337623596191, 0.8398614525794983, -0.5387542247772217, -0.523278534412384, -2.345824718475342, 0.6018604636192322, 0.12045293301343918, 1.374771237373352, -0.22886906564235687, -0.6326390504837036, -0.8723142147064209, 0.09066139906644821, 0.7155131697654724, 0.5029517412185669, 1.6380397081375122, -0.34587520360946655, 0.3166552186012268, -0.20099392533302307, -0.844260573387146, -0.43743985891342163, 1.4529492855072021, -1.8276751041412354, 1.2347203493118286, -0.6522923707962036, -0.13089685142040253, -0.43435391783714294, -0.6096197366714478, 1.687792420387268, 1.218889594078064, -0.8050568103790283, 0.6089577078819275, -3.5673019886016846, 0.13235916197299957, -0.5056430697441101, 0.6124782562255859, -0.6890867352485657, -0.3732217848300934, -1.9169073104858398, 0.19020891189575195, 0.6245309710502625, -0.050097011029720306, 0.807113528251648, 0.10075626522302628, 1.8670854568481445, -0.4278676509857178, -0.8645409345626831, 0.8448072671890259, -0.45449963212013245, -0.6238179206848145, -1.0125126838684082, -0.9735162854194641, -1.7683005332946777, -0.5183499455451965, -0.45385417342185974, -0.5155959725379944, 0.08552086353302002, -0.3087517321109772, 0.32338225841522217, -1.0260584354400635, 0.09638629108667374, -0.8607128262519836, 0.5262099504470825, -1.393835425376892, -0.0824686586856842, 0.11321552097797394, -0.19951705634593964, 0.6869836449623108, -1.0316619873046875, -0.3353433907032013, 0.48493653535842896, 0.764505922794342, 0.6058931946754456, -0.3782961368560791, 0.2613818049430847, -0.8590120673179626, 0.25265756249427795, -0.4367637038230896, 0.8980411291122437, -0.619576632976532, -0.22249072790145874, 0.8687300086021423, -0.03378627076745033, -0.47276440262794495, 0.2799162268638611, 0.10219993442296982, -1.1052626371383667, -0.27226218581199646, -0.6380689144134521, 1.3136019706726074, -0.8116991519927979, 0.2210041880607605, -1.5508545637130737, -1.403246283531189, -0.16192708909511566, 0.13540232181549072, -0.2550061047077179, 0.2768750786781311, -0.7481454014778137, -1.4397534132003784, -1.1715062856674194, 0.9590094685554504, -0.7677964568138123, 2.3279576301574707, -1.1678565740585327, 0.9856082201004028, -0.26327288150787354, 1.0601825714111328, 0.9698340892791748, -0.5588080883026123, -0.3682169020175934, 0.5726801753044128, 0.17686781287193298, -1.6851632595062256, -0.11618321388959885, -0.6491997241973877, 0.3647066652774811, -2.9801039695739746, 0.6222512722015381, -0.4200746417045593, 1.3015810251235962, -0.7283247113227844, -0.19455140829086304, -0.6690456867218018, 1.3527398109436035, 0.06857337802648544, 0.6160815358161926, -0.5393730401992798, 1.7412713766098022, 0.30751875042915344, -0.36465364694595337, 0.3300117254257202, 1.5440963506698608, 0.6613186001777649, 0.28289324045181274, -0.03776809573173523, -2.033646821975708, 1.6959789991378784, -2.2997140884399414, -0.3696361780166626, -1.1454572677612305, 0.15519137680530548, -0.2218741476535797, 0.5998916625976562, -1.8252421617507935, 0.15937745571136475, 0.2825770378112793, -1.9164330959320068, 0.019483350217342377, 0.535336971282959, 0.38078489899635315, -0.6014492511749268, -0.33883583545684814, -2.367354154586792, -0.7005154490470886, 0.3264123499393463, -0.5502170920372009, -1.6416436433792114, 0.8209441900253296, 0.2478150874376297, -1.4526361227035522, 0.6116436123847961, 2.0312957763671875, -0.5803869962692261, 1.667396068572998, 1.8990020751953125, 0.30823633074760437, 1.0377137660980225, 0.5232380032539368, -0.2870139479637146, 1.206059217453003, 0.5771340727806091, 0.04336252436041832, 0.34062066674232483, 1.3448967933654785, -0.1541317254304886, -1.3397517204284668, -0.35462555289268494, -1.304579496383667, -0.8120525479316711, -1.3311152458190918, 0.7294657826423645, 0.9270042777061462, 1.9848605394363403, -1.3743253946304321, -0.04201004281640053, 0.4398854076862335, 0.09062765538692474, 0.4021538496017456, 2.5491671562194824, 0.6730988621711731, -1.5831302404403687, -0.6437485814094543, 1.4632636308670044, -1.1834675073623657, 1.9755737781524658, 2.0760672092437744, -0.7921455502510071, 1.3619484901428223, -1.8195209503173828, -0.3113616406917572, 0.39834877848625183, -0.8068146705627441, 0.12566712498664856, -0.9415177702903748, 0.8026949763298035, 1.5669670104980469, 1.7494953870773315, -0.6085540056228638, 1.5230413675308228, -1.0853430032730103, -0.36033663153648376, 0.6726157665252686, -1.0612958669662476, -0.08059170097112656, -1.2764041423797607, 1.774037480354309, -1.1878204345703125, 1.445346713066101, 1.3829985857009888, -0.4148533046245575, 0.29364293813705444, -0.9161414504051208, -0.7330148816108704, 2.2251338958740234, 1.0436245203018188, -0.9262099266052246, -1.0998096466064453, -0.2448273003101349, -1.5931192636489868, 0.5148633122444153, 2.9963669776916504, -0.2815517783164978, -0.09470577538013458, -1.0648128986358643, 0.17716841399669647, -0.246526837348938, 0.5193862915039062, 1.7253620624542236, -0.34227943420410156, 0.46247172355651855, -0.4444585144519806, -1.738452434539795, 0.5469908118247986, 0.7663493156433105, 0.3359984755516052, -0.0478830449283123, -2.2105939388275146, 0.38805875182151794, 0.9846127033233643, -1.3091206550598145, -0.39401116967201233, 0.6608238220214844, -1.6115660667419434, -0.3415260314941406, 0.07464581727981567, -1.6781302690505981, 1.0040072202682495, 1.3714300394058228, -1.1677416563034058, 0.21312257647514343, 0.5903096795082092, 0.2437107264995575, -0.3167651891708374, 0.650907576084137, -1.3110843896865845, -1.6441477537155151, 0.028711630031466484, -0.8426538109779358, 1.3599005937576294, 0.681584358215332, -0.04558519273996353, -2.1603481769561768, -0.6471628546714783, 1.3315461874008179, 1.7194304466247559, -0.31005391478538513, 0.09190256148576736, 1.5210065841674805, 0.9322324991226196, -0.194098562002182, 1.231263518333435, -1.8439204692840576, -0.6437793970108032, -0.7843500375747681, 1.0080257654190063, 2.414170980453491, -0.4051431119441986, 0.4017070233821869, 0.5430465936660767, -0.057488199323415756, -1.1588720083236694, 1.0698130130767822, -0.6632864475250244, 1.808676838874817, 2.029114007949829, 0.9249211549758911, 0.4179368019104004, 1.1521042585372925, -0.26916301250457764, 0.14742803573608398, 0.4931744933128357, -0.5084952116012573, -0.237893745303154, 1.8175835609436035, 1.6141319274902344, -0.21377801895141602, 0.7454184889793396, 2.666595935821533, -0.21593084931373596, 1.4475797414779663, 2.1280956268310547, -0.4664507210254669, 1.1103315353393555, -0.1062583178281784, -1.677376389503479, 1.046012043952942, 2.306828737258911, 1.6354484558105469, 0.8524677157402039, -0.3596930503845215, -0.3006361126899719, -0.9767610430717468, 0.41699227690696716, 0.28875961899757385, 0.2734125852584839, -1.2503559589385986, 0.5774098038673401, 1.6637122631072998, 0.7211600542068481, 0.6178462505340576, -0.0018332441104575992, 0.9616193771362305, 1.3453227281570435, -0.613405704498291, -0.06959985941648483, 0.2052125632762909, 1.337617039680481, -1.3585476875305176, 1.3036653995513916, 1.2779792547225952, 1.8137751817703247, 1.5749590396881104, -1.032178282737732, -0.7495907545089722, 1.6115150451660156, -0.9462671279907227, -0.4041132628917694, 1.9398562908172607, 1.9917283058166504, -0.9219620227813721, 0.5776676535606384, 0.2958868145942688, -1.628498911857605, -0.8205413818359375, -0.058525729924440384, -1.742491364479065, -0.1974242776632309, 0.4137197732925415, 0.1425677090883255, 0.8054239153862, 0.15116660296916962, -0.32456958293914795, -0.6988222599029541, -0.4643888771533966, -0.025243068113923073, -0.39771658182144165, -1.1975210905075073, 1.32191801071167, 0.054017044603824615, -0.7492539882659912, 0.12186432629823685, 0.6595790386199951, 2.1337313652038574, 1.2809762954711914, -0.7352215051651001, -0.03364512696862221, 0.2502637207508087, -0.025970852002501488, -0.8448286652565002, -0.5043429136276245, 2.460632801055908, -0.029371874406933784, -0.5909130573272705, 1.0161914825439453, 1.0568065643310547, -0.4431495666503906, -0.2877243757247925, -0.048781316727399826, -1.5541476011276245, -0.7813295125961304, 0.05221550539135933, 1.9847807884216309, 1.2447093725204468, 0.41796019673347473, 1.339439034461975, 0.8295568823814392, -1.1135623455047607, 0.24259498715400696, -1.2168421745300293, -1.034637451171875, -0.4464532732963562, -0.1250927448272705, -0.3649637699127197, -0.29905688762664795, -1.3699533939361572, 0.012148727662861347, 1.120712399482727, -1.6486058235168457, 0.39541104435920715, 0.28031328320503235, -1.7424458265304565, 0.5690951943397522, -0.4069186747074127, -0.6990140676498413, 0.5531984567642212, 0.22597768902778625, 0.12147686630487442, -0.19151422381401062, -0.827537477016449, 1.4327460527420044, -1.1303696632385254, -0.6291237473487854, 0.13202230632305145, -0.9949733018875122, 1.3690773248672485, -0.43082135915756226, -0.7905290126800537, -0.470438152551651, 1.4523773193359375, -0.45036497712135315, 1.0801103115081787, 1.1899545192718506, 1.2010680437088013, 0.6370734572410583, 0.9841129183769226, 0.2364884614944458, -1.1424009799957275, 0.3992299437522888, 0.9762090444564819, 1.276215672492981, -0.7518392205238342, -0.020444881170988083, 0.8425690531730652, 0.5452105402946472, 0.7890783548355103, -0.05847027152776718, 0.9407802224159241, 0.14618411660194397, -1.6531856060028076, 0.33170175552368164, -1.529941439628601, -1.1538280248641968, -0.5633106231689453, -2.1028218269348145, 0.684083104133606, -0.009679255075752735, -0.41757816076278687, -1.29104483127594, 0.707435667514801, 0.15101686120033264, 0.3156651258468628, -1.9380170106887817, -1.8258566856384277, 0.48511895537376404, -1.8893048763275146, -0.8447389006614685, 0.4097328186035156, -2.167954206466675, -0.5780210494995117, 0.34977641701698303, 0.6490760445594788, 1.1907289028167725, 0.7841066718101501, -0.5308780074119568, -0.48422303795814514, 0.12665309011936188, 0.5652027130126953, -0.07504506409168243, -0.011786799877882004, 1.3348957300186157, 0.9892857074737549, 0.5690614581108093, -0.3559773564338684, 0.8588225245475769, -1.0217483043670654, 0.17807619273662567, -0.44334477186203003, -0.6916666030883789, 1.4365873336791992, -0.9884365797042847, -0.0595816895365715, -0.03484339639544487, -1.6685328483581543, -0.6737463474273682, -1.5110607147216797, 0.10862607508897781, -0.6462985277175903, 2.2986247539520264, -0.33588907122612, 0.11091536283493042, 0.08107472956180573, 0.32911229133605957, -0.2480592280626297, 0.34066006541252136, -0.8712257146835327, 1.3402695655822754, 0.45152193307876587, 0.5105233192443848, 1.6179980039596558, 0.4440308213233948, 0.02111382968723774, 0.2399418205022812, 0.6097821593284607, -0.05016883462667465, 0.39253267645835876, -0.005912700202316046, -0.9491528868675232, -0.0790509432554245, 0.4871872067451477, -0.3964393138885498, 0.3690659999847412, -0.03479405492544174, 1.0680789947509766, 0.368710994720459, -0.002047204179689288, -0.7929198741912842, 0.31684115529060364, -0.1058209016919136, 0.5608875751495361, 0.4755859076976776, 0.21409474313259125, 0.28989848494529724, -0.5520952343940735, 0.8184353709220886, -0.9237565398216248, 0.6397095322608948, 1.9855018854141235, 1.607064127922058, -0.09333755075931549, -0.877250611782074, 0.14878611266613007, 1.018292784690857, -0.031801339238882065, 0.15097318589687347, -0.4430500864982605, 0.7319957613945007, -0.6817741990089417, -1.869963526725769, -0.22254960238933563, 1.2634910345077515, 0.5552249550819397, 1.0760424137115479, 0.504944920539856, 0.09806887805461884, 0.37729036808013916, 0.15677958726882935, 0.3984849750995636, -2.419079542160034, -1.9622739553451538, -0.7665964365005493, 1.414786458015442, 1.14386785030365, 1.9506003856658936, -0.8910230398178101, -1.796312689781189, 0.6040725708007812, -0.07891549170017242, 0.09268812090158463, 0.6946695446968079, -1.1000829935073853, -0.2885470390319824, 0.37579789757728577, -0.015457232482731342, -0.3532918393611908, 0.7608826756477356, -0.18768948316574097, -0.32011014223098755, 0.37051957845687866, -0.5546618103981018, 0.4083716571331024, -0.7487733960151672, 0.5777994394302368, -0.6621351838111877, -1.4147566556930542, -0.9184944033622742, 1.027595043182373, -1.606824517250061, -1.844796895980835, 0.5868285894393921, -0.2311399281024933, 0.5952156782150269, -1.9370615482330322, -1.003720760345459, -0.8364951610565186, 0.04460691288113594, -0.40033596754074097, -1.8641363382339478, 0.6307634711265564, 1.3764839172363281, 1.1527888774871826, -1.351736068725586, 1.7619839906692505, 0.5526049733161926, 0.19904561340808868, 0.4752465784549713, 1.70212721824646, 0.6272722482681274, -1.267735242843628, -2.345872640609741, -0.10289065539836884, -0.09944334626197815, -1.2412164211273193, 1.4306795597076416, 0.759703516960144, 0.8243697881698608, 0.2904095947742462, -0.22053129971027374, 0.72901850938797, -0.19410008192062378, -0.6621055006980896, 0.2145806849002838, 1.226813554763794, -1.0298678874969482, 1.0441166162490845, -1.2303885221481323, -1.249227523803711, 0.5335709452629089, -0.13036608695983887, 0.5406877994537354, -1.540340781211853, 0.513418972492218, 1.4291290044784546, 0.37077102065086365, -0.9682027101516724, -0.015403538942337036, -0.4777461588382721, -0.3053191900253296, 0.8730780482292175, -0.1620061993598938, -1.439560055732727, -0.540541410446167, -2.394273519515991, 1.4825572967529297, -0.1933748424053192, -1.3721193075180054, -0.7448217272758484, -0.6804488301277161, -0.2873721718788147, -0.6024680137634277, -0.526667058467865, -0.4133338928222656, -1.6418743133544922, 0.3114975094795227, -0.4383244514465332, 2.0988845825195312, 0.5997116565704346, -0.2904927134513855, 0.11589930206537247, -0.7367687225341797, 0.6411008834838867, 0.02303856983780861, 2.375018358230591, -0.7168028950691223, -0.6916148662567139, -1.0438785552978516, -0.8360361456871033, 1.2875066995620728, 0.03575112298130989, -0.5926814675331116, 0.2877478301525116, 0.4356112778186798, -2.4865753650665283, 0.8838826417922974, 1.5291028022766113, -0.5620602965354919, 0.3309328258037567, 1.503838300704956, 1.2434072494506836, 0.5627745389938354, -0.1477230191230774, -0.6963433623313904, -0.677537202835083, 2.1738476753234863, -0.8264269828796387, -0.4639166295528412, 0.8984783291816711, 0.7254264950752258, -0.5769482851028442, 0.4010867476463318, -0.06759513169527054, -0.6920831799507141, -0.38906899094581604, -0.13960805535316467, 1.3250887393951416, 1.7386358976364136, -1.3989670276641846, 2.3263328075408936, -0.0854203999042511, 0.49209052324295044, 1.402785062789917, 0.5682656168937683, -0.8021600246429443, -1.3030070066452026, 0.8001224398612976, 0.2919464111328125, -0.5195525288581848, 2.151857852935791, 0.2307548075914383, 0.7515247464179993, -1.6249995231628418, 1.0272170305252075, -1.4972208738327026, -0.9785753488540649, -0.9087153077125549, -1.0429580211639404, -1.1105799674987793, -0.1409228891134262, -1.2337480783462524, 2.1873226165771484, -2.168605089187622, 0.9935443997383118, -0.362218976020813, -0.1689065396785736, -0.7572482228279114, -1.6859675645828247, -0.5635733008384705, 0.17576557397842407, 0.5489427447319031, 1.7867112159729004, -0.23115593194961548, -1.3477532863616943, -1.5483440160751343, -0.18162277340888977, -1.7568033933639526, 0.21450139582157135, 1.3264453411102295, -0.9246891736984253, 1.1708343029022217, 0.5687677264213562, -1.1471060514450073, 0.9904868602752686, -0.16492299735546112], [0.5196601748466492, 0.3442573547363281, -1.2658143043518066, 1.0713891983032227, -3.2573256492614746, 1.7531687021255493, 0.28733712434768677, -0.20339861512184143, 0.989229142665863, -0.2093353569507599, 0.3198475241661072, -0.1380995810031891, 0.10734374821186066, -0.824411153793335, -0.3688121736049652, -0.5973363518714905, -2.055626153945923, -0.43578052520751953, 1.590950608253479, -0.03654859587550163, -1.1894826889038086, -0.20023474097251892, -0.38782960176467896, 0.12137195467948914, 0.11420795321464539, -0.6891223192214966, 1.3012950420379639, -0.980060875415802, 1.4882949590682983, -0.46641018986701965, -0.8188351988792419, -1.1738017797470093, 0.9410136938095093, -1.381561279296875, -1.2320809364318848, -0.4246038496494293, -0.14958378672599792, 1.7242136001586914, 1.0246013402938843, -0.8642914891242981, -0.4293193519115448, 0.5836081504821777, 0.2951827347278595, -1.659864902496338, -0.36335527896881104, -1.5239862203598022, -0.00873956736177206, 0.37089017033576965, 0.03764266148209572, 0.24686463177204132, 0.24972504377365112, -0.12155262380838394, 0.8369075059890747, -0.427632212638855, -0.19446176290512085, -1.4578062295913696, 0.6335375308990479, -0.8790875673294067, -0.36649253964424133, -0.07700952142477036, 1.4807075262069702, 0.12606458365917206, -1.0521924495697021, -0.2927502989768982, -1.690765142440796, -0.6244939565658569, 0.7657992243766785, 0.9215762615203857, 0.7627559900283813, -1.084426999092102, -1.1309781074523926, 0.5630684494972229, 0.27405399084091187, 0.06523208320140839, -0.8331985473632812, 1.0209530591964722, 0.9593649506568909, 1.6869066953659058, -0.21091195940971375, -1.0188722610473633, 0.20888489484786987, 1.044816493988037, -2.5863196849823, 0.5553922653198242, -0.0959317609667778, 1.2467055320739746, 1.8052678108215332, -0.9984890222549438, -0.3385619819164276, 0.08171139657497406, -0.11174783855676651, 1.649869680404663, -1.348541498184204, 1.2617111206054688, 0.8034568428993225, 1.2062455415725708, -0.44296205043792725, -0.06332844495773315, 0.8584219217300415, -0.8909450173377991, -2.5517141819000244, -0.8557952642440796, 1.140302300453186, -0.39327293634414673, -0.20519965887069702, 0.017098987475037575, -0.19116099178791046, 0.3924236297607422, 0.6573539972305298, 1.8233972787857056, 0.7119783759117126, 1.1951488256454468, 1.3117619752883911, -0.3059103786945343, 1.4797465801239014, 1.4361711740493774, 1.504406452178955, 1.0383633375167847, -0.05864349380135536, -1.0501060485839844, -0.0026031469460576773, -1.1454927921295166, -1.4255131483078003, 1.6981934309005737, -0.20438548922538757, -1.1497756242752075, 0.6279611587524414, -0.6090515851974487, 0.30505627393722534, 1.7131720781326294, 1.603958010673523, 0.056710049510002136, 0.9842175245285034, -1.4159095287322998, 1.7529923915863037, -0.08811474591493607, -0.789830207824707, 2.535398006439209, -0.3707532286643982, -1.6926082372665405, -0.6325574517250061, 0.3803330957889557, 1.2622852325439453, -0.5657774806022644, -0.13778869807720184, -0.7172585725784302, 1.511049747467041, -0.9256738424301147, -0.802935779094696, -0.8360876441001892, 0.041658733040094376, -0.20800229907035828, -0.5529001951217651, 0.8534133434295654, 2.2350144386291504, 0.1659221351146698, -0.8226180672645569, 1.5067423582077026, 0.1888161599636078, -1.77656888961792, 0.44602203369140625, -0.23605754971504211, 0.610387921333313, 1.6262664794921875, -0.8926836848258972, -0.21953293681144714, -1.548195481300354, 1.2544971704483032, 0.8876330256462097, 1.127519965171814, 0.5669263601303101, -1.733580231666565, -0.19963210821151733, -0.03413497656583786, -0.25616857409477234, -0.06707166135311127, 1.2877197265625, -1.3218518495559692, -0.5917098522186279, 0.054401472210884094, 0.8957613110542297, -0.25148552656173706, -0.48978665471076965, 1.5423133373260498, 1.5619508028030396, -0.34551966190338135, 0.7814162969589233, -0.007670524995774031, -1.9471180438995361, -0.25913742184638977, 1.6904542446136475, -0.6159297823905945, 0.2301001101732254, -0.4911080002784729, -0.5705959796905518, -0.04695351421833038, 0.20957998931407928, 1.418246865272522, 1.7676137685775757, 2.8239002227783203, -1.3003031015396118, -0.2878997027873993, -0.03450877219438553, 1.034471035003662, 0.20330829918384552, -1.0028809309005737, 0.7100529074668884, 1.2023987770080566, 2.2388088703155518, -0.7042859196662903, 2.982713460922241, -1.7459347248077393, 0.4331531226634979, 0.9151442646980286, 0.0035263069439679384, -0.6586722731590271, 1.2214233875274658, -1.3330514430999756, 0.30248889327049255, -1.530895709991455, -0.7623867392539978, -0.5508326292037964, -0.8556548953056335, -0.07730648666620255, -0.3518565595149994, 0.8385811448097229, 0.5967875719070435, -0.48166191577911377, -0.07423795759677887, -0.6949899792671204, 0.3680785894393921, -1.9342479705810547, 0.958081841468811, -1.2628626823425293, -0.4778235852718353, 0.12740130722522736, -0.7552725672721863, 0.47295045852661133, -1.842787265777588, 1.159338116645813, -0.8319479823112488, 0.24829813838005066, 0.6500304341316223, 0.2991810142993927, -1.5274713039398193, -1.1983823776245117, 1.997664213180542, 0.5940046310424805, 1.4778010845184326, -1.3771611452102661, -0.20288890600204468, 0.11742985248565674, -0.5530650615692139, 0.3429993987083435, -0.06647886335849762, -0.8364983797073364, -1.349051594734192, 0.3590691089630127, 0.29367026686668396, -1.7516330480575562, -0.5386654734611511, -2.516587018966675, -0.9214177131652832, 0.7969661951065063, -0.722679615020752, 0.04958004131913185, -1.8615593910217285, -0.4363420903682709, 0.7889508008956909, 0.737065851688385, -0.2890847623348236, 0.39627838134765625, 1.5303274393081665, -0.5946551561355591, 1.5269938707351685, -1.039912223815918, -1.1069899797439575, -0.405164510011673, -0.7491359114646912, -0.19908763468265533, -0.2540455758571625, -0.7566393613815308, 0.948675811290741, -2.5284390449523926, 1.399690866470337, 0.5144311189651489, 1.576350450515747, -0.4798792898654938, -0.49215975403785706, 0.1903931349515915, -1.6800297498703003, 0.048733342438936234, 1.2366485595703125, 1.2335147857666016, -1.638790249824524, 0.7762638330459595, 0.12763701379299164, -0.48553675413131714, 0.34494203329086304, -1.6807383298873901, 1.8221937417984009, -2.5934038162231445, -0.5082535743713379, -1.2676235437393188, 1.766204833984375, 0.24462133646011353, 1.581276297569275, -1.1418405771255493, 2.1396443843841553, -0.13497759401798248, -0.4868777096271515, 0.9847397208213806, 2.1578726768493652, 0.7853488922119141, -0.4108680486679077, -0.03480564430356026, 0.913580060005188, 0.28009071946144104, -1.9681427478790283, -1.290069580078125, 1.5561957359313965, 1.4657269716262817, -0.5757930278778076, -0.9122341275215149, -1.3731968402862549, -0.6050107479095459, 1.9732214212417603, -0.6197714805603027, 0.6681331992149353, 0.9177385568618774, 0.3777845501899719, 0.8784165382385254, -1.3881924152374268, 1.5735998153686523, -1.7912821769714355, 0.5054524540901184, 0.29457417130470276, 0.33010610938072205, 0.7414352297782898, 0.7170736193656921, -0.05880112946033478, 0.290558397769928, 0.20300108194351196, 0.42732876539230347, 0.6687762141227722, 0.5435386300086975, -1.7121402025222778, 0.4865078926086426, 0.44186216592788696, -1.9759634733200073, 1.4757498502731323, 0.27871137857437134, -1.894830584526062, -0.8224891424179077, -0.6310526132583618, -0.9654967784881592, -0.355326384305954, -0.09299016743898392, 1.2563896179199219, 0.7655307054519653, 0.6385114789009094, -0.9667850732803345, 1.4346224069595337, -1.1710816621780396, -0.45951715111732483, 1.250454306602478, -0.1185726672410965, 0.5624945163726807, -1.0327668190002441, 0.7311326861381531, -0.6936218738555908, -0.5281127691268921, 0.20537003874778748, 1.6094027757644653, -1.5292505025863647, 0.45924699306488037, 0.6520936489105225, -0.4233434498310089, 1.0657644271850586, 0.143129363656044, -1.111387848854065, -1.3455239534378052, -0.7134366631507874, 0.6753814220428467, -1.7705070972442627, 0.3504246175289154, 0.5739310383796692, -0.42569562792778015, -0.7289420962333679, 0.495800256729126, 1.2641645669937134, -0.6265324354171753, -0.6475191116333008, 1.5510724782943726, 0.615848183631897, 0.4591997563838959, 1.5563762187957764, 1.703843593597412, -0.8231090903282166, 1.4120630025863647, -0.05713563784956932, 0.528623640537262, -0.08442264795303345, -0.6690577864646912, 0.13154786825180054, -1.5503610372543335, 0.8264398574829102, -1.849379301071167, 0.3840165436267853, -1.454375147819519, -0.09806662797927856, -0.44008708000183105, -1.7142316102981567, 0.028760822489857674, 1.8688695430755615, 0.4708549976348877, -1.0973323583602905, 0.3494904339313507, -1.0592628717422485, -0.21784847974777222, -1.5952199697494507, -0.44140011072158813, -1.3097790479660034, -0.6324912309646606, 0.08117424696683884, 1.0239006280899048, 1.1091432571411133, 2.0601227283477783, 1.9960522651672363, -0.7703786492347717, -0.8449783325195312, 2.4392435550689697, 0.442484050989151, 0.4021693170070648, 0.7392634153366089, 1.0436006784439087, -0.3939868211746216, -0.4229888916015625, 1.0554382801055908, 0.028017714619636536, -0.9325914978981018, 0.8462439179420471, -0.5299870371818542, -1.3733148574829102, -0.013142562471330166, -0.3546645939350128, 0.9421577453613281, -1.2745157480239868, 0.5501880049705505, 1.7742267847061157, -0.1331239491701126, -0.48887521028518677, -1.0655919313430786, 1.1906417608261108, -0.44363051652908325, 1.4520068168640137, -0.2238316833972931, 0.6435953378677368, 0.00485208909958601, -0.3232935965061188, -0.2545263469219208, 0.3090665638446808, 0.43502387404441833, -1.9188292026519775, -0.3389143645763397, 0.7508545517921448, -0.16000039875507355, -1.4481801986694336, -0.42795470356941223, 0.3684631884098053, 0.2263021618127823, 0.2094162255525589, 1.9164327383041382, -0.8464493751525879, 0.39238178730010986, -0.4042106568813324, -1.2207144498825073, 0.9243815541267395, 1.7275466918945312, 0.9123496413230896, 0.4243764579296112, -1.4066261053085327, -0.20538851618766785, -0.7614759802818298, 0.399515837430954, -2.124303102493286, -0.78208327293396, -2.240868330001831, 0.47124576568603516, 0.7578064799308777, -0.35794755816459656, -0.16875365376472473, -1.510509967803955, 1.1409904956817627, -0.7610234618186951, -0.7557171583175659, 0.09172527492046356, -1.629592776298523, -0.3126625716686249, 0.6775509119033813, 0.2923479676246643, -0.3149004578590393, 0.28140464425086975, -0.38643285632133484, 1.1573817729949951, -0.21326476335525513, 1.367673397064209, 0.16599683463573456, 0.5747506618499756, 0.10895848274230957, 0.7515326738357544, -1.8835160732269287, 2.0098445415496826, 0.11351646482944489, -1.1983702182769775, 0.1510530561208725, -1.518027663230896, -0.35182416439056396, -0.01915423944592476, -0.021285530179739, -1.3439031839370728, -1.5060476064682007, -0.2916964292526245, 1.0249372720718384, 0.4941871762275696, 0.9660978317260742, 0.3104841411113739, 0.3221244513988495, 0.16418980062007904, -1.1765891313552856, -0.3956623077392578, 1.9031089544296265, 0.7482218146324158, 0.33562445640563965, 0.05796722695231438, 2.870389938354492, 0.6834790110588074, 0.14505180716514587, -1.6288949251174927, 2.5298779010772705, -1.4224390983581543, 0.5224103331565857, -1.504357099533081, -1.2507052421569824, 1.3215744495391846, -2.3351309299468994, 0.36361485719680786, 1.2639508247375488, 0.7659823298454285, -0.6696151494979858, 0.7367604970932007, -0.39575618505477905, -0.5167528390884399, 0.8578292727470398, 0.9074533581733704, -1.2654242515563965, -0.24674710631370544, 0.6133460998535156, -0.045792847871780396, 2.805830955505371, 1.719910740852356, 1.5364725589752197, -0.28304746747016907, -0.4343113601207733, -0.32840844988822937, 1.658226728439331, 1.0766024589538574, 1.8207612037658691, -0.3646463453769684, 0.020758094266057014, -0.8568524718284607, -0.7902193665504456, -1.8435864448547363, -1.0409071445465088, 1.4830718040466309, 0.11027474701404572, 1.153605580329895, 0.6977463960647583, 0.0828242376446724, 1.689627766609192, 1.3149938583374023, -1.736536979675293, 0.713429868221283, -0.20162363350391388, -0.9181776642799377, 0.888645350933075, 0.1272154003381729, -2.0880515575408936, -0.8713048100471497, -0.2314935326576233, -1.675140380859375, -0.5106891989707947, 1.0102412700653076, 0.6444833874702454, -0.5332087874412537, 1.5402449369430542, 0.29431018233299255, -0.040697865188121796, 0.2522583305835724, -0.22912278771400452, 0.1414608359336853, -0.6221248507499695, -0.37199869751930237, -2.0744810104370117, -1.578711748123169, 0.7258629202842712, -1.262133002281189, 1.3863673210144043, 0.545042097568512, 0.6357711553573608, -0.28768816590309143, 1.0693374872207642, -0.4902336001396179, -0.15417172014713287, 1.1298106908798218, 0.3567052483558655, -1.2581608295440674, -2.8170509338378906, -1.8948373794555664, 1.8943973779678345, 2.116103172302246, -0.2637036442756653, -0.87239009141922, 1.048907995223999, -0.9396749138832092, 2.051252841949463, 0.7362497448921204, 0.5402699708938599, -0.9570164680480957, -1.4925775527954102, -0.3260995149612427, 0.8998542428016663, 0.36878710985183716, -1.3011735677719116, -0.3411193788051605, 0.2721145749092102, -1.38691246509552, 2.2435107231140137, 0.5175741910934448, 1.3737874031066895, -1.5552685260772705, 0.26048508286476135, 1.010486364364624, -0.1315714567899704, -1.0189344882965088, 1.4596558809280396, -0.5246357917785645, 1.6332165002822876, -0.2129775434732437, 1.4641319513320923, -1.2594332695007324, 0.13056418299674988, -0.4589438736438751, -2.2947652339935303, 0.8023496866226196, 2.0411999225616455, -0.5854113101959229, 0.5195322036743164, -0.5864320397377014, -0.16167066991329193, -2.303596258163452, -0.44734713435173035, -2.0941076278686523, -0.17988532781600952, -0.2622244954109192, -0.6898289322853088, 0.757978618144989, 0.45552971959114075, -0.45996594429016113, -1.220900058746338, 1.3538146018981934, -0.6505889296531677, -0.21391652524471283, -0.429482102394104, 0.6451598405838013, 1.268505573272705, 1.1075981855392456, -0.098631851375103, -0.8669654130935669, -1.290259599685669, -0.7056592106819153, 0.8589742183685303, -1.6703583002090454, -2.2821381092071533, 1.193908452987671, 0.8055943846702576, -0.6873555183410645, -0.7566829323768616, 0.2639603316783905, -1.007724642753601, -1.0428314208984375, -0.9479349851608276, -0.23057116568088531, -1.9520080089569092, 1.5981943607330322, -3.064828634262085, -0.025177594274282455, -0.002323198365047574, 1.598664402961731, -1.8955885171890259, 0.9134536981582642, 0.32493042945861816, -0.9990633130073547, -0.39015117287635803, 0.5057687163352966, 0.5888047814369202, -0.04378832131624222, -0.2355235069990158, -0.2490864396095276, 0.9479909539222717, 0.2565014362335205, 1.375957727432251, -0.27788853645324707, 0.7673066258430481, -0.26189592480659485, -0.5103501677513123, -0.9232831597328186, -0.5776379108428955, 0.23352350294589996, -1.6265208721160889, -0.8531803488731384, -0.5377969741821289, 0.8385929465293884, -0.45338842272758484, -0.6806280612945557, 0.13257895410060883, 0.5036070942878723, 0.030943885445594788, 1.132684350013733, 0.39860257506370544, 0.5714156627655029, -1.4992780685424805, 0.16844189167022705, -0.6906214356422424, -1.404356598854065, -0.028292544186115265, -0.2342420220375061, 0.11517442762851715, 2.228884220123291, -0.5199477672576904, -0.2714580297470093, -0.09760617464780807, -0.4912167489528656, -1.8153619766235352, 2.493278741836548, -1.9131426811218262, -1.0949004888534546, -0.005960110574960709, -0.982485294342041, 0.4900171756744385, 0.607724130153656, 0.579132080078125, 0.6497640609741211, 0.2942565679550171, 1.2372411489486694, -0.577783465385437, 0.5141669511795044, -0.10841775685548782, 0.5537154078483582, -1.270014762878418, -0.44747766852378845, -0.12426438927650452, -0.35636183619499207, 0.6770754456520081, 1.2751672267913818, -0.6892005205154419, 0.1145772710442543, 1.405420184135437, 0.07341022789478302, -0.18476836383342743, -0.7095656991004944, 1.813925862312317, 1.247190237045288, -0.7693077921867371, 1.4648234844207764, 0.11659141629934311, -1.246016263961792, -1.77048659324646, 0.26497137546539307, -0.5972923040390015, -0.5656479597091675, 1.0752252340316772, 0.6724578142166138, 1.1605178117752075, -1.1925324201583862, -0.613400936126709, -1.7882964611053467, 2.20088529586792, -1.2570738792419434, 0.8286410570144653, 0.30054759979248047, -0.94427090883255, 0.15557371079921722, 2.1509344577789307, 0.3477426767349243, 0.3552079200744629, 0.2993698716163635, 0.5052455067634583, 0.4153897166252136, -0.2535371482372284, 0.8502840995788574, -0.515474259853363, -0.5466687083244324, -0.08490210026502609, -0.5401239991188049, -0.5079793930053711, 0.08347015082836151, 0.2864895761013031, 1.5261071920394897, -1.9223088026046753, 0.780375599861145, -0.5071380734443665, 0.2681829631328583, -0.3373902440071106, 2.330341100692749, -0.5087526440620422, 0.3329406678676605, -1.3369648456573486, -0.7471554279327393, 2.4855854511260986, -0.4299517571926117, 1.0670931339263916, 1.4287176132202148, 0.8580089211463928, -0.3128286898136139, 1.593856930732727, 1.1348168849945068, 0.46826207637786865, -1.7542296648025513, 0.09241370856761932, 0.762191116809845, -0.6264975070953369, 0.141681969165802, -1.3401294946670532, -0.2595021426677704, -0.279874324798584, 1.4041935205459595, 0.941098690032959, 0.8359390497207642, 0.26342740654945374, -0.494771808385849, -1.9474999904632568, -0.5342544317245483, 1.3209439516067505, -0.046995483338832855, -1.2111986875534058, 0.5815420150756836, 0.7042384743690491, -0.9193727970123291, 0.5606990456581116, 1.077627420425415, 0.5348214507102966, -1.6450233459472656, 0.7768579125404358, -0.5047972202301025, 0.5058262944221497, 0.7091442942619324, 0.49782416224479675, 0.08201346546411514, 0.6415920257568359, 1.3289594650268555, -0.3557840883731842, -1.1913267374038696, -2.228778123855591, -1.1042506694793701, -0.6949780583381653, -0.18462562561035156, -0.8991572260856628, -0.6973040699958801, -1.0859452486038208, -0.5469352602958679, 1.621208906173706, 0.5272176861763, -1.2798255681991577, -0.6409768462181091, 0.28041568398475647, -0.8837785124778748, 0.41063418984413147, -1.5359625816345215, -0.5342183709144592, 0.2649974524974823, -1.4862462282180786, 0.5271056890487671, 1.0464961528778076, 0.8193293213844299, 0.0816323459148407, 0.17913936078548431, -0.1534411907196045, 0.02247033827006817, -0.8023596405982971, -0.21606174111366272, 0.45399054884910583, -0.1538495123386383, -1.1016793251037598, 0.396830677986145, -0.5985304713249207, 1.4430793523788452, -0.36557620763778687, 0.5298744440078735, 0.773553192615509, 1.092944622039795, -1.1782511472702026, 1.3333759307861328, 0.5991251468658447, 0.0005262453923933208, -1.165342092514038, 1.9453438520431519, 0.5557786226272583, 1.1535465717315674, 0.8933107256889343, 1.515206217765808, 0.26631224155426025, 0.8407377004623413, -1.4354767799377441, -0.10764498263597488, 0.925099790096283, 0.39993202686309814, 0.825464129447937, -1.1754257678985596, 1.0757200717926025, -0.9688583016395569, 0.12005455046892166, 1.5679268836975098, 2.2886650562286377, -0.5873113870620728, 0.9413858652114868, 0.3338368535041809, 0.3995060324668884, 0.3802945017814636, -1.1388347148895264, 0.47117218375205994, -0.2305060178041458, -0.7983992695808411, 0.48614537715911865, -1.3458203077316284, 0.34677934646606445, 0.9089463353157043, -1.0632754564285278, -1.278127670288086, -0.17007769644260406, -0.8080798387527466, 1.7378689050674438, -0.813356339931488, -0.6478984355926514, 0.5346484780311584, 0.8529691100120544, 0.02055547945201397, 1.516613245010376, -0.6027694344520569, 1.638985514640808, -1.6312861442565918, 1.5076560974121094, 1.0567255020141602, -0.05426571145653725, 0.3203037977218628, 0.4935879707336426, -4.4672136306762695, -0.8506852388381958, -0.8439865112304688, 0.06572645157575607, 0.3629552125930786, 0.29472726583480835, 0.8667669892311096, -0.24004366993904114, -0.19048023223876953, -0.75853031873703, -1.927205204963684, 1.1956919431686401, -0.012442026287317276, 0.21701113879680634, 1.210550308227539, -0.9919145107269287, 0.7656314969062805, -0.5807334184646606, 0.6406128406524658, 1.070935845375061, 1.1000419855117798, -0.04464033246040344, 0.029431497678160667, 1.316280722618103, -0.08115716278553009, 1.0651805400848389, 0.14212413132190704, 1.573759913444519, -0.07062358409166336, 0.5045861601829529, -1.616263508796692, -0.395272433757782, 1.3932206630706787, -1.1377674341201782, -0.17945082485675812, 1.2024955749511719, -1.684169054031372, 3.211325168609619, 0.795970618724823, 0.0747218132019043, 2.0033342838287354, 0.4387262165546417, -0.7686690092086792, -0.4869360029697418, 1.683868169784546, 0.5435615181922913, 0.0004689349443651736, 0.6593204736709595, -1.8218495845794678, 0.2962125837802887, -1.098663091659546, -1.0390276908874512, 0.1890302151441574, 0.028187500312924385, 0.8884901404380798, -0.8927932381629944, -0.03326542302966118, -0.5005744099617004, 1.0964139699935913, 0.5716128945350647, -1.4022059440612793, 0.4862981140613556, 0.8815004825592041, 0.2303733378648758, 0.276720255613327, 1.6032696962356567, 0.9110164046287537, -0.5184401273727417, -0.24705180525779724, -0.6677318811416626, -1.00428307056427, -0.7176284790039062, 0.7873706221580505, 0.46532294154167175, 0.28055548667907715, -1.8400697708129883, -0.5913835167884827, 1.291609287261963, -1.3248217105865479, 0.14530427753925323, -2.2091386318206787, 0.18325355648994446, 1.5640884637832642, 0.21956804394721985, 1.3876532316207886, 1.0640382766723633, 0.38138464093208313, 0.5049014687538147, -1.1572006940841675, -0.1055370345711708, -0.009303789585828781, 1.0955164432525635, 0.7639223337173462, 0.1391163170337677, -0.31036075949668884, -1.3051798343658447, 0.19862742722034454, -0.16573956608772278, -0.700364351272583, -0.6428564786911011, -1.282155990600586, 1.3832424879074097, -0.954203724861145, -1.165903091430664, 0.5234273076057434, 1.8523783683776855, 0.39072751998901367, 0.025774838402867317, -0.5377864241600037, -0.2612617015838623, -1.0834777355194092, -1.1236132383346558, 1.1045355796813965, -0.5562390685081482, -1.3920655250549316, -1.395846962928772, 0.7696000933647156, 1.1053452491760254, 1.4115245342254639, -2.3013288974761963, 0.38454264402389526, 0.12501446902751923, 1.6900055408477783, -0.16661274433135986, -1.3891582489013672, 1.033475637435913, 1.6625839471817017, -0.40635427832603455, -0.3516487181186676, 1.1716909408569336, -0.07215497642755508, -2.263761520385742, 0.6153950691223145, 1.908799409866333, 1.7332416772842407, -0.5594609379768372, -1.3307421207427979, 0.12105754017829895, 0.5693637132644653, 1.174655795097351, 1.4045891761779785, 0.473890483379364, -1.5998725891113281, 0.19290411472320557, -0.7930330038070679, -0.8045336604118347, 1.0255719423294067, -0.43661028146743774, 0.4679834842681885, 0.015415184199810028, -0.9036957621574402, 1.1703331470489502, 0.157936692237854, 0.015830637887120247, -0.16501455008983612, 0.006742280907928944, -1.9059830904006958, -0.33544623851776123, -0.8512341380119324, -2.486900806427002, 0.8150706887245178, 0.18867044150829315, 0.1656714230775833, -1.3754072189331055, 0.3880508244037628, 0.20827879011631012, 0.9190854430198669, 1.6670323610305786, -0.755156397819519, 0.4851122200489044, -1.1431244611740112, 0.42432278394699097, 0.18732942640781403, 0.8406260013580322, -0.23463638126850128, 1.8928160667419434, 0.6514475345611572, -0.8224900960922241, -0.6263485550880432, 0.3398463726043701, 0.6040781736373901, -1.2738851308822632, -0.7638430595397949, 0.7443161606788635, -1.3594247102737427, -1.0629843473434448, 0.17771214246749878, -0.061488378793001175, 1.301358699798584, 0.14499232172966003, -2.0524818897247314, -0.9145408868789673, -0.5910618305206299, -0.5103505849838257, -0.832873523235321, -0.8247354626655579, 0.15256330370903015, 1.2858164310455322, -2.401898145675659, 1.8087897300720215, 0.8089821934700012, -1.0901414155960083, -0.17741939425468445, -0.8260907530784607, 0.5210051536560059, -0.7689345479011536, -1.3338055610656738, -1.858227252960205, 1.6475907564163208, 2.3879928588867188, -1.3451225757598877, 0.2800864577293396, -0.5147401094436646, 0.44090041518211365, 0.7827703356742859, 0.09339205175638199, -0.10702826082706451, 0.13587598502635956, 0.28075897693634033, 0.8654548525810242, 0.27778857946395874, -0.35536158084869385, -1.220218539237976, 0.6203147768974304, 1.2298663854599, -1.5020272731781006, 0.5298662781715393, 1.4023545980453491, -0.4870009422302246, 0.3805975615978241, -0.7519565224647522, 0.30900484323501587, 1.1516306400299072, -0.062475163489580154, -1.155177116394043, 0.05888665094971657, -1.353639841079712, 0.12971056997776031, 0.9831411242485046, 0.05906357988715172, -0.577433168888092, 0.2693208158016205, -0.49841946363449097, -0.19197770953178406, -1.613429307937622, -0.607809841632843, 3.3361244201660156, 0.892497718334198, -1.392149567604065, 0.2777668833732605, -1.0626529455184937, -0.4384629428386688, 0.1877322494983673, 0.33084505796432495, -0.6568812131881714, -1.142303228378296, -0.3763924539089203, -1.1850497722625732, 0.6353797912597656, 0.23820100724697113, -0.26673653721809387, -0.5740820169448853, 0.509448766708374, -1.2800804376602173, 2.082209587097168, -0.06580640375614166, 0.17069445550441742, 0.10202844440937042, 0.9075289368629456, -0.4943762421607971, 0.5736062526702881, 0.32597586512565613, 0.9153748154640198, -0.2168302834033966, 1.6957253217697144, 0.04255944862961769, 0.8805667757987976, -0.44370517134666443, 0.14419199526309967, 1.0261484384536743, -0.43317463994026184, -0.7233030796051025, -0.1448172628879547, 0.892625629901886, 0.6971006393432617, 1.251707911491394, 1.8578089475631714, -2.129538059234619, 1.1453124284744263, 0.03986223787069321, 0.31618526577949524, 0.05527958646416664, -0.9977815747261047, -0.6344436407089233, 0.28688323497772217, -0.8315291404724121, 0.09434588998556137, 0.0038287697825580835, 0.8880351781845093, -0.8564577698707581, 0.11413848400115967, -0.07607562839984894, -0.45405247807502747, 1.473141074180603, -0.685355544090271, -0.05637846887111664, -0.08016557991504669, 0.18135707080364227, 1.2505370378494263, 1.5109195709228516, -0.7066669464111328, 1.2794287204742432, 0.30086642503738403, -1.6989978551864624, 0.9184624552726746, -0.3131602704524994, 1.313215732574463, -0.6653231382369995, 1.0054445266723633, 1.549525499343872, 1.6111884117126465, 0.37152907252311707, -0.18156850337982178, -1.503290057182312, 0.3601052463054657, 0.5518912672996521, 0.6632131338119507, -0.43944883346557617, -0.06519509106874466, 2.198319673538208, 0.18265603482723236, 2.320000648498535, 1.133934497833252, 0.49182042479515076, 0.3794650137424469, 0.13332314789295197, 0.7701177000999451, -0.036572106182575226, -0.047789230942726135, -0.38592228293418884, 0.6569181084632874, -2.3886094093322754, -1.1214731931686401, 0.6912217736244202, 1.614103078842163, 0.17511795461177826, -0.5839982628822327, -0.20854447782039642, -0.38120704889297485, 0.8055057525634766, -0.45988574624061584, -1.8343318700790405, -0.19298341870307922, -0.6992760896682739, -0.324144572019577, 0.713909387588501, 0.4423595368862152, -0.47954031825065613, -1.3121721744537354, -0.14946943521499634, 0.4577943682670593, -0.21869395673274994, -0.23679345846176147, -0.009271637536585331, -1.0013896226882935, 0.15556757152080536, -0.33399897813796997, -1.9538880586624146, -0.8472024202346802, -2.0404398441314697, -0.8746119141578674, -0.9316437244415283, -0.5879423022270203, -0.7169433832168579, -0.6009561419487, 0.5974408388137817, 1.0243037939071655, 1.065175175666809, -0.8912065029144287, -0.36122071743011475, 0.6815552711486816, 1.7616666555404663, 0.004754764959216118, 1.6321604251861572, 1.447600245475769, -0.4036071002483368, -0.6243627071380615, -0.2753990888595581, -0.5893350839614868, 0.07853275537490845, 0.9498873949050903, 0.8222106099128723, 0.5355640649795532, 0.598453164100647, 0.2929632365703583, 1.1025046110153198, 1.0948578119277954, 0.5688093900680542, 0.3823262155056, -0.590516984462738, 1.055211067199707, 1.0435254573822021, 1.4096589088439941, -0.10157700628042221, 0.7632284760475159, -0.6420523524284363, 0.04805262014269829, -0.1439862996339798, -2.750570058822632, 0.8270658850669861, 1.0995972156524658, 0.4599932134151459, -1.170777440071106, -2.4810597896575928, 1.5289756059646606, -0.2945733368396759, -0.447157621383667, 1.8337985277175903, -0.1652744859457016, 0.23114170134067535, 0.8461247682571411, -0.22289817035198212, 1.2239978313446045, 0.3185092508792877, -0.21122480928897858, 1.4172728061676025, 0.6224626898765564, -1.5361236333847046, 0.9554734826087952, -0.37402769923210144, -0.28805649280548096, 1.4203323125839233, -1.354864478111267, -0.550041139125824, 1.0194965600967407, 0.5143996477127075, 1.504409909248352, -1.5918940305709839, 1.1222827434539795, -0.09464677423238754, -0.7218149900436401, -0.6097248792648315, 0.21658088266849518, -0.12823528051376343, -0.5870127081871033, 0.4403644800186157, -1.0996992588043213, 0.4397960901260376, -1.7880172729492188, 1.026389718055725, -0.44897007942199707, 0.6538313627243042, -0.4305816888809204, 0.18241067230701447, 0.12197411060333252, -0.655994176864624, 0.6576477289199829, 2.135183334350586, -0.28389057517051697, -1.6913083791732788, 1.8535858392715454, 0.7900915145874023, -1.1072633266448975, 1.0210376977920532, 0.7539626955986023, -0.26481685042381287, -0.11173809319734573, 0.06755141913890839, 0.19166144728660583, 0.625270664691925, 0.8411983847618103, 0.5535714626312256, -1.0328433513641357, -0.5703586339950562, 1.7846813201904297, -0.3631165027618408, 0.5286833047866821, 0.9145538806915283, -0.2524549961090088, -0.6782008409500122, -0.9504534602165222, 0.3934774398803711, 1.1121469736099243, 0.9646899700164795, -0.09864085167646408, 0.4087844789028168, 0.6406437754631042, 1.4933958053588867, 1.2485899925231934, -0.2688087821006775, 0.3225065767765045, -2.7074975967407227, 0.5171167850494385, -2.3651764392852783, 2.454566478729248, 0.158274844288826, -0.45587995648384094, 1.612009048461914, -0.19809472560882568, 1.0101048946380615, 2.0174765586853027, -2.952313184738159, -1.1352890729904175, -1.2058953046798706, 0.4323421120643616, 0.3720702826976776, -2.033937454223633, 0.25840625166893005, 0.30418580770492554, -0.24631191790103912, -0.05092773959040642, -0.4321211278438568, 0.02373204380273819, 0.08560602366924286, 0.2857089638710022, -0.2635131776332855, 0.12999069690704346, -1.082046389579773, 0.44454941153526306, 1.225301742553711, -0.19383779168128967, -1.1277610063552856, -0.3920941948890686, -1.245969295501709, -2.488292694091797, 0.8370677828788757, 0.06469302624464035, 1.2452316284179688, 0.6983106136322021, 1.0053791999816895, -0.7275859117507935, 0.4032244086265564, 0.29345977306365967, -0.035295791923999786, 0.0747455507516861, 0.43369060754776, 0.7654913067817688, -1.3551912307739258, 0.7038098573684692, -0.5246783494949341, 0.31558308005332947, 1.2008119821548462, -1.3595541715621948, -1.295084834098816, 0.8920004367828369, -0.09640855342149734, 1.156145691871643, 0.3115714192390442, -0.8567587733268738, -1.2432130575180054, 0.9091284275054932, 1.0893440246582031, -0.27850595116615295, -0.5461596846580505, 0.823408842086792, -0.1475379914045334, 1.156527042388916, 0.5771860480308533, -0.8198392391204834, -0.9118088483810425, 0.027246160432696342, -1.49232017993927, -0.025646911934018135, 0.7042351365089417, -0.7377256751060486, 0.7342563271522522, 0.2760621905326843, 0.00990376342087984, 1.012188196182251, -0.00610437011346221, 0.6585323810577393, -0.8029956817626953, 1.4019465446472168, -0.8549578785896301, -0.019255032762885094, -0.1971888691186905, 1.9905312061309814, 0.30405089259147644, -0.060814548283815384, 2.9184577465057373, -0.557097315788269, -0.5082178711891174, 0.07686469703912735, -0.08017178624868393, -1.1671770811080933, 1.536921501159668, 0.3672782778739929, 0.26390764117240906, -0.6275850534439087, 0.9611550569534302, 0.5070368647575378, 1.1852481365203857, 0.39381739497184753, 0.9295550584793091, -0.6166855692863464, -1.0840339660644531, 1.2013897895812988, -0.9850595593452454, 0.17206014692783356, -0.151722252368927, -1.5371110439300537, 0.31470590829849243, -1.401626706123352, 1.2923201322555542, 0.044860925525426865, 0.5588224530220032, -0.6847012042999268, -0.23184789717197418, -2.4663708209991455, 0.23441646993160248, 2.1061716079711914, 0.5300741791725159, 0.6012793183326721, 0.5900095701217651, 1.0480901002883911, 0.9895235896110535, -0.1535106748342514, 0.6343607306480408, -0.20989330112934113, -0.15097679197788239, 1.9489390850067139, 0.35211706161499023, -1.0737372636795044, 1.4782997369766235, -0.2493905872106552, -0.9959503412246704, -0.3847421109676361, 0.5379166603088379, -0.6154167652130127, 0.6082626581192017, -0.03787262737751007, 0.4176597595214844, -0.44729918241500854, 0.6167359948158264, -1.2735698223114014, 1.8764839172363281, 0.8238983750343323, -1.1881283521652222, -0.6194766163825989, 1.8182027339935303, -0.8539003729820251, -0.6896141767501831, 1.2442649602890015, -0.109723299741745, -0.6526157855987549, 0.6710736751556396, -1.179117202758789, 0.8711166977882385, -0.8328009843826294, 0.36298519372940063, -1.1097967624664307, -0.9746636748313904, -0.6647531986236572, 0.25882428884506226, 0.006899973377585411, -2.6542868614196777, -0.14161895215511322, 1.110062837600708, 0.18149442970752716, -0.7486861944198608, 1.1372120380401611, -0.7923945188522339, 0.49915236234664917, 1.637275218963623, -0.8492988348007202, -1.8898319005966187, 1.4244349002838135, 0.44991540908813477, 1.6684281826019287, 0.6411351561546326, -0.2849096655845642, -0.6107887029647827, 0.2606854736804962, 0.38732239603996277, 1.7432936429977417, 0.18866465985774994, 0.7653854489326477, -0.2706420421600342, -1.2999184131622314, -1.2687095403671265, -0.7967792749404907, 0.6041164398193359, 0.6931672096252441, -0.18435758352279663, -0.3122695982456207, 1.9120194911956787, -0.5125155448913574, -0.12083903700113297, -0.6378797292709351, 0.13354355096817017, 0.3429618775844574, 0.43670639395713806, 2.408782482147217, 0.5400393605232239, 1.286713719367981, -0.9634307026863098, 2.359926462173462, -0.8569916486740112, 1.2451852560043335, -0.6799439191818237, 0.5975291728973389, -2.350461483001709, -0.36341074109077454, -1.5450221300125122, 2.032790184020996, 0.13079984486103058, -0.49991390109062195, -0.09438253194093704, 0.47793763875961304, -0.832589328289032, 0.03700340911746025, 0.11994925886392593, 0.237185537815094, 1.028957486152649, 0.20012147724628448, 1.1165039539337158, 0.5307749509811401, 0.09987738728523254, 0.48251694440841675, 1.1794334650039673, -0.864061713218689, 0.33377230167388916, -0.12591023743152618, -0.8878916501998901, -0.6335504055023193, 0.21115721762180328, -0.14746612310409546, -0.3134869933128357, 0.8861863017082214, -0.7797073125839233, 0.6022413969039917, -0.4504872262477875, -1.019567847251892, 1.2772889137268066, 0.05728228762745857, -0.5671085119247437, 0.2846981883049011, 1.1661255359649658, 2.0409576892852783, 0.7382108569145203, 0.5121510624885559, 1.3862193822860718, -0.7208715081214905, 1.457626223564148, -1.0843859910964966, 0.8713709712028503, 0.2021329253911972, -1.559390902519226, -0.3277818560600281, 1.7459137439727783, -0.44356197118759155, 0.4796168804168701, 0.7531783580780029, -0.028544394299387932, 0.7196903824806213, 0.13622434437274933, 2.6981310844421387, 1.4642882347106934, -1.4635751247406006, -0.45282042026519775, 0.09787621349096298, -0.9487202763557434, 0.7796005010604858, -2.0545003414154053, 1.11387038230896, 0.019012071192264557, -1.5846036672592163, 0.478426456451416, 0.5618743300437927, -1.5712172985076904, 1.3216572999954224, -0.8628806471824646, 1.217193603515625, 1.154983639717102, 1.3801134824752808, -0.6702343225479126, -1.2947781085968018, 0.2725500166416168, -0.6812915205955505, -1.2054885625839233, 1.0999752283096313, 1.6388174295425415, -0.375225305557251, -1.2657065391540527, -0.917296290397644, -1.3069512844085693, -1.1976510286331177, 0.9662769436836243, -0.1345425695180893, -0.13994498550891876, -0.7664360404014587, -1.027722716331482, -0.6977453827857971, 0.7228061556816101, -0.7292807102203369, 1.0756845474243164, 0.41524767875671387, 1.2503162622451782, 0.5978444218635559, 0.43992871046066284, 2.958498477935791, 0.2969532310962677, 1.5880743265151978, -0.642988383769989, 1.1932530403137207, 0.24411946535110474, -1.8502451181411743, 0.009699772112071514, -0.9320178627967834, -1.3456584215164185, 0.11616025120019913, -0.37795090675354004, -0.04921228438615799, -0.40175557136535645, 0.03286290168762207, 0.976142168045044, -1.6462969779968262, -0.8714919686317444, -0.9051911234855652, -1.3620712757110596, 1.2976869344711304, -0.6897866725921631, -0.5492402911186218, 0.1988721489906311, -0.34874069690704346, -0.3134689927101135, -0.4221843481063843, 0.16916052997112274, -0.01733258180320263, 0.38476672768592834, -0.8686152100563049, 1.0044236183166504, -1.1472047567367554, -0.6333323121070862, 2.0419883728027344, 1.458333134651184, -0.6417822241783142, 0.17133991420269012, -0.35933956503868103, -0.15900826454162598, 2.0599772930145264, -1.1431306600570679, -1.3585039377212524, -0.1568351536989212, 0.6060221195220947, -0.8394815325737, -0.030801592394709587, 0.06731415539979935, -1.5097925662994385, -0.3130592703819275, 0.49684199690818787, -0.017041053622961044, 2.9962692260742188, 0.47097188234329224, -0.28073108196258545, 1.2540323734283447, -0.7197411060333252, 0.156121164560318, 1.120226263999939, -0.37726452946662903, -1.2010562419891357, 0.3382575809955597, -0.9181492328643799, -0.7926701307296753, 0.3911505341529846, -0.5699561834335327, 1.4042304754257202, 0.1841401755809784, 0.49797001481056213, 1.6500521898269653, -0.32910969853401184, 0.24622394144535065, -0.4099981188774109, -0.8603529930114746, 0.10269877314567566, -1.2178022861480713, -1.017425537109375, 0.05977974459528923, -0.5663668513298035, -0.4112017750740051, 0.5614245533943176, -2.075770616531372, 1.0397813320159912, -0.3740115165710449, -0.4081243872642517, 2.5321567058563232, -0.08844365924596786, 0.4879451096057892, -0.3361036479473114, 0.9472129940986633, 0.9203516840934753, 0.9738475680351257, -0.8309302926063538, 0.24529311060905457, 0.47380951046943665, -1.5573813915252686, 0.8922947645187378, 1.1040406227111816, 0.08723492175340652, 0.21423988044261932, 2.432603359222412, -0.564075767993927, -1.0563865900039673, -0.24107973277568817, -1.8195854425430298, 0.8299375176429749, 0.6515405178070068, 0.7611451148986816, 2.5789380073547363, -0.9399628639221191, 0.7271772027015686, 0.06829393655061722, -0.2750653326511383, 1.1213271617889404, -2.8144781589508057, 1.3259978294372559, 1.0929697751998901, -0.31203708052635193, -1.9447646141052246, 0.5263437628746033, 0.5141028165817261, 0.822359025478363, 0.20745539665222168, -0.6012450456619263, 0.6642841100692749, 0.5033515691757202, -0.0058147842064499855, -0.6412770748138428, -0.4269891381263733, -0.7731750011444092, 0.4496692717075348, -0.13404077291488647, -1.125952959060669, 0.7803239226341248, -0.11694280058145523, -0.40981677174568176, -0.15993455052375793, -1.6056885719299316, 0.44613537192344666, 0.43531227111816406, 1.6802449226379395, 1.644802212715149, -1.265418291091919, 0.13193286955356598, -0.6966288089752197, 1.0393224954605103, 1.1952420473098755, -0.6170392632484436, 0.9285401105880737, -0.632179319858551, -1.7843260765075684, -0.5211986303329468, 0.15384265780448914, 0.7625362277030945, -0.22182969748973846, 1.0179989337921143, -0.6868414282798767, -0.03957965970039368, 0.19790862500667572, 0.552353024482727, -0.17713844776153564, 1.192389965057373, 0.9546985626220703, 0.3180382549762726, -2.237478017807007, -0.17198894917964935, -1.105013370513916, 0.02110135555267334, -0.9452893137931824, -0.37114179134368896, 0.6708955764770508, -0.7028806805610657, 0.27578574419021606, 1.6291476488113403, 1.6605242490768433, 1.2329211235046387, -2.609405517578125, -0.6539246439933777, -0.7039661407470703, -0.46044740080833435, -0.09333404898643494, -0.30447837710380554, -1.1648908853530884, -2.0165302753448486, 0.5512627959251404, -1.2575187683105469, 0.9343438744544983, 0.3186684548854828, -0.39688315987586975, -0.4021914303302765, -0.3533937931060791, -0.6137296557426453, 1.6841561794281006, -0.4990769624710083, -0.42677542567253113, -0.40694835782051086, 0.3067900240421295, 0.7671458125114441, 0.16230060160160065, 0.21854892373085022, 0.014515584334731102, -1.129373550415039, 0.43668055534362793, -0.3173031806945801, -0.617841899394989, -0.7885721921920776, 0.2793811857700348, 1.7016429901123047, 1.24294912815094, -0.79423588514328, 0.5091050863265991, 0.876965343952179, -0.9410446286201477, -0.6311953067779541, 0.37609967589378357, -0.7729779481887817, 0.9193275570869446, -1.885396122932434, -0.7533950209617615, -0.2560160458087921, 0.05122293159365654, 0.027958594262599945, -0.3211669921875, -0.1484426110982895, -1.3754956722259521, -1.6845414638519287, -0.5659763813018799, -1.1354131698608398, -1.044490098953247, 2.296854257583618, -0.4834457039833069, -0.531154453754425, 0.019706696271896362, 0.8650391697883606, -0.20366144180297852, -0.15121129155158997, -0.3911832273006439, 0.013450575061142445, -0.7792640328407288, -2.0602173805236816, -0.7895569801330566, 0.3577818274497986, -1.770939826965332, -1.9325300455093384, -0.6152361631393433, 0.8255776762962341, 1.4803736209869385, 0.6762884855270386, 0.9329267740249634, 0.8284806609153748, 0.28265905380249023, 0.5081492066383362, 0.35206863284111023, 0.009153835475444794, -1.7166626453399658, -0.28151512145996094, -1.953831434249878, -0.04059309884905815, 0.5560885667800903, 0.27281320095062256, -0.09912259876728058, -0.6784224510192871, -1.466646671295166, 0.14395105838775635, -0.41006505489349365, -0.7004224061965942, -1.1590622663497925, 1.141697883605957, 0.40408435463905334, -0.9358383417129517, 1.0411220788955688, -0.038787659257650375, -0.1973249614238739, -0.6298115849494934, 1.224639654159546, 1.0933319330215454, 0.5290906429290771, 1.2784079313278198, -0.2754400968551636, 0.6127916574478149, -0.9168643951416016, -0.8960584998130798, -0.6253552436828613, 1.3590924739837646, 0.5276631116867065, 0.6141196489334106, 0.9274170994758606, -0.057799436151981354, -0.855083703994751, -1.4143667221069336, -1.2600535154342651, -0.48447421193122864, -0.23545187711715698, -1.0673213005065918, -0.32303696870803833, -0.5409038066864014, 1.0884828567504883, -0.1898442506790161, 1.172973871231079, -0.31787195801734924, -0.6121061444282532, -0.6222986578941345, -0.5214276909828186, -0.33978915214538574, -0.27153119444847107, 0.6615524888038635, 0.8471126556396484, 1.1188392639160156, 1.3607510328292847, -0.44213438034057617, 0.05822816491127014, 0.01707204058766365, -0.7682024836540222, -0.5971857309341431, 0.7387747764587402, 1.0115618705749512, -0.30427801609039307, 0.6433501243591309, 0.5986848473548889, -1.5045835971832275, -1.0173702239990234, -0.9951450228691101, -0.24016737937927246, 0.8556672930717468, 0.822252094745636, 0.9309588670730591, 1.4893783330917358, 0.012364311143755913, -0.5967776775360107, 0.6649343371391296, 0.3032420873641968, -0.6418284773826599, 0.4159296751022339, 0.04312734305858612, -0.9088629484176636, 0.9196292161941528, -0.4571748375892639, 2.0818991661071777, 0.132082000374794, 2.1374549865722656, -0.716601550579071, -0.8304567337036133, -0.3375680446624756, -0.23183512687683105, -0.25368058681488037, 0.4462781846523285, -1.629025936126709, -0.10958167910575867, -0.23142221570014954, -1.167222023010254, -1.645445704460144, -0.8114679455757141, 0.5979276895523071, -0.3037320375442505, -0.2932259738445282, 0.7109262347221375, 1.0015944242477417, 1.4301728010177612, 0.5918170213699341, 0.3188186287879944, 0.4109572172164917, -0.12087620794773102, 0.6003051400184631, 1.1772202253341675, 1.5037610530853271, -0.6212411522865295, -0.12637272477149963, -0.341234415769577, -1.329306960105896, 0.6383721828460693, 1.0347106456756592, -0.03992217034101486, -1.0072970390319824, 0.07515082508325577, 0.16718991100788116, -1.0599247217178345, -0.2760453522205353, 0.0449276864528656, 0.4329066276550293, -1.4657349586486816, -0.32505014538764954, -0.1590162217617035, 0.7409258484840393, 0.2462514489889145, 0.24356955289840698, 1.0890936851501465, 1.978340983390808, 0.13576386868953705, 0.5245996713638306, -0.5215609669685364, -1.1230576038360596, 1.1462846994400024, 0.9302736520767212, 0.46685174107551575, -0.2243814319372177, 0.522922694683075, 0.3538880944252014, 1.2603543996810913, 0.04375985264778137, 0.2482956200838089, -0.6423995494842529, 0.6569473147392273, 0.24167566001415253, 0.03812712803483009, -0.25793930888175964, 0.5431653261184692, 1.021677017211914, -0.04560234397649765, 0.5385863184928894, 0.8524115681648254, 3.143362283706665, -0.25889840722084045, -1.434335708618164, -1.3625743389129639, 0.4927894175052643, -0.6328074336051941, -1.08966064453125, 1.1728410720825195, 0.03792377933859825, 0.1415666788816452, -0.7736970782279968, 0.6159950494766235, 1.8835601806640625, 0.48490583896636963, -0.0466875396668911, -0.7233081459999084, -0.6245351433753967, -1.3039281368255615, -0.8662799000740051, -0.3678922951221466, 1.1797629594802856, -1.320368766784668, -0.2294955998659134, 0.13108131289482117, -0.06698904186487198, 0.6637570858001709, -0.13002118468284607, 1.0372564792633057, -0.018661459907889366, 0.1756903976202011, -0.5791885852813721, -1.8167247772216797, -0.24313665926456451, 1.3928369283676147, 0.21656274795532227, 1.0866243839263916, 0.6587822437286377, -0.4245644509792328, 1.9383043050765991, 0.6190357804298401, -0.6426796317100525, 0.8645181655883789, 0.010898045264184475, 0.6519109010696411, 0.9787341356277466, 0.5299254059791565, 1.3764171600341797, 0.39863353967666626, -1.292155146598816, 0.2682817876338959, -0.4848771393299103, -0.7329654693603516, 0.5934680104255676, 0.2765692174434662, -0.4640076458454132, -0.9873574376106262, -0.9420150518417358, 0.21735896170139313, -1.0815577507019043, -0.8240326046943665, -0.18233560025691986, 0.09369704127311707, -0.6240262985229492, 1.1852158308029175, 0.6112594604492188, -0.035068318247795105, 0.2511645555496216, -0.2862951159477234, -1.7516752481460571, -1.3938801288604736, 0.8968988060951233, 0.33660462498664856, 1.5504857301712036, 0.3085925877094269, 0.4768886864185333, 2.3225417137145996, 0.3147534132003784, 0.2339543104171753, -0.5350257754325867, 0.420113205909729, 0.08915975689888, -1.4903528690338135, 0.2071375548839569, -1.6132696866989136, 1.4678304195404053, -0.8953943252563477, 1.6411911249160767, -0.6614068746566772, -0.1945078819990158, -0.31608057022094727, 0.33506983518600464, -0.4016475975513458, 0.41554486751556396, -0.8892233371734619, -0.0816013440489769, 0.3783484995365143, -0.9982386827468872, 0.10591635853052139, 1.4282902479171753, 0.4013207256793976, 0.09317561239004135, 0.7351353168487549, 0.3497803211212158, 0.1602899134159088, -1.2468039989471436, 0.8827311992645264, 0.6223296523094177, 0.37072035670280457, 2.612888813018799, 0.13737106323242188, -0.5164008140563965, -0.44864389300346375, 0.8975841403007507, 1.074913501739502, -0.5027629137039185, -1.997328519821167, 0.01905241794884205, 2.885448932647705, 1.9625862836837769, 1.0118978023529053, 1.0103658437728882, -0.3091547191143036, 0.9046199917793274, -1.8247956037521362, -0.36890819668769836, -0.14883145689964294, -2.8464553356170654, -2.3028512001037598, -1.093248724937439, -0.22717972099781036, -0.9640273451805115, 0.3545345664024353, -0.4477539360523224, 0.7281822562217712, 0.552748441696167, 0.9787464141845703, -0.4552234411239624, -0.2661600708961487, -0.6782170534133911, 0.28955766558647156, -0.8561700582504272, -1.3925434350967407, -0.8387073874473572, 0.23325540125370026, 0.07958266139030457, 1.0137830972671509, -0.9358499646186829, 0.39732253551483154, 0.2684207558631897, -0.64650559425354, -0.5627334117889404, -0.09017779678106308, 0.6564875245094299, -0.519325852394104, -0.4515943229198456, 0.12981396913528442, -0.5166277289390564, -0.7325663566589355, -1.1670522689819336, 0.6888889670372009, -0.3632134199142456, 1.0685827732086182, -0.273340106010437, -1.7246248722076416, -1.5067089796066284, 1.1499521732330322, 0.8440824151039124, -1.1204766035079956, -0.5975162386894226, -1.0552027225494385, 1.3773986101150513, 0.3466431796550751, 0.2224125862121582, 1.2357136011123657, 0.4133758842945099, -0.266871839761734, -0.8698665499687195, 0.7360851764678955, -1.0924983024597168, 2.3450756072998047, 0.988868236541748, -0.06789271533489227, -0.34783056378364563, -1.0105838775634766, -0.9432777762413025, -1.9951274394989014, 1.205280065536499, 1.1640039682388306, -0.07482177764177322, -0.7437114119529724, -1.2632240056991577, 0.545626163482666, 0.8477864861488342, 0.22359414398670197, 1.184507131576538, -0.34704333543777466, 1.1374605894088745, -1.661565899848938, -0.3639189302921295, 1.2356867790222168, -0.07943662256002426, 0.9337865114212036, 0.0010322507005184889, -0.8148086071014404, -0.019622931256890297, -0.011603880673646927, 0.10349646955728531, -0.13187655806541443, 0.7448787689208984, -1.0952938795089722, 0.15177388489246368, 0.6207225322723389, 1.6658480167388916, -0.6940709948539734, -1.8400050401687622, 1.2179492712020874, -0.4638209939002991, 1.1453510522842407, -0.8334221243858337, 0.362303763628006, 0.2909495234489441, 1.2500174045562744, 2.1549623012542725, 0.22632654011249542, 0.2679016590118408, -0.08266942203044891, -1.3926305770874023, -0.3210156559944153, -0.9385031461715698, 0.606768786907196, 1.2557528018951416, 1.117271900177002, 1.3323347568511963, 0.443888783454895, -0.06107418239116669, 0.4331318140029907, -0.300516277551651, 0.21072693169116974, -0.7275415658950806, -0.7015606760978699, 1.093070149421692, -0.3433964252471924, 0.16885001957416534, 0.14304879307746887, -0.4629933536052704, 0.9744272828102112, -0.50331050157547, -0.8587340116500854, -1.7544316053390503, 0.04923534020781517, 0.9774559736251831, -0.35191184282302856, -0.8559727072715759, 2.1648879051208496, 1.1077531576156616, 0.9678689241409302, -1.0403083562850952, -1.0404061079025269, 1.1434149742126465, 0.021440615877509117, -0.5180090069770813, -0.7303662300109863, 0.0004269479832146317, 0.4353332221508026, 0.8730707764625549, -1.990031123161316, -0.8409419655799866, -0.2024294137954712, 0.42144614458084106, -1.745587706565857, 0.6345821619033813, -0.31659942865371704, -1.0944712162017822, 1.1891800165176392, 0.35931533575057983, -1.3884252309799194, 0.5117361545562744, -1.2104233503341675, -1.902733564376831, 0.3839356005191803, -0.8278005719184875, 0.11928832530975342, -1.441884994506836, 0.49739590287208557, 0.6365011930465698, -1.9118704795837402, 0.4728531539440155, -0.81585294008255, -1.1457723379135132, 0.16800011694431305, 0.267861932516098, -0.43822410702705383, -0.28042349219322205, 1.5660074949264526, 0.0006739662494510412, 1.0121647119522095, 1.0121060609817505, -0.12497276812791824, -0.4411018490791321, 0.5542072057723999, 0.2563110291957855, 0.2645847797393799, 1.610266089439392, -0.6582647562026978, -0.49833643436431885, 1.657804250717163, 0.099777452647686, 1.5169222354888916, 0.5404544472694397, 2.041476249694824, 0.9948027729988098, -0.14501647651195526, -0.5947166085243225, -0.8870517611503601, 1.0559781789779663, -0.670769453048706, 0.5984647870063782, -0.5152280926704407, 0.6350009441375732, -1.4028973579406738, 0.5613036751747131, -1.3613425493240356, -0.6663796305656433, 0.5022425055503845, 1.2436790466308594, 1.1611663103103638, -0.4974287748336792, 0.4385637938976288, -1.7494059801101685, 0.7230571508407593, 1.6930385828018188, 0.25139597058296204, 0.8692891001701355, 1.9137179851531982, -0.530960738658905, 0.10865005850791931, 1.4799156188964844, 1.6309070587158203, -2.0372705459594727, 0.3971502184867859, -0.15613605082035065, -0.526232898235321, 0.9512068033218384, -1.3955192565917969, 0.879514753818512, 0.1397562175989151, 0.6008573174476624, 0.31494444608688354, 1.9227302074432373, 1.038326382637024, 0.5360323190689087, -0.2502996623516083, 1.1798638105392456, 0.8445686101913452, -1.4587762355804443, 0.3224433958530426, -0.6661624908447266, 0.4206007122993469, -0.8857100009918213, -1.1070513725280762, 0.8030674457550049, 0.6344791054725647, 0.1555289924144745, -0.38167324662208557, -0.48074138164520264, -1.150712013244629, -0.053432416170835495, -0.27830949425697327, -0.5767101645469666, -0.43385037779808044, 2.3363804817199707, 0.04667166993021965, 0.7363062500953674, -0.37103796005249023, -1.3397865295410156, -1.3695601224899292, 0.49117499589920044, -1.114398717880249, 0.1446647197008133, -1.839572787284851, 0.7311633229255676, -1.2181751728057861, 1.6727540493011475, 1.112227201461792, -1.9793574810028076, 0.22865669429302216, -0.520058810710907, 0.5274149775505066, 1.09236478805542, 1.1340893507003784, 2.1432478427886963, 0.1990405023097992, -0.6104072332382202, 1.473677396774292, -0.8961968421936035, -0.2559112012386322, -0.18890464305877686, 0.11925189942121506, 0.5047542452812195, -0.8045531511306763, 0.6308408975601196, 1.7094261646270752, 0.8267450928688049, -0.02694191411137581, 1.6976239681243896, -0.34459152817726135, -0.2289188653230667, 0.835256040096283, 0.7933781147003174, 0.5571598410606384, -0.016425110399723053, -0.8348484039306641, -0.5523200631141663, -0.2626325786113739, -0.5197396874427795, -1.2156989574432373, -0.36099183559417725, -2.426578998565674, 0.30320748686790466, 2.1585512161254883, -0.7448736429214478, -1.5209599733352661, -0.1661936491727829, 0.7344101667404175, -1.4571146965026855, 0.2297353595495224, 1.8981750011444092, -1.402806282043457, -0.4295962154865265, 0.5088856220245361, -1.3277223110198975, -0.009049952030181885, 1.3530728816986084, 0.009235364384949207, 0.2792029082775116, -0.5219135284423828, 1.8052608966827393, 1.7904313802719116, -1.1583290100097656, -1.6528981924057007, -0.43597474694252014, -1.1438815593719482, 1.1642532348632812, 0.43007537722587585, -0.19386132061481476, 0.8279292583465576, -2.0580875873565674, -1.2787854671478271, -0.9801729917526245, 0.9783714413642883, 2.293550491333008, -0.13195723295211792, 1.1755088567733765, 1.0255376100540161, 0.27954092621803284, 0.9614019393920898, 0.5812684893608093, 1.0363671779632568, 0.6279704570770264, -1.0021964311599731, 1.1229523420333862, 0.03649063780903816, 1.2448195219039917, 0.7060250639915466, -0.8944121599197388, 2.1310980319976807, 1.2865142822265625, 0.4815293252468109, 3.1322686672210693, 0.17093569040298462, -1.649509072303772, -1.2161967754364014, -0.4149685800075531, -0.44611597061157227, 1.4728978872299194, 0.8140309453010559, -0.23689144849777222, 1.25827157497406, 0.08357353508472443, -1.2002651691436768, -0.9719671607017517, 0.23390716314315796, 0.3402671813964844, -0.6247277855873108, 0.5340501070022583, 1.50812566280365, 0.40201547741889954, -0.15234780311584473, 1.1184918880462646, -0.3224554657936096, 0.960395872592926, 0.8531036376953125, -0.5325915217399597, -0.3112978935241699, 0.7173868417739868, 0.4690149426460266, -0.9410197734832764, -0.7767865061759949, -0.4965318441390991, -0.13975687325000763, 0.308212012052536, 1.0758793354034424, -0.49799883365631104, 0.9997497797012329, 0.5031301379203796, -1.1028519868850708, 1.2770277261734009, -0.6037809252738953, -0.6107940077781677, 0.35950952768325806, 0.40741032361984253, 1.1522244215011597, -0.2572745978832245, 0.3832905888557434, -0.25704076886177063, 0.15395639836788177, 2.864278554916382, 1.138289213180542, -1.0215399265289307, 1.8557308912277222, 0.2447304129600525, -0.11445581912994385, -0.7548418641090393, 0.3499370813369751, 0.8938863277435303, 1.5374648571014404, 1.0031462907791138, 0.1612563580274582, -1.323683261871338, -1.625699520111084, -0.47670742869377136, -0.404574453830719, 0.7524788975715637, 0.15561972558498383, 0.3855096399784088, 1.1405105590820312, 0.1782781481742859, -0.4615989029407501, -0.7655670046806335, -1.4808597564697266, -0.270396888256073, -0.05681697651743889, -1.7107057571411133, 0.9825486540794373, -0.2591262757778168, 0.2872023284435272, -0.25917482376098633, -0.10387342423200607, 0.2135525345802307, -2.0840861797332764, 0.8205083012580872, 1.2284541130065918, 0.9533572793006897, -1.6996169090270996, -0.046248190104961395, -1.3780791759490967, 0.9861540794372559, -0.11313426494598389, -1.6711972951889038, 0.9857211112976074, -1.447309970855713, -0.8808517456054688, 0.5807133913040161, 0.5508121252059937, 0.08869316428899765, -0.4179192781448364, -0.3305681049823761, 0.2645457684993744, 1.8095245361328125, 0.004683222156018019, -1.0139508247375488, 0.5906562209129333, 0.10717015713453293, 0.6358819603919983, -1.2614891529083252, -1.04102623462677, 0.38502237200737, -0.2980346083641052, 0.7556147575378418, -0.81892991065979, -0.5173414349555969, 0.6019631028175354, -2.5061240196228027, -0.8987072706222534, 0.22671398520469666, 0.9189764857292175, 0.03439183533191681, 1.6071856021881104, -1.015893578529358, 0.02913268469274044, 0.16058112680912018, -0.04121346399188042, -0.16835419833660126, -0.7776656746864319, 0.18028922379016876, -1.3526641130447388, 0.6829569935798645, 0.03812630847096443, -1.0607643127441406, -0.11442682892084122, -0.3105185329914093, -1.410241723060608, 0.5226362347602844, -1.0420302152633667, 0.15770429372787476, -0.8426543474197388, -1.0101559162139893, 0.14924699068069458, 0.7979822158813477, -0.7891350388526917, -0.657269299030304, 1.2041248083114624, 0.9991146922111511, 0.26053571701049805, 0.24386824667453766, -2.6735973358154297, -1.0642510652542114, -1.321042537689209, -1.0627918243408203, 0.20159317553043365, 0.35122957825660706, -0.0076358383521437645, 0.7265117764472961, 0.061793748289346695, 1.5821436643600464, 1.2593101263046265, 2.153669834136963, -0.2516306936740875, -2.222590684890747, 2.1143362522125244, 1.1358647346496582, -0.9147569537162781, 0.9585742354393005, 0.7439708709716797, -1.6024963855743408, -1.6224944591522217, 0.5046840906143188, 0.8727113604545593, 0.8765608668327332, -1.2721565961837769, -0.2348843216896057, -0.7632836103439331, -0.8138798475265503, 1.5865097045898438, 0.275233656167984, 1.4913967847824097, 0.6256330013275146, -2.0504095554351807, -1.2709318399429321, 0.9339474439620972, -0.053719811141490936, -0.9980819821357727, 1.3810020685195923, 0.64285808801651, 1.6798495054244995, -0.9282506704330444, -0.945010781288147, -0.727647602558136, 2.051574230194092, 1.940421223640442, 1.0305052995681763, 0.1472274363040924, -0.6670507192611694, 0.986091673374176, -0.5850183367729187, 0.05353827401995659, 0.4588206708431244, -1.3674745559692383, -1.0804507732391357, 0.48673883080482483, -1.8161088228225708, 1.2764852046966553, -0.9574304223060608, 0.519496500492096, 2.444695234298706, 1.07794988155365, 0.36627739667892456, 0.47664979100227356, 0.6161471009254456, 0.888396143913269, 1.3921576738357544, -0.09492896497249603, 0.17635948956012726, -0.6946343183517456, -1.2544432878494263, -0.11156988143920898, 0.6188080906867981, 0.3979962468147278, -1.578877329826355, 1.0754517316818237, 0.22094228863716125, 0.976568341255188, -0.21214336156845093, -1.6491047143936157, -0.9265517592430115, -0.905900239944458, -1.023208737373352, 0.8307486772537231, -0.4234652817249298, 1.094735860824585, 0.9689943194389343, 1.2051584720611572, 1.401106834411621, 0.6278548836708069, -0.6308592557907104, 0.03698934242129326, 0.04388387128710747, -1.662218451499939, -1.6834837198257446, -0.433634489774704, 0.07690152525901794, -0.9369410872459412, 0.6887729167938232, -0.6805769205093384, 0.013415969908237457, 1.2423840761184692, 0.7236199378967285, -0.8097308278083801, -0.19221320748329163, 1.257774829864502, 1.0107488632202148, -1.343360424041748, -0.6627627015113831, 0.7078920006752014, -0.1899232268333435, -1.1064924001693726, -0.40429240465164185, 1.907545804977417, 0.2781868875026703, -0.18954509496688843, 0.14989188313484192, -1.8117308616638184, 0.8488419055938721, 1.8757365942001343, 1.0832605361938477, -1.6889228820800781, -0.039870668202638626, -0.06078048050403595, -0.875749409198761, -0.15556606650352478, 0.4085135757923126, -0.7005813121795654, 1.2512472867965698, -1.2848598957061768, 0.9691000580787659, 1.4327243566513062, -0.09957374632358551, 0.022058561444282532, -1.7869837284088135, -0.7078214883804321, 2.1784250736236572, -0.7571443915367126, 0.7237721681594849, 1.5297802686691284, 0.3191392421722412, -1.0116865634918213, -0.7989096641540527, -1.287327527999878, -0.7969018220901489, -1.0235308408737183, 1.2635451555252075, 1.3473175764083862, 0.40761569142341614, 0.2632071375846863, -1.2901170253753662, 2.177626848220825, -2.6547586917877197, 0.027286775410175323, 0.281090646982193, -0.23721618950366974, 0.8026720881462097, -0.4981330335140228, -0.3567095398902893, 1.149857521057129, -0.4446019232273102, -1.7791244983673096, -1.0657131671905518, -0.9879127144813538, -0.28932225704193115, -0.5331124663352966, -1.2347031831741333, -1.6611549854278564, -0.21336951851844788, 2.0694656372070312, 2.218339204788208, 0.03863465040922165, -0.12834881246089935, -0.6625155806541443, 0.25627580285072327, 1.0757330656051636, 0.36095163226127625, 0.19737306237220764, 0.4986554980278015, -0.4243106544017792, 0.4226182699203491, 1.6132036447525024, -1.8690707683563232, 0.023373035714030266, -0.4440072178840637, -1.0582225322723389, -0.2739472985267639, 0.21429631114006042, -0.5607152581214905, -0.42182695865631104, -1.1051685810089111, -0.7730323076248169, 0.4547736644744873, 1.400789499282837, 0.9888126254081726, 1.1087034940719604, -0.17144878208637238, -0.9000251293182373, -0.3392859101295471, 0.2134377509355545, 1.1850717067718506, 1.010334849357605, -2.1035263538360596, 1.116901159286499, -0.5864229798316956, 0.8282496333122253, 0.9892692565917969, -0.5357556343078613, 1.1539963483810425, -0.9209802746772766, -1.2602925300598145, -0.6484764814376831, 1.1377612352371216, -0.5459474921226501, 0.6271098852157593, 0.39455723762512207, 0.4590696394443512, 0.012644553557038307, -1.8468049764633179, -0.13160717487335205, 1.1010466814041138, -0.7437129616737366, 0.7505430579185486, 0.7001129388809204, 0.46609270572662354, 0.31799858808517456, 0.9520200490951538, -0.4825490415096283, 0.1005014181137085, 1.168161392211914, 0.05051350221037865, -0.23209573328495026, -0.4898243248462677, -0.3438419699668884, -0.019419318065047264, 0.08873937278985977, 1.369266390800476, -1.8555631637573242, -0.12171326577663422, -0.5296763777732849, -1.001699686050415, -0.5727905631065369, -1.2631785869598389, 0.5834372043609619, -0.27166110277175903, -0.43784141540527344, -1.318469762802124, -0.4027465581893921, -0.29698535799980164, -1.5730646848678589, 0.46177437901496887, 0.07554030418395996, 0.008521964773535728, -0.9458768367767334, -0.29505589604377747, 0.7425740361213684, -0.579788327217102, -0.8555629253387451, -0.021478157490491867, 0.4979235529899597, -0.6784462928771973, 2.0269970893859863, 0.6041437387466431, -1.8247966766357422, -0.636860728263855, -0.8383317589759827, -0.4410380423069, -0.5816932320594788, 0.5664237141609192, -0.8598650097846985, -1.5947986841201782, 0.6489664912223816, -0.6840663552284241, -0.3796946704387665, -0.40701669454574585, -0.46528738737106323, -0.8001647591590881, 1.3811914920806885, -0.30624476075172424, 0.30403250455856323, 0.33083269000053406, 2.307241439819336, -0.15161068737506866, 1.9041835069656372, 0.9016286134719849, 0.5820881724357605, 2.71736741065979, 1.8570975065231323, 0.30735963582992554, 0.4681977331638336, 0.8580366969108582, -0.5763064026832581, 1.154564380645752, 0.3893907368183136, 1.3169058561325073, 0.18830269575119019, -0.9704107642173767, -0.058752886950969696, 1.3160853385925293, -0.8564064502716064, -0.9618973731994629, 2.339954376220703, 1.9980789422988892, -1.5289244651794434, 0.044300224632024765, -0.0004000855260528624, 0.8568838834762573, -0.3038121163845062, 1.5536688566207886, 0.7971456050872803, -2.7138161659240723, -0.7484201788902283, -0.8857529163360596, -0.41720712184906006, -1.4626858234405518, -2.098109006881714, 1.3716120719909668, -1.1427671909332275, -0.12569265067577362, 0.2967405319213867, -1.1042792797088623, -0.21402089297771454, 0.2705095410346985, 0.010486844927072525, -0.0845538079738617, -0.8847829103469849, -1.1059401035308838, 0.4598694145679474, 0.4736311435699463, 1.0424827337265015, -1.4573354721069336, 0.8631730675697327, 0.08236370235681534, 0.015583041124045849, 0.07839607447385788, 0.2824060618877411, 0.3063371181488037, -1.4527192115783691, 2.18723201751709, 1.8683875799179077, 0.585729718208313, -0.05729823186993599, -0.48158740997314453, -0.8319424390792847, 0.7038760185241699, 0.34345924854278564, 1.7565839290618896, 0.9620396494865417, -0.533722996711731, -0.21534469723701477, -0.3243667185306549, -1.127424955368042, -1.162113070487976, -0.028608456254005432, -1.1658458709716797, -0.5415295958518982, 0.030083931982517242, 0.8920477628707886, -0.4438387453556061, -1.7924119234085083, 0.09255300462245941, -0.41338393092155457, -1.0752663612365723, 0.2859852910041809, -1.6739161014556885, -1.0434200763702393, 1.2408931255340576, -0.9716926217079163, -0.03038678877055645, 0.2691764533519745, 0.05660538747906685, 0.5547698140144348, -1.0153441429138184, 0.08480248600244522, -0.13337485492229462, -1.3806596994400024, -0.8723844289779663, -0.8494912981987, -0.9094542860984802, -0.13927212357521057, 0.15779244899749756, 0.06008467823266983, -0.11156339943408966, 0.18919400870800018, -0.7289494872093201, -0.6587035655975342, 0.20922675728797913, -0.5321303009986877, 1.1838021278381348, -0.5430994033813477, -1.0840994119644165, -0.09724964946508408, -0.21513497829437256, 0.31350764632225037, 0.7005189061164856, -2.3849070072174072, 2.3331894874572754, -0.19790762662887573, 0.1721835881471634, -0.8156779408454895, 1.53402578830719, -0.2119380682706833, -0.3048602044582367, 0.8403455018997192, 0.9047940969467163, 2.5861411094665527, 0.6797088384628296, 2.21207594871521, -0.6779342889785767, 0.1663389950990677, 0.6063643097877502, -0.1414097100496292, 1.2553961277008057, 0.07087019830942154, -0.672633171081543, 0.7016177773475647, 1.3537485599517822, -2.0273730754852295, -0.28727245330810547, -0.8453599214553833, -0.2090788036584854, 0.4294781982898712, 1.2861180305480957, 0.31918033957481384, 1.3265119791030884, -0.8381661772727966, 0.20613637566566467, -0.4322558641433716, 1.113014817237854, -0.7530865669250488, -0.6464154124259949, -1.027327537536621, -0.6359935998916626, 1.2122421264648438, 0.12871243059635162, -0.9253196120262146, 0.48892927169799805, 0.1378486603498459, 0.6245538592338562, 0.30328014492988586, 0.34528160095214844, 0.6592363715171814, -0.07593703269958496, 0.9832451343536377, -0.6281535029411316, 0.8739629983901978, -0.14532162249088287, 0.9013752341270447, -0.9786537885665894, -1.4432343244552612, -0.6866896748542786, -1.5431370735168457, -0.577819287776947, -0.6666707396507263, -0.7632317543029785, -0.803268551826477, -0.9754583835601807, 1.0747358798980713, 0.8481786847114563, -0.09892629832029343, 1.52825927734375, 0.053127970546483994, 0.9513425827026367, 0.10431087762117386, 1.4502300024032593, -0.5536816716194153, -0.36046135425567627, -0.3512849509716034, 0.9293935298919678, 0.2832960784435272, 0.5826323628425598, 0.046182580292224884, 3.0347189903259277, 0.8124065399169922, 0.5456287264823914, 0.6013204455375671, 0.8729690909385681, 0.7036709189414978, 1.4321861267089844, 0.7376517653465271, -0.7451217770576477, 0.9471463561058044, -0.6766945719718933, -1.0090703964233398, 1.0718669891357422, -0.32744652032852173, 0.08198577910661697, 0.0605006143450737, 1.5125106573104858, -1.069980263710022, -0.5719763040542603, 0.3030155599117279, -0.3106001913547516, -1.0371828079223633, -1.2505799531936646, 0.433390349149704, -0.06622487306594849, 0.2348470389842987, -0.5033931732177734, -0.04218785837292671, 0.9450094699859619, 1.8944823741912842, -1.0715551376342773, -0.013954907655715942, 0.18538883328437805, 1.3133724927902222, 0.548116147518158, -0.02600003033876419, 0.5021847486495972, -1.562509298324585, 1.069344162940979, 0.6279521584510803, -0.11872889846563339, 1.8265228271484375, -0.5390812754631042, 0.5083498358726501, 0.27274689078330994, -0.04845636710524559, -0.4266015291213989, 0.8157944083213806, 0.13932055234909058, 1.507978916168213, 0.1724514216184616, -1.8204902410507202, 0.2863841652870178, -0.707953155040741, -0.23623602092266083, 0.679774284362793, 0.20277532935142517, 1.325321078300476, -0.3747267723083496, 0.19395872950553894, -0.7134021520614624, 0.20699287950992584, 0.08146445453166962, 1.7500724792480469, -1.3390005826950073, -1.2899396419525146, 2.1805262565612793, -0.4341958463191986, 0.11598996818065643, 0.30999863147735596, 0.5361424684524536, 1.8143208026885986, 0.06582633405923843, 0.33003050088882446, -0.7248334884643555, 1.326082468032837, 1.421340823173523, -1.2771790027618408, -0.9245902895927429, -0.9580501317977905, 0.2054113745689392, -0.8226974010467529, -0.8019232749938965, -0.6402420997619629, 0.49002566933631897, 1.0118366479873657, -0.2521546185016632, -0.9866498708724976, -0.7291275262832642, 0.9484267234802246, -0.3276427090167999, 1.2108277082443237, 0.942314863204956, -0.29201042652130127, 1.5866634845733643, -0.8736113905906677, 1.1467450857162476, 1.993341326713562, -0.28715455532073975, -1.4614360332489014, 1.9365285634994507, -0.04450218006968498, -0.4628139138221741, -1.1719521284103394, -0.17982664704322815, -0.490913450717926, 1.420415997505188, -0.28712204098701477, -0.5301780700683594, -0.9532638192176819, 0.32508838176727295, -1.091853141784668, 1.879340648651123, -0.5073530673980713, -0.12230168282985687, 0.22827118635177612, -0.3621305823326111, 0.7281841039657593, -1.295589804649353, 0.631696343421936, -1.029631495475769, -0.3537931442260742, -0.7682270407676697, -0.6903290152549744, 0.809551477432251, 0.6458530426025391, -0.022412965074181557, -1.5282949209213257, -2.029125452041626, 2.287771463394165, 0.8130717277526855, 0.29452234506607056, 0.4110441505908966, 2.0276169776916504, 0.05275910720229149, 1.2521028518676758, -0.5546702146530151, 0.6475111842155457, 2.1273386478424072, -1.6519625186920166, 0.20140516757965088, 0.9577893018722534, 0.04815128818154335, 0.058683328330516815, -0.6261433959007263, -0.8780676126480103, -0.1400991976261139, 0.6682462096214294, -0.7512520551681519, 1.0217187404632568, 0.8379245400428772, 0.2961946427822113, -0.17400036752223969, 0.6183635592460632, -0.9254254102706909, 0.00179873441811651, -1.4976205825805664, 0.09380277246236801, 0.8019322752952576, 2.2663941383361816, -0.4915311634540558, 0.7786846160888672, -0.15609890222549438, -1.7617031335830688, -1.5013940334320068, 0.3481959104537964, -1.181876540184021, 0.8390801548957825, -0.6451008915901184, 1.4431978464126587, 0.42408376932144165, 0.1803630292415619, -0.31483128666877747, -1.4110989570617676, 0.3424381911754608, 1.724424123764038, 1.2144193649291992, 0.7385978102684021, 0.41734573245048523, 0.15243974328041077, 0.24404725432395935, -0.4339591860771179, -1.3713164329528809, -1.6147816181182861, -0.7119070291519165, -0.07388967275619507, -1.5233314037322998, 1.1410013437271118, -1.7698462009429932, -1.1162010431289673, 0.4361813962459564, -0.006132641341537237, -0.12722106277942657, -0.23923425376415253, -0.5155083537101746, 1.8453644514083862, 1.4003432989120483, 2.675252676010132, -1.3579673767089844, 1.1777377128601074, 0.44741812348365784, -0.6565726399421692, 0.5045629143714905, -0.6503568291664124, -1.5904934406280518, 0.08932112157344818, -0.47406449913978577, 0.3844229578971863, -0.4345240294933319, 0.8249037265777588, -0.38067299127578735, 1.7870389223098755, 1.725319504737854, -0.5831544995307922, -1.3245147466659546, -0.3539382219314575, -0.6256641745567322, -0.1609281301498413, -0.31795138120651245, -0.6862483024597168, -0.8842148780822754, -1.2542153596878052, 0.565739095211029, 0.2232280671596527, -2.6347856521606445, 0.4393627941608429, 1.6256872415542603, -1.0326871871948242, 0.12376583367586136, 0.4502713084220886, -1.4369863271713257, -0.3085975646972656, -1.0299651622772217, -0.5598850250244141, 0.7528887987136841, -0.5669155120849609, -1.1779958009719849, -1.495609998703003, -1.4338995218276978, -1.1589668989181519, 1.0159685611724854, 0.08188406378030777, -0.20223689079284668, -0.7009837627410889, 0.3701309859752655, 0.16583044826984406, -0.5073449611663818, -1.7701891660690308, -0.6685668230056763, 0.8585766553878784, 0.29137662053108215, -0.7187159061431885, 0.8661257028579712, -1.1138103008270264, 1.0558580160140991, 0.3123386800289154, 0.5148490071296692, 1.3544577360153198, 0.6789493560791016, -0.8875541687011719, 1.7626409530639648, 1.3766387701034546, 0.8468397259712219, -0.06572666019201279, 0.24775314331054688, -0.3150988221168518, 1.9107236862182617, 1.242592453956604, 1.0744256973266602, 0.4737323820590973, -0.24168774485588074, 0.4931214153766632, -1.0796167850494385, 0.41186296939849854, 1.3271574974060059, 0.02017585188150406, 2.1353275775909424, 0.5022853016853333, -0.3300994336605072, 0.48301348090171814, 0.5003286600112915, 1.2309882640838623, -0.9842969179153442, 0.4808425009250641, 0.7555541396141052, -1.5404229164123535, -0.4155464470386505, 1.8667758703231812, -3.0605597496032715, -1.6196845769882202, 1.6816065311431885, 1.928728699684143, 1.269901156425476, -0.29298821091651917, -0.02876490168273449, 0.2611572742462158, 1.6401896476745605, 0.03351093456149101, 1.86765718460083, -0.23920072615146637, -0.014223177917301655, -0.16384147107601166, 0.6918527483940125, -0.2593627870082855, 0.41457366943359375, 0.798069953918457, 0.6450115442276001, 1.9603904485702515, -1.1028834581375122, -0.8985236287117004, -0.8527257442474365, 0.306808739900589, 0.19459833204746246, -0.5729045867919922, -1.8328421115875244, 1.7382985353469849, -1.1146554946899414, -1.4927339553833008, -0.3195456862449646, 0.39702826738357544, 0.12392507493495941, 1.7227003574371338, 1.4601993560791016, -0.2225675880908966, -0.32813552021980286, 0.3280356228351593, 0.44107112288475037, 0.782760739326477, 0.9867275953292847, 1.7792266607284546, -0.05367368459701538, 0.9071647524833679, 1.1770628690719604, 0.22613631188869476, 1.078790307044983, -1.3853392601013184, -0.417829692363739, -0.014926007017493248, -0.28389546275138855, -1.4681665897369385, -0.9335262775421143, 0.8640994429588318, 0.3462154269218445, 0.4532228410243988, 1.635758876800537, 1.2994388341903687, 0.013182621449232101, 1.6056283712387085, -1.412129282951355, -1.1648982763290405, -0.47665876150131226, 0.7077203989028931, 0.7299628257751465, 0.8495684266090393, -0.5890758633613586, -1.027979850769043, 0.4570379853248596, -0.3281399607658386, 1.6828559637069702, 0.8217865228652954, -0.566677987575531, 0.3150804936885834, 0.6207879185676575, -1.676261067390442, -0.18125389516353607, -0.1811293661594391, 0.6696070432662964, 0.8665983080863953, -0.794141411781311, 0.8719663619995117, -0.8607810139656067, -0.8964961767196655, 0.4841638207435608, -0.6521782875061035, 0.8436158895492554, 0.9532470107078552, 0.14845342934131622, -0.429808646440506, -0.22154925763607025, -0.8810808658599854, 0.09181278944015503, -1.6851388216018677, -1.4407449960708618, 0.5042995810508728, -1.443858027458191, -1.0909117460250854, -0.3695334494113922, 0.666646420955658, -1.698134422302246, 0.13625691831111908, 0.6473804712295532, -1.0854904651641846, 0.4255923628807068, 0.16546957194805145, -0.20009882748126984, 0.3422299921512604, -2.3015947341918945, 0.12339811027050018, 0.12749549746513367, 0.48336002230644226, -0.07384935021400452, 1.7197166681289673, -0.2622356116771698, -0.03879530355334282, 0.14630624651908875, 1.0374956130981445, 1.3543106317520142, 0.5581098198890686, 1.16233229637146, -0.4412955939769745, -0.7082927823066711, -0.8636243343353271, 2.395709991455078, -1.008291244506836, -0.03903679922223091, 1.4514166116714478, 4.488816921366379e-05, -0.27654680609703064, -0.279646635055542, 0.48494455218315125, -0.11333753168582916, 1.5169936418533325, 0.7310850620269775, -0.9143275618553162, -0.8293398022651672, 0.41962939500808716, 0.5343437194824219, -1.1938292980194092, -1.1269543170928955, 2.1737046241760254, -1.9143314361572266, -0.2932259142398834, -1.0996837615966797, 0.07696729153394699, -0.09488583356142044, 0.14108392596244812, -0.20584462583065033, -1.4678456783294678, 1.0524085760116577, -0.6016293168067932, -1.3036197423934937, 0.011532343924045563, 0.860541820526123, 0.38307473063468933, 0.5551775097846985, 0.3067519962787628, -0.5113861560821533, -0.3610210716724396, -1.3476221561431885, 0.6469137668609619, 1.1400926113128662, -0.1675938367843628, -0.4429401159286499, 0.9131024479866028, -1.3193641901016235, 1.3361762762069702, 0.09415043145418167, 2.2406020164489746, 1.1736257076263428, 0.6992838382720947, 1.4628866910934448, 0.4616389274597168, -0.39686083793640137, 0.403144896030426, -0.957020103931427, -1.128784418106079, -1.5216162204742432, -0.7863847017288208, 0.9542955160140991, 0.3288900554180145, -0.5239207744598389, -0.5096772313117981, -1.5912054777145386, -1.2378169298171997, 1.169894814491272, -0.22455716133117676, -0.3528575599193573, -0.9937254786491394, 1.5803568363189697, 0.710684061050415, -0.6710270643234253, -0.7717874646186829, -0.7569456100463867, 0.3365614414215088, -1.9853363037109375, 0.4128747284412384, 1.3843177556991577, -0.30090805888175964, 0.8687242865562439, -0.7902683615684509, 0.12292494624853134, 2.0563604831695557, 0.3858011066913605, 0.9295104742050171, -2.4260377883911133, -1.134375810623169, 0.680418848991394, 0.39228934049606323, -1.482085108757019, -1.3947736024856567, -1.0095925331115723, -1.3246433734893799, 0.3825347423553467, -0.669221818447113, 0.085088349878788, 1.3327900171279907, 0.38285544514656067, -0.9167201519012451, -0.7484408020973206, 0.595187783241272, -0.2554660439491272, -0.37000808119773865, 0.2844429612159729, 0.02351086027920246, -0.0910923108458519, -1.0376216173171997, 1.57827889919281, 1.3399051427841187, -0.5867774486541748, 0.44111892580986023, 1.1024854183197021, 0.5746568441390991, 0.49147018790245056, -0.04984089732170105, -0.9829577803611755, 0.28183451294898987, 0.6084496974945068, -0.2180013656616211, -0.5752292275428772, -1.6521995067596436, -0.09346708655357361, -0.1245085597038269, -0.5267913341522217, 0.4468547999858856, 0.44702041149139404, 0.07842370867729187, -0.2996729910373688, -1.7208937406539917, -0.328751802444458, 1.3701103925704956, 0.15830160677433014, -1.0959582328796387, -0.5804958939552307, 1.2744460105895996, -0.8368887305259705, 0.47519025206565857, 0.6453066468238831, -0.05700994282960892, -1.1054152250289917, -2.6929471492767334, 0.8141557574272156, -0.6612460017204285, 0.581119179725647, 1.2589267492294312, 0.8412104845046997, -0.8704883456230164, 1.204400897026062, 0.5038548707962036, 0.9707937240600586, -0.02078348584473133, -1.2872895002365112, -1.4414317607879639, -0.49945512413978577, 0.12949012219905853, 0.14879724383354187, 0.2396097183227539, 0.43370625376701355, 1.75674307346344, 0.33458447456359863, 1.3057304620742798, 0.08150403946638107, 0.011658382602036, 1.2634283304214478, 0.08329711109399796, -0.23567886650562286, -0.04382661357522011, 0.4247297942638397, 0.5450665950775146, -0.008395558223128319, -0.08087745308876038, 1.2955429553985596, 1.046410083770752, 0.24215348064899445, -1.3510940074920654, 0.2609597444534302, 0.8614314794540405, 1.0683066844940186, -1.1082124710083008, 4.092591762542725, -0.2523918151855469, -2.0733582973480225, 0.7692974805831909, 0.026703204959630966, -0.5344344973564148, -0.9935312867164612, 0.6308664679527283, -1.1854420900344849, -1.9427837133407593, 0.9056988954544067, -1.0183275938034058, -1.0085710287094116, 0.7612875699996948, 0.3850233852863312, -0.23878756165504456, 1.0850681066513062, -1.3013951778411865, 0.09917417913675308, 0.8016358613967896, -0.6139178276062012, -0.15290452539920807, -0.9693903923034668, -0.31753942370414734, 0.1754760444164276, 1.2460602521896362, -0.2534264326095581, -0.3433257043361664, 0.2721598148345947, 0.17943823337554932, 1.9484524726867676, 1.404496192932129, -0.3562743067741394, 0.47594377398490906, -0.4847787320613861, 0.09501467645168304, -0.15784844756126404, -0.513592004776001, -0.7582188844680786, 0.6021265387535095, 0.07433143258094788, 1.0792804956436157, 0.6986507773399353, 0.8879050612449646, 1.8772151470184326, -0.9501746296882629, 0.6914160251617432, 0.7978857755661011, -0.969541072845459, 1.0212541818618774, -0.7953223586082458, -0.15686306357383728, -1.1972198486328125, 0.06341490149497986, 0.8056542873382568, -0.0761454850435257, -1.4524940252304077, 1.2528488636016846, 0.6343891024589539, -0.8035129904747009, 0.44315069913864136, -1.8973698616027832, 2.080449342727661, -0.6333266496658325, 0.010567182675004005, 1.697827935218811, -0.6853210926055908, -0.1787610799074173, 0.3400349020957947, 0.5698755979537964, 0.11079659312963486, 0.1383572369813919, -0.3620005249977112, 0.6597297191619873, 1.2018107175827026, 0.5114770531654358, -0.5325974225997925, -0.8071665167808533, -0.27361881732940674, -1.3037536144256592, 0.2129759043455124, -0.589869499206543, -2.372640371322632, 1.7771700620651245, -0.9933710098266602, -0.20361705124378204, -1.0490630865097046, -0.15782517194747925, -1.444199562072754, -0.992978572845459, 0.2117513120174408, 0.18518222868442535, -0.6581400036811829, -1.1769753694534302, 2.07755446434021, -0.6728332042694092, -0.28167498111724854, -1.3311123847961426, -0.8330322504043579, -0.5314568281173706, -0.16386370360851288, -0.10818197578191757, -0.08417309820652008, 0.44510534405708313, -1.1764627695083618, -1.3736424446105957, 0.78400719165802, 0.17374424636363983, -0.6398605108261108, 0.32911428809165955, 0.9200688004493713, 0.7355174422264099, -0.7244381904602051, 0.3559141457080841, -0.4271794259548187, -0.46921291947364807, -1.0831472873687744, 0.7315258979797363, 0.23153717815876007, 0.4061836004257202, -0.16566604375839233, -0.5255122780799866, -0.20359399914741516, 0.17480692267417908, -0.692365288734436, -0.07059022784233093, -0.16359707713127136, 1.8088397979736328, -1.5314342975616455, -0.3067185580730438, -0.2608107924461365, 0.445555180311203, 0.03301562741398811, -0.5364722013473511, 1.209632158279419, -1.8120942115783691, 0.2545739710330963, -0.9436526894569397, 0.04887015372514725, -1.7777378559112549, 0.56316739320755, 0.7128860354423523, -1.738455057144165, 0.17571070790290833, 1.031948208808899, -0.7158308625221252, -0.26851001381874084, 0.4001881182193756, -0.614806056022644, 1.7495479583740234, -0.7343501448631287, 0.2993680238723755, 1.0604737997055054, -0.5302247405052185, 0.2984152138233185, -0.2058139592409134, -0.03606836870312691, -0.2843218445777893, -0.23809991776943207, 0.27836135029792786, -1.0945674180984497, -0.0951780378818512, 1.3878798484802246, 0.219851553440094, 0.6476401686668396, 0.6599149703979492, -0.0731014609336853, -0.7548196911811829, 0.13787944614887238, 0.37044626474380493, 2.288581132888794, 0.1811431348323822, 2.175752878189087, 1.2080122232437134, 0.3411617875099182, -0.06892842799425125, -0.08597902953624725, 0.4024518132209778, -0.707730233669281, -0.6977077722549438, 0.3513590097427368, -0.2771047055721283, 0.9566382169723511, 0.34361958503723145, 0.6893666982650757, 0.42287498712539673, 2.040841817855835, -1.1774981021881104, 2.5525355339050293, 1.114164113998413, -0.31070736050605774, 0.5207058787345886, 0.1008019894361496, 1.4988791942596436, -0.07916811108589172, -1.0239572525024414, 0.6809290051460266, 1.0558472871780396, 0.8897351622581482, 0.40811866521835327, 1.2628178596496582, 0.3814636170864105, 0.2911165654659271, -0.2057098001241684, 0.3366602957248688, 0.7471277117729187, -0.8780797719955444, -0.09463578462600708, -0.10899336636066437, -0.5567388534545898, 0.24025306105613708, 1.931326150894165, -0.7384029030799866, -0.3086535334587097, 0.7937562465667725, 0.019384169951081276, 1.6170527935028076, 1.0567362308502197, -0.770160436630249, -0.692537784576416, 0.7149274349212646, -0.7387300133705139, -0.9240394830703735, -0.2872481346130371, 0.1353362649679184, -0.1956545114517212, 1.0404784679412842, 1.45072340965271, -0.985310971736908, 0.43182244896888733, 0.5422981977462769, 0.7571671009063721, 0.9513401985168457, 1.43955659866333, -0.43884193897247314, 1.281009316444397, 0.9524579644203186, -0.9566121697425842, 0.6911864876747131, 0.6118841767311096, -0.9628490805625916, -0.19415611028671265, 0.4008338451385498, -0.07674290239810944, -1.1264694929122925, -1.4899601936340332, 0.37022748589515686, 0.7758795619010925, -0.15782202780246735, 0.3212495744228363, 0.34814366698265076, -0.28800898790359497, 0.8811393976211548, 0.11353708803653717, -0.9175866842269897, 1.0437543392181396, 1.1445120573043823], [0.30817094445228577, -0.6962103247642517, -0.5637988448143005, 0.07817167788743973, 0.2311122566461563, 0.4864749312400818, 1.0755760669708252, 1.2972080707550049, -1.956157922744751, 0.08384114503860474, 0.552560031414032, 0.14829732477664948, -0.17911022901535034, 0.06827276200056076, -0.5647016763687134, -0.8829445242881775, 0.5689705014228821, 0.8244786858558655, -1.114967942237854, -0.11029733717441559, -0.2304321676492691, 2.453801155090332, 0.2890379726886749, 0.1625296175479889, 1.0086512565612793, -0.36291924118995667, -0.9198417663574219, -0.890497624874115, -0.5326242446899414, -1.060040831565857, -0.3777676522731781, 0.38811078667640686, 0.4933216869831085, -0.062182143330574036, 1.5502208471298218, 1.3728641271591187, -1.3452519178390503, -0.5685908794403076, 0.27284204959869385, 0.08182504028081894, 1.2047936916351318, 0.5062419176101685, -1.5035027265548706, -0.3960038721561432, -0.31483086943626404, -2.4310317039489746, -0.04335205256938934, -1.4106630086898804, 1.4549390077590942, 1.1735942363739014, -0.11523741483688354, 0.9898187518119812, -1.472409963607788, -0.7588621973991394, 1.370347023010254, 0.28788864612579346, 1.1833240985870361, -1.8618477582931519, -0.8032544851303101, -0.6824212074279785, 0.3157733082771301, 0.7950317859649658, 0.8480581641197205, 0.845106303691864, -1.4683358669281006, 1.0821523666381836, -1.6464707851409912, 1.1564677953720093, 0.4672872722148895, -0.15108005702495575, 0.6413801908493042, -0.7667431831359863, 0.5643097162246704, 0.4108794033527374, -1.0235124826431274, 0.484533429145813, 0.9691962599754333, 1.2912272214889526, 0.3834664821624756, 0.06016106903553009, -0.3533551096916199, -1.0330194234848022, 0.6705932021141052, -0.021109651774168015, 0.45385149121284485, 1.2224336862564087, -0.1467302292585373, -0.4561864733695984, -0.8120051026344299, 0.40860116481781006, -0.2851041257381439, -1.7543015480041504, 0.5947290658950806, 0.1714518964290619, -0.31461411714553833, 2.3180091381073, 1.1197909116744995, -0.7173013687133789, -0.49922046065330505, 0.29044124484062195, -0.9047970175743103, -0.5266330242156982, 1.5372955799102783, 0.5693046450614929, 0.7770241498947144, 1.3223035335540771, -1.4409674406051636, 0.950310230255127, 1.8749181032180786, 0.658703088760376, -0.3576727509498596, 0.9038270115852356, 0.529213011264801, 0.9425466656684875, 0.2557591497898102, -0.05679365247488022, -0.01564166136085987, -0.20369680225849152, -0.9457004070281982, 0.8151525855064392, -0.7464953660964966, -1.0840767621994019, 0.14806924760341644, -0.09432502090930939, -0.904294490814209, 0.07592953741550446, 0.33934086561203003, -0.5950079560279846, -2.100004196166992, 0.47394344210624695, -0.8084478378295898, -0.9936403036117554, 1.7133177518844604, 1.7155388593673706, 1.753958821296692, -0.6167553663253784, -0.2529585361480713, -0.2600412368774414, 0.8268912434577942, 0.7334602475166321, 2.5142273902893066, 0.20544975996017456, -0.43785202503204346, 0.38596978783607483, 0.2497928887605667, 0.08445867896080017, -0.8056563138961792, -1.8209952116012573, 0.2487892210483551, 0.43341514468193054, -0.5990028977394104, 0.7906917333602905, -0.7882726192474365, 0.01929030567407608, 0.3959437608718872, -1.1683554649353027, 0.2725430727005005, -0.3159478008747101, -1.6183456182479858, 0.2714087665081024, 0.6246219277381897, 0.7369387745857239, -0.9586305618286133, 1.7859991788864136, 0.6418823599815369, 0.7764330506324768, -0.9916332364082336, 0.5838155746459961, -0.861004114151001, -1.0553416013717651, 0.1256808191537857, 0.27058500051498413, -0.298221617937088, 0.9521585702896118, -1.4738713502883911, -0.45774170756340027, 0.3593091070652008, -0.14542625844478607, -1.8599119186401367, 1.2743595838546753, 0.05165807902812958, -0.5003721117973328, 2.4513254165649414, -1.8693089485168457, 0.8900966048240662, 0.2804693877696991, -0.838811993598938, -0.19293473660945892, -1.5297967195510864, -0.07761231064796448, 0.5761432647705078, -0.7210827469825745, 1.0317060947418213, -0.9289647340774536, 0.6705257296562195, -0.2465442419052124, 0.02456398494541645, -0.1321098357439041, 1.3338453769683838, 0.5063512921333313, 0.42766278982162476, -0.31718528270721436, 0.28571534156799316, 1.6293381452560425, 0.30955207347869873, 0.9886035919189453, -0.038494229316711426, -1.761706829071045, 0.9077615737915039, -0.21989299356937408, -0.3764273524284363, -0.3504994213581085, 1.8181194067001343, 0.474689245223999, 1.516332745552063, 0.08778326213359833, -0.11159256845712662, -0.5736069083213806, 0.12550854682922363, -0.9764556288719177, -0.3637777864933014, 0.5056204795837402, 0.8142892122268677, 0.7763887643814087, 0.24219779670238495, -0.21922262012958527, 0.13958637416362762, -0.450764924287796, -0.754708468914032, 0.03898625820875168, -1.445679783821106, 0.7080537676811218, 1.1057957410812378, -1.0046651363372803, -0.3792742192745209, 0.871342658996582, 1.0539692640304565, 0.40682998299598694, 1.521856427192688, -1.197060465812683, 1.4861328601837158, -0.29048675298690796, -1.2341116666793823, 0.2116544544696808, -1.2546148300170898, -0.36980345845222473, -0.5066553950309753, -0.4246138036251068, -0.2656187117099762, -0.8497807383537292, -0.8406615853309631, 0.509638249874115, -0.8106613159179688, -0.6746652126312256, 0.414429247379303, -0.5094727277755737, 0.7741156816482544, -0.7674926519393921, -0.13694553077220917, 0.7282304167747498, -0.32287096977233887, 0.12789000570774078, 1.9408437013626099, -0.9809208512306213, 0.13624772429466248, 0.15913750231266022, -0.20239976048469543, -2.2496418952941895, -0.3133193254470825, 0.09357298165559769, -1.6623812913894653, -1.51443612575531, 1.2268377542495728, 0.30916500091552734, 1.0934348106384277, -0.07094813138246536, 0.11674803495407104, -0.6592071652412415, -0.22702954709529877, -1.1794342994689941, 2.4346139430999756, -0.035233255475759506, 1.4087975025177002, 1.0784400701522827, 0.5538814067840576, 0.5990412831306458, -0.8973795175552368, 0.628166913986206, 0.9295080304145813, -1.2198448181152344, -0.45409300923347473, 0.012912929989397526, 1.7219798564910889, 0.9380003213882446, 0.9664724469184875, -0.1662580370903015, -0.6316236257553101, -0.49500545859336853, -1.2419874668121338, 0.8632203340530396, -0.7984291911125183, 3.1514813899993896, 0.4406849443912506, -0.36886313557624817, 0.4005361795425415, 0.5592126846313477, 0.0588950589299202, 1.3885473012924194, -1.9307852983474731, -0.8806264996528625, 0.711276650428772, -0.34717774391174316, 1.1466715335845947, -2.249833583831787, -0.5952537655830383, 1.30011785030365, -1.2072174549102783, 1.952782154083252, -0.18717893958091736, -0.8722882866859436, 2.0230507850646973, 0.45714303851127625, 0.3525239825248718, 0.13213256001472473, -1.1393723487854004, 0.5366532802581787, 2.1610257625579834, -1.1840578317642212, -0.32256436347961426, -0.1497214287519455, -0.28396159410476685, -2.317983627319336, 0.20250087976455688, 0.5671985149383545, 1.0363579988479614, -0.31214720010757446, -0.366190642118454, -1.1890208721160889, -1.520082950592041, 1.3453971147537231, -0.003548314794898033, 0.5807564854621887, 0.43818584084510803, -1.8739815950393677, -1.4013453722000122, 0.9221885800361633, 1.1386983394622803, -0.7559746503829956, -0.34202757477760315, 2.2330713272094727, 0.379143625497818, 0.6848908066749573, 0.4426998496055603, 1.493929147720337, 0.1296655833721161, 0.8075195550918579, -0.7501381039619446, -0.8235716819763184, -0.26922816038131714, -0.053326670080423355, -0.49787044525146484, 0.9782792925834656, -0.18719010055065155, -0.8113808035850525, -0.2626728415489197, 1.1322497129440308, 1.515820026397705, 1.8899084329605103, 0.4349552094936371, 0.6504775881767273, -2.1450552940368652, -0.09159073233604431, -1.201740026473999, -3.7262349128723145, 1.593693733215332, 0.44544368982315063, -0.7581202983856201, 1.2304229736328125, -0.3020493984222412, -0.972778856754303, -0.5063358545303345, -0.5736894607543945, -0.37344545125961304, 0.7390409111976624, 0.7606177926063538, 0.1554824709892273, 0.47627755999565125, 1.6216431856155396, 0.10902618616819382, 0.1408037394285202, 0.01742956042289734, 0.19064554572105408, -0.2546987235546112, 0.01978764310479164, -0.007239909376949072, 1.4262162446975708, 0.6660780906677246, 0.4684966206550598, -0.5428392887115479, -0.3211524784564972, 0.5039049983024597, -0.18058957159519196, -0.8511766791343689, 0.308499276638031, 0.8264694809913635, 0.684843122959137, -0.16476549208164215, -0.35099849104881287, -0.10551995038986206, -1.5938986539840698, 0.9748544096946716, -1.7201460599899292, 0.3508850336074829, 0.8804486989974976, 0.2855377495288849, 1.5703836679458618, 0.02738477662205696, -1.1743749380111694, -1.1003614664077759, 0.470631867647171, -0.5992834568023682, -0.4882826805114746, -0.7827426791191101, -1.1170350313186646, 1.517038106918335, 1.1603609323501587, -0.4545625150203705, -0.6933764219284058, 0.4431350827217102, 1.4307085275650024, -1.1632903814315796, 1.1202266216278076, -0.9224517345428467, -0.5619173645973206, 0.5359483957290649, -0.008272494189441204, -0.7176589965820312, -0.45305395126342773, -1.6114894151687622, 0.5201817750930786, -1.4992570877075195, 1.1758434772491455, -0.7199093103408813, -0.7630032300949097, 0.26677030324935913, -0.742944061756134, 0.5956655144691467, -0.11301721632480621, -1.1050814390182495, 0.39764681458473206, -0.4818023443222046, -1.2562583684921265, -1.9118832349777222, 1.1924901008605957, 0.24202507734298706, 1.7346657514572144, -0.316330224275589, 1.4538233280181885, -0.7895811200141907, 0.9293603897094727, 0.5856695771217346, 1.3535147905349731, -0.045491572469472885, 0.11451229453086853, -0.8561441898345947, -0.6685120463371277, -1.6121941804885864, -1.0506923198699951, -1.433548092842102, -0.5433754920959473, -1.3787883520126343, 0.1611011028289795, 0.11105785518884659, 0.9927485585212708, 0.6344937682151794, -0.26110461354255676, -1.1667170524597168, 1.280204176902771, 0.28114965558052063, -0.8825564384460449, 1.2249070405960083, 1.0527914762496948, -0.03152326121926308, -0.7083093523979187, -0.7853385806083679, 0.23801225423812866, 0.694768488407135, 0.6219533085823059, -1.6146819591522217, 2.876159906387329, -0.4659108519554138, 0.1989247053861618, 1.2013695240020752, 1.2697120904922485, -0.845295786857605, -0.5136834383010864, -1.5755548477172852, 0.314407080411911, 1.0293428897857666, -0.06985151022672653, -2.0368471145629883, 1.2850124835968018, -0.0760541707277298, 0.6497987508773804, -0.32973548769950867, -0.7119992971420288, 1.2489323616027832, -0.5185011625289917, 0.8222969770431519, -0.182353675365448, 0.6257670521736145, 0.5950567722320557, -1.0676114559173584, 1.4941158294677734, -0.5173412561416626, -2.0663042068481445, 1.3263646364212036, -2.231279134750366, 0.6465494632720947, -0.347188264131546, -0.721588671207428, 1.7749677896499634, -0.015640581026673317, -0.8285271525382996, -1.9295562505722046, 0.6432758569717407, -0.9742359519004822, 0.6166343092918396, 1.9911221265792847, -2.2543673515319824, 0.24046333134174347, -0.26115623116493225, -1.4374077320098877, -0.20969939231872559, -1.162548303604126, 0.09055721014738083, 2.2672359943389893, 0.7068222165107727, 0.16740907728672028, 0.5631632804870605, -1.1226345300674438, -0.1301194131374359, 1.5922244787216187, -0.07171206176280975, -0.6918481588363647, -1.0540274381637573, 1.1821092367172241, 0.64969801902771, 0.6683204770088196, -0.06209379807114601, -0.43902266025543213, 0.2660019099712372, 1.0159838199615479, 1.309279203414917, -0.4676581621170044, 3.182934045791626, 0.1524328738451004, 0.49219146370887756, -0.7486182451248169, -0.5655561685562134, -2.310948371887207, -0.2960008382797241, 0.5986431837081909, 2.0790750980377197, 0.146748885512352, -1.02888023853302, 0.80645751953125, 0.45737460255622864, 2.542879343032837, 1.4960228204727173, 0.7953038811683655, -0.5309862494468689, 0.13425546884536743, -1.3475322723388672, -0.6577858328819275, -0.3817017078399658, 0.13478747010231018, 1.009584665298462, 1.4756097793579102, -0.582216739654541, -0.03606162220239639, 0.8668599724769592, 0.38152986764907837, -0.22421367466449738, -0.02236027643084526, 1.4667363166809082, 0.8017299771308899, -1.6570366621017456, -0.27086952328681946, 0.27809205651283264, -0.9860822558403015, -0.7232213616371155, -1.7142283916473389, -1.4304475784301758, 1.1442217826843262, 0.21510112285614014, 0.49411192536354065, -2.1637048721313477, -0.8368983268737793, -2.130255937576294, 0.07782769203186035, -0.2054990828037262, 0.5140069127082825, -0.08880272507667542, 0.7057450413703918, -1.543860912322998, -1.688822627067566, -0.6347718834877014, -1.4208309650421143, -1.9579360485076904, 2.2661867141723633, -0.5527273416519165, -0.3008488416671753, 0.23554259538650513, 0.5504506230354309, 0.6754893660545349, -1.9017900228500366, -0.6376455426216125, -0.250598669052124, -0.1507168710231781, 0.7713460326194763, 0.5897441506385803, -0.6875869631767273, -0.4510107934474945, -0.11724498867988586, 0.3317825198173523, 0.8662722706794739, -2.097766399383545, -0.37104085087776184, -1.0799888372421265, -0.5022680163383484, -1.1669515371322632, 0.7430363297462463, 1.3645422458648682, -0.4636399745941162, 0.7179082632064819, -0.4247657060623169, -1.4387985467910767, -2.2792932987213135, 0.8156763911247253, 0.8281949758529663, -1.3387874364852905, -0.7474346160888672, -1.6809501647949219, 0.4098215103149414, -0.21412819623947144, 0.39600440859794617, -1.4746273756027222, 0.609494686126709, 0.8895916938781738, 0.7824954986572266, 1.0534796714782715, 1.309531569480896, 0.47378337383270264, 1.1092137098312378, 0.9852675199508667, 1.2965812683105469, 0.6210474967956543, -1.610446572303772, 1.1402474641799927, -1.8510534763336182, -0.5327702760696411, -1.0215948820114136, 0.3493558168411255, -0.35166874527931213, -1.6769428253173828, -0.08833148330450058, 1.1473621129989624, 0.9001508951187134, -0.2091394066810608, -0.27618739008903503, 0.1805819720029831, -0.02209574356675148, -0.21361872553825378, 0.23858918249607086, -0.16720128059387207, 0.5318201184272766, 2.2977514266967773, -0.4074721932411194, -0.3537258505821228, 0.2590773403644562, 1.4356942176818848, -0.9270502328872681, 0.7953271865844727, 1.3446478843688965, -0.9039243459701538, -0.2566586434841156, -0.6156107783317566, 1.4753389358520508, 0.9040241241455078, -0.6880302429199219, -0.4111950993537903, -2.0889382362365723, -0.4552852213382721, -0.544080376625061, 0.33135196566581726, -0.44391775131225586, -1.0785069465637207, 1.314699411392212, 2.4055368900299072, -2.7432751655578613, 1.7643826007843018, -0.9135398864746094, -1.4049937725067139, 0.04883654788136482, -0.7132174968719482, 0.47314757108688354, 0.042402949184179306, 0.46745285391807556, 0.46044647693634033, -0.5130075812339783, 1.9405443668365479, 1.0141555070877075, 1.5721733570098877, -1.3112664222717285, -0.9920439720153809, 0.3258169889450073, -1.4623631238937378, -0.23867228627204895, 1.8211945295333862, -1.2441325187683105, 0.8899657726287842, -0.260489284992218, -0.33826759457588196, -1.4255839586257935, -2.0255274772644043, -0.8755767345428467, -1.712675929069519, 1.6000112295150757, -1.7258926630020142, -0.8723551630973816, 1.2140884399414062, -0.7477144002914429, 1.7658523321151733, 0.1722608506679535, -0.7310157418251038, -1.0059832334518433, -1.2165626287460327, -0.6659011244773865, -0.6829606890678406, -1.512439250946045, 0.7312048673629761, 0.7446611523628235, -0.8671934604644775, 0.8698192238807678, 0.17998522520065308, 0.4652644991874695, 0.21082882583141327, 2.4341936111450195, 0.23239737749099731, 1.5488426685333252, -0.5095317363739014, 0.1666092425584793, -0.14935480058193207, 0.6902180910110474, -0.17580927908420563, -0.6884989142417908, 0.26442983746528625, 0.8484440445899963, -0.031135298311710358, 0.17344561219215393, 0.34620705246925354, 0.5862028002738953, 0.5639551877975464, -0.9243700504302979, 0.6142598390579224, -1.1617628335952759, -0.5467398166656494, 0.22136825323104858, -0.3219333589076996, -0.12791702151298523, -0.792218804359436, -0.723006546497345, -1.2812978029251099, 0.8681313991546631, -1.5005654096603394, -0.1826481819152832, 1.5038957595825195, 0.5900020003318787, 1.5367838144302368, 0.9430728554725647, 0.28715717792510986, -0.04286639392375946, -0.37797921895980835, -0.8187896013259888, 0.5627525448799133, 1.13917875289917, 0.7919732332229614, 2.6506524085998535, 0.6367619037628174, 0.0610373429954052, -0.5082648396492004, -2.2783362865448, 0.5689413547515869, -0.5738340020179749, 0.11730926483869553, 1.4375323057174683, 0.35669398307800293, 1.1793431043624878, 0.508048415184021, 0.4709937572479248, 0.2722744345664978, -0.9981922507286072, 0.7685131430625916, 0.3236335217952728, -0.32555511593818665, 0.6920346617698669, -0.24471241235733032, 0.10092494636774063, -0.2932634949684143, -1.0034600496292114, 0.9911127686500549, -0.9790920615196228, 0.5944916605949402, -0.14691831171512604, 0.20217540860176086, 0.6063681244850159, 0.7349873185157776, -1.3248891830444336, -0.4190659523010254, 0.15474191308021545, -0.06410194933414459, 1.7788889408111572, 0.43662160634994507, 0.8987725973129272, -1.3738136291503906, 0.18343423306941986, 0.5369004607200623, -0.24229936301708221, -0.2838113605976105, -1.1144280433654785, -1.0883126258850098, -0.380587100982666, 1.1617060899734497, 0.1752958446741104, 0.08442647755146027, 0.913874089717865, 1.1771167516708374, 0.2122945785522461, 0.7317372560501099, -0.19004227221012115, 1.3280435800552368, 0.25404906272888184, 0.7793115973472595, 1.540575623512268, 1.5249013900756836, 0.3417583703994751, 0.9358435869216919, -0.7265956997871399, 2.478410482406616, 0.49450919032096863, -0.2121385633945465, 0.8333323001861572, -0.6036534309387207, -0.7489495873451233, -0.3873390853404999, 0.3661397099494934, 1.7430031299591064, -0.5043102502822876, 1.7068802118301392, 1.5705941915512085, -2.296759605407715, 0.38684695959091187, 0.621794581413269, 0.7358913421630859, -0.6693779230117798, 0.24225208163261414, 0.9501000642776489, 0.228404700756073, -0.025143465027213097, -0.8585938811302185, -0.09340649098157883, -0.05555877089500427, -0.824368417263031, 0.2953653335571289, -0.07138960808515549, 0.3733501434326172, 0.34044185280799866, -0.5845913887023926, -0.4105519950389862, -0.030733997002243996, 0.7520841360092163, -0.29343312978744507, 0.11896707117557526, 0.5336121916770935, 1.2777925729751587, 0.037457581609487534, -1.284375786781311, 0.31007903814315796, 0.7120022177696228, 0.3785194158554077, -1.3212908506393433, -0.3738480806350708, -0.46193447709083557, 0.16029606759548187, -0.22259864211082458, 1.4493387937545776, -0.43491241335868835, -0.7283848524093628, 1.4873480796813965, -0.301154762506485, 0.8216529488563538, -0.7691187858581543, -0.5425326824188232, 1.7497658729553223, 1.9471224546432495, 1.269357442855835, 0.3340613842010498, -1.0628886222839355, 0.1695365458726883, -0.4477294087409973, -0.4231433570384979, 0.2291213572025299, 0.29823243618011475, 1.0767135620117188, -0.006860542111098766, -1.519695520401001, -0.24633820354938507, -0.7714995741844177, 0.8695256114006042, -1.605770468711853, -1.387406587600708, -2.441438913345337, 0.13948510587215424, -0.5765846967697144, -0.5351776480674744, -0.7531070709228516, -0.6868716478347778, 2.2776477336883545, -0.1508173942565918, -1.417396068572998, -1.5778447389602661, 1.4303065538406372, -1.3490544557571411, 0.007416986860334873, -0.6225581169128418, -0.9058893918991089, 0.41766586899757385, -1.1387689113616943, -0.14071059226989746, 1.194320559501648, 2.050414800643921, 0.7407740354537964, -0.38960641622543335, -1.616024136543274, -0.6276727318763733, -0.21188639104366302, -0.3626863360404968, -0.012329346500337124, 1.505193829536438, -0.7648798823356628, -0.6514967679977417, 0.460953027009964, -0.39915284514427185, 0.34001320600509644, -1.2256569862365723, -0.1607954502105713, 0.5806828141212463, 1.6851061582565308, -0.5132846236228943, 0.8918853402137756, -0.2753930985927582, -0.05428396537899971, -0.48856446146965027, -1.4864202737808228, -0.6575280427932739, -0.03723573684692383, -1.1568281650543213, 1.6260130405426025, -0.019927116110920906, -0.34554097056388855, 0.6655263304710388, -1.8116865158081055, -1.1830661296844482, 0.32584893703460693, -0.8627309799194336, -1.4468648433685303, 0.2487160563468933, -0.6617331504821777, 1.3600836992263794, -0.45575863122940063, 0.3771004378795624, 1.3916877508163452, 0.23293215036392212, -0.4156990051269531, -0.3433417081832886, -1.4257549047470093, -0.43400895595550537, 0.01319070439785719, -0.7114424109458923, -0.26107704639434814, 0.5504574775695801, -2.2890288829803467, 0.926139235496521, 0.9886624217033386, 1.0267654657363892, -1.3485767841339111, -0.46591198444366455, -1.4548059701919556, -0.780712366104126, -0.6376397609710693, -0.573599636554718, -0.8685618042945862, -2.3405492305755615, -0.44314563274383545, 1.5902646780014038, 0.2239261269569397, 0.8787551522254944, -0.5805812478065491, 1.277923822402954, -0.49032798409461975, -0.3238203227519989, -0.04487817734479904, -2.189577341079712, -1.2211264371871948, 1.3312416076660156, -1.0753779411315918, -0.6327565908432007, -0.9844185709953308, -1.2224087715148926, -0.27699407935142517, 0.9707697033882141, 0.5748921036720276, -1.2393791675567627, 0.4485841989517212, 0.584228515625, -0.786410391330719, 1.0865354537963867, 0.25057119131088257, 0.3526950180530548, 0.24819275736808777, -1.3375293016433716, -0.4266456961631775, -1.0051159858703613, 0.7581446766853333, -1.2855231761932373, 0.18925192952156067, -0.5060566663742065, -0.4900762736797333, 0.7908153533935547, -0.28110453486442566, 0.5656488537788391, 2.028028726577759, 0.7894075512886047, 0.2981090247631073, -0.3082118630409241, 0.08890271186828613, 0.22540424764156342, -0.7663147449493408, -0.4071497619152069, 0.6630143523216248, -1.1678693294525146, -0.31831303238868713, -1.8762072324752808, 0.204941064119339, -0.9109780192375183, 0.1800510734319687, -0.7607457637786865, 1.550789475440979, 0.9768925309181213, 0.6584060788154602, -0.7532938718795776, -1.466184377670288, 0.7179756164550781, 1.2544054985046387, -1.2359704971313477, -0.0567382276058197, -0.9392004013061523, 0.8242771625518799, 0.37333181500434875, -0.9368816614151001, -0.6460713148117065, -0.5583017468452454, -0.14338791370391846, -1.0900102853775024, -0.44077736139297485, 0.6100364327430725, -0.9660399556159973, 2.4985523223876953, -0.7402973771095276, 0.0376909039914608, 1.8298290967941284, -1.3640942573547363, -1.6346080303192139, -1.1091302633285522, 0.22985999286174774, 0.029807262122631073, -0.18951326608657837, -0.54600590467453, -0.9518551826477051, 2.34277606010437, -0.3800588846206665, -1.5822652578353882, 0.8780028820037842, -0.31774476170539856, 0.5983392596244812, 0.529352605342865, 0.7532854080200195, 1.5931150913238525, 0.6409816145896912, 0.46546000242233276, -1.4775633811950684, -0.985528290271759, -1.1639734506607056, -0.547199547290802, 0.6750097870826721, 0.8954618573188782, -0.0920477882027626, 1.1611944437026978, -0.23969168961048126, -0.031651776283979416, -0.3117797374725342, 1.041799783706665, 1.1034806966781616, 0.5160612463951111, 0.45151907205581665, -0.5224965214729309, -0.5141215324401855, -0.6473338007926941, -1.2423564195632935, -0.7407182455062866, 0.6877310872077942, 0.5364890694618225, 0.9400025606155396, 0.5238208770751953, 0.5867414474487305, -0.30887946486473083, 1.911112666130066, 0.2624165117740631, 0.6915587186813354, -0.6190677881240845, 1.3533087968826294, -0.2722703218460083, 1.334063172340393, -0.29567012190818787, 1.180328369140625, 0.7386672496795654, 0.27032867074012756, -1.4466499090194702, 0.24910220503807068, 1.5254954099655151, 1.3671163320541382, -1.7379775047302246, 1.5295346975326538, -1.4187740087509155, 0.9259051084518433, -0.11116248369216919, 0.05059774965047836, 0.8077976107597351, 2.4627914428710938, -1.3088970184326172, 0.7085680961608887, 0.059606753289699554, 0.8314704298973083, 0.7334742546081543, -0.0696970671415329, -0.2403102070093155, 1.5235496759414673, -1.0594730377197266, 0.31140100955963135, 1.2448365688323975, -0.9219859838485718, 2.0672593116760254, 0.9797137975692749, 0.911686360836029, -0.8809583187103271, 0.46302902698516846, 0.031481143087148666, 1.0457466840744019, -0.07079320400953293, 1.380151629447937, 1.9154880046844482, -0.11024428904056549, 1.6462841033935547, -0.4734375476837158, 1.1035633087158203, 0.23283059895038605, -1.3324517011642456, 0.9907865524291992, -2.3318991661071777, -0.1618276983499527, -0.14119389653205872, 0.09069228172302246, 2.0291237831115723, -0.5586683750152588, -0.07247161865234375, -1.0089075565338135, -0.5989047884941101, -0.31456226110458374, -0.7346678376197815, 0.7734379768371582, 1.1398706436157227, 1.1126893758773804, -0.17938879132270813, 2.1638119220733643, 1.0750561952590942, -0.07807737588882446, -1.497437596321106, 0.5513955354690552, -1.3288406133651733, 0.5011974573135376, -1.4850449562072754, 0.9802877902984619, 1.0121657848358154, -0.11450675874948502, -0.5010386109352112, -0.1877460926771164, -0.6325211524963379, 0.18382269144058228, 0.29024583101272583, -0.2032957524061203, -0.22446498274803162, 0.54729163646698, 0.5381879210472107, -1.5218250751495361, -1.64432692527771, -0.8680071234703064, 0.8429467678070068, -1.5487794876098633, -1.213571548461914, 1.3358843326568604, 0.12969671189785004, 1.1152960062026978, -1.4716802835464478, -0.41151097416877747, -0.1586034595966339, 0.0970621109008789, 1.0054702758789062, -0.15190313756465912, -0.44397109746932983, -0.7847016453742981, -0.01164195965975523, 0.1565704494714737, -1.3504222631454468, 0.3117172420024872, 0.1615414023399353, 1.8594250679016113, -1.0308797359466553, 0.17899973690509796, -0.6895585060119629, 0.18695341050624847, 0.6111655235290527, 0.8070669174194336, -0.8093743920326233, 0.7997131943702698, -0.6262995004653931, -0.2890474200248718, 0.44817206263542175, 0.14947721362113953, -0.26874327659606934, 0.62991863489151, 0.7216204404830933, -0.49548354744911194, -0.8007601499557495, 2.846423387527466, 1.8383591175079346, 0.2628117799758911, 0.2247428596019745, -0.510049045085907, 2.3273136615753174, 0.2506488263607025, -0.0871141254901886, -1.3148608207702637, -1.28428316116333, -1.0086297988891602, -0.9102668166160583, -1.7497221231460571, 1.0602184534072876, 1.0463656187057495, -1.3752682209014893, -1.0153518915176392, -1.3326878547668457, 0.001241955324076116, -1.2964999675750732, 0.20592761039733887, -0.39338454604148865, -0.5296096801757812, 0.5279815793037415, 0.959481418132782, -0.6728643178939819, 0.6668019890785217, 0.04517410695552826, 0.39362406730651855, 0.07653724402189255, 0.3599734306335449, 0.048017147928476334, 1.5018665790557861, 2.224266529083252, 0.43665552139282227, 2.4018025398254395, -0.16819581389427185, 0.970172107219696, 0.3653301000595093, 0.4009275436401367, 0.6000240445137024, -0.951762318611145, 0.31487977504730225, -0.30657491087913513, 0.9110912084579468, -1.2615426778793335, -1.7275760173797607, 0.04965284466743469, -1.1487327814102173, 0.5740089416503906, 2.124030351638794, -0.6481484770774841, 0.4294237494468689, 1.2497957944869995, -0.8516115546226501, 1.0187753438949585, -0.49733585119247437, -0.2239283174276352, -0.021561196073889732, 0.5348365306854248, 1.2555533647537231, -1.124228835105896, -0.35527729988098145, -1.4535696506500244, -0.17344413697719574, -0.13725633919239044, 0.9609082341194153, -1.0840059518814087, -0.8126649856567383, -0.7047721147537231, -0.5424835085868835, -3.168671131134033, 0.0778551697731018, 0.7442068457603455, -2.693852663040161, -3.260464906692505, -0.04214818403124809, 0.03678823262453079, 0.5777179002761841, 0.8646149635314941, -1.7092567682266235, 0.5691647529602051, -1.6913397312164307, -0.2559156119823456, -0.9417505264282227, 0.7220398783683777, 0.6404814720153809, 0.11792051792144775, 0.0026749265380203724, -0.3857482969760895, -0.000757537956815213, 1.0588773488998413, -0.9576407074928284, 0.6922123432159424, 1.023440957069397, -0.5121358036994934, 0.9333938360214233, -1.118579626083374, -0.02283298969268799, 3.6164557933807373, -0.8510997295379639, 2.026638984680176, -0.911242663860321, 0.10404708236455917, 0.9367277026176453, 1.7268836498260498, -0.3197666108608246, 0.12131097912788391, 1.6867873668670654, 1.6393089294433594, -0.2622830867767334, -0.7177727818489075, -0.6024590730667114, -0.33952754735946655, 1.5172494649887085, 0.005302084609866142, -0.10162831097841263, -0.5547147989273071, -0.35088133811950684, -0.8214392066001892, -0.9385660290718079, 0.7496629953384399, -0.1392558217048645, 0.0863993763923645, 0.24311481416225433, -0.41113507747650146, 0.5439071655273438, 1.14535653591156, 0.44477352499961853, 1.2591015100479126, -0.8157183527946472, -0.6243900060653687, 1.3081985712051392, -0.10366444289684296, 0.8095571398735046, 0.9900913834571838, 0.9426555037498474, 0.05869964510202408, -1.0312918424606323, 1.4708349704742432, 0.29345032572746277, 0.7098755836486816, -1.8259215354919434, 1.045772910118103, 0.3210674524307251, 0.24329541623592377, -0.27341046929359436, 1.6672567129135132, -0.3081856667995453, -0.28414905071258545, 1.1248244047164917, 0.2028728574514389, -1.925405502319336, 2.3528308868408203, 0.85501629114151, -1.6003340482711792, 0.36153632402420044, -0.4856656789779663, 0.6977145075798035, -0.5190554857254028, 1.1096233129501343, 0.3764704465866089, 0.15930695831775665, 0.08815120905637741, -0.18069152534008026, -0.9726263284683228, -1.0294349193572998, -0.7699450850486755, 1.8482565879821777, -0.4920393228530884, 0.17022046446800232, -0.2634081542491913, 1.060860514640808, 1.3034172058105469, -0.5241149663925171, -0.19759978353977203, 0.16046185791492462, -0.024148717522621155, 0.3175741136074066, 0.48258817195892334, 0.36224687099456787, 1.0536282062530518, 0.6994621157646179, 0.9522237181663513, -0.20048844814300537, -0.6088278293609619, 0.4605804681777954, 0.4752448499202728, -0.5333729982376099, -2.0248842239379883, 0.7532118558883667, 0.511178731918335, 0.2501469552516937, 0.11500526964664459, -1.4519121646881104, -0.1310310661792755, 0.4024120569229126, 0.024629417806863785, 0.49530908465385437, -0.6822183728218079, -0.4983364939689636, 0.24021115899085999, -0.9933432936668396, 0.3135119676589966, 0.2780371308326721, -1.3768166303634644, -0.6844059228897095, -2.995936155319214, -0.44521769881248474, 0.6836271286010742, -0.5280197858810425, 1.3455244302749634, 0.46461373567581177, 0.9355330467224121, -0.6135331392288208, 0.862869143486023, 1.803290843963623, -0.252001017332077, -1.7379850149154663, -0.30562713742256165, 2.0535333156585693, -0.9095511436462402, -0.36071258783340454, 0.39010849595069885, -0.15250228345394135, 0.9407402873039246, 0.13755401968955994, -1.2681457996368408, -1.0780243873596191, 0.582266628742218, 1.386393666267395, 1.74809992313385, -0.30779755115509033, -1.586217999458313, 0.3962664008140564, 0.3086176812648773, -0.5097467303276062, -0.31372785568237305, 0.04189443588256836, -0.9632896184921265, -0.199737086892128, 0.017263595014810562, -0.4421289265155792, -1.1934696435928345, 0.3832101821899414, 0.3737293779850006, -0.7851340174674988, 1.9495036602020264, -0.9925945997238159, -0.11158880591392517, -0.44942769408226013, -0.5293762683868408, 1.0367367267608643, 0.26935213804244995, 2.3056328296661377, 1.5330890417099, 0.7402456402778625, 0.4237523674964905, -0.6850618720054626, 1.416059970855713, 0.45828577876091003, -1.1911077499389648, -0.9258684515953064, -2.2548089027404785, 0.3783968985080719, 1.0511449575424194, 1.3464354276657104, 0.20976218581199646, -1.3862509727478027, 0.43923062086105347, -1.1089376211166382, 1.1621347665786743, 2.1589295864105225, 1.1756179332733154, -0.3472069799900055, -1.5594664812088013, -0.029074909165501595, 1.9090237617492676, -0.7131383419036865, -0.2142656445503235, 1.9455838203430176, 1.3026305437088013, 0.07452674210071564, -0.4890112280845642, 0.4749046266078949, 0.3297761082649231, 0.044078998267650604, -0.8416863083839417, 0.5863390564918518, -1.2080591917037964, 0.09907358884811401, -0.9905282258987427, -0.07371652871370316, -0.4530779719352722, 0.27419567108154297, 0.6709569096565247, -0.5745446681976318, 1.5486376285552979, -0.24594202637672424, 0.7467076778411865, 0.09669388830661774, -1.3029927015304565, 2.4390721321105957, -0.6906731128692627, 1.1571670770645142, 1.0591444969177246, -1.3130004405975342, -0.2581425905227661, -1.0591367483139038, -0.47461986541748047, 0.2723828852176666, 1.1557118892669678, -0.5046278238296509, -0.5020447969436646, -0.4608966112136841, -0.14774656295776367, 0.9982179403305054, -0.9585336446762085, 0.05800957605242729, 0.14905224740505219, 0.704593300819397, -1.4609651565551758, 1.3363133668899536, 0.053661394864320755, 0.8725911378860474, -1.5411977767944336, 0.16225992143154144, -0.5920373797416687, -0.28619858622550964, -0.2965618968009949, -0.6204400658607483, 0.11361762881278992, -0.04213149473071098, 0.8262629508972168, -0.40608352422714233, -0.0344308502972126, -0.21404878795146942, 0.36550799012184143, -0.22303444147109985, 0.9160725474357605, -1.2986644506454468, 0.08761889487504959, -0.21632692217826843, 0.9091068506240845, -0.1133127436041832, -1.2120580673217773, 0.19647857546806335, -0.5249847173690796, -0.4937950372695923, 0.005267187021672726, -0.32910609245300293, -1.1045969724655151, 0.5709046721458435, 1.0971643924713135, 0.039837613701820374, 0.46841004490852356, -1.0622234344482422, -1.8612476587295532, -0.883601725101471, -0.7200526595115662, -0.7665089964866638, -0.10788628458976746, -0.944412887096405, 1.2166361808776855, 0.6638567447662354, -2.6173548698425293, 0.23595084249973297, -0.2804369032382965, -0.5577843189239502, -0.7172231078147888, -0.40082916617393494, 2.140624761581421, 1.136107087135315, 1.1121059656143188, -1.1702325344085693, 0.07173231989145279, 0.6176630854606628, -1.0074436664581299, -0.2576344907283783, -0.5098942518234253, 2.2043778896331787, 0.3547557592391968, 1.8036781549453735, -0.37166720628738403, 0.5482465028762817, -1.2116879224777222, -2.0879712104797363, -2.233771800994873, 0.42707744240760803, -1.3139138221740723, -1.5555074214935303, 0.8731740713119507, 1.915106177330017, -0.7627085447311401, 0.24030230939388275, -0.32808929681777954, -0.6281417608261108, -1.5228511095046997, 1.6849523782730103, -0.8212268352508545, -0.3319830000400543, -0.3428083062171936, -0.40085718035697937, -0.4153062403202057, -1.177786946296692, 2.5025811195373535, 0.007622195407748222, -0.43171176314353943, -1.2485716342926025, 0.7511671781539917, -0.8345581293106079, -1.7237728834152222, 0.5318241715431213, 1.179136037826538, -0.1991662085056305, -0.6838926672935486, 1.6669303178787231, 0.5676488876342773, -0.4709652066230774, 0.06986220180988312, -0.7612845301628113, -0.8205459713935852, 1.3995492458343506, -1.9001938104629517, -1.6391916275024414, 0.9454411864280701, -0.8853294849395752, -0.24286167323589325, 0.6571380496025085, 0.9423854947090149, 0.026922527700662613, -0.2758312523365021, -0.7600584626197815, 0.9400740265846252, -0.5323881506919861, 0.5959787964820862, -0.869442880153656, -0.6103681921958923, 0.6530673503875732, -0.7168328762054443, -0.779742419719696, -1.6620444059371948, -0.3779992461204529, 1.2732884883880615, 0.04816050827503204, 0.11384794861078262, 0.5707452893257141, 0.775476336479187, 0.6454987525939941, 1.4111911058425903, -1.8081010580062866, -0.3678892254829407, -0.5884419083595276, 0.1315414160490036, -0.8217309713363647, -0.5529285073280334, -1.3767632246017456, -0.29599761962890625, -0.2218308001756668, 0.6743995547294617, -0.47410616278648376, -0.05145437642931938, 0.5037432312965393, -0.45118749141693115, 0.5606957077980042, -0.14052921533584595, 0.8557443618774414, 0.4530690610408783, -0.5869914293289185, 0.9330598711967468, -0.7055829167366028, 0.28848814964294434, 0.2839818298816681, -1.1305792331695557, 2.3146402835845947, 0.7528697848320007, 1.0033973455429077, 1.5713614225387573, -1.393159031867981, 0.19239145517349243, -0.5456458926200867, -0.45454689860343933, -0.5238112211227417, -0.06762760132551193, -0.3122415244579315, -0.7426672577857971, -1.9882068634033203, 0.2215949147939682, 0.2700323164463043, -2.1284162998199463, 0.7064216136932373, -0.23710477352142334, 0.05443190783262253, -0.4455861747264862, 1.8009129762649536, -0.1520557403564453, 0.8054699897766113, 0.7227069139480591, 0.3914699852466583, 0.06354779750108719, 0.708830714225769, 0.30633744597435, -0.3020678460597992, 2.5496203899383545, -0.453926682472229, -0.0024314080365002155, -0.4201047420501709, -0.38435637950897217, 0.7325712442398071, -0.23300935328006744, -1.3398972749710083, -1.15501070022583, -1.5014760494232178, 1.3338627815246582, 1.311453104019165, -0.5768397450447083, 0.7175676226615906, 1.114086389541626, -0.36367446184158325, 0.5588821172714233, -0.7875518798828125, 1.6147481203079224, 0.7846102714538574, -0.08054465055465698, -0.4897846281528473, -0.6230489015579224, -1.276963710784912, -0.3186982274055481, -0.9933699369430542, 2.357426404953003, 0.9347593188285828, -1.2972652912139893, 1.191757321357727, 0.12013179063796997, -1.0127720832824707, -0.7666499614715576, -0.016743596643209457, -0.308336466550827, -0.6454899907112122, -1.2672985792160034, 0.3175030052661896, -1.2117749452590942, 0.009015410207211971, -0.012182326056063175, -1.1139214038848877, -0.6947239637374878, 2.238147258758545, 0.9203981161117554, -1.244572639465332, -0.14091695845127106, -1.2914496660232544, -1.2170482873916626, -1.7188036441802979, 1.5056484937667847, 0.3200531601905823, 0.3199787139892578, 2.3041980266571045, -1.7648184299468994, 0.9918258786201477, -1.019985318183899, 0.9410395622253418, -1.2694259881973267, 1.2376517057418823, -1.6074647903442383, -0.8142994046211243, -0.1430816501379013, -0.22358332574367523, 0.8338929414749146, -0.02804122492671013, -0.9400785565376282, 2.134155750274658, -1.187257170677185, 0.33882030844688416, -0.8331136703491211, -0.8789084553718567, 0.22647088766098022, -0.754256010055542, 1.0950928926467896, -0.6570122241973877, -1.3716768026351929, 0.06988437473773956, -0.6878718733787537, 0.13289068639278412, -0.42779919505119324, 0.22703135013580322, 0.849886417388916, 1.6713966131210327, 1.3745777606964111, -0.3061502277851105, 2.031979560852051, -0.33232858777046204, 0.5539831519126892, -0.05071232467889786, -0.2921053171157837, 0.1580073982477188, -1.610646367073059, -0.09912404417991638, 1.361234426498413, -0.1672111451625824, -0.12216468900442123, -0.13967666029930115, 1.4967012405395508, -0.9882457256317139, 0.6432788372039795, 0.8389703631401062, -0.7201952338218689, 0.0970921665430069, -0.24488644301891327, -0.5420399308204651, -0.44916269183158875, 0.562218189239502, 0.05435450002551079, 0.09606240689754486, -0.04469858109951019, 0.9819337129592896, 0.017091890797019005, 0.4593525826931, 2.5466485023498535, 0.8243495225906372, -0.6872786283493042, -0.5617852210998535, 0.18415462970733643, -0.47813740372657776, 1.2784337997436523, -0.8441134691238403, -1.078871488571167, -0.46785077452659607, -0.18231506645679474, -0.5153795480728149, 0.10452195256948471, -0.5727584362030029, 0.36857324838638306, 0.5773088335990906, 1.4055612087249756, -0.8903121948242188, -1.3741703033447266, 1.1671470403671265, -0.7210080623626709, 1.3335261344909668, 0.7551457285881042, 1.273503303527832, 1.1527800559997559, 0.4686869978904724, -0.9077464938163757, 0.39246031641960144, 0.8137215971946716, 0.013947423547506332, 0.08537966012954712, 1.5635020732879639, 1.486158013343811, -0.32743608951568604, 0.4022738039493561, -1.2001230716705322, 2.638145923614502, 0.2568020522594452, 0.581271231174469, 1.6894004344940186, -0.384769082069397, -0.2898622751235962, -0.8101919889450073, 0.21998478472232819, 0.774172842502594, -0.7901638150215149, -0.17295321822166443, -0.7475448250770569, -0.1568908542394638, -1.5695738792419434, -0.04548544064164162, 0.3047954738140106, 0.7222967147827148, -0.7308357954025269, 0.003442642744630575, 0.5723187923431396, -0.63484126329422, -0.7595980763435364, -0.21683412790298462, 0.6555741429328918, 0.7050008177757263, -0.9125930666923523, -0.013261842541396618, -0.2615588307380676, -0.5561917424201965, -0.8096861243247986, 0.04554692283272743, 0.21879994869232178, -0.19070783257484436, 1.0704151391983032, 0.5418334603309631, 0.5289578437805176, -0.08816123753786087, -1.086348295211792, 0.1943747103214264, -0.1571146547794342, -0.7868884801864624, 0.9430820941925049, -0.6352570652961731, -1.0291149616241455, 1.083099603652954, -0.19484026730060577, -0.8786171078681946, -0.6359883546829224, -1.6240530014038086, -1.3743185997009277, 1.908888578414917, 0.4859517514705658, -0.26117655634880066, -0.22615620493888855, 0.139070063829422, -0.2792103588581085, -0.21273452043533325, -0.48312318325042725, 0.20412397384643555, -1.7194072008132935, -1.3986467123031616, 0.45430511236190796, 1.5692944526672363, 0.5457682013511658, -1.7899627685546875, 2.1253316402435303, 0.19907771050930023, -1.35298490524292, -1.3606523275375366, 0.05804036930203438, -0.4407653212547302, -0.877971887588501, 0.33374258875846863, 1.2817307710647583, 0.8050312995910645, -0.619198739528656, -0.7092077136039734, 0.26351600885391235, -0.8679066896438599, 0.28729528188705444, 1.1965477466583252, -0.39156341552734375, 0.55849289894104, 1.6694587469100952, 1.7593830823898315, 1.6237809658050537, -0.46993687748908997, -0.12726040184497833, -0.5703067183494568, -0.14994682371616364, -0.4544937312602997, -0.0910845547914505, 0.5809805989265442, -0.5969673991203308, -0.7348392009735107, -0.2990717887878418, -0.20606543123722076, 0.6445618867874146, -0.18850404024124146, 1.5890777111053467, 0.12461452186107635, -0.8808395266532898, 0.9817566871643066, -0.442490816116333, -0.13492776453495026, -2.0775625705718994, -0.23030413687229156, -0.7805129885673523, 1.2282963991165161, -0.11548780649900436, -1.0528408288955688, -1.149287462234497, 0.32833701372146606, -0.04593651369214058, 0.04273175075650215, 0.8596840500831604, 0.45443519949913025, -0.7704473733901978, -0.2390577793121338, 0.3056611716747284, -0.7879740595817566, -1.3125168085098267, -2.258528470993042, 1.0675803422927856, -0.12177801877260208, 0.07919082790613174, 0.6947895288467407, -0.22334426641464233, 0.727521538734436, -0.18904347717761993, 0.6233145594596863, -0.6844205856323242, 1.2828669548034668, -1.3417381048202515, -2.302029848098755, -0.4425014555454254, -1.2263262271881104, -1.9205302000045776, 0.42577803134918213, -0.24154944717884064, 0.9269845485687256, -0.48032569885253906, -0.852780282497406, 0.14703907072544098, 0.5163851976394653, 1.2099815607070923, -0.3294573724269867, -0.8789740204811096, -2.096457004547119, 1.2683472633361816, 0.2690333425998688, 0.8328644633293152, 0.6770128607749939, -0.48089954257011414, -1.1572303771972656, 0.8015154004096985, 1.3323625326156616, -1.3679275512695312, -0.22981910407543182, 1.2710531949996948, -0.20134088397026062, -0.051282450556755066, 0.27873101830482483, 0.1209384873509407, -0.05812983587384224, -1.0726865530014038, -1.4127694368362427, 0.6209253668785095, 0.6885901093482971, 0.28684571385383606, 0.9847667813301086, 0.10076592117547989, -0.22034481167793274, -0.5277456641197205, 1.4721039533615112, 2.9110987186431885, -1.0248703956604004, 1.2739598751068115, -0.11635326594114304, -0.9406569004058838, -0.17169755697250366, 0.10032842308282852, -0.3830087184906006, 1.189455270767212, 0.32891085743904114, 0.07652907073497772, 0.9576942324638367, 0.5601828098297119, -0.45200175046920776, 0.21657152473926544, 0.7242000699043274, -0.12001040577888489, -0.020472193136811256, -1.5635509490966797, -1.4418607950210571, -0.20464935898780823, -0.5898975133895874, -0.6107136011123657, 1.3272061347961426, 0.20842896401882172, 1.216699481010437, -1.3323689699172974, 0.7198691368103027, 0.11166248470544815, -1.6608871221542358, 0.2280663102865219, -1.4817578792572021, 0.7486128211021423, 0.7697572708129883, -0.5717458724975586, -0.44786420464515686, -0.4358867406845093, -0.18295209109783173, -1.1650049686431885, 0.05692850425839424, 0.3224128782749176, -0.8621812462806702, 1.9396082162857056, -0.8456915616989136, 0.21265827119350433, 0.3863356411457062, 1.0458275079727173, 1.4547864198684692, -0.23375169932842255, -0.7249026298522949, -0.6284168362617493, -1.1934024095535278, -0.5732328295707703, -0.9855180382728577, -1.1575630903244019, -1.5224230289459229, 0.9883456230163574, 0.2350424826145172, 1.2456928491592407, -0.5648902058601379, -0.8852108120918274, 1.5419882535934448, -0.4600982069969177, -0.2688431739807129, -0.7059393525123596, 1.4793200492858887, -0.9349061250686646, 1.049590826034546, 0.1265586018562317, -1.4360170364379883, 0.26023250818252563, 0.31079551577568054, 0.6556174159049988, -0.21848498284816742, -0.6518983244895935, 0.6721715331077576, 1.2748656272888184, -0.3694232702255249, 2.9029500484466553, 1.4290883541107178, 0.279679536819458, 1.3924098014831543, -0.4767962396144867, -0.7667360305786133, -0.18180911242961884, 0.08085262030363083, -1.5866612195968628, -1.4797872304916382, 0.5836659669876099, 0.7927640676498413, 1.444667935371399, -0.34832799434661865, 0.592232882976532, 0.8202115893363953, 0.5429969429969788, 1.348087191581726, -0.6529176831245422, -0.07121644914150238, 1.1706597805023193, 0.4838643968105316, -0.18577027320861816, -1.0428519248962402, 1.0891642570495605, -0.3665659725666046, -0.06762690097093582, -1.5172560214996338, 1.9770957231521606, 1.524960994720459, -0.7607956528663635, -1.3384653329849243, 0.926748514175415, 0.9550601243972778, 1.6874171495437622, 0.03293831646442413, 0.8900817632675171, -1.4401829242706299, 0.3612285554409027, 0.7482163310050964, -0.7065136432647705, -1.306398868560791, 0.49613264203071594, -0.046226438134908676, 0.8811928629875183, 0.6162340044975281, 0.9978709816932678, -0.7060437798500061, -1.2414640188217163, -1.2321323156356812, -0.5024037957191467, 0.14225991070270538, -0.31691816449165344, 1.3198626041412354, 1.013532042503357, 0.0804668664932251, -0.8677881956100464, 1.2345809936523438, -0.3423563539981842, 0.28970327973365784, -0.5932673215866089, -0.7801081538200378, -0.11187360435724258, -0.5017332434654236, 0.8360399603843689, -1.2354243993759155, 0.04060368612408638, 0.10889622569084167, -0.5751094222068787, -1.7365636825561523, -0.542201817035675, -0.4180081784725189, -0.5210677981376648, -0.3643539249897003, -0.0525563508272171, -0.7735863924026489, -0.597144365310669, 1.1466357707977295, -0.11647745221853256, 0.24798059463500977, -0.14663973450660706, -1.0144109725952148, 0.5761300921440125, 0.41508427262306213, -1.4705122709274292, -0.3068220317363739, -0.21682704985141754, 0.6604570150375366, 0.603196382522583, 0.3438345193862915, -1.3545528650283813, 0.3014872670173645, -1.557596206665039, -1.2398451566696167, -1.6576695442199707, -0.5682318806648254, -0.32601749897003174, -0.31744569540023804, 0.2867277264595032, -1.6560790538787842, 1.2540967464447021, 1.2549123764038086, 0.9464232325553894, 0.8664200305938721, -0.23604419827461243, -0.8261554837226868, -0.9947699308395386, -0.9942283630371094, 0.7911149263381958, 0.5800946354866028, 1.0157207250595093, 1.7641363143920898, 1.7800248861312866, 0.4595637023448944, 0.9430511593818665, 1.3738925457000732, 0.7263818979263306, 0.5977703928947449, 0.9523105025291443, 0.8610721230506897, 1.4273182153701782, -0.12853901088237762, 1.9812822341918945, -2.136805295944214, 1.214571475982666, 1.2347431182861328, -1.6414437294006348, -0.757548451423645, 0.33943501114845276, -0.5649940371513367, 0.8062363862991333, 1.217842936515808, 0.46233683824539185, -1.3549531698226929, 0.7957040667533875, -1.1317230463027954, -0.7313040494918823, -0.587231457233429, 0.9494417905807495, -0.05643649399280548, -0.4432683289051056, -0.6630769968032837, 1.1082738637924194, -0.7273913025856018, 0.24823057651519775, -1.321237325668335, -0.6236632466316223, -2.588954210281372, 1.0388811826705933, -1.7949938774108887, 1.081552267074585, 2.0637142658233643, -0.20771479606628418, 1.4783430099487305, 0.41002005338668823, -0.25921377539634705, 0.42778971791267395, 0.800540030002594, 0.7127546668052673, 0.4861842393875122, -0.09051555395126343, -1.4012805223464966, 0.09143199026584625, -0.9294572472572327, -1.1867835521697998, -0.13907957077026367, -0.07163561135530472, -0.9792715311050415, 1.0616549253463745, -0.03473808243870735, 0.7787282466888428, 0.7050427794456482, 0.9792681336402893, 2.0999088287353516, -1.0096839666366577, 0.316943883895874, -0.9747252464294434, -0.947238564491272, 1.4231048822402954, -0.44519126415252686, -1.350562334060669, 0.6843661665916443, -0.7435809969902039, -1.4208295345306396, 0.8427402377128601, 0.7095861434936523, -0.335972398519516, 0.10759248584508896, 0.8262329697608948, 0.7065075039863586, 1.0046701431274414, -0.5077798962593079, -0.28565728664398193, -0.8105174899101257, -1.2879494428634644, 0.7062346935272217, -0.541543185710907, -0.014994335360825062, 0.08876311779022217, -0.7439842224121094, -1.247306227684021, 0.5017350912094116, 2.049900531768799, 0.17098753154277802, -1.9526405334472656, 1.5515577793121338, 0.9584619402885437, -0.330520361661911, 1.4447238445281982, -0.4317498803138733, -2.0548853874206543, -0.9542973041534424, -0.0314701683819294, -1.1608647108078003, -0.6620491147041321, -0.5612387657165527, -0.29253247380256653, 0.06591726839542389, 0.1370500922203064, 1.4851367473602295, 1.1820732355117798, 0.8673837780952454, 0.6251690983772278, 0.12196412682533264, -0.05933768302202225, -0.6608532071113586, -1.0718914270401, 0.6323763728141785, 3.196474075317383, -0.6218317151069641, 0.7352610230445862, -0.15710124373435974, 0.4192248284816742, 0.23159201443195343, 0.38699260354042053, 0.12139081209897995, -0.17154037952423096, 1.127119779586792, -0.3012508451938629, -0.3325031101703644, -1.0313743352890015, -0.6901156902313232, 1.2383973598480225, -1.5965570211410522, 0.32140928506851196, 0.3280903995037079, 2.1950888633728027, -0.3808998465538025, 0.9347511529922485, -0.689619243144989, -0.08312657475471497, 0.8395838141441345, 1.599735140800476, -1.0811282396316528, -0.6882236003875732, 0.38441959023475647, -0.587641716003418, 0.2026291787624359, 1.644655704498291, 1.0094836950302124, -0.1274970918893814, -0.5556319952011108, 0.5263673067092896, -0.5083044767379761, -0.17276346683502197, 0.0569654144346714, 0.016421055421233177, 0.596852719783783, 0.30755874514579773, -0.4824739396572113, 0.6776251792907715, -0.7396585941314697, -1.0983946323394775, -1.046190619468689, -0.1409972757101059, -0.47259166836738586, 1.217787742614746, -0.47887134552001953, 0.4351300895214081, 0.21453800797462463, 0.31250882148742676, -0.2220103144645691, -1.4205961227416992, -0.9776332974433899, 1.256142258644104, -0.7246419787406921, -0.5774064660072327, -1.493540644645691, -0.5969614386558533, -0.1733117401599884, 0.7733227014541626, -0.7753722071647644, -0.8492092490196228, 1.052667260169983, -2.010646104812622, 1.3735641241073608, 2.343635082244873, 0.024073468521237373, 0.28471148014068604, -0.19342245161533356, -0.9749521613121033, -1.0795669555664062, -1.115676760673523, -0.1479392647743225, 1.2441810369491577, -0.4579693377017975, -2.2108852863311768, -0.516462504863739, -1.4809353351593018, -0.7532307505607605, 0.4745723009109497, 0.08063392341136932, 0.9878464937210083, 1.1356863975524902, -0.7702709436416626, 0.7427972555160522, 0.42553648352622986, 1.1178258657455444, -0.9020575284957886, -1.0321723222732544, -0.8665578365325928, 0.49635472893714905, -0.5944109559059143, 0.7841355800628662, -0.504733681678772, -0.6605511903762817, -1.1447206735610962, -0.36155304312705994, 2.384601593017578, -0.5058503150939941, -1.106412410736084, -0.4849852919578552, -0.4268234074115753, -0.6132364273071289, 0.9667428731918335, 0.03381543233990669, 2.2540743350982666, 0.08661346137523651, 0.8530863523483276, 0.011132783256471157, 0.514510989189148, -2.3737387657165527, 0.4197830855846405, -1.573835015296936, -0.4973827302455902, -0.6087694764137268, -0.8627035617828369, -0.4133862555027008, 0.40626397728919983, -1.0829302072525024, -0.43440112471580505, 1.2061069011688232, 1.0727322101593018, -0.0355968214571476, -0.5231958627700806, 0.7677586078643799, -0.45897045731544495, -1.0574814081192017, -0.3398829996585846, 0.3870985805988312, -0.7206828594207764, -1.4653651714324951, -0.6019275784492493, -1.0332708358764648, -0.8482580780982971, 1.6483570337295532, -0.6457782983779907, 1.401271939277649, 0.8440433144569397, -0.010754585266113281, -0.5567205548286438, -2.067634105682373, -0.15719619393348694, -0.6540911197662354, -0.6788762807846069, 0.6859260201454163, -0.7329833507537842, 0.6202783584594727, -0.4112580418586731, 0.22101256251335144, -0.47790512442588806, -0.07440708577632904, 0.8874115347862244, 1.0536681413650513, -0.04539503529667854, -0.3765806257724762, -1.0352929830551147, 0.7479190230369568, 2.4763753414154053, -1.2915942668914795, -0.4429226815700531, -0.09619554877281189, 0.746942937374115, 0.039475057274103165, -1.0499178171157837, -0.7995729446411133, -0.3450959622859955, -0.258066862821579, 1.182204008102417, -1.7381154298782349, -0.48052841424942017, 0.8835939764976501, -0.8822795748710632, -0.2565763592720032, 0.8002719283103943, -0.19560283422470093, -1.1481643915176392, 0.10838263481855392, 1.1844279766082764, 0.7565038204193115, -0.7106721997261047, -1.3634247779846191, -0.5794677734375, 1.870889663696289, 0.05318889394402504, -0.4068155288696289, 2.251765489578247, -0.8396274447441101, -0.6160182952880859, 0.31993624567985535, 0.26140525937080383, -0.33612653613090515, 1.6454293727874756, 0.4258032441139221, -0.8571107387542725, 0.0755729004740715, 0.6326658129692078, 1.1746426820755005, 1.0377148389816284, 0.9754847288131714, 1.0640244483947754, 0.6912798285484314, 2.543604850769043, 1.3188080787658691, -0.2973233759403229, -0.23964689671993256, 0.029220355674624443, -0.7438450455665588, 0.8122612833976746, 0.5301328301429749, -2.066983222961426, -0.6159985065460205, 0.6084211468696594, 0.32267212867736816, 1.2558709383010864, -0.4898959994316101, -0.5695870518684387, 0.6221617460250854, 1.532334327697754, 0.1299460083246231, -1.0540474653244019, 0.09906624257564545, -1.1933338642120361, 1.7573611736297607, -0.16915981471538544, -0.4451221823692322, -1.0432184934616089, 1.555548906326294, 0.1385829597711563, -1.6306123733520508, 1.0364737510681152, 1.5299931764602661, -0.39221569895744324, 2.501342296600342, -0.5147535800933838, -0.14754465222358704, 0.2710099220275879, -1.4679304361343384, 0.2518174946308136, 1.950166940689087, 0.6043729782104492, -0.01130295917391777, 1.0230693817138672, -1.6307876110076904, 0.0692167654633522, -1.0293008089065552, 0.7189255356788635, 1.3886595964431763, -0.15264295041561127, 1.0461046695709229, 2.355701446533203, -1.3734571933746338, -0.1816890388727188, 1.9808974266052246, 1.1138274669647217, 0.6047961115837097, -2.3000965118408203, -0.996085524559021, 0.07938088476657867, -1.7577265501022339, -0.3180873394012451, -0.991007387638092, 0.48294419050216675, 0.4751569926738739, -0.33542290329933167, -0.2057592272758484, 2.467217445373535, 0.3310830891132355, 0.7759392261505127, -0.3409135639667511, 0.5790307521820068, -2.3107292652130127, -0.19183963537216187, -0.5107108354568481, -0.9542626738548279, -0.5061548352241516, -0.3916963040828705, 0.9097661375999451, 0.35007527470588684, -0.6624705791473389, -0.19801510870456696, -0.5785824060440063, 0.8607074618339539, 0.43868008255958557, -0.06730210781097412, 0.4771406352519989, -1.8359044790267944, -2.155754327774048, 0.27880918979644775, -1.3378320932388306, 0.7186538577079773, -0.30531904101371765, -0.4049105942249298, 0.9371458292007446, -0.24608725309371948, 0.6032463312149048, 0.6784735918045044, -1.192702054977417, -1.4929800033569336, -0.34571778774261475, 0.19671496748924255, 0.2852364182472229, -0.32084307074546814, -1.6776132583618164, 0.5346804261207581, -1.2361600399017334, -0.11647021025419235, -0.8116138577461243, -0.6072080731391907, -1.3412671089172363, 0.6112183928489685, 0.431946337223053, 0.5080348253250122, 1.2444244623184204, 0.34708741307258606, -1.1931520700454712, -0.7350801229476929, 0.4197309613227844, 2.0927510261535645, 0.40129825472831726, 1.4062585830688477, 0.06365689635276794, 1.170413613319397, 1.9480693340301514, 0.055575624108314514, 0.5624350309371948, 0.5968168377876282, 0.6261339783668518, 0.8658428192138672, 0.3265717029571533, 1.4819588661193848, -1.4626981019973755, -0.4453217089176178, -0.7024694085121155, -0.1970279961824417, 2.0351836681365967, 1.5978034734725952, -0.1849765181541443, 1.6185858249664307, -0.0963413342833519, 0.29377827048301697, -1.5874167680740356, -0.27626174688339233, -0.33092668652534485, 0.1122436448931694, -0.8643084168434143, -0.27992963790893555, -0.6647518873214722, 0.030402570962905884, 1.9926567077636719, -0.5370252132415771, 0.8865988254547119, -0.10935555398464203, -1.7867212295532227, 0.015918361023068428, -0.8468946218490601, 0.2509421110153198, 0.27860352396965027, -0.45744380354881287, -0.4575226604938507, 0.32062089443206787, -0.3912900388240814, -0.5453701615333557, 0.0445091687142849, -0.6933731436729431, -0.45696714520454407, -1.671102523803711, 0.632183849811554, 0.19098733365535736, -0.04653141275048256, 2.5138473510742188, -1.212119698524475, -0.1943373680114746, -0.9396803379058838, -0.7597191333770752, 0.743226170539856, -0.14955994486808777, -0.6965435743331909, 1.3013768196105957, 0.37631529569625854, -0.03342695161700249, -1.623674750328064, -0.010316595435142517, 1.3418588638305664, -0.18538181483745575, -0.46048933267593384, 0.1085200384259224, -0.17061792314052582, -1.6445173025131226, -1.550386667251587, -0.6709368824958801, 0.38779088854789734, 0.9735932946205139, 0.5276181101799011, -0.03961165249347687, -0.23744754493236542, 0.9626915454864502, 1.1367673873901367, 1.1259822845458984, -0.8889744877815247, 0.26998278498649597, -0.13226114213466644, -0.8778929710388184, 0.3361869752407074, 0.34574323892593384, 1.0907964706420898, 1.6585114002227783, 0.8715772032737732, -0.46835291385650635, 1.1817822456359863, -0.0207289457321167, 0.5686145424842834, -0.3154437243938446, -0.06816308945417404, 0.1439206451177597, 0.1615273803472519, -0.441570907831192, 1.4574991464614868, -1.4625787734985352, -0.02965814433991909, 0.15444213151931763, -0.16040216386318207, 0.9541844129562378, -0.21728254854679108, -1.095986247062683, 0.9643735885620117, -2.2905216217041016, 0.5580878257751465, -0.4173736870288849, -1.3339203596115112, 0.03835578262805939, -2.9602839946746826, -1.0445759296417236, 1.9463483095169067, 1.0119099617004395, -1.8348439931869507, -1.8041318655014038, -1.129716396331787, 1.083195686340332, 0.7206490635871887, 0.21910694241523743, -1.464800477027893, -1.2203832864761353, 0.2258896678686142, 0.3777976334095001, 0.49769920110702515, -0.20548880100250244, 0.7862297296524048, 0.3529454469680786, -0.2713674008846283, -0.055569350719451904, 0.37329375743865967, 1.428153395652771, 1.2468836307525635, -1.3902734518051147, 0.38317111134529114, -1.7273529767990112, 0.6545289754867554, -1.5466454029083252, 0.7251660823822021, 0.947878360748291, -0.3818725645542145, -0.7839598059654236, 1.2341229915618896, -1.3469003438949585, -0.06977968662977219, -0.060104284435510635, 0.5022348165512085, 0.03741173446178436, 0.39494848251342773, 1.4371830224990845, -0.5471547842025757, 0.6152737736701965, -1.5545377731323242, 0.49597790837287903, 1.925168752670288, -0.9681324362754822, -0.09944413602352142, -0.8097532987594604, -1.570865273475647, 0.3763340413570404, 0.5011006593704224, -1.741371989250183, -0.4861949682235718, -1.168128490447998, -0.12261755019426346, -0.2232259213924408, -1.2302770614624023, 0.3755004405975342, 1.5358833074569702, 0.8022739887237549, 0.12650993466377258, 1.512127161026001, 0.15958024561405182, -1.0343207120895386, -0.593624472618103, -1.162276029586792, -1.0563287734985352, -0.7342476844787598, -0.512976884841919, -0.39854443073272705, -0.21084220707416534, 1.0620619058609009, -0.49199724197387695, -0.09685082733631134, -0.8393188714981079, 1.3339101076126099, -0.6364955902099609, -1.5635508298873901, 0.18608133494853973, -1.6314395666122437, -0.4279252588748932, 1.7724336385726929, 0.4064732491970062, 1.3503670692443848, 0.5217588543891907, 0.4213368594646454, 0.9544373750686646, 0.39634159207344055, 0.5901300311088562, 1.0799403190612793, 0.8741488456726074, -1.6829406023025513, -2.128147840499878, -0.12352318316698074, 1.6927117109298706, -0.18678125739097595, 0.10228991508483887, -0.21061132848262787, -0.5451372861862183, -1.940872311592102, -0.7831013202667236, 0.9362002015113831, -0.7977797389030457, 0.8279827833175659, 2.071288585662842, 0.5823355913162231, 0.8242948651313782, -0.880662739276886, 0.39655351638793945, -0.4756246507167816, 0.8012135028839111, -0.050578802824020386, 0.7434006333351135, -0.5913207530975342, 0.6159900426864624, -3.2132835388183594, 0.8390283584594727, 0.09237142652273178, 0.41472262144088745, 0.7606752514839172, 0.3411591053009033, 1.1277905702590942, -1.026922583580017, 1.0248291492462158, 1.8376615047454834, 1.258703351020813, 0.02921663224697113, -0.5084714889526367, 0.330883264541626, 0.07365845888853073, 0.02617921307682991, 1.440405011177063, -0.5038226246833801, -0.8930405378341675, 0.5589894652366638, -0.4398770332336426, -2.5672430992126465, 1.0724042654037476, 0.46270301938056946, -0.20294684171676636, -2.0628609657287598, -1.2457119226455688, -0.4147626757621765, 0.9130944609642029, -2.0933008193969727, -1.7973508834838867, 1.027658224105835, -0.5354862809181213, -0.7007448077201843, -0.3776465952396393, 0.2738152742385864, 0.6027746796607971, 0.4520762860774994, -0.44983160495758057, 0.6021115779876709, -1.9247018098831177, 0.08701067417860031, 0.06409134715795517, -1.1226693391799927, -0.7266127467155457, -0.530888020992279, -0.22116558253765106, 0.20021876692771912, -1.1391417980194092, 0.09565986692905426, 1.3290951251983643, -0.48570677638053894, 1.1442562341690063, -0.9967947006225586, 0.19585709273815155, 0.8652313947677612, 1.0107983350753784, 0.7472690343856812, -0.2875196933746338, 0.7299672961235046, 1.6076453924179077, -1.5760968923568726, 0.7592698335647583, 0.2570262551307678, 0.034747518599033356, 0.11090108007192612, -0.4563169479370117, 0.3650498390197754, 1.0354520082473755, -0.5350748300552368, 0.30232834815979004, 1.219300627708435, -1.7324706315994263, -0.26946061849594116, -1.6615173816680908, 1.0082486867904663, 0.5098826289176941, 0.08676169812679291, 0.048729319125413895, 0.6176369786262512, -0.23102377355098724, 0.17234033346176147, -0.5679476261138916, 1.3060932159423828, 1.2380634546279907, -1.746256947517395, -0.823035478591919, 0.8914444446563721, -0.17691920697689056, 0.5095636248588562, 0.14478860795497894, -0.37140750885009766, 1.019689917564392, -0.23477312922477722, -0.5516421794891357, -0.7654751539230347, 1.9898780584335327, 1.2305184602737427, -0.04029703512787819, -0.3474361300468445, -1.2845723628997803, -1.1939489841461182, 0.2346782088279724, 0.6003058552742004, 1.4847781658172607, -1.6416198015213013, -0.5214897394180298, -0.73863685131073, -0.3539925813674927, -0.6219291090965271, 0.35779091715812683, 0.3923807442188263, 0.3742467761039734, -0.09044062346220016, 1.442990779876709, 0.3675937056541443, 0.6256298422813416, -1.5054936408996582, -0.7645560503005981, 0.5343822836875916, -0.5691503882408142, -2.4667296409606934, -0.7110426425933838, 0.47537899017333984, 0.1553596705198288, 0.5924443602561951, 0.636918842792511, 1.1487594842910767, 1.8364046812057495, 0.6187679767608643, -0.024854417890310287, 0.8672845959663391, -0.08994514495134354, -0.8640034794807434, -1.2792420387268066, 1.756777048110962, -1.5954060554504395, 1.8388313055038452, -1.9825584888458252, 0.5981734991073608, 0.22722020745277405, 0.9676842093467712, 1.3663780689239502, -0.04506004601716995, -0.001061557326465845, 0.03145787492394447, -2.4460439682006836, -1.590491771697998, -3.001373052597046, -0.26362407207489014, 0.37725481390953064, 1.4213895797729492, -0.40716469287872314, -0.3925797641277313, 1.3736430406570435, -1.4843871593475342, 0.049446988850831985, -0.7013049125671387, -0.8301695585250854, 1.2800484895706177, -0.6286715865135193, 2.1529433727264404, -1.2256227731704712, -1.147073745727539, -1.412506341934204, -1.0495072603225708, 0.04609781876206398, -0.20195786654949188, -0.2375968098640442, -1.6962286233901978, 0.4097951650619507, 1.1106668710708618, 0.6515383124351501, 0.5171176195144653, -0.8722770810127258, 0.42731088399887085, -0.019201794639229774, -0.8628811836242676, 0.6978169083595276, 0.3844461143016815, -0.5705645680427551, 0.13826988637447357, -1.5708222389221191, 1.4830819368362427, 0.9620857834815979, -0.3444027602672577, 0.9098595380783081, -1.270653247833252, -0.28507712483406067, 0.09387929737567902, -0.6091160178184509, 0.8444253206253052, 1.478727102279663, -0.4342145621776581, 0.2201496660709381, 0.5361868739128113, -1.6190062761306763, 0.3676389753818512, -0.9311364889144897, -0.9738271236419678, 0.3460458517074585, -0.5055038928985596, -0.7538668513298035, -1.319456696510315, -0.3962376117706299, -0.8214520812034607, 0.12910610437393188, 0.2219497561454773, -0.04557616263628006, 1.0824474096298218, 0.9060139060020447, 0.2792310416698456, -1.46675705909729, 0.08398255705833435, -0.12684941291809082, 1.9496357440948486, -0.7258092761039734, 1.1343353986740112, 0.8830015063285828, -0.1535663902759552, -0.35704395174980164, 0.3140609562397003, -0.15998370945453644, 0.16160805523395538, -1.2949360609054565, -0.048009779304265976, 0.02782629244029522, -0.32020336389541626, -1.1107466220855713, 0.9954748749732971, -0.07754532992839813, 0.8362317681312561, 1.6482981443405151, -1.5867984294891357, 2.06002140045166, -0.7704406976699829, -0.19850941002368927, -0.12664426863193512, 1.1915892362594604, 1.1235685348510742, -0.012732189148664474, -0.3357005715370178, 0.5701570510864258, -0.6441300511360168, -0.3533231019973755, -1.7644810676574707, -0.5258352756500244, -1.6327385902404785, -1.1200459003448486, 0.7033066749572754, -0.8830273151397705, -0.8835430145263672, 1.6045787334442139, -1.3908812999725342, 1.3079197406768799, 0.37499016523361206, -0.06717890501022339, -1.1984505653381348, 0.17008483409881592, 0.8708271980285645, -0.3526162803173065, -1.6800312995910645, 1.0596307516098022, 0.18577946722507477, -0.20334206521511078, 0.5235434174537659, 0.33414846658706665, 1.7620421648025513, -0.8313493132591248, -0.21453244984149933, 0.06323271989822388, 0.022333674132823944, 1.1365519762039185, 0.7348361015319824, -0.9589927792549133, -2.3523662090301514, -0.7513911128044128, 0.034092169255018234, 0.3674200475215912, -1.3382655382156372, 1.5386481285095215, 0.3053435683250427, 0.023596782237291336, 1.4176721572875977, 1.1464636325836182, -0.847244143486023, 1.4093397855758667, -1.3794113397598267, -0.3346227705478668, 0.4366835951805115, 0.08190342038869858, 0.5661567449569702, -1.9581618309020996, -2.938431978225708, -1.109786033630371, -2.4239706993103027, -1.2252418994903564, -0.26952826976776123, -1.21488618850708, 0.24282492697238922, 1.90583074092865, 0.36660075187683105, -1.3920987844467163, 2.8150298595428467, 0.32306480407714844, -0.17059025168418884, 0.35744577646255493, -1.0944535732269287, 1.644919753074646, -1.6554512977600098, -0.8736943006515503, -0.2154950499534607, -0.997890055179596, 0.9199960231781006, 1.555989146232605, -1.1889317035675049, -0.04586426913738251, -0.5419853925704956, -1.9807580709457397, -0.2961442768573761, -1.1473042964935303, 0.361016184091568, -1.0812737941741943, -0.14911311864852905, 0.3303377032279968, -0.23760458827018738, 0.20282043516635895, 0.9020341038703918, 0.42977458238601685, -0.5424825549125671, -2.3627564907073975, 0.14206638932228088, 1.1684865951538086, -1.441709041595459, 1.1393183469772339, 1.129582166671753, -1.0760271549224854, 0.10401321202516556, 0.4737057685852051, 0.31131529808044434, -1.6912717819213867, -0.9365305304527283, 0.8149537444114685, 0.10648147761821747, 2.197478771209717, -0.21806444227695465, 1.8290880918502808, 0.2915835380554199, 0.15859010815620422, 0.13074463605880737, 0.7917320728302002, -0.6133689284324646, 1.4122775793075562, -0.3865146040916443, 0.08173440396785736, 0.8353163003921509, -0.7432992458343506, -0.49590346217155457, -1.9245514869689941, -0.5386040210723877, 0.640088677406311, -1.1897166967391968, -0.6801816821098328, -0.682759702205658, -0.6107207536697388, 1.3941967487335205, -1.0581995248794556, -0.21754398941993713, -0.23472805321216583, 0.9665147662162781, -0.1176762804389, -0.5886661410331726, -0.38016602396965027, -1.4203823804855347, -0.08224344998598099, 0.01775384321808815, -2.162339210510254, -0.6018677353858948, -0.1694754660129547, -0.2693743407726288, -1.8014733791351318, -0.13111580908298492, 0.14849326014518738, 0.5327901244163513, -0.7046614289283752, 1.2830451726913452, -1.8688493967056274, -0.9773620963096619, -1.1766554117202759, -0.5966998338699341, -0.1128389909863472, 0.9763141870498657, 2.0646896362304688, 1.5839654207229614, -0.21285703778266907, -0.19126556813716888, -0.1791400909423828, 3.494701862335205, -0.3586898148059845, 0.3412327468395233, -1.0134443044662476, 0.7900715470314026, 0.4075281620025635, -0.012869847938418388, -0.5363834500312805, 0.0020775641314685345, 0.05584312602877617, 0.30025994777679443, -0.8554159998893738, -0.22526217997074127, 0.30524763464927673, -1.4821985960006714, 0.7022022604942322, 0.33354344964027405, 0.23285233974456787, 0.3512950837612152, 0.6298038959503174, -0.0943637490272522, 0.2680620551109314, 0.6600232720375061, 0.7512249946594238, 1.5744553804397583, 0.5924150347709656, 1.3035211563110352, 0.7714843153953552, -0.25320905447006226, 1.0631742477416992, 0.9363723397254944, -2.7589874267578125, 0.1325131058692932, -0.3073659837245941, -1.0119481086730957, -0.2371721863746643, -0.10094685107469559, -0.6471298336982727, 0.6990516781806946, -0.12034963816404343, -0.35114777088165283, 1.3979945182800293, 0.08028053492307663, 0.6868159770965576, 0.838120698928833, 1.365915298461914, 2.008976459503174, -0.8485790491104126, -1.1316858530044556, -1.1532584428787231, 0.458982914686203, 1.1627966165542603, 0.20979872345924377, -0.27049872279167175, 0.4011249542236328, 0.5772587656974792, 0.6801179647445679, -0.1841021180152893, -2.287933349609375, 0.5046616196632385, 1.1200473308563232, -1.0566412210464478, 0.10044203698635101, -0.4209834635257721, 0.07761876285076141, -0.5739850997924805, 0.23413041234016418, 0.14823298156261444, -0.6887956857681274, 0.911197304725647, -0.12172883003950119, -0.6124699115753174, 0.5001530647277832, -0.3254022002220154, 0.2480195015668869, -1.9443740844726562, -0.8886588215827942, -0.5073016285896301, 0.3538273274898529, -0.3279692530632019, 0.5623468160629272, -0.05618782714009285, 0.3468021750450134, 0.16606442630290985, -1.1363569498062134, -0.4965352416038513, 0.7827483415603638, -1.3426601886749268, 0.3382534086704254, -0.035706255584955215, -0.22919175028800964, -1.1986215114593506, -0.04885440319776535, 2.16760516166687, -0.5008763670921326, -0.763813316822052, 0.05168019235134125, -0.8475221991539001, -0.5784148573875427, -1.406697392463684, -0.7904714345932007, -0.269647479057312, 0.9995243549346924, -0.9012638330459595, 1.2129732370376587, -0.9723520278930664, -1.6053134202957153, 1.128251552581787, 1.6296218633651733, -1.7735100984573364, -0.3630264401435852, 1.1796425580978394, 0.03253912925720215, -0.4666907489299774, -0.22152818739414215, 0.3161085546016693, -0.1868882179260254, -0.32863694429397583, 1.1478476524353027, -2.2503247261047363, -0.3316546678543091, 1.3921531438827515, -1.7805216312408447, -0.0796990916132927, 0.8502423763275146, -0.49186113476753235, -0.5954825282096863, -1.864427924156189, -0.7960565686225891, 1.034205675125122, -0.7697776556015015, 1.3299535512924194, 2.0615994930267334, -1.4184869527816772, -0.9586214423179626, -0.05143820866942406, 0.07396513968706131, -1.152070164680481, -1.335341215133667, 1.0291342735290527, 0.09193406254053116, -0.6081909537315369, 0.7622411847114563, -1.1080273389816284, -1.1456029415130615, -0.6893045902252197, -0.12779508531093597, 0.4203703701496124, -0.62006014585495, -1.947118878364563, 0.19984786212444305, 0.44045135378837585, -0.6924605369567871, 1.3448718786239624, 0.9915850162506104, 0.8924368023872375, 0.9386165738105774, 0.7980819940567017, -0.5673298239707947, -0.7079452276229858, 1.6366064548492432, -0.2582317292690277, 0.8598782420158386, 0.3969593644142151, -1.5564008951187134, 1.9245531558990479, 1.3384807109832764, 1.3760567903518677, 1.2843565940856934, 0.6074532270431519, 0.6188889145851135, 0.8992303609848022, 0.3408801853656769, 1.2425904273986816, 0.46688804030418396, 0.0692414939403534, 0.19307385385036469, 1.2138749361038208, -0.6212903261184692, -2.1431891918182373, -0.5564477443695068, -0.22953161597251892, 1.122380256652832, -2.41001296043396, -2.216171979904175, -0.37401676177978516, 0.08047391474246979, -0.4186544120311737, 0.13898780941963196, -2.123701572418213, 1.0506770610809326, -0.04083947092294693, -1.2759519815444946, 0.20467939972877502, 0.004087598994374275, -1.7184807062149048, -1.0094059705734253, -1.3684937953948975, -0.7539282441139221, 1.1090235710144043, -0.6480981707572937, -2.0740232467651367, 1.5821384191513062, -0.1324341893196106, -1.3958523273468018, -2.39109206199646, -0.2552816569805145, 0.27865976095199585, 0.33497458696365356, 0.9085506796836853, 1.0924166440963745, 0.8216851949691772, 2.321864128112793, 0.4770936369895935, 1.008480191230774, -0.6692754030227661, 0.2342781275510788, 1.1301578283309937, 1.2226725816726685, -0.6441232562065125, -0.4283793866634369, -0.8225716352462769, -1.1326237916946411, 0.30533507466316223, 1.4247004985809326, 0.4056319296360016, 0.614337682723999, 0.1511981040239334, 0.29128485918045044, 0.44140559434890747, -0.7406724691390991, 0.1892540156841278, 0.00845316145569086, 1.7784366607666016, -1.2099274396896362, -0.3372579514980316, -0.5278496146202087, 1.631017804145813, 1.2625876665115356, -0.5936276316642761, 1.2465919256210327, 0.5163654685020447, 0.2916260063648224, 0.340748131275177, 1.0173107385635376, 0.6809464693069458, -1.6777797937393188, -0.21386007964611053, 0.23262599110603333, 0.3511931300163269, 1.5018938779830933, 0.5340144038200378, -0.5400632619857788, -0.5913429856300354, -1.5504043102264404, 1.3017058372497559, -1.4988373517990112, -0.27325862646102905, 1.3643417358398438, 1.5384728908538818, -0.6822748780250549, -3.246670722961426, 0.35154053568840027, 0.3714168965816498, 0.4151698350906372, 1.8078985214233398, 0.021732380613684654, 2.2854089736938477, 0.5266827344894409, -0.13316747546195984, -1.9513550996780396, 0.1482016146183014, -0.11027523875236511, -0.7827085852622986, -0.315091609954834, -1.6170580387115479, -0.361640065908432, -1.1046550273895264, -0.7824491262435913, -0.22912487387657166, -0.26560741662979126, -0.5664563179016113, -2.1063973903656006, 0.5871868133544922, 0.6922845244407654, 1.0123497247695923, 0.578301727771759, 0.27296900749206543, 1.4977874755859375, -1.0695548057556152, 0.33542490005493164, -0.8614839315414429, 0.48652327060699463, 2.2188937664031982, 0.3962443470954895, 0.49087589979171753, -1.018123984336853, 0.8495671153068542, -1.8463926315307617, 0.5545784831047058, 0.7420065402984619, -1.0297338962554932, 0.3169555962085724, -0.06289388984441757, -0.2357024848461151, 0.6459209322929382, -0.685635507106781, 1.6309115886688232, 0.4606229364871979, -1.1737644672393799, 0.9413540959358215, 0.664289116859436, 1.2891559600830078, 0.935549795627594, 0.8047831058502197, 0.26418808102607727, 1.2730028629302979, -0.6294464468955994, -0.5153511762619019, -1.429948091506958, 1.3459097146987915, 0.3049781620502472, 1.0985686779022217, -0.5747493505477905, -0.12674784660339355, 0.5839552879333496, -0.06623458862304688, 0.8411797285079956, -0.47673940658569336, 0.9135984182357788, -0.16964010894298553, 1.7240004539489746, -1.8889362812042236, 0.2431490272283554, 0.8629394769668579, 1.9574038982391357, -0.7643904685974121, 1.06450355052948, -1.8061137199401855, 0.5222665071487427, -0.06720083951950073, 0.07057512551546097, 0.8429073095321655, -1.4900320768356323, -0.5875689387321472, 0.2775304317474365, -0.47461026906967163, -2.2827200889587402, 0.8503130078315735, -0.6922556161880493, -0.9323021173477173, 1.87456214427948, 1.622624158859253, 0.35320502519607544, 1.6111432313919067, 2.049548625946045, 0.6880301237106323, 0.02661999687552452, -0.19022706151008606, 0.3825364410877228, 0.38210317492485046, 0.17675548791885376, -1.251827597618103, 1.379834532737732, 0.5793641805648804, -0.1093192994594574, 0.5776060819625854, 0.4323987364768982, 0.5151385068893433, -0.9278174638748169, -0.33150702714920044, 0.6645061373710632, -0.8486015796661377, 0.46784737706184387, -2.416358232498169, 1.3141231536865234, -0.29145559668540955, 0.6763173937797546, 1.2801461219787598, -0.9257147908210754, 2.380058526992798, -0.15873757004737854, 0.34500014781951904, 0.8387638330459595, -0.41756680607795715, -0.04907980561256409, 0.36324143409729004, 1.1598870754241943, 0.9734074473381042, -1.064487338066101, 0.41178590059280396, 0.5600025057792664, -1.2341845035552979, -1.848175048828125, -0.2346751093864441, 2.200634717941284, 1.7394222021102905, 1.2552672624588013, 1.5819671154022217, -1.1904065608978271, 1.5177572965621948, -0.8454104661941528, -0.56842440366745, -1.6335629224777222, -0.12034568190574646, -0.7324094772338867, -0.9826135039329529, -1.4545718431472778, -0.7849869728088379, -1.7898471355438232, -0.6291226148605347, 2.180513620376587, 0.06388979405164719, -0.12118102610111237, 0.34498050808906555, -0.6373867392539978, -0.44575169682502747, -1.5561541318893433, -0.7619199752807617, 0.27302080392837524, 1.3172701597213745, 1.2519170045852661, -0.0738256573677063, 1.872416615486145, -1.2577393054962158, -0.6088021993637085, 1.122997522354126, -0.16346494853496552, -0.12552112340927124, -0.35220709443092346, -1.100905179977417, 0.3702678680419922, 0.45890432596206665, -0.5275217890739441, 0.3358893096446991, -1.6693446636199951, 0.8754801750183105, -0.7340406775474548, -0.2734293043613434, -0.43716996908187866, 1.182273268699646, 0.24780134856700897, 0.851852536201477, 0.020652391016483307, -0.8964262008666992, 0.788856029510498, 0.41781264543533325, 0.048523541539907455, 1.2841869592666626, -1.4398330450057983, -2.2337708473205566, -0.5591973066329956, 0.8000314831733704, 0.3905944526195526, -0.4409615695476532, -0.6578602194786072, -2.323608636856079, -0.19061937928199768, 1.1132358312606812, 0.2748473882675171, 0.8488649725914001, 1.3401418924331665, -1.0021686553955078, -0.8991696834564209, -0.6020470261573792, 1.883457064628601, 2.5710818767547607, -0.7266342639923096, 1.659970998764038, -1.662670612335205, -0.7492448687553406, -2.5699222087860107, -1.0590782165527344, 0.10728918761014938, 0.9397172927856445, -0.12039212137460709, -1.1872968673706055, 0.52166348695755, -1.2259489297866821, 0.3978191912174225, 0.5119510889053345, 1.2078620195388794, -0.2193944901227951, 2.062432050704956, -0.0772385448217392, 0.06219063699245453, 0.8769524693489075, -0.7315060496330261, -0.7987232804298401, -0.22448848187923431, -0.601442813873291, 0.25259149074554443, 0.016695009544491768, -1.9732273817062378, -1.229230284690857, -1.0055806636810303, -0.27292144298553467, 1.253867506980896, -0.7353315949440002, 0.5371828675270081, -1.0437685251235962, 0.7257242202758789, -0.39233317971229553, 0.6304872632026672, -0.8409907221794128, -0.10656248033046722, 1.4996647834777832, 0.5925540328025818, 0.4998648464679718, 1.017852544784546, -0.8100489377975464, -0.2664589583873749, 0.0014228629879653454, 0.1519787311553955, -0.8368906378746033, -1.9233813285827637, -1.0957387685775757, -0.5402980446815491, 1.2446390390396118, -0.08170977234840393, -0.29443979263305664, -0.02385622449219227, 1.3289207220077515, 0.8912685513496399, 0.762030303478241, -1.1396889686584473, -0.7859759330749512, -0.5672420263290405, -0.18442173302173615, 0.08138429373502731, 0.20741093158721924, 0.11722534894943237, -0.7761402726173401, 0.25830498337745667, 1.1177141666412354, 0.2703891694545746, -0.2211378663778305, 1.9827969074249268, 1.0349009037017822, 0.23839719593524933, 1.4326794147491455, 0.07149185985326767, 0.21815642714500427, -1.3549872636795044, -0.6968842148780823, 0.034862443804740906, 1.3120051622390747, -0.2008153200149536, -0.4570564031600952, -0.07457981258630753, 1.5439276695251465, 1.6827863454818726, 2.75797963142395, -1.719886302947998, -0.5328118801116943, -0.19066691398620605, -0.424099326133728, -0.5035483241081238, 1.1053054332733154, 0.9083077907562256, 0.11486906558275223, -0.9007701277732849, 0.7469751834869385, 0.009851481765508652, -0.2630327343940735, 1.7201989889144897, -0.5821860432624817, 0.5732866525650024, -0.7216301560401917, -0.2758476138114929, 0.6345359683036804, 1.7237095832824707, -0.8478277921676636, 0.5203571915626526, 1.7317429780960083, -1.4736465215682983, 0.584621787071228, 0.9960817098617554, -0.7713260650634766, 0.4722941815853119, -0.2765083312988281, 0.24301306903362274, 0.6221781373023987, 0.21003714203834534, 0.5289352536201477, 0.1690351665019989, -1.0352928638458252, -0.8714094758033752, -0.1426985263824463, 0.6627539992332458, 1.285727620124817, -1.0340397357940674, -0.13554580509662628, -0.3688679337501526, -0.17373602092266083, -2.2693729400634766, 0.6760988235473633, 2.0126256942749023, 0.632439374923706, 0.6557163000106812, -0.4415833353996277, -2.571118116378784, -0.20646986365318298, -1.5540797710418701, 0.6517331600189209, -0.996593177318573, 0.06831712275743484, 0.7264114022254944, -0.07093662768602371, -2.063608407974243, -0.3071208596229553, -1.8647903203964233, -0.491031289100647, -0.7827649116516113, 1.6282243728637695, -0.29785430431365967, 1.1806102991104126, 0.8428804278373718, -0.04343492165207863, 1.4253523349761963, 1.1318403482437134, 1.6324036121368408, 0.33278921246528625, 1.9432436227798462, 0.5644756555557251, -0.7010418772697449, -0.1492001712322235, 1.145046353340149, 1.082619071006775, 0.7172906994819641, -0.8984360694885254, 2.0014069080352783, 1.6958088874816895, -1.4752709865570068, 0.7443375587463379, -0.12475328892469406, 2.4709055423736572, 0.2667678892612457, 0.7575609087944031, -1.4062106609344482, -0.9703329205513, 0.11824420839548111, -0.14716313779354095, -1.2388114929199219, -0.6467603445053101, -1.7381876707077026, -0.8233311772346497, -0.3297367990016937, -1.5028120279312134, -0.8711365461349487, -0.7872718572616577, 0.0017906157299876213, 0.5872617363929749, -1.2257708311080933, -0.6863577961921692, -0.13331694900989532, -1.5041590929031372, -1.0868853330612183, 0.7796744704246521, 0.6132504343986511, 0.08485468477010727, -1.1934829950332642, 0.010882111266255379, -1.1703600883483887, -1.0822834968566895, 0.6999395489692688, 0.5841392874717712, -0.972877025604248, 0.826359212398529, 1.3788855075836182, -1.8500138521194458, -0.23997896909713745, -0.012700974009931087, 0.318308025598526, -1.2479734420776367, -0.7103648781776428, -0.08365297317504883, 1.0405209064483643, 0.13902729749679565, -0.39561983942985535, 0.6200575232505798, -0.43449798226356506, 1.7107592821121216, -0.29947441816329956, -0.2673380970954895, 0.7524480223655701, 1.466487169265747, -0.19584405422210693, 1.1677343845367432, 0.7902804613113403, -0.5546700358390808, 0.11494018137454987, -1.6363086700439453, -2.0383546352386475, 0.1558125913143158, -1.6744599342346191, 0.9181849956512451, -0.021821005269885063, -1.1830384731292725, -0.3576178252696991, -1.4049814939498901, -0.6761216521263123, 0.25644567608833313, -1.1833430528640747, 0.2563978433609009, 1.3353192806243896, 1.877220869064331, -0.517366349697113, 0.374220609664917, 0.30443838238716125, -0.595619797706604, 0.4105837941169739, -0.4039427638053894, 0.5994197726249695, -0.424332857131958, 0.8061175346374512, -0.07417187839746475, -0.17509150505065918, 0.9127006530761719, -2.0662002563476562, 0.5678129196166992, -1.0166900157928467, -1.3075891733169556, -0.4103321135044098, 1.1786280870437622, -0.5593303442001343, 0.9168287515640259, -0.8073418736457825, -0.5314292311668396, 0.05453504994511604, 1.2581923007965088, -1.6443943977355957, 0.5343099236488342, 0.04052335396409035, 0.4172675609588623, -1.6046030521392822, -0.3217630982398987, 0.18542420864105225, 0.6330537796020508, 0.5332760810852051, -1.4090769290924072, 0.3839946687221527, -1.2283239364624023, 0.2921757102012634, 1.22951340675354, 0.7698829770088196, -0.16815809905529022, -0.6446889638900757, 0.9589918255805969, -0.28309768438339233, 1.5225859880447388, -1.6008213758468628, -0.1562822312116623, 0.7133026719093323, -1.5741840600967407, -0.05677775293588638, -0.36760029196739197, -1.202215552330017, -0.07954120635986328, 0.11806470155715942, 1.1198517084121704, 1.396266222000122, 1.233464002609253, -0.6395092606544495]], "dtype": "float32"}, {"__nd__": [4097, 4096, 2049, 0], "dtype": "int64"}, 0.0883883], "e": {"__nd__": [[0.00025778290000744164, 0.00022811951930634677, 0.0002658829907886684, 0.00022633987828157842, 0.00023067332222126424, 0.00024219487386289984, 0.0002478368696756661, 0.00024205939553212374, 0.00021925596229266375, 0.00023960971157066524, 0.0002124009479302913, 0.0002793929888866842, 0.00021308813302312046, 0.00023311635595746338, 0.0002873718331102282, 0.00024981985916383564, 0.00024611823027953506, 0.00026143138529732823, 0.00024413815117441118, 0.0002569684584159404, 0.00023076626530382782, 0.00022897162125445902, 0.0002515288651920855, 0.00026479133521206677, 0.00024420866975560784, 0.00026363544748164713, 0.00021955129341222346, 0.0002656239958014339, 0.0002267870440846309, 0.00022388514480553567, 0.00023136184609029442, 0.0002433036861475557, 0.0002695351140573621, 0.00025518363690935075, 0.0002455642679706216, 0.00022014851856511086, 0.00023053414770402014, 0.0002942953142337501, 0.0002487837627995759, 0.0002576243714429438, 0.00025063895736820996, 0.0002280543849337846, 0.000209844161872752, 0.0002335289609618485, 0.00023737800074741244, 0.00022755136888008565, 0.000195251326658763, 0.00020307747763581574, 0.0002711610868573189, 0.00021425580780487508, 0.00022727886971551925, 0.0002510740887373686, 0.00022776154219172895, 0.00023670871451031417, 0.000253404927207157, 0.00021516370179597288, 0.00023716253053862602, 0.00030353214242495596, 0.00027879784465767443, 0.00022307902690954506, 0.0002221051399828866, 0.00024924793979153037, 0.00021974195260554552, 0.00022848737717140466, 0.0001935158798005432, 0.00024844054132699966, 0.00021529779769480228, 0.0002089826884912327, 0.0002263618225697428, 0.0002595815167296678, 0.00023178666015155613, 0.00023915068595670164, 0.00027918530395254493, 0.0002524521842133254, 0.0002820025256369263, 0.00021341131650842726, 0.0002414080809103325, 0.00023375393357127905, 0.0002731088316068053, 0.0002181382878916338, 0.000252108700806275, 0.0002518538967706263, 0.00024086808844003826, 0.00024775363272055984, 0.000270389806246385, 0.00021704249957110733, 0.0001875409361673519, 0.00027141228201799095, 0.0002103231818182394, 0.00027064309688284993, 0.00026099177193827927, 0.00023272099497262388, 0.00023162146680988371, 0.0002529538469389081, 0.00023189756029751152, 0.00021132126857992262, 0.0002495236985851079, 0.000255097373155877, 0.0002486537559889257, 0.0002502849674783647, 0.00023706286447122693, 0.00021918106358498335, 0.00025177610223181546, 0.0002514236548449844, 0.0002589757787063718, 0.00023423261882271618, 0.0002451679902151227, 0.000233250975725241, 0.0002635975833982229, 0.00023203158343676478, 0.0002479148097336292, 0.00023412166046909988, 0.0002341699437238276, 0.0002470917534083128, 0.0002621868043206632, 0.00021840752742718905, 0.000227490600082092, 0.0002572662488091737, 0.00023222238814923912, 0.00021857331739738584, 0.00022919432376511395, 0.00024045722966548055, 0.00022420015011448413, 0.0002536722458899021, 0.00024038895207922906, 0.0002473563945386559, 0.0002443658304400742, 0.00021460709103848785, 0.00027327839052304626, 0.0002587438211776316, 0.00020500084792729467, 0.0002930895716417581, 0.0002781979274004698, 0.0002577771374490112, 0.000279838073765859, 0.00028168578865006566, 0.00024176304577849805, 0.00023905973648652434, 0.00022406410425901413, 0.00024900794960558414, 0.0002537582768127322, 0.0002432482287986204, 0.00025566580006852746, 0.0002382183010922745, 0.0002614252152852714, 0.00027394702192395926, 0.0002476927766110748, 0.0002723648212850094, 0.0002413293841527775, 0.00022963852097745985, 0.0002439525560475886, 0.00025191644090227783, 0.00026915609487332404, 0.00020699923334177583, 0.000262248155195266, 0.00024792118347249925, 0.00022279948461800814, 0.00022600410738959908, 0.00021946485503576696, 0.0002534251834731549, 0.00028661146643571556, 0.00023598993720952421, 0.00023036912898533046, 0.00026339225587435067, 0.000284251436823979, 0.0002545244060456753, 0.00025971446302719414, 0.00024593283887952566, 0.00025846395874395967, 0.00021193754218984395, 0.00022217708465177566, 0.00023493169283028692, 0.00026734083076007664, 0.0002521629212424159, 0.0001993277110159397, 0.00022251781774684787, 0.00023259336012415588, 0.00022301966964732856, 0.0002281653432874009, 0.00025031628319993615, 0.00023305982176680118, 0.0002795516629703343, 0.00021379714598879218, 0.00021428693435154855, 0.00020998145919293165, 0.00025146163534373045, 0.0002420519303996116, 0.00026348960818722844, 0.00026185569004155695, 0.00025972939329221845, 0.00022182462271302938, 0.00024059218412730843, 0.00019834213890135288, 0.00025257508968934417, 0.00024851516354829073, 0.00024274236056953669, 0.00023656224948354065, 0.00025040641776286066, 0.00024048758496064693, 0.0002480378607288003, 0.00023490677995141596, 0.00022173150500748307, 0.0002665346837602556, 0.0002332920703338459, 0.00023477159265894443, 0.00022763985907658935, 0.00023528399469796568, 0.00021632052084896713, 0.0002589290088508278, 0.0002377234777668491, 0.0002932376810349524, 0.00023651366063859314, 0.0001893115259008482, 0.0002787184785120189, 0.00020944076823070645, 0.00021079304860904813, 0.0002730634296312928, 0.00023301708279177547, 0.00023366579262074083, 0.00023457604402210563, 0.00023635663092136383, 0.0002462825505062938, 0.00024911100626923144, 0.00025734066730365157, 0.0002649743983056396, 0.0002506437886040658, 0.00025095429737120867, 0.00021933684183750302, 0.00023292278638109565, 0.00023145414888858795, 0.0002710479311645031, 0.00022594313486479223, 0.00021199417824391276, 0.00024099160509649664, 0.0002447006991133094, 0.0002869832969736308, 0.0002494262589607388, 0.00028942953213118017, 0.00026306757354177535, 0.0002662131446413696, 0.00022586682462133467, 0.00024196409503929317, 0.0002509636979084462, 0.00028100356576032937, 0.00024404030409641564, 0.0002707826788537204, 0.0002448044833727181, 0.00024018010299187154, 0.00021439112606458366, 0.00021517759887501597, 0.00020312253036536276, 0.0003358708636369556, 0.00028432003455236554, 0.00023158032854553312, 0.0002312794531462714, 0.0002547356125432998, 0.00022407389769796282, 0.00026546698063611984, 0.0002379103534622118, 0.0002954648225568235, 0.000233600425417535, 0.00022622296819463372, 0.0002706422528717667, 0.00024673703592270613, 0.00022583255486097187, 0.00022489088587462902, 0.0002325356617802754, 0.00024262025544885546, 0.00025277899112552404, 0.00022807196364738047, 0.00023850354773458093, 0.0002682423801161349, 0.00027934537502005696, 0.0002488325408194214, 0.00023634139506611973, 0.00022834197443444282, 0.0002554601524025202, 0.00023594516096636653, 0.0002823467075359076, 0.00023025869450066239, 0.0002737308677751571, 0.00028852507239207625, 0.00026437576161697507, 0.00024890433996915817, 0.00022696531959809363, 0.00027756483177654445, 0.0002366553817410022, 0.00023866796982474625, 0.0002592035161796957, 0.00023802326177246869, 0.0002657757140696049, 0.00023849713033996522, 0.0002333649608772248, 0.0002597446436993778, 0.00023714911367278546, 0.00023643021995667368, 0.00024109883815981448, 0.0002879418316297233, 0.00023108556342776865, 0.00026461377274245024, 0.00024490049690939486, 0.00024388519523199648, 0.00022564241953659803, 0.00025437562726438046, 0.00023119074467103928, 0.00020926397701259702, 0.00023748639796394855, 0.00023405758838634938, 0.0002771807194221765, 0.00022390036610886455, 0.00025365297915413976, 0.00023693061666563153, 0.00023721822071820498, 0.00024362812109757215, 0.0002770895080175251, 0.00023281443282030523, 0.00023861246882006526, 0.00023201582371257246, 0.00020585081074386835, 0.00021810701582580805, 0.00025069425464607775, 0.0002874157507903874, 0.00024020634009502828, 0.0002559974091127515, 0.00025506209931336343, 0.00023062998661771417, 0.00020017278438899666, 0.0002230905811302364, 0.00023168600455392152, 0.00022031411936040968, 0.0002811267040669918, 0.00021383794955909252, 0.0002735652669798583, 0.00024082211893983185, 0.00023902225075289607, 0.00022455603175330907, 0.00028221384854987264, 0.0002556929539423436, 0.0002608834474813193, 0.00024933970416896045, 0.0002754479937721044, 0.0002475149231031537, 0.00022257753880694509, 0.00022327966871671379, 0.0002531058853492141, 0.0002869146119337529, 0.0002468312450218946, 0.0002157935086870566, 0.00024952267995104194, 0.0002341074141440913, 0.00021679072233382612, 0.00026743774651549757, 0.00026715861167758703, 0.00021283590467646718, 0.0002835011691786349, 0.00025572284357622266, 0.00021297387138474733, 0.0002261129266116768, 0.0002824412367772311, 0.000266648072283715, 0.00024091431987471879, 0.0002256977022625506, 0.0001922071969602257, 0.00026261541643179953, 0.00027731055160984397, 0.00024187512462958694, 0.00024655810557305813, 0.0002504416916053742, 0.00023400664213113487, 0.0002207353973062709, 0.0002306833048351109, 0.000209168138098903, 0.00021976417338009924, 0.0002699647448025644, 0.00024981412570923567, 0.00029628814081661403, 0.00021612907585222274, 0.00021765277779195458, 0.00026194521342404187, 0.0001936120679602027, 0.00023275484272744507, 0.00026983494171872735, 0.00023563583090435714, 0.0002698710886761546, 0.0002620656741783023, 0.00026029726723209023, 0.00023644296743441373, 0.00024273473536595702, 0.00019402126781642437, 0.00023648001661058515, 0.0002732064458541572, 0.0002593392273411155, 0.0002623687614686787, 0.000226395161007531, 0.00028205159469507635, 0.00025494684814475477, 0.00022067787358537316, 0.00025134169845841825, 0.0002679914759937674, 0.00021492125233635306, 0.00024021287390496582, 0.00022972864098846912, 0.0002348117996007204, 0.00023387058172374964, 0.00024177311570383608, 0.00023136945674195886, 0.0002808930876199156, 0.00027045293245464563, 0.0001970338198589161, 0.00023423995298799127, 0.00022640044335275888, 0.0002952938375528902, 0.000263448862824589, 0.0002550673671066761, 0.00026571404305286705, 0.00024263441446237266, 0.0002186644560424611, 0.00024797339574433863, 0.0002594955440144986, 0.000259433378232643, 0.0002566118782851845, 0.00023168610641732812, 0.0002220652240794152, 0.0002209518861491233, 0.0002745547972153872, 0.0002779325295705348, 0.0002519262779969722, 0.00024280414800159633, 0.0002035800280282274, 0.00022438846644945443, 0.00028508028481155634, 0.0002355620381422341, 0.00024669014965184033, 0.00022330458159558475, 0.0002672391419764608, 0.00030619994504377246, 0.00024926866171881557, 0.00022232555784285069, 0.00025799969444051385, 0.00020956533262506127, 0.0002746134123299271, 0.00024818142992444336, 0.0002144063328159973, 0.00022378555149771273, 0.00024238058540504426, 0.00022850280220154673, 0.0002538222470320761, 0.00022177361825015396, 0.00021414709044620395, 0.00022680367692373693, 0.0002376733027631417, 0.00023992460046429187, 0.00026066042482852936, 0.00028786962502636015, 0.00024374683562200516, 0.00021441631542984396, 0.0002577527775429189, 0.0002668265369720757, 0.00019681427511386573, 0.00022918899776414037, 0.000227876371354796, 0.0002532606013119221, 0.00022552172595169395, 0.00019152385357301682, 0.00025264217401854694, 0.00023894730838946998, 0.00023874727776274085, 0.00023947814770508558, 0.00025020266184583306, 0.0002888454182539135, 0.00024902549921534956, 0.00021596750593744218, 0.00023332650016527623, 0.00024590789689682424, 0.00025822402676567435, 0.00024819871759973466, 0.00025505086523480713, 0.0002562823938205838, 0.00023986509768292308, 0.00025596815976314247, 0.0002482867566868663, 0.0002657976874615997, 0.00023628976487088948, 0.00026102227275259793, 0.00025669470778666437, 0.00023180989956017584, 0.00021314749028533697, 0.00024141353787854314, 0.00024698994820937514, 0.0002481563133187592, 0.0002536933170631528, 0.00024518510326743126, 0.0002578604326117784, 0.00024186204245779663, 0.0002228529192507267, 0.00020717094594147056, 0.00027572299586609006, 0.00024677548208273947, 0.00023229797079693526, 0.00023997187963686883, 0.00024256001051981002, 0.0002561556175351143, 0.00028771988581866026, 0.0002515203377697617, 0.000233963510254398, 0.00022281920246314257, 0.0002823406539391726, 0.00023297958250623196, 0.0002542055444791913, 0.00024258332268800586, 0.00023002500529401004, 0.00024372909683734179, 0.0002348064153920859, 0.00023054427583701909, 0.0002208629302913323, 0.0002440855751046911, 0.000282062916085124, 0.0002524577721487731, 0.00028615674818865955, 0.00020606628095265478, 0.00017624723841436207, 0.000262838089838624, 0.0002227899240097031, 0.00024787368602119386, 0.00026610345230437815, 0.00023348066315520555, 0.00025852007092908025, 0.00027121175662614405, 0.00022435245045926422, 0.00022169764270074666, 0.0002616554847918451, 0.000240460634813644, 0.0002470870967954397, 0.00024062127340584993, 0.0002341593208257109, 0.0002259751781821251, 0.00023594383674208075, 0.0002464423596393317, 0.00024870745255611837, 0.00026143318973481655, 0.0002188794023822993, 0.0002503436990082264, 0.00028378848219290376, 0.00025146990083158016, 0.0002740253403317183, 0.0002587336639408022, 0.000232008722377941, 0.0002522989234421402, 0.00025340428692288697, 0.000244048802414909, 0.0002766755351331085, 0.00020761236373800784, 0.0002734971640165895, 0.0002237217704532668, 0.00016222904378082603, 0.000222561284317635, 0.0002604832989163697, 0.00018535967683419585, 0.0002423013502266258, 0.00027334876358509064, 0.0002366732805967331, 0.0002307877002749592, 0.00025447102962061763, 0.0002375449548708275, 0.0002530232595745474, 0.00023604670423083007, 0.0002487536403350532, 0.00022563023958355188, 0.00025892743724398315, 0.00022918377362657338, 0.00020425028924364597, 0.0002231532707810402, 0.0002711791603360325, 0.0002472057531122118, 0.0002310112613486126, 0.00021935980475973338, 0.0002455762878526002, 0.00023650949879083782, 0.000201106391614303, 0.00023613071243744344, 0.00024059080169536173, 0.00023892860917840153, 0.00024101381131913513, 0.0002155414695153013, 0.00024682117509655654, 0.00026081729447469115, 0.00023434267495758832, 0.00023559229157399386, 0.0002219225134467706, 0.0002396612981101498, 0.00022215934586711228, 0.0003026406338904053, 0.00022199875093065202, 0.0002563699090387672, 0.00027643769863061607, 0.00026289161178283393, 0.00023079740640241653, 0.00022635344066657126, 0.00021529696823563427, 0.00022907413949724287, 0.00023389095440506935, 0.0002693293208722025, 0.0002532867656555027, 0.00023696446442045271, 0.00023519581009168178, 0.0002444635902065784, 0.0002515672822482884, 0.0002594979596324265, 0.00025009457021951675, 0.0002435528440400958, 0.00026750058168545365, 0.00025159327196888626, 0.00025400263257324696, 0.00025901684421114624, 0.00023744204372633249, 0.0002617956488393247, 0.00020692685211542994, 0.00021268129057716578, 0.00021198198373895139, 0.00029622571310028434, 0.0002592272649053484, 0.00021879328414797783, 0.00023192512162495404, 0.00023702900216449052, 0.00021204738004598767, 0.00024223046784754843, 0.00023696226708125323, 0.00022002894547767937, 0.00020765879889950156, 0.00021880495478399098, 0.00025693274801597, 0.00026476799394004047, 0.0002234512649010867, 0.00021352041221689433, 0.00023783516371622682, 0.00021957843273412436, 0.0002145973703591153, 0.00020529879839159548, 0.0002584596222732216, 0.00021286016271915287, 0.00020857035997323692, 0.0002368603163631633, 0.00022729985357727855, 0.00022606560378335416, 0.00025814701803028584, 0.00022801556042395532, 0.0002368443092564121, 0.00028129288693889976, 0.00026612821966409683, 0.000229127355851233, 0.00023854592291172594, 0.00022116930631455034, 0.00027555233100429177, 0.00024049294006545097, 0.00023349245020654052, 0.00022153621830511838, 0.0002464896533638239, 0.00025329890195280313, 0.00023893953766673803, 0.0002739260671660304, 0.00021500651200767606, 0.00022210787574294955, 0.0002529228513594717, 0.0002676674339454621, 0.0002244145580334589, 0.00024673016741871834, 0.00026509418967179954, 0.00024495774414390326, 0.0002311012358404696, 0.00023885926930233836, 0.00022119906498119235, 0.0002654366544447839, 0.00024541557650081813, 0.00021990202367305756, 0.0002346189576201141, 0.00024996339925564826, 0.00025893692509271204, 0.00024955786648206413, 0.0002483554999344051, 0.0002610395895317197, 0.00025536719476804137, 0.00024936406407505274, 0.00026249547954648733, 0.00024717862834222615, 0.00022669894678983837, 0.00024690080317668617, 0.0002503211435396224, 0.0002225761563749984, 0.0002496666566003114, 0.00021976786956656724, 0.0002551130310166627, 0.00024222627689596266, 0.000254178186878562, 0.00025180098600685596, 0.0002474926004651934, 0.0001948594581335783, 0.0002452176413498819, 0.0002374073665123433, 0.00021811475744470954, 0.0002437811199342832, 0.000227306445594877, 0.00025571734295226634, 0.0002234023268101737, 0.0002399192744633183, 0.00020935646898578852, 0.0002512299979571253, 0.00024019349075388163, 0.00023719147429801524, 0.00022393034305423498, 0.00025080915656872094, 0.0002606139751151204, 0.00024131135432980955, 0.00023536653316114098, 0.00023348795366473496, 0.00025300291599705815, 0.00024290283909067512, 0.0002530889178160578, 0.0002287115785293281, 0.000248264754191041, 0.0002884863642975688, 0.00026228735805489123, 0.0002918724494520575, 0.00024305084662046283, 0.00021980925521347672, 0.0002704584039747715, 0.00022917587193660438, 0.0002701836347114295, 0.00023297582811210304, 0.00024988973746076226, 0.00024928952916525304, 0.00027265463722869754, 0.0002379431971348822, 0.0002307234681211412, 0.0002147995837731287, 0.0002447773877065629, 0.00027410834445618093, 0.00024340538948308676, 0.00021615586592815816, 0.0002446178696118295, 0.00023342127678915858, 0.0002531473874114454, 0.0002347237168578431, 0.00023720262106508017, 0.00019883154891431332, 0.0002633141411934048, 0.00019508838886395097, 0.0002539660199545324, 0.0002741965581662953, 0.0002511215570848435, 0.0002508240577299148, 0.0002294974256074056, 0.00024524301989004016, 0.00025927566457539797, 0.0002329148119315505, 0.00023640092695131898, 0.0002566633920650929, 0.00023188497289083898, 0.00026758271269500256, 0.00022768093913327903, 0.00024392269551753998, 0.00025475554866716266, 0.00023623424931429327, 0.0002375538897467777, 0.00020819934434257448, 0.00024045472673606128, 0.000281438697129488, 0.0002572760568000376, 0.00021309898875188082, 0.0002496321976650506, 0.000218963687075302, 0.00023869614233262837, 0.0002242436312371865, 0.000251962395850569, 0.00026574271032586694, 0.00022952961444389075, 0.0002646658103913069, 0.00025054573779925704, 0.00024230040435213596, 0.00024644340737722814, 0.0002710419939830899, 0.0002871135075110942, 0.0002276802115375176, 0.00029984270804561675, 0.00023389689158648252, 0.00023446579871233553, 0.00026820661150850356, 0.0002754672896116972, 0.00020389779820106924, 0.0003093781997449696, 0.0002565435424912721, 0.00024114926054608077, 0.0002241869951831177, 0.0002445906575303525, 0.0002329978160560131, 0.00022813370742369443, 0.00022706594609189779, 0.00023543002316728234, 0.0002529703197069466, 0.00023620946740265936, 0.0002352707670070231, 0.0002244198985863477, 0.00023627631890121847, 0.00022595847258344293, 0.00024281391233671457, 0.0002671687980182469, 0.000258801766904071, 0.00022954553423915058, 0.0002709130058065057, 0.00023843171948101372, 0.00025084608932957053, 0.000275881466222927, 0.00022701734269503504, 0.0002452486369293183, 0.00023376905301120132, 0.0002416630304651335, 0.00023446815612260252, 0.00022636271023657173, 0.00025508867111057043, 0.00018826617451850325, 0.00024671724531799555, 0.0002860825043171644, 0.0002625907363835722, 0.00023305907961912453, 0.0002495160442776978, 0.0002511900383979082, 0.0002292637072969228, 0.0002569588541518897, 0.00027007731841877103, 0.00022848778462503105, 0.00025815871777012944, 0.00021115891286171973, 0.00023785342636983842, 0.00024345274141523987, 0.00019672139023896307, 0.0002690404071472585, 0.00022740007261745632, 0.0002334165619686246, 0.0002681283513084054, 0.0002317361650057137, 0.00019354262622073293, 0.0002214498381363228, 0.0002526103053241968, 0.00024844551808200777, 0.00022863488993607461, 0.00027545701595954597, 0.00023952573246788234, 0.00024191902775783092, 0.0002731941931415349, 0.0002466808364260942, 0.0002545411407481879, 0.00022218329831957817, 0.00022439991880673915, 0.00022495050507131964, 0.00023680612503085285, 0.000253310747211799, 0.00031058117747306824, 0.0002235700230812654, 0.0002260849141748622, 0.00022895386791788042, 0.00023809527920093387, 0.00024010601919144392, 0.00023579972912557423, 0.0002241486363345757, 0.00024947652127593756, 0.0002349760616198182, 0.00023774664441589266, 0.00020747707458212972, 0.00022894058201927692, 0.000250835029873997, 0.00023177546972874552, 0.0002513049403205514, 0.00024328948347829282, 0.00025630072923377156, 0.0002500908449292183, 0.00021652111900039017, 0.0002466173027642071, 0.0002289218973601237, 0.00027246971148997545, 0.0002480953698977828, 0.00021533087419811636, 0.0002275752049172297, 0.00022976109175942838, 0.00027799129020422697, 0.0002453827182762325, 0.000239038941799663, 0.0002239932946395129, 0.0002485792792867869, 0.00023513771884609014, 0.00027222666540183127, 0.00025594892213121057, 0.0002042517444351688, 0.00023774270084686577, 0.00022973373415879905, 0.0002445893769618124, 0.0002614417753648013, 0.0002364594693062827, 0.000246871670242399, 0.00024628720711916685, 0.00021966743224766105, 0.00026035780319944024, 0.0002640509628690779, 0.00026619245181791484, 0.00022942564100958407, 0.00022221883409656584, 0.00028070129337720573, 0.00024382091942243278, 0.0002340129140065983, 0.00022018446179572493, 0.00023673083342146128, 0.0002669900131877512, 0.00021867126633878797, 0.00022149254800751805, 0.00024980472517199814, 0.00021869594638701528, 0.00024286513507831842, 0.0002449887979310006, 0.00025485260994173586, 0.00021509590442292392, 0.00029819420888088644, 0.0002268226962769404, 0.0002460137475281954, 0.00026961456751450896, 0.00023558319662697613, 0.000228098186198622, 0.0002640275633893907, 0.0002469166065566242, 0.00028892391128465533, 0.00024370259779971093, 0.00025456969160586596, 0.00024570003733970225, 0.00022974291641730815, 0.0002480408875271678, 0.00024675161694176495, 0.0002950984053313732, 0.00022681933478452265, 0.00027497680275700986, 0.0002976366668008268, 0.0002360717044211924, 0.00025460057076998055, 0.00023683560721110553, 0.0002469756582286209, 0.00019430837710388005, 0.00024853262584656477, 0.00026347022503614426, 0.0002153439272660762, 0.00021461572032421827, 0.00026916354545392096, 0.00023121136473491788, 0.00025104210362769663, 0.00020510712056420743, 0.00025221373653039336, 0.0002753145236056298, 0.00025832990650087595, 0.0002565281465649605, 0.00023863615933805704, 0.0002211113169323653, 0.00025970579008571804, 0.00023791253624949604, 0.0002562353038229048, 0.00021047938207630068, 0.0002639465674292296, 0.00021447439212352037, 0.0002440340322209522, 0.00022896588779985905, 0.0002561231958679855, 0.00023790860723238438, 0.0002462828706484288, 0.0002528400218579918, 0.00023817319015506655, 0.00022750798962078989, 0.00023600801068823785, 0.00024313663016073406, 0.00024839278194122016, 0.00021859246771782637, 0.0002116328541887924, 0.000243404894717969, 0.00025184015976265073, 0.0002592223172541708, 0.00024372819461859763, 0.00023014900216367096, 0.00023745406360831112, 0.00022768399503547698, 0.00021947140339761972, 0.00023031843011267483, 0.00024094522814266384, 0.0002484658907633275, 0.0002800634829327464, 0.00022090792481321841, 0.00023379492631647736, 0.00022989498393144459, 0.0002922808635048568, 0.00023312571283895522, 0.0002289065159857273, 0.00022708758478984237, 0.00021575976279564202, 0.0002462487027514726, 0.00020646322809625417, 0.00023809545382391661, 0.00022456022270489484, 0.00023201125441119075, 0.00024828684399835765, 0.00022754399105906487, 0.00026640662690624595, 0.00022243286366574466, 0.00024449004558846354, 0.0002474005159456283, 0.0002496442466508597, 0.000242905443883501, 0.00027139094891026616, 0.00024498684797436, 0.00020587249309755862, 0.0002149672363884747, 0.00023534384672529995, 0.00029825433739461005, 0.00029642711160704494, 0.00021849623590242118, 0.00025028438540175557, 0.00021345799905247986, 0.00022486438683699816, 0.00028488351381383836, 0.0002642884792294353, 0.00023015546321403235, 0.00024366486468352377, 0.00025341464788652956, 0.00024179038882721215, 0.00020726687216665596, 0.00021635973826050758, 0.00025467824889346957, 0.00025697966339066625, 0.0002461991389282048, 0.00022556414478458464, 0.0002792256127577275, 0.0002765907265711576, 0.00022563642414752394, 0.000235759696806781, 0.00024090726219583303, 0.00023213772510644048, 0.00024959660368040204, 0.00023643342137802392, 0.00021779835515189916, 0.00021923406166024506, 0.00023053200857248157, 0.00026749432436190546, 0.00022554189490620047, 0.00023870645964052528, 0.00024721494992263615, 0.0002431395259918645, 0.00022013277339283377, 0.0002486425801180303, 0.0002417053037788719, 0.00027072668308392167, 0.0002685675281099975, 0.00023843436792958528, 0.0002451823966111988, 0.0002374944742769003, 0.00025811159866862, 0.0002469765895511955, 0.0002623758919071406, 0.0002261792542412877, 0.00023371714632958174, 0.00026566218002699316, 0.0002494090877007693, 0.0002748247643467039, 0.00022652935876976699, 0.0002557799743954092, 0.00025919274776242673, 0.00020804646192118526, 0.0002163021854357794, 0.00023238459834828973, 0.00023380763013847172, 0.00021922378800809383, 0.0002399568329565227, 0.00021264978568069637, 0.0002716299204621464, 0.00023261406749952585, 0.0002584103785920888, 0.00026085504214279354, 0.0002472048217896372, 0.0002690230030566454, 0.00024026971368584782, 0.00027685757959261537, 0.00020726944785565138, 0.00030418502865359187, 0.0002725505328271538, 0.00026934739435091615, 0.0002547850599512458, 0.00026214378885924816, 0.00023401103680953383, 0.0002483535499777645, 0.00021624259534291923, 0.0002596247068140656, 0.0002544125309213996, 0.00023009846336208284, 0.00023051879543345422, 0.00023931633040774614, 0.0002410426241112873, 0.0002463076962158084, 0.00022153303143568337, 0.00026187102776020765, 0.00026683841133490205, 0.0002684424980543554, 0.00023389463603962213, 0.0002360287617193535, 0.0002532541111577302, 0.00024721032241359353, 0.00026833664742298424, 0.0002463730052113533, 0.00023673083342146128, 0.00023367002722807229, 0.00024604712962172925, 0.00022266925952862948, 0.00022589891159441322, 0.00022694589279126376, 0.00026278343284502625, 0.00023631940712220967, 0.00022230851755011827, 0.00029932387406006455, 0.00020998370018787682, 0.0002618311846163124, 0.0002672297996468842, 0.00023995338415261358, 0.0002971793001051992, 0.00026389569393359125, 0.00021397702221293002, 0.00023473742476198822, 0.0002415855706203729, 0.00026166002498939633, 0.0002668240340426564, 0.0002759869967121631, 0.0002542664296925068, 0.00021772239415440708, 0.00024210213450714946, 0.00031701367697678506, 0.00020291880355216563, 0.00022188313596416265, 0.00020202840096317232, 0.00024281948572024703, 0.00021386069420259446, 0.0002631327952258289, 0.00024895285605452955, 0.00023896043421700597, 0.00025452455156482756, 0.00024233281146734953, 0.00019108132983092219, 0.00026890350272879004, 0.00025100266793742776, 0.00022802948660682887, 0.00022578171046916395, 0.00027397667872719467, 0.00024170549295376986, 0.0002748662082012743, 0.00026236556004732847, 0.00025117292534559965, 0.00022800327860750258, 0.00024193459830712527, 0.00023523012350779027, 0.00023959134705364704, 0.00022998193162493408, 0.0002660939935594797, 0.0002827899297699332, 0.0002523533476050943, 0.0002787790144793689, 0.0002216193388449028, 0.0002466090954840183, 0.00023037215578369796, 0.00029021286172792315, 0.00024323820252902806, 0.00024118134751915932, 0.00021281727822497487, 0.00024772860342636704, 0.00022166348935570568, 0.0002432919864077121, 0.000248406024184078, 0.00024874889641068876, 0.000213034130865708, 0.00031966017559170723, 0.0002486705197952688, 0.0002394034672761336, 0.0002546014147810638, 0.00024340594245586544, 0.00022619205992668867, 0.00030992235406301916, 0.0002346259861951694, 0.00027204366051591933, 0.00021434808149933815, 0.00023441387747880071, 0.0002464280987624079, 0.00025561696384102106, 0.00026012901798821986, 0.0002551706857047975, 0.0002329210110474378, 0.00021932943491265178, 0.0002505063312128186, 0.0002496768720448017, 0.00023485459678340703, 0.0002358630154049024, 0.00021472445223480463, 0.0002515512169338763, 0.00022793174139223993, 0.0002163889876101166, 0.0002551150682847947, 0.00024505401961505413, 0.00023556397354695946, 0.0002492452913429588, 0.00026521357358433306, 0.00023825351672712713, 0.00023857010819483548, 0.0002556579711381346, 0.00024314725305885077, 0.0002330351126147434, 0.00027450628113001585, 0.0002542453003115952, 0.0002446571015752852, 0.00023846134718041867, 0.00028721903800033033, 0.0002310168492840603, 0.00029603781877085567, 0.00026956890360452235, 0.00026887902640737593, 0.00021841973648406565, 0.0002363640523981303, 0.00021824099530931562, 0.00022448954405263066, 0.00023878489446360618, 0.00024887494510039687, 0.000259214750258252, 0.00022534349409397691, 0.0002160205622203648, 0.00027650545234791934, 0.0002427126164548099, 0.00024899819982238114, 0.00024052710796240717, 0.00024321791715919971, 0.00023702852195128798, 0.0002705097140278667, 0.00023973931092768908, 0.0002272144192829728, 0.00023137146490626037, 0.00026734176208265126, 0.00020644751202780753, 0.00027776978095062077, 0.00022072337742429227, 0.00027554656844586134, 0.0002765747776720673, 0.00022666809672955424, 0.00029761475161649287, 0.0002374018367845565, 0.00024101720191538334, 0.00025121396174654365, 0.0002530852216295898, 0.00023494703054893762, 0.00026837486075237393, 0.00023818299814593047, 0.00027034751838073134, 0.00032974465284496546, 0.00025797137641347945, 0.0002395230403635651, 0.00027314008912071586, 0.00020405600662343204, 0.00023281217727344483, 0.00024507121997885406, 0.0002203891781391576, 0.0002838340005837381, 0.00020787421090062708, 0.0002133740927092731, 0.00020717612642329186, 0.00023982004495337605, 0.00024711625883355737, 0.00024802901316434145, 0.00023580290144309402, 0.00022655610518995672, 0.00025665038265287876, 0.0002691680274438113, 0.0002336408942937851, 0.00026312482077628374, 0.00026709274970926344, 0.0002594184479676187, 0.0002482056152075529, 0.0002387450949754566, 0.0002547173062339425, 0.00026665092445909977, 0.00022544284001924098, 0.00022608395374845713, 0.0002489362086635083, 0.000228197401156649, 0.00024297807249240577, 0.0002880424144677818, 0.00024657489848323166, 0.00023213673557620496, 0.00023734208662062883, 0.0002472632040735334, 0.00024334329646080732, 0.00025060854386538267, 0.0002730690175667405, 0.00023368810070678592, 0.00022261339472606778, 0.00024924660101532936, 0.00027938830316998065, 0.00019901280757039785, 0.00025708304019644856, 0.00023469897860195488, 0.00020606823090929538, 0.00024368525191675872, 0.0002561270084697753, 0.00022494762379210442, 0.00022469557006843388, 0.00024791565374471247, 0.00021872809156775475, 0.00023089196474757046, 0.00025924877263605595, 0.00023439251526724547, 0.00023654208052903414, 0.00024657274479977787, 0.00020046001009177417, 0.00022380232985597104, 0.00024385978758800775, 0.000228502947720699, 0.00023071393661666662, 0.00023430098372045904, 0.0002317421167390421, 0.0003136192972306162, 0.00021387408196460456, 0.0002065214212052524, 0.00019415662973187864, 0.00023771185078658164, 0.00025733147049322724, 0.0002255627914564684, 0.0002676017757039517, 0.00023347226670011878, 0.0002310752752237022, 0.00022537691984325647, 0.00024329200095962733, 0.00024201649648603052, 0.00026007762062363327, 0.00023329236137215048, 0.0002729050465859473, 0.00025775222457014024, 0.0002466763253323734, 0.00024202179338317364, 0.00024118252622429281, 0.00030480834539048374, 0.0002713814319577068, 0.00023377490288112313, 0.0002118374832207337, 0.00022935915330890566, 0.00023207759659271687, 0.00022316593094728887, 0.00023486881400458515, 0.0002510731283109635, 0.0002629286318551749, 0.0002552220830693841, 0.0002524432202335447, 0.0002510258636903018, 0.00023076427169144154, 0.00024987958022393286, 0.0002464834542479366, 0.00030019599944353104, 0.00021460856078192592, 0.00022867326333653182, 0.00024254509480670094, 0.00024718433269299567, 0.00024224047956522554, 0.00021620468760374933, 0.00024610996479168534, 0.0002532422367949039, 0.00029015276231803, 0.0002186883066315204, 0.00025673050549812615, 0.0002092035283567384, 0.0002550699864514172, 0.00025455208378843963, 0.00022116675972938538, 0.00020977550593670458, 0.0002646412467584014, 0.00020584174490068108, 0.00023933615011628717, 0.00022226643341127783, 0.000268275587586686, 0.00026679428992792964, 0.0002382145175943151, 0.00023810850689187646, 0.00026179186534136534, 0.0002772608713712543, 0.00026049945154227316, 0.00024914994719438255, 0.00020462482643779367, 0.00025603504036553204, 0.00022092726430855691, 0.00026742214686237276, 0.00024334786576218903, 0.00022487800742965192, 0.00023312499979510903, 0.0002791629813145846, 0.00023262880858965218, 0.0002675440046004951, 0.0002541103749535978, 0.00020588729239534587, 0.00024902037694118917, 0.0002277328894706443, 0.0002505487354937941, 0.00026385256205685437, 0.00023292744299396873, 0.0002588504576124251, 0.0002435552451061085, 0.00022475625155493617, 0.0002486805897206068, 0.0002015206846408546, 0.00024106301134452224, 0.0002042916021309793, 0.0002380038786213845, 0.0002687563537620008, 0.0002396140480414033, 0.0002224337076768279, 0.00023307037190534174, 0.00027611482073552907, 0.00024530323571525514, 0.0002557024417910725, 0.0002528845507185906, 0.000248123164055869, 0.00028787952032871544, 0.0002677642332855612, 0.00026022110250778496, 0.0002605291665531695, 0.00022272761270869523, 0.00027158085140399635, 0.00028452897095121443, 0.00024758046492934227, 0.0002652042603585869, 0.0002693468704819679, 0.00022480871120933443, 0.0002461973635945469, 0.00023166697064880282, 0.00028554891468957067, 0.00021648207621183246, 0.000251491874223575, 0.0002482271520420909, 0.00021669342822860926, 0.00020770161063410342, 0.00024634276633150876, 0.0002308476105099544, 0.00026295240968465805, 0.00025264560827054083, 0.00024041901633609086, 0.00021375407231971622, 0.000275642320048064, 0.00025195805937983096, 0.0002468949242029339, 0.0002626632631290704, 0.0002262185444124043, 0.00021475247922353446, 0.0002492382191121578, 0.00022364778851624578, 0.0002562771551311016, 0.0002324613305972889, 0.0002594723191577941, 0.00021140607714187354, 0.0002202974137617275, 0.0002286360104335472, 0.00022652166080661118, 0.00025700998958200216, 0.00027306104311719537, 0.00025423578335903585, 0.00021614303113892674, 0.0002458840317558497, 0.00021972149261273444, 0.0002233232225989923, 0.00023306006914936006, 0.00025034535792656243, 0.0002490007900632918, 0.00028450210811570287, 0.00025777510018087924, 0.00023386433895211667, 0.0002684525097720325, 0.0002529976482037455, 0.0002252081612823531, 0.0002688729437068105, 0.0002550014469306916, 0.00023484877601731569, 0.00024146339274011552, 0.00028164146351628006, 0.0002455997164361179, 0.00026674155378714204, 0.00026208951021544635, 0.0002553936210460961, 0.00020789452537428588, 0.0002595342812128365, 0.0002793535531964153, 0.0002201348979724571, 0.00024713532184250653, 0.00024072114320006222, 0.0002583896566648036, 0.00025528884725645185, 0.00021114711125846952, 0.000261384790064767, 0.00020390396821312606, 0.0002653748670127243, 0.0002859818050637841, 0.00021360153914429247, 0.00024033390218392015, 0.0002107339823851362, 0.0002619657607283443, 0.0002311996795469895, 0.0002464129065629095, 0.00028597796335816383, 0.0002633401018101722, 0.0002435595088172704, 0.00025053726858459413, 0.0002524656301829964, 0.00025944746448658407, 0.00022996145708020777, 0.0002080229896819219, 0.00024664934608154, 0.0002964456216432154, 0.00024266188847832382, 0.0002384720282861963, 0.00022322044242173433, 0.00024140019377227873, 0.00025057524908334017, 0.0002431998000247404, 0.0002387968124821782, 0.0002399054792476818, 0.0002334873133804649, 0.00024936904083006084, 0.00020784555817954242, 0.00025319308042526245, 0.00021228985860943794, 0.00024510969524271786, 0.00025794535758905113, 0.00022047407401259989, 0.00023993744980543852, 0.0002364948159083724, 0.00024842345737852156, 0.0002639279409777373, 0.00025475665461272, 0.00028525968082249165, 0.0002541259746067226, 0.00023080238315742463, 0.00026352371787652373, 0.00027065727044828236, 0.00024400113034062088, 0.0002670924295671284, 0.00023538597451988608, 0.00023833477462176234, 0.0002581627923063934, 0.00023507222067564726, 0.00026396921020932496, 0.00024051319633144885, 0.0002458062954246998, 0.00022930771228857338, 0.0002212509571108967, 0.00022582249948754907, 0.00023820990463718772, 0.00026422730297781527, 0.0002227577642770484, 0.00022046369849704206, 0.0002029683964792639, 0.00024175207363441586, 0.00027710283757187426, 0.00023363860964309424, 0.00022882303164806217, 0.00022253842325881124, 0.00022707818425260484, 0.000246797309955582, 0.00021569097589235753, 0.0002497207897249609, 0.0002563774469308555, 0.00030058014090172946, 0.00022877125593367964, 0.0002099405537592247, 0.0002641016326379031, 0.00024363842385355383, 0.00023620831780135632, 0.0002384438121225685, 0.0002484813448973, 0.0002463494020048529, 0.0002629774680826813, 0.00023453707399312407, 0.00024032195506151766, 0.0002657733566593379, 0.0002553424856159836, 0.00021435717644635588, 0.0002489741600584239, 0.0002777035697363317, 0.00022069422993808985, 0.00022305746097117662, 0.00022560861543752253, 0.000247528194449842, 0.00022690946934744716, 0.0002338637423235923, 0.00023168956977315247, 0.0002641368191689253, 0.0002457925584167242, 0.00024031841894611716, 0.00022448209347203374, 0.00024959331494756043, 0.00021409985492937267, 0.00018873623048420995, 0.0002404091792413965, 0.00024174358986783773, 0.00026311277179047465, 0.00023717619478702545, 0.0002243501367047429, 0.00023163226433098316, 0.00023432048328686506, 0.0002466658188495785, 0.00024752813624218106, 0.00022277973766904324, 0.00029185775201767683, 0.00022798673307988793, 0.0002501476265024394, 0.00023226774646900594, 0.0001982500107260421, 0.0002478581154718995, 0.00023449258878827095, 0.00023858669737819582, 0.0002860400709323585, 0.0002556297986302525, 0.0002615427947603166, 0.00026845053071156144, 0.0002522535505704582, 0.0002261039480799809, 0.00033002288546413183, 0.00026044409605674446, 0.00020145879534538835, 0.00023577696993015707, 0.0002445426944177598, 0.0001795487478375435, 0.00026092820917256176, 0.00023291503021027893, 0.00021667960390914232, 0.00022761865693610162, 0.0002241736074211076, 0.00025105499662458897, 0.000231105339480564, 0.00025500808260403574, 0.0002836424100678414, 0.00026121988776139915, 0.00025273580104112625, 0.0002539570559747517, 0.0002595991245470941, 0.0002509896585252136, 0.0002306658134330064, 0.00023481575772166252, 0.000246955722104758, 0.0002788763667922467, 0.0002597027050796896, 0.0002503436990082264, 0.00025816450943239033, 0.00023086211876943707, 0.0002492625790182501, 0.000210832484299317, 0.00024149223463609815, 0.00023415133182425052, 0.0002551063953433186, 0.0002461926778778434, 0.000280804350040853, 0.0002238710439996794, 0.0002543394803069532, 0.00022379115398507565, 0.000259474094491452, 0.00022465472284238786, 0.00020972346828784794, 0.00023003123351372778, 0.00024355671484954655, 0.00026377191534265876, 0.00021535862470045686, 0.0002204203774454072, 0.00023086015426088125, 0.00024250103160738945, 0.0002399226650595665, 0.00026640499709174037, 0.00026570266345515847, 0.00027039903216063976, 0.00025556585751473904, 0.0002629069203976542, 0.00020710672833956778, 0.00027674378361552954, 0.0002208099904237315, 0.0002558696724008769, 0.00020517247321549803, 0.00023575016530230641, 0.0002876241342164576, 0.0002623771724756807, 0.0002483050338923931, 0.00020587138715200126, 0.00022077170433476567, 0.0002396886411588639, 0.0002559819840826094, 0.0002490866172593087, 0.00022694589279126376, 0.0002565497125033289, 0.0002857426879927516, 0.0002692653506528586, 0.00023711063840892166, 0.00021451979409903288, 0.0002762335061561316, 0.0002391056768829003, 0.0002264461654704064, 0.00023038116341922432, 0.0002223251067334786, 0.00025217223446816206, 0.00024914953974075615, 0.00024227150424849242, 0.00022571782756131142, 0.0002494016371201724, 0.00022719864500686526, 0.00023737309675198048, 0.00022740256099496037, 0.00022845038620289415, 0.0002491974155418575, 0.00021069590002298355, 0.0002831115562003106, 0.00028801022563129663, 0.00025639645173214376, 0.0002807181153912097, 0.0002922762359958142, 0.00024813556228764355, 0.00027268336270935833, 0.00027415750082582235, 0.00024263867817353457, 0.00022243465355131775, 0.0002209535741712898, 0.00026982740382663906, 0.00021762809774372727, 0.000245386065216735, 0.00024225373635999858, 0.00022036385780666023, 0.00022451292898040265, 0.00022228615125641227, 0.00027884935843758285, 0.00023498869268223643, 0.0002495624066796154, 0.0002503029827494174, 0.0002452355984132737, 0.0002607567876111716, 0.00022627602447755635, 0.00024987146025523543, 0.0002585708862170577, 0.0002631142269819975, 0.00022339342103805393, 0.0002432461769785732, 0.00023332974524237216, 0.00023327059170696884, 0.00023949613387230784, 0.00022003303456585854, 0.0002450182510074228, 0.000269091542577371, 0.00025251571787521243, 0.00022634757624473423, 0.0002473279309924692, 0.00026228040223941207, 0.00021140176977496594, 0.00021224693045951426, 0.00023717382282484323, 0.0002244819770567119, 0.00028350422508083284, 0.0002445850695949048, 0.0002428390725981444, 0.00023930032330099493, 0.0002298779145348817, 0.00023832681472413242, 0.0002837121719494462, 0.00023610977223142982, 0.0002725566446315497, 0.0002484058204572648, 0.000239693108596839, 0.00023662135936319828, 0.00020925959688611329, 0.00019981028162874281, 0.00029027278651483357, 0.00025930581614375114, 0.00029579061083495617, 0.0002510946651455015, 0.00026651116786524653, 0.000244912167545408, 0.0002884907880797982, 0.00022533278388436884, 0.0002802667149808258, 0.0002034270582953468, 0.00024635912268422544, 0.00024112338724080473, 0.00023319550382439047, 0.00021331777679733932, 0.00022837529832031578, 0.0002489527396392077, 0.0002846206771209836, 0.00024143068003468215, 0.0002644816704560071, 0.00020955494255758822, 0.00024254371237475425, 0.00024356995709240437, 0.0002703628852032125, 0.0002423689147690311, 0.00026470847660675645, 0.0002736586320679635, 0.0002515883243177086, 0.00026098237140104175, 0.0002324001834494993, 0.0002279088948853314, 0.00028420519083738327, 0.00021996542636770755, 0.0002403213584329933, 0.00026884552789852023, 0.0002545574971009046, 0.00021878472762182355, 0.0002462647680658847, 0.00025558745255693793, 0.0002461087715346366, 0.0002437860966892913, 0.00026618025731295347, 0.00025179414660669863, 0.0002439263043925166, 0.0002558814303483814, 0.00020297893206588924, 0.00023026112467050552, 0.0002711296547204256, 0.00023861519002821296, 0.00025520860799588263, 0.00026138254906982183, 0.00019737188995350152, 0.00021611005649901927, 0.00024702647351659834, 0.00025166457635350525, 0.0002319812192581594, 0.0002869480231311172, 0.00025783185265026987, 0.0002468987659085542, 0.00022882578196004033, 0.00018755231576506048, 0.00023481916286982596, 0.0002246320218546316, 0.0002223406481789425, 0.00022892972629051656, 0.00026943645207211375, 0.00022326924954541028, 0.0002370208821957931, 0.000236773572396487, 0.0002208322985097766, 0.00023177430557552725, 0.00024329333973582834, 0.00024206051602959633, 0.00021099534933455288, 0.00028659121016971767, 0.0002842874964699149, 0.00024154312268365175, 0.00024165476497728378, 0.00022735889069736004, 0.00021929795912001282, 0.0002532058279030025, 0.00022905557125341147, 0.0002329259441467002, 0.00024252728326246142, 0.00024375863722525537, 0.00024980271700769663, 0.00024174661666620523, 0.0002867574221454561, 0.00021762409596703947, 0.0002266170922666788, 0.0002682875201571733, 0.00023841598886065185, 0.000250166398473084, 0.00024252402363345027, 0.00023237834102474153, 0.00023194379173219204, 0.0002205569326179102, 0.00024118454894050956, 0.0002860067179426551, 0.0002704986836761236, 0.0002955135714728385, 0.00026289623929187655, 0.000244723487412557, 0.00026245531626045704, 0.0002457579830661416, 0.0002227209770353511, 0.00022623258701059967, 0.00027283449890092015, 0.00023410796711686999, 0.0002165964979212731, 0.00023778017202857882, 0.00026784485089592636, 0.0002650936949066818, 0.00024701980873942375, 0.00021537378779612482, 0.00024373123596888036, 0.00023368322581518441, 0.00030407620943151414, 0.00024018016119953245, 0.0002429992164252326, 0.00022317955153994262, 0.0002108707558363676, 0.0002425845741527155, 0.00024612530251033604, 0.00027988615329377353, 0.00023501146642956883, 0.0002410255401628092, 0.0002817943459376693, 0.00025763449957594275, 0.0002600529114715755, 0.0002602583263069391, 0.00020332960411906242, 0.00025395044940523803, 0.00030241633066907525, 0.00023190074716694653, 0.0002244159986730665, 0.00026869692374020815, 0.0002802208182401955, 0.00024870302877388895, 0.00020164871239103377, 0.0002184554614359513, 0.00022442116460297257, 0.00022725474264007062, 0.00022169618750922382, 0.00023735406284686178, 0.0002319461782462895, 0.00024079538707155734, 0.000246274983510375, 0.0002415460185147822, 0.00022598612122237682, 0.00025469253887422383, 0.00022567878477275372, 0.00023603778390679508, 0.00023456437338609248, 0.0002591364609543234, 0.00024049176136031747, 0.00024903708253987134, 0.00021891480719204992, 0.00025574714527465403, 0.00024370146275032312, 0.00024059368297457695, 0.00026555880322121084, 0.0002249582757940516, 0.00025448310771025717, 0.00020674796542152762, 0.00022456036822404712, 0.00027020866400562227, 0.00024092743115033954, 0.00021600564650725573, 0.00027130625676363707, 0.00024789245799183846, 0.00024965201737359166, 0.00021264799579512328, 0.00025982307852245867, 0.00025529981940053403, 0.00023626667098142207, 0.0002423828118480742, 0.00022784326574765146, 0.0002446678699925542, 0.00021734644542448223, 0.0002535817329771817, 0.0002478095702826977, 0.0002227066725026816, 0.00024319965450558811, 0.0002257545420434326, 0.00024091215163934976, 0.00021577917505055666, 0.00023499730741605163, 0.0002846300776582211, 0.00025620279484428465, 0.0002703665813896805, 0.000287559611024335, 0.0002229261299362406, 0.00024342138203792274, 0.00022608695144299418, 0.0002803970710374415, 0.00027280073845759034, 0.00028233195189386606, 0.00021694581664633006, 0.00027149595553055406, 0.0002438407245790586, 0.00022906140657141805, 0.00024943106109276414, 0.00021479422866832465, 0.00025812824605964124, 0.00022056627494748682, 0.0002306083624716848, 0.00025728699984028935, 0.00026124250143766403, 0.00026033702306449413, 0.00024516216944903135, 0.00023663199681323022, 0.0002464882272761315, 0.00023734994465485215, 0.00026345031801611185, 0.00025197959621436894, 0.0002441255492158234, 0.00026002441882155836, 0.00022011964756529778, 0.00022453743440564722, 0.00026518222875893116, 0.0002551110228523612, 0.00024393602507188916, 0.00025261318660341203, 0.00023766850063111633, 0.00023148783657234162, 0.0002491024788469076, 0.00026467532734386623, 0.000269832496996969, 0.0002523543080314994, 0.00024272370501421392, 0.00021832676429767162, 0.0002487180463504046, 0.00023464013065677136, 0.0002600829757284373, 0.00022714847000315785, 0.00023728606174699962, 0.00022228348825592548, 0.00026264460757374763, 0.0002563097805250436, 0.0002080081612803042, 0.0002275629376526922, 0.0002442029654048383, 0.00027247320394963026, 0.00028746118186973035, 0.00026825605891644955, 0.00023934432829264551, 0.0002549957425799221, 0.00022728811018168926, 0.00027854411746375263, 0.00021855767408851534, 0.00020953724742867053, 0.00025077685131691396, 0.0002659244055394083, 0.0002728142135310918, 0.00025330999051220715, 0.00025755309616215527, 0.00023476217756979167, 0.00023523150593973696, 0.00022397928114514798, 0.0002494115033186972, 0.00022145402908790857, 0.00022551869915332645, 0.00022056634770706296, 0.0002529284101910889, 0.00025199397350661457, 0.0002464777498971671, 0.00026512276963330805, 0.00021177991584409028, 0.0002454430505167693, 0.0002705558727029711, 0.000270667253062129, 0.00022320158313959837, 0.0002498772519174963, 0.0002336029865546152, 0.00023939333914313465, 0.0002301586209796369, 0.00022776237165089697, 0.0002509567711967975, 0.0002813610772136599, 0.00023541823611594737, 0.00024139285960700363, 0.00021082500461488962, 0.00024097376444842666, 0.00024367027799598873, 0.00024100606970023364, 0.0002604545734357089, 0.00027209139079786837, 0.00022356478439178318, 0.00022535784228239208, 0.0002233350824099034, 0.00023451403831131756, 0.00022365814947988838, 0.00024847700842656195, 0.00021530843514483422, 0.0002135461545549333, 0.00021571980323642492, 0.00021907116752117872, 0.0002488421159796417, 0.00025250468752346933, 0.0002488548052497208, 0.00025107586407102644, 0.00024350074818357825, 0.00026892591267824173, 0.0002375105395913124, 0.0002544476301409304, 0.000281043176073581, 0.0002256148000014946, 0.00022584314865525812, 0.00024009248591028154, 0.00028559041675180197, 0.00029361245105974376, 0.0002310791314812377, 0.0002869479067157954, 0.0002386340347584337, 0.00022118858760222793, 0.00024981118622235954, 0.00024036437389440835, 0.00023944437270984054, 0.00021855358500033617, 0.0002248144446639344, 0.00025008400552906096, 0.00025680166436359286, 0.00023621547734364867, 0.00021785592252854258, 0.0002411688765278086, 0.00024262594524770975, 0.00022307565086521208, 0.00024793538614176214, 0.00023602753935847431, 0.0002633855910971761, 0.00022926455130800605, 0.000257949490332976, 0.0002687702071852982, 0.00021729838044848293, 0.00023444033286068588, 0.00021465489408001304, 0.00024065365141723305, 0.0002400002849753946, 0.0002242166519863531, 0.0002612029202282429, 0.00024703756207600236, 0.00024367697187699378, 0.00023653038078919053, 0.00022097292821854353, 0.00026931386673823, 0.00023416457406710833, 0.0002594821562524885, 0.00019021969637833536, 0.00024131206737365574, 0.00020337672322057188, 0.0002312319993507117, 0.00026215368416160345, 0.0002426964492769912, 0.0002203601907240227, 0.00026569649344310164, 0.00021933000243734568, 0.0002130482462234795, 0.00024029541236814111, 0.00020982146088499576, 0.0003120696055702865, 0.00027915360988117754, 0.00022509938571602106, 0.0002754398446995765, 0.0002562377485446632, 0.00021607772214338183, 0.000266796414507553, 0.00023900072847027332, 0.0002502939896658063, 0.00025507406098768115, 0.00025734613882377744, 0.00022242825070861727, 0.000266506802290678, 0.00023930467432364821, 0.0002954879601020366, 0.00023428544227499515, 0.00021894760720897466, 0.0002479934773873538, 0.00020798675541300327, 0.00019651137699838728, 0.000241348214331083, 0.0002617652644403279, 0.0002519540721550584, 0.00022691192862112075, 0.0002339374623261392, 0.00022111939324531704, 0.00023682501341681927, 0.00023434455215465277, 0.00026116709341295063, 0.0002367536217207089, 0.0002880810934584588, 0.0002258442691527307, 0.00025209764135070145, 0.00025916361482813954, 0.0002526844327803701, 0.00026154855731874704, 0.0003006671031471342, 0.0002673778508324176, 0.00026233124663122, 0.0002669662644620985, 0.0002265471703140065, 0.0002157206181436777, 0.00023366734967567027, 0.00022097838518675417, 0.00025653516058810055, 0.00019615422934293747, 0.0002482288982719183, 0.000255158927757293, 0.0002251326513942331, 0.00026438620989210904, 0.0002436463546473533, 0.00025531399296596646, 0.00023059468367137015, 0.0002605757617857307, 0.0002939398982562125, 0.00024857401149347425, 0.00025416677817702293, 0.00026285843341611326, 0.0002518720575608313, 0.00022245860600378364, 0.00025657590595074, 0.00020269426750019193, 0.0002562318404670805, 0.00022557082411367446, 0.0002584629401098937, 0.0002759872586466372, 0.0002201600291300565, 0.0002424937265459448, 0.00024001843121368438, 0.00023573341604787856, 0.0002361388469580561, 0.0002175375266233459, 0.00028193771140649915, 0.00023222595336847007, 0.00021160850883461535, 0.0002595560799818486, 0.00022415033890865743, 0.00023944981512613595, 0.00024353789922315627, 0.0002112760121235624, 0.000255211052717641, 0.00025219444069080055, 0.00027120020240545273, 0.0002122806035913527, 0.00024340071831829846, 0.0002495161024853587, 0.00024514549295417964, 0.0002697977761272341, 0.0002443923440296203, 0.00023942453844938427, 0.0002436157810734585, 0.00023835813044570386, 0.00025330649805255234, 0.00022817366698291153, 0.00023515937209594995, 0.00025052798446267843, 0.0002681911864783615, 0.00024685816606506705, 0.0002675433934200555, 0.0002298531326232478, 0.00023871359007898718, 0.0002615326957311481, 0.00023036803759168833, 0.0002737049071583897, 0.00019619239901658148, 0.00025829722289927304, 0.00022883510973770171, 0.00024777959333732724, 0.0002436089562252164, 0.0002573124074842781, 0.0002128440683009103, 0.0002354767348151654, 0.0002728019899223, 0.0002530594065319747, 0.00023331337433774024, 0.0002547773765400052, 0.00023188977502286434, 0.0001977325155166909, 0.00022892473498359323, 0.0002479415852576494, 0.00021168304374441504, 0.0002171407686546445, 0.00024972917162813246, 0.0002733787405304611, 0.00024512672098353505, 0.00021745232515968382, 0.0002484239812474698, 0.00024371367180719972, 0.00023946049623191357, 0.00024387431039940566, 0.00022477655147667974, 0.00022467896633315831, 0.00021872084471397102, 0.0002833769249264151, 0.00026814869488589466, 0.00021870715136174113, 0.00022969934798311442, 0.00022284408623818308, 0.0002587305789347738, 0.0002578938438091427, 0.000274226360488683, 0.0002498732937965542, 0.00023345722001977265, 0.0002481936535332352, 0.0002553689409978688, 0.0002345678658457473, 0.00022896107111591846, 0.00022185775742400438, 0.00026930891908705235, 0.00024292539455927908, 0.0002796979097183794, 0.00026429383433423936, 0.0002467080485075712, 0.00024403881980106235, 0.0002480993571225554, 0.00022477706079371274, 0.00022666025324724615, 0.00024965786724351346, 0.00021835426741745323, 0.00028105394449084997, 0.00027221866184845567, 0.00023423010134138167, 0.00024158298037946224, 0.00025042847846634686, 0.00022733227524440736, 0.0002279644540976733, 0.00023793049331288785, 0.00027876472449861467, 0.000247480085818097, 0.00026594457449391484, 0.0001998481893679127, 0.0002600870793685317, 0.00025425475905649364, 0.00024918734561651945, 0.00021343055414035916, 0.0002550022618379444, 0.00024499904247932136, 0.00024731314624659717, 0.0002118483535014093, 0.00028761819703504443, 0.00023855296603869647, 0.0002219344605691731, 0.0002409951266599819, 0.00025235654902644455, 0.00023974012583494186, 0.00022104979143477976, 0.0002525402815081179, 0.00027536405832506716, 0.0002323644730495289, 0.0002434497291687876, 0.00025120042846538126, 0.0002496009401511401, 0.00022666467702947557, 0.0002617362479213625, 0.00021451727661769837, 0.0002411160385236144, 0.00025570308207534254, 0.00027197078452445567, 0.00020382029470056295, 0.0002710181288421154, 0.0002687297237571329, 0.0002192068932345137, 0.00028533345903269947, 0.00023753407003823668, 0.00023283653717953712, 0.00026073973276652396, 0.00022362063464242965, 0.00026809319388121367, 0.00019971965230070055, 0.0002374483592575416, 0.00020377154578454792, 0.0002710956323426217, 0.0002519090776331723, 0.00020033925829920918, 0.00023194277309812605, 0.0002822369278874248, 0.000240804860368371, 0.00018079523579217494, 0.00025851864484138787, 0.00020551076158881187, 0.0002378783538006246, 0.0002990656648762524, 0.0002468973689246923, 0.00025980101781897247, 0.00027167078224010766, 0.00021907352493144572, 0.0002313649165444076, 0.00023576030798722059, 0.0002649493981152773, 0.00023228090140037239, 0.00023259020235855132, 0.00023816763132344931, 0.00022520951461046934, 0.0002441628312226385, 0.00022660904505755752, 0.00024573804694227874, 0.00025590514997020364, 0.0002255053404951468, 0.00023118441458791494, 0.00024848332395777106, 0.00021942288731224835, 0.00027232340653426945, 0.00022387626813724637, 0.00024695240426808596, 0.00024390409816987813, 0.00023274694103747606, 0.0002571467775851488, 0.0002442956902086735, 0.00020480918465182185, 0.00023181637516245246, 0.0002690634282771498, 0.00021801120601594448, 0.0003155391023028642, 0.00028800457948818803, 0.0002284309157403186, 0.000280209380434826, 0.00021653994917869568, 0.0002300311898579821, 0.0002725642407312989, 0.00029555472428910434, 0.00026562868151813745, 0.0002699131437111646, 0.00023446384875569493, 0.00023869425058364868, 0.00022993388120085, 0.0002340511418879032, 0.0002531221543904394, 0.00026760317268781364, 0.00031499401666224003, 0.00020277938165236264, 0.00022362711024470627, 0.00025526367244310677, 0.0002638405712787062, 0.0002494723885320127, 0.00022853832342661917, 0.00021546875359490514, 0.0002814754843711853, 0.00023863786191213876, 0.00023883095127530396, 0.0002472364285495132, 0.00026662289747036994, 0.00028781354194507003, 0.00027914991369470954, 0.0002635668497532606, 0.0002423587575322017, 0.00025863302289508283, 0.00024892869987525046, 0.00022464434732683003, 0.00024157097504939884, 0.00022559206990990788, 0.0002141709701390937, 0.00026890498702414334, 0.00024383378331549466, 0.000213395687751472, 0.0002485636214260012, 0.00023739800963085145, 0.00022591053857468069, 0.0002470622712280601, 0.00029878577333875, 0.0002577317936811596, 0.00024216729798354208, 0.00024107146600726992, 0.00024058946291916072, 0.00022850549430586398, 0.00022548444394487888, 0.0002714940346777439, 0.00022379726578947157, 0.00025412830291315913, 0.00021692122390959412, 0.00026137629174627364, 0.00021686256513930857, 0.0002506343589629978, 0.00028922781348228455, 0.0002226750657428056, 0.0002580653235781938, 0.0002549341006670147, 0.0002323301014257595, 0.0002385902189416811, 0.00026375980814918876, 0.00021789125457871705, 0.00021159784228075296, 0.00023065203276928514, 0.0002200122835347429, 0.00023345275258179754, 0.0002109404158545658, 0.0002463342098053545, 0.00024461362045258284, 0.0002589360810816288, 0.00023053167387843132, 0.0002671315160114318, 0.0002654078707564622, 0.0002622325555421412, 0.00023712331312708557, 0.000239958826568909, 0.0002957841206807643, 0.0002265360380988568, 0.00022379413712769747, 0.00025419227313250303, 0.0002285096561536193, 0.00022297473333310336, 0.00023083084670361131, 0.0002641027676872909, 0.00020957297238055617, 0.00022759329294785857, 0.0002834429615177214, 0.00022843711485620588, 0.00021518224093597382, 0.000217862194404006, 0.00023850779689382762, 0.0002520029665902257, 0.0002249557146569714, 0.00021220896451268345, 0.00022663877462036908, 0.00024695374304428697, 0.00024500576546415687, 0.0002803515235427767, 0.0002457690134178847, 0.0002411461027804762, 0.00029024016112089157, 0.00026708756922744215, 0.000247617659624666, 0.00022576794435735792, 0.0002569922071415931, 0.00022178389190230519, 0.00023788760881870985, 0.00023840993526391685, 0.00025424998602829874, 0.0002727703540585935, 0.0002147890190826729, 0.00029603156144730747, 0.0002345777174923569, 0.0002835627237800509, 0.00021299438958521932, 0.00025323303998447955, 0.0002213259576819837, 0.00021046597976237535, 0.00023560428235214204, 0.00022783387976232916, 0.00020666145428549498, 0.0002229607052868232, 0.0002567135379649699, 0.0002546531322877854, 0.00023342896020039916, 0.0002322854270460084, 0.0002578468411229551, 0.00022864261700306088, 0.00021829447359777987, 0.00025306217139586806, 0.00023224450706038624, 0.0002456869406159967, 0.0002617368008941412, 0.0003152195131406188, 0.00024618307361379266, 0.00022571389854419976, 0.0002708160027395934, 0.00022026583610568196, 0.00025020004250109196, 0.00026324589271098375, 0.00023209680512081832, 0.000248716794885695, 0.0002386895357631147, 0.00022121267102193087, 0.0002570906071923673, 0.0002928980393335223, 0.0002744635276030749, 0.0002294217119924724, 0.00024965329794213176, 0.00021421023120637983, 0.000258963176747784, 0.00023320040781982243, 0.0002218150330008939, 0.00022031812113709748, 0.00024782033869996667, 0.00024526691413484514, 0.00021526114142034203, 0.0002743238292168826, 0.00020518952806014568, 0.00023609015624970198, 0.0002520542766433209, 0.00025461165932938457, 0.0002077319659292698, 0.0002209271478932351, 0.00025325483875349164, 0.00026787276146933436, 0.00021899798593949527, 0.00023447324929293245, 0.0002652971015777439, 0.00024954750551842153, 0.00023562749265693128, 0.00023447336570825428, 0.00026430681464262307, 0.00025781261501833797, 0.0002452456101309508, 0.00026340378099121153, 0.0002301678468938917, 0.00024592949193902314, 0.0002673449052963406, 0.00025612383615225554, 0.00025174373877234757, 0.0002120461140293628, 0.0002571904915384948, 0.0002263130445498973, 0.0002568706404417753, 0.00020667770877480507, 0.00026358183822594583, 0.00023152155335992575, 0.0001963820104720071, 0.00027131102979183197, 0.00021984076010994613, 0.0002228584053227678, 0.0002261892077513039, 0.00023719953605905175, 0.0002811289159581065, 0.0002553552621975541, 0.00026648782659322023, 0.0002815434127114713, 0.00022376740525942296, 0.00022578946663998067, 0.00022679052199237049, 0.00026385337696410716, 0.00025016270228661597, 0.00023559480905532837, 0.00027506312471814454, 0.0002459647657815367, 0.0002547174517530948, 0.00024933277745731175, 0.00024034854141063988, 0.00023433488968294114, 0.00024029490305110812, 0.000223955386900343, 0.0002681721234694123, 0.000259016320342198, 0.00025997351622208953, 0.00025370041839778423, 0.00024280704383272678, 0.0001903405791381374, 0.0002598987484816462, 0.00025300111155956984, 0.00022744404850527644, 0.00024409223988186568, 0.00024609104730188847, 0.00027403520653024316, 0.00024440354900434613, 0.0002280988119309768, 0.0002775972825475037, 0.00021444098092615604, 0.0002340364590054378, 0.00019912105926778167, 0.00030899886041879654, 0.00024555003619752824, 0.00021247891709208488, 0.0002173602260882035, 0.0002553305239416659, 0.00023554581275675446, 0.00022128601267468184, 0.0002728549006860703, 0.00024485503672622144, 0.0002433040936011821, 0.00031257819500751793, 0.0002824533439707011, 0.00022069524857215583, 0.00023665335902478546, 0.00023656903067603707, 0.00023609248455613852, 0.00022282268037088215, 0.0002735734742600471, 0.00025777716655284166, 0.0002455425274092704, 0.00021557783475145698, 0.0002466571459081024, 0.00023136456729844213, 0.0002868484880309552, 0.00024813238997012377, 0.000242770736804232, 0.00025761473807506263, 0.00024057543487288058, 0.0002303949004271999, 0.0003324593708384782, 0.00023258154396899045, 0.00026881101075559855, 0.00020834399037994444, 0.00025753016234375536, 0.0002711617562454194, 0.0002553560771048069, 0.0002672372793313116, 0.0002086724416585639, 0.00023133539070840925, 0.00022219965467229486, 0.0002595341647975147, 0.00025291048223152757, 0.0002561873698141426, 0.00022925018856767565, 0.0002543182927183807, 0.00024254288291558623, 0.00023935266654007137, 0.00022178857761900872, 0.0002516048843972385, 0.00025733435177244246, 0.0002523590810596943, 0.0002244561619590968, 0.0002456716902088374, 0.0002975837851408869, 0.0002291888667969033, 0.00025888020172715187, 0.00024585009668953717, 0.00022076608729548752, 0.00021419464610517025, 0.00026236841222271323, 0.00024715758627280593, 0.0002315506717422977, 0.00030163879273459315, 0.00023355762823484838, 0.00025479504256509244, 0.00023371228598989546, 0.0002344505046494305, 0.0002481164992786944, 0.0002214906271547079, 0.0002736529568210244, 0.00021336418285500258, 0.0002669591340236366, 0.0002370857837377116, 0.0002395388000877574, 0.00025731007917784154, 0.0002612480893731117, 0.00022394854750018567, 0.00021841851412318647, 0.0002616035344544798, 0.0002559034328442067, 0.0002646329812705517, 0.0002842069370672107, 0.0002610471856314689, 0.00021661905338987708, 0.00024705627583898604, 0.00024328457948286086, 0.00024287855194415897, 0.00027267407858744264, 0.00024360539100598544, 0.0002749984560068697, 0.00023309273819904774, 0.00027084266184829175, 0.00021390912297647446, 0.0002465150610078126, 0.0002407393913017586, 0.00021674699382856488, 0.00022456883743871003, 0.0002514428924769163, 0.00026612868532538414, 0.0002559340500738472, 0.0002523578586988151, 0.00025923605426214635, 0.0002338445046916604, 0.0002448046288918704, 0.00023074181808624417, 0.00024545929045416415, 0.00023300753673538566, 0.00023980814148671925, 0.00019133421301376075, 0.00021538256260100752, 0.00026615060050971806, 0.0002342003135709092, 0.0002299932821188122, 0.00023507590231020004, 0.0002619681472424418, 0.00027388351736590266, 0.00026971218176186085, 0.0002710432163439691, 0.00022572700981982052, 0.00024845617008395493, 0.00020337317255325615, 0.00023062789114192128, 0.00024789696908555925, 0.0002659816818777472, 0.0002759597555268556, 0.00020236709679011256, 0.00024270114954560995, 0.00029026399715803564, 0.00022542040096595883, 0.00022959988564252853, 0.00024485032190568745, 0.0002876348153222352, 0.00020375566964503378, 0.0002344122331123799, 0.00028832018142566085, 0.0002403692196821794, 0.00022082832583691925, 0.00022160273510962725, 0.0002736534515861422, 0.0002506397431716323, 0.0002674137067515403, 0.00027085765032097697, 0.0002573285310063511, 0.0002562718582339585, 0.00024640493211336434, 0.00026604687445797026, 0.0002190258092014119, 0.0002626485947985202, 0.0002242663613287732, 0.0002338070480618626, 0.00025997101329267025, 0.00023328553652390838, 0.00019620386592578143, 0.0002283704816363752, 0.00023262889590114355, 0.00024177448358386755, 0.0002288241230417043, 0.00027003578725270927, 0.00022228859597817063, 0.00023955201322678477, 0.0002205969940405339, 0.0002567211049608886, 0.00025985599495470524, 0.00026059491210617125, 0.00026155656087212265, 0.0002459453244227916, 0.0002677404845599085, 0.0002335316821699962, 0.00022151395387481898, 0.00023132942442316562, 0.0002622819447424263, 0.0002614387485664338, 0.00023653988318983465, 0.00024136104912031442, 0.00021683197701349854, 0.00023435894399881363, 0.0002282746572745964, 0.0002567994815763086, 0.00023342535132542253, 0.0003114839200861752, 0.00026563351275399327, 0.00022354579414241016, 0.00023971610062289983, 0.0001986937568290159, 0.0002301513886777684, 0.0002497277164366096, 0.00020744783978443593, 0.00024228738038800657, 0.00023103354033082724, 0.0002296523452969268, 0.00024997792206704617, 0.00019897018501069397, 0.00022752094082534313, 0.00023654008691664785, 0.0002737429749686271, 0.00021590726100839674, 0.00028322552680037916, 0.0002575694234110415, 0.00023333175340667367, 0.00025670198374427855, 0.00024010695051401854, 0.00022904544312041253, 0.00027552421670407057, 0.00022657518275082111, 0.00027868468896485865, 0.00023880990920588374, 0.0002472990599926561, 0.00023836510081309825, 0.0002605011686682701, 0.00023167877225205302, 0.0002511508937459439, 0.0002495839144103229, 0.00023457317729480565, 0.00025783400633372366, 0.00022892642300575972, 0.0002607732021715492, 0.00026579396217130125, 0.00028410827508196235, 0.00024343334371224046, 0.00024448655312880874, 0.00023759002215228975, 0.0002277530584251508, 0.00021331419702619314, 0.00027463494916446507, 0.000223054681555368, 0.00022903975332155824, 0.0002489211910869926, 0.0002948598994407803, 0.00020997179672122002, 0.0002191898092860356, 0.00024935457622632384, 0.0002532490761950612, 0.00024334668705705553, 0.0002330905554117635, 0.00023489903833251446, 0.00023906011483632028, 0.0002628112561069429, 0.0002691131376195699, 0.00023820900241844356, 0.00025354023091495037, 0.00027904572198167443, 0.00025232843472622335, 0.00022235146025195718, 0.0002535615931265056, 0.00023321236949414015, 0.00024409066827502102, 0.0002471586049068719, 0.00026062081451527774, 0.00024470986681990325, 0.0002707023231778294, 0.000257540843449533, 0.0002675898722372949, 0.00023917938233353198, 0.0002799169160425663, 0.00023258168948814273, 0.00024398844107054174, 0.0002105767634930089, 0.00023839579080231488, 0.0002804873511195183, 0.00021716582705266774, 0.00024386767472606152, 0.0002415631024632603, 0.00026383184012956917, 0.000306758243823424, 0.00023857674386817962, 0.0002442410623189062, 0.00022328530030790716, 0.00026579046971164644, 0.00026435963809490204, 0.0002485731092747301, 0.00026735299616120756, 0.0002443613775540143, 0.0002560243592597544, 0.00024285008839797229, 0.00023766867525409907, 0.00028447710792534053, 0.00020706745272036642, 0.00022343522869050503, 0.00023120215337257832, 0.00024776210193522274, 0.0002457819937262684, 0.00027020034031011164, 0.00025099594495259225, 0.00025348502094857395, 0.0002771340077742934, 0.00024649061379022896, 0.0002657763834577054, 0.0002772635198198259, 0.00023330208205152303, 0.0002742629440035671, 0.0002855608472600579, 0.0003038625291083008, 0.0002438425290165469, 0.00026049138978123665, 0.00025251254555769265, 0.0002627976064104587, 0.0002241330803371966, 0.0002266676165163517, 0.0002311408461537212, 0.0002362547384109348, 0.00024389107420574874, 0.00019824050832539797, 0.00027661374770104885, 0.0002401733072474599, 0.0002202551404479891, 0.00023421966761816293, 0.00020136262173764408, 0.00025897269370034337, 0.0002829446457326412, 0.0002414878981653601, 0.0002579732099547982, 0.00024295574985444546, 0.0002498074609320611, 0.0002647348737809807, 0.0002521116111893207, 0.0002194917033193633, 0.00024438719265162945, 0.00025343536981381476, 0.00028759799897670746, 0.00021346917492337525, 0.00028155260952189565, 0.0002301373751834035, 0.00026824744418263435, 0.00027950972435064614, 0.000221198468352668, 0.00022009717940818518, 0.00022711645578965545, 0.00023383117513731122, 0.00028773624217137694, 0.00022324254678096622, 0.00026255028205923736, 0.0001894239685498178, 0.00021327608556021005, 0.0002164835314033553, 0.00027003095601685345, 0.0002214570704381913, 0.0002520423731766641, 0.000244372698944062, 0.0002467513259034604, 0.0002157674462068826, 0.00022229789465200156, 0.000224041155888699, 0.00023269954544957727, 0.00025747515610419214, 0.00025297762476839125, 0.00023838580818846822, 0.0002601687447167933, 0.00022302931756712496, 0.00022903940407559276, 0.00021510398073587567, 0.00025351636577397585, 0.0002341805666219443, 0.000234275619732216, 0.0001985128765227273, 0.00024363688135053962, 0.00025053456192836165, 0.00025283286231569946, 0.0002345465763937682, 0.00025050342082977295, 0.00026451749727129936, 0.00025567738339304924, 0.00024906269391067326, 0.00029991689370945096, 0.00025332870427519083, 0.00027531356317922473, 0.00022767721384298056, 0.00029239736613817513, 0.00024920800933614373, 0.0002508925972506404, 0.00021180765179451555, 0.00023664339096285403, 0.00023419315402861685, 0.00021444579761009663, 0.0002577163395471871, 0.0002469177998136729, 0.0002498699468560517, 0.0002752927248366177, 0.00023142679128795862, 0.00023238922585733235, 0.00024965149350464344, 0.0002962809812743217, 0.0002654562995303422, 0.00022756238467991352, 0.00021522094903048128, 0.0002340049686608836, 0.00025856701540760696, 0.00022166046255733818, 0.0002482907730154693, 0.00022384400654118508, 0.00023748670355416834, 0.00025338545674458146, 0.00024510748335160315, 0.00021366920555010438, 0.0002481045376043767, 0.0002184466866310686, 0.0002542729489505291, 0.0002573540841694921, 0.00020065454009454697, 0.00021516649576369673, 0.0002562789595685899, 0.00023677441640757024, 0.0002551104698795825, 0.00026446193805895746, 0.00024270283756777644, 0.0002447728475090116, 0.0002540738496463746, 0.0002417829673504457, 0.00021600189211312681, 0.00023172884539235383, 0.00023437707568518817, 0.0002162383170798421, 0.0002348076377529651, 0.0002358983037993312, 0.0002507755998522043, 0.00025306266616098583, 0.00023067228903528303, 0.0002264530776301399, 0.00022911938140168786, 0.0002659265010152012, 0.0002026090514846146, 0.00026114648790098727, 0.0002327410038560629, 0.000249734177486971, 0.00022270534827839583, 0.0002599144645500928, 0.00023672333918511868, 0.00024881993886083364, 0.00024190390831790864, 0.00022969131532590836, 0.00027870526537299156, 0.0002529486082494259, 0.00025123771047219634, 0.00027110788505524397, 0.00019273970974609256, 0.00022173415345605463, 0.00024709373246878386, 0.00024158520682249218, 0.0002716481685638428, 0.00025414745323359966, 0.0002285551599925384, 0.0002578059211373329, 0.00023438975040335208, 0.00023846779367886484, 0.0002599387080408633, 0.00027059632702730596, 0.0002372139715589583, 0.000262933608610183, 0.00022815058764535934, 0.0002677249431144446, 0.00025422993348911405, 0.0002468815946485847, 0.00026923202676698565, 0.0002623927139211446, 0.0002576092374511063, 0.00023771662381477654, 0.000237109066802077, 0.00024195380683522671, 0.0002710277622099966, 0.00022633645858149976, 0.00023707118816673756, 0.0002412681351415813, 0.00026620205608196557, 0.00023799009795766324, 0.00022349653590936214, 0.00024987850338220596, 0.000230394332902506, 0.000248755794018507, 0.0002560218272265047, 0.0002775250468403101, 0.00022998670465312898, 0.00024031403881963342, 0.0002372702001594007, 0.00026514416094869375, 0.00024229867267422378, 0.00024149991804733872, 0.00025116276810877025, 0.00029662565793842077, 0.00024511993979103863, 0.00022139507927931845, 0.00024124911578837782, 0.00022499611077364534, 0.00022643803094979376, 0.00024645045050419867, 0.00025279427063651383, 0.0002397861098870635, 0.0002440473617753014, 0.00023878389038145542, 0.00022869584790896624, 0.00025908922543749213, 0.00026905618142336607, 0.0002387419226579368, 0.00023620238061994314, 0.0002712779678404331, 0.0002805376425385475, 0.0002386240375926718, 0.00023735662398394197, 0.0002497089153621346, 0.00023313879501074553, 0.00025317311519756913, 0.00020934642816428095, 0.0002723993093241006, 0.00026172175421379507, 0.0002696894807741046, 0.00026288515073247254, 0.0002541932335589081, 0.00023516618239227682, 0.0002569378702901304, 0.00025937112513929605, 0.00023923971457406878, 0.0002361443330300972, 0.00026647685444913805, 0.00022840908786747605, 0.00020557122479658574, 0.0002470611361786723, 0.0002541769645176828, 0.0002427087601972744, 0.00027019510162062943, 0.00025092996656894684, 0.0002765477111097425, 0.0002454128407407552, 0.0002643491025082767, 0.00022469802934210747, 0.00023740812321193516, 0.0002812788006849587, 0.0002837397623807192, 0.000222378657781519, 0.00023435099865309894, 0.00024037351249717176, 0.0002074652147712186, 0.00023875650367699564, 0.00023153152142185718, 0.00022442972112912685, 0.00023305932700168341, 0.00025361223379150033, 0.000279308034805581, 0.00023906906426418573, 0.0002348373382119462, 0.00023541484551969916, 0.00026390160201117396, 0.00023196318943519145, 0.0002553118101786822, 0.0002382839738857001, 0.00025742893922142684, 0.0002452260523568839, 0.00021700235083699226, 0.00023197082919068635, 0.0002126606268575415, 0.00024056535039562732, 0.00025722436839714646, 0.00024227358517237008, 0.0002256326552014798, 0.00023912765027489513, 0.00025580907822586596, 0.0002579166030045599, 0.00026171442004851997, 0.0002496214001439512, 0.0002513573272153735, 0.00023455226619262248, 0.0002925419539678842, 0.00028287345776334405, 0.00026470786542631686, 0.0002506978053133935, 0.00024951453087851405, 0.00020364098600111902, 0.00023243033501785249, 0.00024564340128563344, 0.0002545331954024732, 0.0002617360441945493, 0.00022481929045170546, 0.00026621102006174624, 0.0002682848135009408, 0.00025988765992224216, 0.00024829813628457487, 0.00027399329701438546, 0.00024238300102297217, 0.000228788674576208, 0.0002441553515382111, 0.00024880809360183775, 0.0002797404595185071, 0.00023929808230604976, 0.0002388229186180979, 0.00025057236780412495, 0.00026527821319177747, 0.00022156935301609337, 0.00025822786847129464, 0.00022819859441369772, 0.000254171114647761, 0.00023343943757936358, 0.00022926136443857104, 0.0002621594467200339, 0.00022560410434380174, 0.0002555395767558366, 0.000255065766396001, 0.0002573361271061003, 0.00024107996432576329, 0.00023286281793843955, 0.00025660055689513683, 0.00029472491587512195, 0.000255793594988063, 0.0002923161955550313, 0.0002696402370929718, 0.0002699158794712275, 0.0002703983918763697, 0.00027798282098956406, 0.0002439470699755475, 0.00021590088726952672, 0.0002469552564434707, 0.00026942265685647726, 0.0002419707307126373, 0.00024997242144308984, 0.0002856729843188077, 0.00026816895115189254, 0.0002452594053465873, 0.00023225616314448416, 0.00023908686125651002, 0.00023606322065461427, 0.00022985768737271428, 0.00025065449881367385, 0.0002475657092873007, 0.00023415949544869363, 0.00022920483024790883, 0.00024949447833932936, 0.0002230405661975965, 0.00022864276252221316, 0.00022040923067834228, 0.00025835621636360884, 0.0002589020295999944, 0.00022070584236644208, 0.00020358750771265477, 0.00025053080753423274, 0.0002002687833737582, 0.00021256237232591957, 0.00021196241141296923, 0.00027470439090393484, 0.00026509727467782795, 0.0002256295847473666, 0.00023807153047528118, 0.00025627706781961024, 0.00021896063117310405, 0.00023469033476430923, 0.0002837989595718682, 0.00025075406301766634, 0.00023394552408717573, 0.0003034846449736506, 0.00026447628624737263, 0.00023111746122594923, 0.00022375228581950068, 0.0002775785396806896, 0.00024093354295473546, 0.00024707705597393215, 0.00026970947510562837, 0.0002292819699505344, 0.0002534438681323081, 0.0002574544632807374, 0.0002523083530832082, 0.000237263273447752, 0.00023132158094085753, 0.00022613992041442543, 0.0002644038468133658, 0.00028828115318901837, 0.0002453352208249271, 0.00022342342708725482, 0.00024030781059991568, 0.00026092040934599936, 0.00026158842956647277, 0.00021204777294769883, 0.0002460748655721545, 0.00022340672148857266, 0.00023778786999173462, 0.0002464807766955346, 0.0002857200743164867, 0.00024277328338939697, 0.0002302179200341925, 0.00026493245968595147, 0.0002641812898218632, 0.0002340740174986422, 0.000249452335992828, 0.00023160593991633505, 0.00021521026792470366, 0.00022499510669149458, 0.00023918377701193094, 0.00021881941938772798, 0.00023765199875924736, 0.00022515682212542742, 0.0002429856831440702, 0.00026667106430977583, 0.000244035356445238, 0.00021462795848492533, 0.0002256619045510888, 0.00023732737463433295, 0.00023903808323666453, 0.0002598563442006707, 0.00022767689370084554, 0.00020262815814930946, 0.0002537623222451657, 0.0002590721705928445, 0.0002495856606401503, 0.00019101184443570673, 0.00024365342687815428, 0.0002315520978299901, 0.0002407686406513676, 0.00025897027808241546, 0.00027096280246041715, 0.00024068461789283901, 0.0002576310944277793, 0.00026135763619095087, 0.00023335564765147865, 0.00020551377383526415, 0.000270022515906021, 0.00022309186169877648, 0.0002550435601733625, 0.0002478325041010976, 0.0002756914764177054, 0.000208495301194489, 0.00023913316545076668, 0.00023130045155994594, 0.0002332535950699821, 0.0002474898356013, 0.0002935668162535876, 0.00023095536744222045, 0.00023823125229682773, 0.0002660987083800137, 0.00024792688782326877, 0.0002475334331393242, 0.00021015883248765022, 0.00025641583488322794, 0.00026810754206962883, 0.0002155468100681901, 0.0002583884634077549, 0.00020535053045023233, 0.00025786596233956516, 0.0002317819744348526, 0.00023268820950761437, 0.00020659300207626075, 0.00023230996157508343, 0.00026856022304855287, 0.0002335583121748641, 0.00023087186855264008, 0.0002184196637244895, 0.00024503536405973136, 0.00025571355945430696, 0.0002264887880301103, 0.00019891737611033022, 0.00024273968301713467, 0.0002840452070813626, 0.000253357837209478, 0.00022082305804360658, 0.00026796269230544567, 0.0002406306884950027, 0.00022426653595175594, 0.0002329018898308277, 0.0001912023581098765, 0.00023709719243925065, 0.0002395443298155442, 0.0002038374077528715, 0.00022029528918210417, 0.00022055972658563405, 0.000232366583077237, 0.0002536889514885843, 0.00022495178563985974, 0.00021052722877357155, 0.00025395973352715373, 0.00023758805764373392, 0.00026543682906776667, 0.0002565178438089788, 0.0002383117243880406, 0.00026606503524817526, 0.00023562497517559677, 0.00024331167514901608, 0.00022595684276893735, 0.00026227085618302226, 0.00025344171444885433, 0.0002445052668917924, 0.00023581237473990768, 0.00023337596212513745, 0.00022937366156838834, 0.00021546485368162394, 0.000264264817815274, 0.0002723477373365313, 0.0002479316608514637, 0.00026933287153951824, 0.00023721536854282022, 0.00020501334802247584, 0.0002124011080013588, 0.0002823626855388284, 0.00022852813708595932, 0.00023680058075115085, 0.0002508563920855522, 0.00021430946071632206, 0.0002505397133063525, 0.00022584140242543072, 0.0002763833326753229, 0.00025279668625444174, 0.0002651466056704521, 0.00025022999034263194, 0.00020243317703716457, 0.00023829344718251377, 0.0002274496655445546, 0.00024763945839367807, 0.00022485469526145607, 0.00024786501307971776, 0.00022820514277555048, 0.00025256365188397467, 0.00020629745267797261, 0.0002344495296711102, 0.00025469547836109996, 0.000250127020990476, 0.00021224343799985945, 0.000275961821898818, 0.00028109378763474524, 0.00025790664949454367, 0.00022639945382252336, 0.0002380244550295174, 0.0002351153380004689, 0.00022620780509896576, 0.00020281510660424829, 0.00019271734345238656, 0.000266861985437572, 0.0002448167069815099, 0.00023091633920557797, 0.0002739545307122171, 0.00023457020870409906, 0.0002257057058159262, 0.00026544640422798693, 0.000221867929212749, 0.0002720195916481316, 0.00023551477352157235, 0.00029850052669644356, 0.00022174004698172212, 0.00024859854602254927, 0.00021099780860822648, 0.00025200992240570486, 0.0002666525833774358, 0.00023530605540145189, 0.0002582278975751251, 0.0002294900914421305, 0.0002597570128273219, 0.00026922329561784863, 0.00024042364384513348, 0.00019750272622331977, 0.00023637745471205562, 0.0002570982906036079, 0.00027909319032914937, 0.00025448043015785515, 0.00021271209698170424, 0.00022255298972595483, 0.00024702303926460445, 0.0002851301396731287, 0.00028577333432622254, 0.00023198961571324617, 0.0002476782537996769, 0.00024238266632892191, 0.00022481544874608517, 0.0002375634794589132, 0.0002344601962249726, 0.0002545011229813099, 0.0002882592671085149, 0.0002376762859057635, 0.00024024469894357026, 0.0002452319022268057, 0.0002276077284477651, 0.00023533139028586447, 0.00024390840553678572, 0.00024251626746263355, 0.0002305908565176651, 0.0002534574014134705, 0.0002786486002150923, 0.00020224576292093843, 0.00022778133279643953, 0.0002757703769020736, 0.00020330838742665946, 0.00028395658591762185, 0.00022391114907804877, 0.00024408812168985605, 0.00020687848154921085, 0.0002660533064045012, 0.0002522003196645528, 0.0002465572324581444, 0.0002448793384246528, 0.00020731694530695677, 0.00026736719883047044, 0.0002693936985451728, 0.0002655158459674567, 0.0002533242222853005, 0.0002790663274936378, 0.00023209673236124218, 0.00022079423069953918, 0.00022756341786589473, 0.0002401502861175686, 0.0002352812880417332, 0.00025296976673416793, 0.0002793313760776073, 0.00023400504142045975, 0.00022575042385142297, 0.00023200827126856893, 0.0002344841050216928, 0.00024349585874006152, 0.00024883932201191783, 0.0002218291920144111, 0.00024249809212051332, 0.00023981640697456896, 0.0002441767428535968, 0.00023368853726424277, 0.00025155916227959096, 0.0002693487040232867, 0.0002568814961705357, 0.0002753241278696805, 0.00025836584973149, 0.0002527554170228541, 0.0002544877934269607, 0.0002135139366146177, 0.00026409144629724324, 0.00024251031572930515, 0.00024875745293684304, 0.0002727933751884848, 0.0002562429872341454, 0.00025198771618306637, 0.0002203583571827039, 0.0002462663105688989, 0.00025018828455358744, 0.0002482874260749668, 0.00024097006826195866, 0.00021722989913541824, 0.00021957924764137715, 0.00024370253959205002, 0.00023305619833990932, 0.00021410944464150816, 0.0002554400416556746, 0.0002463860728312284, 0.0002587155031505972, 0.0002682698832359165, 0.00021293495956342667, 0.00027940570726059377, 0.00024834141368046403, 0.00028220631065778434, 0.0002526177268009633, 0.00022983216331340373, 0.0002659929159563035, 0.0002577738487161696, 0.0002205975033575669, 0.00026726897340267897, 0.0002392414171481505, 0.00021598231978714466, 0.00020850308646913618, 0.00023112916096579283, 0.0002471776388119906, 0.0002515678061172366, 0.0002797314082272351, 0.00024870026390999556, 0.000257745647104457, 0.0002660672180354595, 0.0002293725119670853, 0.00023524247808381915, 0.00022062193602323532, 0.00022367550991475582, 0.00022891920525580645, 0.00024777467478998005, 0.00021719517826568335, 0.0002651372633408755, 0.00024412086349911988, 0.0002107077743858099, 0.0002849524898920208, 0.0002351625735173002, 0.0002303279034094885, 0.0002534314407967031, 0.00024009188928175718, 0.00026655299006961286, 0.0002267575910082087, 0.00023991147463675588, 0.00021620022016577423, 0.0002797028864733875, 0.0002633125113788992, 0.00020940248214174062, 0.000251302175456658, 0.00023436199990101159, 0.00024258870689664036, 0.0002656106371432543, 0.0002496246306691319, 0.00021765557175967842, 0.0002480097464285791, 0.0002133039088221267, 0.00022271568013820797, 0.00026776365120895207, 0.0002478082897141576, 0.0002578450075816363, 0.00024076433328446, 0.00024535853299312294, 0.00023326188966166228, 0.00023381217033602297, 0.00025380399893037975, 0.00026944518322125077, 0.00028455493156798184, 0.0002209763479186222, 0.00023337059246841818, 0.00030660341144539416, 0.00022568157874047756, 0.0002489754988346249, 0.0002628567744977772, 0.0002765814133454114, 0.00023235029948409647, 0.0002240078174509108, 0.00022334368259180337, 0.00022669308236800134, 0.00021339693921618164, 0.0002051112096523866, 0.0002741985663305968, 0.00027109915390610695, 0.0002048766182269901, 0.00021622750500682741, 0.00020568945910781622, 0.00024084716278593987, 0.0002232414553873241, 0.00025649231974966824, 0.00021404468861874193, 0.00023212688392959535, 0.0002674528514035046, 0.0002731543208938092, 0.00022506111417897046, 0.00024087865313049406, 0.00022890813124831766, 0.00022982651717029512, 0.00020894572662655264, 0.00024214014410972595, 0.00021338567603379488, 0.00025183105026371777, 0.0002176362177124247, 0.00023531833721790463, 0.00031518092146143317, 0.00023731906549073756, 0.00026361815980635583, 0.000261023233179003, 0.00024107657372951508, 0.00025652797194197774, 0.00023386826796922833, 0.00023110336042009294, 0.000271784228971228, 0.000211807680898346, 0.00022533170704264194, 0.00024952756939455867, 0.00025210369494743645, 0.00025043776258826256, 0.0002601207233965397, 0.0002865964488591999, 0.0002591538068372756, 0.0002357681660214439, 0.00030248743132688105, 0.00022503736545331776, 0.00025245611323043704, 0.00020200370636302978, 0.00028004954219795763, 0.00022725548478774726, 0.0002357477496843785, 0.00020309831597842276, 0.0002517192915547639, 0.00024074059911072254, 0.00023035387857817113, 0.00021266608382575214, 0.00023609098570887, 0.00030074029928073287, 0.0002409048902336508, 0.00024666349054314196, 0.0002029781462624669, 0.00022216349316295236, 0.00027227308601140976, 0.00021429489424917847, 0.0002077066747006029, 0.00026013748720288277, 0.00023906957358121872, 0.00023649448121432215, 0.00027468567714095116, 0.0002546984178479761, 0.00025113692390732467, 0.0002453773922752589, 0.00026052704197354615, 0.0002077532117255032, 0.00029334420105442405, 0.00023562446585856378, 0.0002395469055045396, 0.00023203981982078403, 0.00022208754671737552, 0.00025138756609521806, 0.00025552805163897574, 0.000245315779466182, 0.0003140545159112662, 0.00024653185391798615, 0.00022280508710537106, 0.00025924225337803364, 0.0002336097531951964, 0.0002672175760380924, 0.00026723582413978875, 0.00025331799406558275, 0.00027426978340372443, 0.00025462539633736014, 0.00021934544201940298, 0.00022279091353993863, 0.00026472951867617667, 0.0002401311503490433, 0.00025365198962390423, 0.0002654982963576913, 0.00022930526756681502, 0.0002917934616561979, 0.00021037676197011024, 0.0002513409417588264, 0.0002424890553811565, 0.0002838492800947279, 0.0002378301287535578, 0.00021953803661745042, 0.0002313876320840791, 0.0002767546975519508, 0.0002437517832731828, 0.00024391505576204509, 0.0002297847968293354, 0.0002970509813167155, 0.00021572054538410157, 0.00022098889166954905, 0.0002471031039021909, 0.0002584956237114966, 0.00021344235574360937, 0.0002404564875178039, 0.00020972307538613677, 0.00022497185273095965, 0.00028261993429623544, 0.00024664521333761513, 0.0002573929959908128, 0.00025375839322805405, 0.00022551006986759603, 0.00021260077483020723, 0.00023769614927005023, 0.00022275334049481899, 0.0002339209895581007, 0.0002697408781386912, 0.00021173845743760467, 0.0002515511878300458, 0.00024367327569052577, 0.00023934541968628764, 0.0002078533434541896, 0.0002954256779048592, 0.00025560351787135005, 0.0002506232412997633, 0.0002640412130858749, 0.00023447787680197507, 0.00024392308841925114, 0.00027723194216378033, 0.000220546979107894, 0.0002668575325515121, 0.00025505252415314317, 0.0002468559832777828, 0.00021697474585380405, 0.00020681078603956848, 0.00025599676882848144, 0.00026162283029407263, 0.0002491273917257786, 0.00023819381021894515, 0.00023180534481070936, 0.0002721677883528173, 0.0002558119304012507, 0.0002681850746739656, 0.000256270490353927, 0.00024913152446970344, 0.0002514064544811845, 0.00026023213285952806, 0.00025004002964124084, 0.00021671423746738583, 0.00022463711502496153, 0.00019243384303990752, 0.0002343061933061108, 0.0002258389285998419, 0.00022142281522974372, 0.000292098120553419, 0.0002535227395128459, 0.00023239947040565312, 0.0002950132475234568, 0.00024125378695316613, 0.00023751980916131288, 0.00022619462106376886, 0.00022259863908402622, 0.0002881832479033619, 0.0002705305232666433, 0.0002342077495995909, 0.00023940012033563107, 0.00022735728998668492, 0.00022692668426316231, 0.00027724652318283916, 0.00023077170772012323, 0.0002771757426671684, 0.0002489816688466817, 0.00026147247990593314, 0.00026033923495560884, 0.00024840267724357545, 0.00027090354706160724, 0.00022740766871720552, 0.00025827233912423253, 0.00020547928579617292, 0.0002292937715537846, 0.00025068537797778845, 0.00024933493114076555, 0.0002407379070064053, 0.00023808992409612983, 0.00024920047144405544, 0.0002637438883539289, 0.000260889355558902, 0.00025062012718990445, 0.00023546224110759795, 0.00025523206568323076, 0.00024425232550129294, 0.00024290906731039286, 0.00020097274682484567, 0.0002525690069887787, 0.0002061453997157514, 0.00024154405400622636, 0.00026138030807487667, 0.0002877295482903719, 0.0002277111925650388, 0.0002422617981210351, 0.00024017896794248372, 0.00027078556013293564, 0.0002468943130224943, 0.0002337432379135862, 0.00024892232613638043, 0.00022482848726212978, 0.00022663861454930156, 0.00021251625730656087, 0.0002446114376652986, 0.00024169539392460138, 0.0002143997116945684, 0.00022445328067988157, 0.00023188289196696132, 0.00023987663735169917, 0.00023416159092448652, 0.00027572765247896314, 0.00026588625041767955, 0.00025009067030623555, 0.00030599901219829917, 0.00027654392761178315, 0.0002692750422284007, 0.00023781933123245835, 0.0002721614728216082, 0.00023386756947729737, 0.0002594657998997718, 0.0002348122652620077, 0.00026972003979608417, 0.00021913596719969064, 0.00025944161461666226, 0.00025951029965654016, 0.00023989063629414886, 0.00024293700698763132, 0.0002643560292199254, 0.00024229359405580908, 0.0002645180793479085, 0.0002521678397897631, 0.000264256406808272, 0.00029065145645290613, 0.0002641488390509039, 0.00031714269425719976, 0.00023251461971085519, 0.00023112705093808472, 0.0002319004270248115, 0.00023497101210523397, 0.00023612190852873027, 0.00025220250245183706, 0.00023016304476186633, 0.00024424129514954984, 0.00024247703549917787, 0.00026002625236287713, 0.00020341954950708896, 0.00025405013002455235, 0.0002711837587412447, 0.0002657648583408445, 0.00025199868832714856, 0.00028440161258913577, 0.000285058980807662, 0.0002514964435249567, 0.00024058834242168814, 0.00024266463879030198, 0.0001956563937710598, 0.0002267064555780962, 0.00025542633375152946, 0.00023759079340379685, 0.00024538818979635835, 0.0002779949572868645, 0.0002715720038395375, 0.000252036057645455, 0.0002693290007300675, 0.00022461937624029815, 0.000266608374658972, 0.00022720611013937742, 0.0002338226040592417, 0.0002487213059794158, 0.0002501269627828151, 0.00021714855392929167, 0.00021687016123905778, 0.0002855567436199635, 0.00024525256594642997, 0.00025075057055801153, 0.00021278196072671562, 0.00021256734908092767, 0.00020182551816105843, 0.00023674839758314192, 0.00026778242317959666, 0.00026438318309374154, 0.00023875635815784335, 0.00028344555175863206, 0.00024241958453785628, 0.00024225463857874274, 0.0002324734814465046, 0.00026165798772126436, 0.00023124729341361672, 0.0002478606766089797, 0.0002602615859359503, 0.00026175021775998175, 0.00022871205874253064, 0.00023544557916466147, 0.00021398755779955536, 0.0002450434840284288, 0.00021315517369657755, 0.00024521688465029, 0.00020431684970390052, 0.0002512968785595149, 0.00022676379012409598, 0.0002254563441965729, 0.00022918905597180128, 0.00022967011318542063, 0.00023400006466545165, 0.000266127084614709, 0.00022960547357797623, 0.00027280740323476493, 0.00028849029331468046, 0.0002457174414303154, 0.00023606573813594878, 0.00024241946812253445, 0.00021122253383509815, 0.00022888736566528678, 0.0002733322908170521, 0.0002406642452115193, 0.00021769222803413868, 0.0002845404378604144, 0.0002573272795416415, 0.00021454371744766831, 0.0002708215033635497, 0.00024122497416101396, 0.0002453132765367627, 0.00024363478587474674, 0.00026319053722545505, 0.00025147348060272634, 0.00021933075913693756, 0.00025135857868008316, 0.00022487052774522454, 0.00026143487775698304, 0.0002573419187683612, 0.00025298260152339935, 0.00022795052791479975, 0.00028945336816832423, 0.00022577925119549036, 0.00021908523922320455, 0.00026587769389152527, 0.00023454189067706466, 0.00025900817126967013, 0.00021547135838773102, 0.00024729460710659623, 0.00024286934058181942, 0.000267908995738253, 0.00024015858070924878, 0.00026104331482201815, 0.000263003574218601, 0.00023952529591042548, 0.00025071355048567057, 0.0002266107767354697, 0.00023323277127929032, 0.00023235355911310762, 0.00023438241623807698, 0.00022910130792297423, 0.0002793527382891625, 0.00025673440541140735, 0.0002602683671284467, 0.0002438948577037081, 0.0002815864863805473, 0.00027256543398834765, 0.00022424591588787735, 0.00027066259644925594, 0.000249081029323861, 0.00025786241167224944, 0.00026096610235981643, 0.0002725552476476878, 0.00022755147074349225, 0.0002870651369448751, 0.0002617686113808304, 0.0002624986518640071, 0.00023921296815387905, 0.00025401319726370275, 0.00022365126642398536, 0.0002582692832220346, 0.00019671351765282452, 0.00024291512090712786, 0.00024007988395169377, 0.0002440104726701975, 0.00021856543025933206, 0.00025177845964208245, 0.00024846591986715794, 0.00023503319243900478, 0.00025969435228034854, 0.00025983157684095204, 0.00023309556127060205, 0.00023829306883271784, 0.0002468318853061646, 0.0002508993202354759, 0.0002731368876993656, 0.00023428308486472815, 0.0002560110588092357, 0.0002704341895878315, 0.00024433067301288247, 0.0002269619144499302, 0.00028591559384949505, 0.00024398788809776306, 0.0002294211444677785, 0.00022384579642675817, 0.00022980777430348098, 0.00021939081489108503, 0.0002530343481339514, 0.0002319288469152525, 0.00022209007875062525, 0.00023238506400957704, 0.0002430361055303365, 0.00023632217198610306, 0.00021907266636844724, 0.00022265760344453156, 0.00023377280740533024, 0.00022646367142442614, 0.00024183614004869014, 0.00028548677801154554, 0.0002433315385133028, 0.0002288238174514845, 0.00021660226047970355, 0.00022664856805931777, 0.00022418337175622582, 0.00021821740665473044, 0.0002506421005818993, 0.00023177903494797647, 0.00022349839855451137, 0.0002363224484724924, 0.00027981825405731797, 0.0002453452907502651, 0.0002366609260207042, 0.00021815596846863627, 0.00022915092995390296, 0.0002493814390618354, 0.00025889804237522185, 0.00026824098313227296, 0.00027536979177966714, 0.00027120482991449535, 0.00022004346828907728, 0.00023420796787831932, 0.00023055981728248298, 0.00022991152945905924, 0.0002668290981091559, 0.00022773671662434936, 0.00026130001060664654, 0.00023693646653555334, 0.00022294881637208164, 0.00024242168001364917, 0.00028328821645118296, 0.00018950726371258497, 0.00025247936719097197, 0.00027690065326169133, 0.0002209483936894685, 0.00032382961944676936, 0.0002691165136639029, 0.00023939026868902147, 0.0002521697897464037, 0.00024094924447126687, 0.00023230933584272861, 0.00026297237491235137, 0.00019621422688942403, 0.00023547487217001617, 0.00025365411420352757, 0.00022369049838744104, 0.0002376885968260467, 0.0002558800915721804, 0.0002545267634559423, 0.00027680513449013233, 0.00023659877479076385, 0.00024035217938944697], [0.0002749516861513257, 0.00022040578187443316, 0.00023118687386158854, 0.00020166096510365605, 0.00023327865346800536, 0.00027516932459548116, 0.00025663411361165345, 0.0002475812507327646, 0.00023506201978307217, 0.00028362966259010136, 0.00024261804355774075, 0.0002569898497313261, 0.00022964723757468164, 0.0002548699558246881, 0.00024249544367194176, 0.00023285398492589593, 0.000266312767053023, 0.0001992640463868156, 0.0002724369987845421, 0.00024247090914286673, 0.0002732319990172982, 0.0002679012541193515, 0.0002685804502107203, 0.00024839965044520795, 0.0002442566037643701, 0.0002927891327999532, 0.00022896798327565193, 0.00021322148677427322, 0.00021697803458664566, 0.00027862118440680206, 0.00022716185776516795, 0.00018741122039500624, 0.00023424024402629584, 0.00021525012562051415, 0.0002606958441901952, 0.00023407286789733917, 0.0002483097487129271, 0.0002502711722627282, 0.00027317815693095326, 0.00026406816323287785, 0.00024093477986752987, 0.0002156930131604895, 0.0002529494813643396, 0.00020523597777355462, 0.0002368883287999779, 0.0002213102561654523, 0.0002498509711585939, 0.00022917655587662011, 0.00026698256260715425, 0.0002814433828461915, 0.0002575973339844495, 0.00021530613594222814, 0.00028770160861313343, 0.0002434606576571241, 0.00025298731634393334, 0.00024303719692397863, 0.00025982619263231754, 0.00025091093266382813, 0.0002525996242184192, 0.0002481401024851948, 0.0002561397268436849, 0.00022844396880827844, 0.0002071929775411263, 0.00027131944079883397, 0.00024176637816708535, 0.00022766951587982476, 0.0002307079266756773, 0.00025366913178004324, 0.00024470448261126876, 0.00023493330809287727, 0.0002298012695973739, 0.0002509893965907395, 0.00023680967569816858, 0.00023690008674748242, 0.00021846566232852638, 0.0002578035928308964, 0.00027537980349734426, 0.0002396039926679805, 0.00022663547133561224, 0.00022902089403942227, 0.00025126346736215055, 0.0002615728590171784, 0.00021094725525472313, 0.00022870648535899818, 0.00024094691616483033, 0.00022133892343845218, 0.0002325494133401662, 0.0002390677109360695, 0.00025217406800948083, 0.00024445229792036116, 0.00022326614998746663, 0.00023746304214000702, 0.00024295196635648608, 0.0002197649737354368, 0.0002610382507555187, 0.00021551807003561407, 0.0002598232531454414, 0.0002535040257498622, 0.00023079880338627845, 0.00023304304340854287, 0.00023373407020699233, 0.00024559596204198897, 0.0002691406407393515, 0.0002550529898144305, 0.00024325343838427216, 0.00023035702179186046, 0.00024394049250986427, 0.00020593186491169035, 0.0002544880553614348, 0.00025170730077661574, 0.0002307366521563381, 0.00026444424293003976, 0.00021182133059483021, 0.0002762160438578576, 0.00023447882267646492, 0.00021905452013015747, 0.0002650810347404331, 0.00023773583234287798, 0.000228771343245171, 0.00020058370137121528, 0.00020918925292789936, 0.00025933037977665663, 0.00025753004592843354, 0.00025623413966968656, 0.00022818335855845362, 0.00024176010629162192, 0.00023290353419724852, 0.0002089332847390324, 0.00022144343529362231, 0.00024772968026809394, 0.00025549184647388756, 0.0002659215242601931, 0.0002224738127551973, 0.00024797781952656806, 0.00026743870694190264, 0.00022065028315410018, 0.00023699711891822517, 0.0002683780330698937, 0.0002140637516276911, 0.00027067511109635234, 0.0002160027506761253, 0.00023401810904033482, 0.000236257299548015, 0.00025239630485884845, 0.0002536420652177185, 0.0002315724123036489, 0.00024140662571880966, 0.0002342825900996104, 0.0002616999263409525, 0.00021197136084083468, 0.00026946395519189537, 0.00026073551271110773, 0.00023601400607731193, 0.00020775821758434176, 0.00019830149540212005, 0.0002784802345559001, 0.00023843137023504823, 0.00022485261433757842, 0.0002243303315481171, 0.0002300501655554399, 0.0002508733596187085, 0.00025688507594168186, 0.00022004016500432044, 0.0002457080117892474, 0.0002388062421232462, 0.0002592486853245646, 0.00025203012046404183, 0.00026218770653940737, 0.00027517820126377046, 0.00020281036267988384, 0.00023770031111780554, 0.00022635232016909868, 0.0002757597540039569, 0.00023791124112904072, 0.00023763753415551037, 0.0002309807314304635, 0.00021551601821556687, 0.0002608145005069673, 0.00025953867589123547, 0.0002507694880478084, 0.0002938057004939765, 0.00025293356156907976, 0.0002539973647799343, 0.00023398466873914003, 0.00022140670625958592, 0.00023161795979831368, 0.00024449615739285946, 0.0002642999461386353, 0.00024196994490921497, 0.00022442947374656796, 0.00026987033197656274, 0.0002257395681226626, 0.0002547381154727191, 0.0002248576347483322, 0.00023717527801636606, 0.00029929063748568296, 0.0002283692010678351, 0.00022481214546132833, 0.00029224861646071076, 0.00022180068481247872, 0.00021669099805876613, 0.00026859596255235374, 0.0002448119630571455, 0.0002615029225125909, 0.0002421167300781235, 0.0002646629873197526, 0.00022206908033695072, 0.00027996368589811027, 0.0002965973981190473, 0.0002603621396701783, 0.0002645236672833562, 0.00020729005336761475, 0.00027177928132005036, 0.00021457327238749713, 0.00025554042076691985, 0.00019222668197471648, 0.00024597233277745545, 0.0002190947998315096, 0.0002178977883886546, 0.00023761996999382973, 0.00026114037609659135, 0.00022139294014777988, 0.0002371286100242287, 0.00026140708359889686, 0.0002282664063386619, 0.0002358926140004769, 0.00026011341833509505, 0.00023431320732925087, 0.00020255932759027928, 0.00027245041565038264, 0.00026228727074339986, 0.000249631266342476, 0.00024514144752174616, 0.0002223621995653957, 0.0002550463832449168, 0.0002681329206097871, 0.00024383931304328144, 0.00022566285042557865, 0.00020274125563446432, 0.0002656497817952186, 0.00024311896413564682, 0.00021485271281562746, 0.0002375071489950642, 0.00022935951710678637, 0.00023382579092867672, 0.0002449808525852859, 0.00024610458058305085, 0.0002192376705352217, 0.0002421313402010128, 0.00029124566935934126, 0.0002443186822347343, 0.0002645576896611601, 0.00022152584278956056, 0.00024222885258495808, 0.0002360892976867035, 0.00018046933109872043, 0.00022500622435472906, 0.0002575052494648844, 0.0002331235009478405, 0.00021996544091962278, 0.0002642480831127614, 0.00024322261742781848, 0.00025635832571424544, 0.00023717536532785743, 0.0002617195714265108, 0.00023174351372290403, 0.00023163194418884814, 0.00026225586771033704, 0.00024006812600418925, 0.0002564303285907954, 0.00022984367387834936, 0.00022470232215709984, 0.0002649109228514135, 0.00022854057897347957, 0.00021277660562191159, 0.00019769903155975044, 0.00029239675495773554, 0.00021368562011048198, 0.00023593651712872088, 0.00025126858963631094, 0.0002166732301702723, 0.00024058147391770035, 0.0002716792223509401, 0.00025218454538844526, 0.000270452699624002, 0.00028163299430161715, 0.00025866489158943295, 0.00025267081218771636, 0.0002187659265473485, 0.0002227616059826687, 0.0002469635510351509, 0.0002371246664552018, 0.00022288173204287887, 0.00024687990662641823, 0.00024097959976643324, 0.00021796264627482742, 0.00022872092085890472, 0.0002606936322990805, 0.00024708500131964684, 0.00024155271239578724, 0.00024211640993598849, 0.00023436956689693034, 0.00022537782206200063, 0.00024215778103098273, 0.0002539750130381435, 0.0002412907051621005, 0.0002243972267024219, 0.00023291140678338706, 0.0002123425656463951, 0.0002663257473614067, 0.0002724480291362852, 0.00023959740065038204, 0.00023460698139388114, 0.00023862128728069365, 0.00021264310635160655, 0.00023436412448063493, 0.00025462289340794086, 0.0002758323971647769, 0.00024517416022717953, 0.00029847369296476245, 0.00024804408894851804, 0.00023116118973121047, 0.00021059509890619665, 0.000242919908487238, 0.0002810865989886224, 0.00024097615096252412, 0.00022045880905352533, 0.00022583527606911957, 0.00024633994325995445, 0.00022928616090212017, 0.00025746459141373634, 0.0002543024020269513, 0.00024726148694753647, 0.00024218011822085828, 0.00023716897703707218, 0.0002507049939595163, 0.0002786343393381685, 0.00022353387612383813, 0.0002185513876611367, 0.0002437453658785671, 0.00025641158572398126, 0.00021996985014993697, 0.00024976086569949985, 0.0002449610910844058, 0.00023364456137642264, 0.0002263335045427084, 0.00023922068066895008, 0.00028105618548579514, 0.00024212620337493718, 0.0002333708107471466, 0.0002311842399649322, 0.00026904826518148184, 0.0002674304414540529, 0.00023257179418578744, 0.0002855457423720509, 0.00025379066937603056, 0.00025211164029315114, 0.00025840578018687665, 0.00027490101638250053, 0.00025036511942744255, 0.0002505884622223675, 0.00022497789177577943, 0.0002473591302987188, 0.0002061675622826442, 0.0002405480045126751, 0.00020568784384522587, 0.00022672089107800275, 0.00024834301439113915, 0.00023189933563116938, 0.0002468316233716905, 0.0002308980911038816, 0.0002588521165307611, 0.00022628888837061822, 0.00021686693071387708, 0.0002771070576272905, 0.0002690207911655307, 0.00022836978314444423, 0.00024126797507051378, 0.00022853763948660344, 0.0002551671350374818, 0.000260258762864396, 0.00025932490825653076, 0.00026953592896461487, 0.00024061527801677585, 0.0002590868389233947, 0.00023672357201576233, 0.00020497606601566076, 0.00024874633527360857, 0.00023971016344148666, 0.00019657611846923828, 0.0002529763150960207, 0.0002457450027577579, 0.00029013046878390014, 0.00023509515449404716, 0.0002571442455518991, 0.00023513706400990486, 0.00029014662140980363, 0.00021836951782461256, 0.0002207986981375143, 0.00027996848803013563, 0.00020218300051055849, 0.0002501626149751246, 0.000249547854764387, 0.00019448243256192654, 0.00023011640587355942, 0.0002543050213716924, 0.00022615194029640406, 0.0002251352125313133, 0.00026093338965438306, 0.00027335702907294035, 0.00023985908774193376, 0.00020582797878887504, 0.00021937780547887087, 0.0002232920960523188, 0.00025000792811624706, 0.00028218646184541285, 0.000222881804802455, 0.00026558066019788384, 0.0002852325851563364, 0.00023385597160086036, 0.00019824282207991928, 0.00026911948225460947, 0.00024397899687755853, 0.0002873402845580131, 0.00024739833315834403, 0.0002470047038514167, 0.0002699189935810864, 0.00028225273126736283, 0.00025144830578938127, 0.0002453264605719596, 0.0002576997794676572, 0.00023518693342339247, 0.00020035586203448474, 0.0002549996133893728, 0.00022989834542386234, 0.00022642807743977755, 0.00025696438387967646, 0.00023963538114912808, 0.00023631243675481528, 0.00025405859923921525, 0.00024267453409265727, 0.00021735778136644512, 0.0002575820544734597, 0.000257844862062484, 0.00022562408412341028, 0.0002732429711613804, 0.00022853385598864406, 0.00023410102585330606, 0.0002796147018671036, 0.00022329726198222488, 0.00022164033725857735, 0.000243814880377613, 0.00026011650334112346, 0.00021726197155658156, 0.00024205453519243747, 0.00023852703452575952, 0.00025391593226231635, 0.00024127191863954067, 0.0002525036979932338, 0.00023293198319151998, 0.00024927244521677494, 0.0002508587494958192, 0.00024751329328864813, 0.0002459041716065258, 0.00023997003154363483, 0.0002518629189580679, 0.0002420956880087033, 0.00020745540678035468, 0.00021812398335896432, 0.0002667406515683979, 0.0002221656177425757, 0.0002534266095608473, 0.00023172605142462999, 0.0002287052193423733, 0.00026021539815701544, 0.00026560627156868577, 0.0002820143708959222, 0.00022761888976674527, 0.00019753948436118662, 0.00024493131786584854, 0.000253046746365726, 0.0002527887118048966, 0.00028235893114469945, 0.00026145510491915047, 0.0002737873583100736, 0.000249378674197942, 0.00026980339316651225, 0.0002620307495817542, 0.0002430457971058786, 0.0002450364117976278, 0.0002245622599730268, 0.00024338698131032288, 0.000268196890829131, 0.0002514836669433862, 0.00025862950133159757, 0.00022385163174476475, 0.0002279342443216592, 0.00022188184084370732, 0.0002468597376719117, 0.00023549780598841608, 0.0002439317322568968, 0.00023531183251179755, 0.00023685628548264503, 0.0002571554796304554, 0.00023051253810990602, 0.00022612859902437776, 0.00019451066327746958, 0.0002681523037608713, 0.0002513255749363452, 0.00023972726194188, 0.00023608592164237052, 0.0002607201167847961, 0.00023196893744170666, 0.0002499165420886129, 0.0002470691397320479, 0.0002635439159348607, 0.0002210818784078583, 0.00022753141820430756, 0.00024272166774608195, 0.0002594089601188898, 0.00023575132945552468, 0.00025195349007844925, 0.00024261182988993824, 0.00022034076391719282, 0.0002519326808396727, 0.0002561375149525702, 0.00027499839779920876, 0.00028972586733289063, 0.0002226501819677651, 0.00027759603108279407, 0.00021232910512480885, 0.0002477661764714867, 0.00022160133812576532, 0.00021572349942289293, 0.00026188671472482383, 0.00026859898935072124, 0.0002263410424347967, 0.00023010048607829958, 0.000247297459281981, 0.000216605985770002, 0.0002508557809051126, 0.00023188670456875116, 0.0002254363353131339, 0.000254002952715382, 0.00022633708431385458, 0.0002220089954789728, 0.00023351627169176936, 0.00025698341778479517, 0.00023092988703865558, 0.00024800674873404205, 0.0002768572885543108, 0.00029675636324100196, 0.00023351560230366886, 0.00025653804186731577, 0.00023666699416935444, 0.00024956956622190773, 0.0002560376888141036, 0.00025862129405140877, 0.00024735991610214114, 0.000254537386354059, 0.0002573249221313745, 0.000238752385484986, 0.0001991955650737509, 0.00020451299496926367, 0.00023952111951075494, 0.00023815964232198894, 0.0002633161493577063, 0.00024375670182053, 0.00024062884040176868, 0.0002788329147733748, 0.00023812484869267792, 0.00021856019156984985, 0.0002824003458954394, 0.00025466791703365743, 0.0002404428960289806, 0.0001972236786969006, 0.0002355233591515571, 0.00023123044229578227, 0.00021162656776141375, 0.00023033012985251844, 0.00022419345623347908, 0.000229231794946827, 0.00023333291755989194, 0.00020891192252747715, 0.00022184368572197855, 0.00022274936782196164, 0.00022903323406353593, 0.0002267421077704057, 0.00024201655469369143, 0.00023858805070631206, 0.00022410156088881195, 0.00023378328478429466, 0.00023401109501719475, 0.00023277355649042875, 0.00023437055642716587, 0.00024740383378230035, 0.0002580451837275177, 0.0002388315333519131, 0.0002257253072457388, 0.00023506391153205186, 0.0002585669863037765, 0.00023538550885859877, 0.0002215233544120565, 0.0002474038628861308, 0.00022697570966556668, 0.0002257351588923484, 0.00024395168293267488, 0.00024277711054310203, 0.00023128466273192316, 0.0002633638505358249, 0.00025393255054950714, 0.0002736132009886205, 0.00024702519294805825, 0.00023427396081387997, 0.00026680855080485344, 0.00029658066341653466, 0.0002504629082977772, 0.00023725077335257083, 0.0002495315275155008, 0.0002364879474043846, 0.00023914156190585345, 0.00023057282669469714, 0.0002892354968935251, 0.0002409896405879408, 0.00023663530009798706, 0.0002238667366327718, 0.0002474126813467592, 0.00025142505182884634, 0.00021462631411850452, 0.0002500214905012399, 0.00025521349743939936, 0.00024245383974630386, 0.00026548575260676444, 0.00027517511625774205, 0.0002457460213918239, 0.00022775221441406757, 0.0002490135084372014, 0.00023500843963120133, 0.00021098002616781741, 0.000226939344429411, 0.00023536312801297754, 0.0002280227781739086, 0.000239284461713396, 0.0002499828115105629, 0.00024986424250528216, 0.00023013443569652736, 0.00027495119138620794, 0.00022057055321056396, 0.000248173251748085, 0.00024231475254055113, 0.00027393430355004966, 0.00026476048515178263, 0.00024075867258943617, 0.0002602721215225756, 0.00021451928478199989, 0.00024863510043360293, 0.00023617151600774378, 0.0002150115615222603, 0.0002651952381711453, 0.00022580444056075066, 0.00023743884230498224, 0.00023757736198604107, 0.00023165717720985413, 0.00022646509751211852, 0.0002456932852510363, 0.00023682547907810658, 0.0002651870308909565, 0.000208063778700307, 0.0002268295647809282, 0.0002346385590499267, 0.00024016786483116448, 0.00022241307306103408, 0.0002122363803209737, 0.00027379050152376294, 0.00024497078265994787, 0.00022071009152568877, 0.00023932187468744814, 0.00021941513114143163, 0.00024183859932236373, 0.00025674913194961846, 0.0002542783331591636, 0.00021861775894649327, 0.0002223883057013154, 0.0002796444750856608, 0.0002753939770627767, 0.0002578161074779928, 0.0002508019970264286, 0.0002711961860768497, 0.0002531980280764401, 0.0002447089646011591, 0.00026153854560106993, 0.00023272204271052033, 0.00025961350183933973, 0.00024956496781669557, 0.0002255190775031224, 0.00026922329561784863, 0.0002058067184407264, 0.00021312051103450358, 0.00021976031712256372, 0.00026789840194396675, 0.0002368419081903994, 0.00026801147032529116, 0.0002337718615308404, 0.00026694178814068437, 0.0002178715803893283, 0.00022307659673970193, 0.00024307900457642972, 0.00026131709455512464, 0.00021069853391963989, 0.00025461005861870944, 0.0002507535682525486, 0.0002503304276615381, 0.00024487811606377363, 0.00025860019377432764, 0.0002674507850315422, 0.000246695737587288, 0.00027227698592469096, 0.00022430352692026645, 0.00023715027782600373, 0.0002541367430239916, 0.0002630389353726059, 0.0002518954570405185, 0.0002488081809133291, 0.00026536310906521976, 0.00024260384088847786, 0.0002455382200423628, 0.00025556914624758065, 0.00022725954477209598, 0.0003000398282893002, 0.0002283458161400631, 0.0002379898796789348, 0.00023515101929660887, 0.00025948890834115446, 0.00026636026450432837, 0.0002366613334743306, 0.000246478128246963, 0.00025950136478058994, 0.00024043030862230808, 0.00025169068248942494, 0.00026536319637671113, 0.00023719604359939694, 0.0002496811212040484, 0.00023231006343849003, 0.0002412022149655968, 0.00022404378978535533, 0.00030445316224358976, 0.0002351401635678485, 0.0002629873051773757, 0.00025121564976871014, 0.0002643559128046036, 0.0002611230011098087, 0.00023234447871800512, 0.00029953481862321496, 0.0002605711342766881, 0.00022556610929314047, 0.00019724961020983756, 0.0002298379986314103, 0.00026407657423987985, 0.000269063952146098, 0.0002232720871688798, 0.00024225781089626253, 0.00022511673159897327, 0.0002802094095386565, 0.000264002155745402, 0.0002119581331498921, 0.00023074683849699795, 0.0002436952490825206, 0.0002436013164697215, 0.0002858708321582526, 0.00026739557506516576, 0.00024867680622264743, 0.00022963345691096038, 0.0002666617510840297, 0.00025545700918883085, 0.00024309750006068498, 0.0002695141010917723, 0.00024020184355322272, 0.0002056376833934337, 0.00019353217794559896, 0.00025339017156511545, 0.0002291701384820044, 0.0002530348428990692, 0.00029266279307194054, 0.00025017326697707176, 0.00026624600286595523, 0.0002338586637051776, 0.00023542740382254124, 0.00022265488223638386, 0.0002551001089159399, 0.00027364108245819807, 0.000237023807130754, 0.0002610535884741694, 0.0002657051372807473, 0.00022566443658433855, 0.00027528146165423095, 0.00021650259441230446, 0.00026151491329073906, 0.00022762590378988534, 0.00022120641369838268, 0.00021678248594980687, 0.0002568002382759005, 0.0002369717403780669, 0.00024857858079485595, 0.00027316497289575636, 0.00026279501616954803, 0.00027722460799850523, 0.00025942851789295673, 0.00024265078536700457, 0.00022609252482652664, 0.0002480075054336339, 0.00024139349989127368, 0.0002545091847423464, 0.0002498890389688313, 0.0002459961688145995, 0.000274995865765959, 0.00024208342074416578, 0.00024819676764309406, 0.0002390471490798518, 0.00026471426826901734, 0.00024824138381518424, 0.00020668689103331417, 0.00024110311642289162, 0.0002687915985006839, 0.0002055959339486435, 0.0002506913151592016, 0.0002604143228381872, 0.0002832729951478541, 0.00020750060502905399, 0.00023767755192238837, 0.00024237993056885898, 0.00027916370891034603, 0.00023834338935557753, 0.00022427955991588533, 0.0002570476208347827, 0.0002359611535212025, 0.00024042061704676598, 0.0002832723839674145, 0.0002539668930694461, 0.0002807137498166412, 0.00023376326134894043, 0.00027176266303285956, 0.00029223880846984684, 0.00030803869594819844, 0.00023631923249922693, 0.00024957346613518894, 0.0002424646372674033, 0.0002424433914711699, 0.0002335070603294298, 0.0002260006294818595, 0.00023430422879755497, 0.0002488187747076154, 0.00024845890584401786, 0.00023555458756163716, 0.00024073428357951343, 0.0002435986971249804, 0.0002586790651548654, 0.0002537416876293719, 0.0002291517739649862, 0.00022151958546601236, 0.000259132037172094, 0.0002305761445313692, 0.00025016767904162407, 0.00025812641251832247, 0.0002323270309716463, 0.00024756096536293626, 0.00021504538017325103, 0.0002246447402285412, 0.00024549008230678737, 0.00021626841044053435, 0.0002624725748319179, 0.0002348596026422456, 0.00023362343199551105, 0.0002587237977422774, 0.000244931026827544, 0.00021844774892088026, 0.00022246396110858768, 0.000245466799242422, 0.0002573828387539834, 0.00024665804812684655, 0.0002479955437593162, 0.00022836096468381584, 0.0002259660977870226, 0.00025696924421936274, 0.00023457083443645388, 0.00020589267660398036, 0.000232062884606421, 0.0002305631060153246, 0.00021321751410141587, 0.00024518874124623835, 0.0002266391966259107, 0.0002347488043596968, 0.00024326020502485335, 0.0002854849735740572, 0.0002811722515616566, 0.00023508374579250813, 0.00023450274602510035, 0.0002493392676115036, 0.0002308016992174089, 0.00024780345847830176, 0.00023168980260379612, 0.0002452416520100087, 0.00024496964761056006, 0.00022779006394557655, 0.0002502828720025718, 0.00022253766655921936, 0.0002268430544063449, 0.00023648890783078969, 0.00023909550509415567, 0.00022226250439416617, 0.00023509061429649591, 0.0002399844815954566, 0.0002321441425010562, 0.00022624632401857525, 0.00026524782879278064, 0.00023799578775651753, 0.00026145827723667026, 0.00024855980882421136, 0.0002819012152031064, 0.0002455873473081738, 0.00024389340251218528, 0.0002756070753093809, 0.00021928705973550677, 0.0002649088855832815, 0.00024507948546670377, 0.0002499947149772197, 0.00023574562510475516, 0.00024542436585761607, 0.00024605143698863685, 0.0002968374465126544, 0.00021470640785992146, 0.00026443658862262964, 0.00025900895707309246, 0.0002489227626938373, 0.00022346693731378764, 0.00023336046433541924, 0.00021879789710510522, 0.000232052625506185, 0.0002340987412026152, 0.0002824830589815974, 0.0002214679989265278, 0.000229915720410645, 0.00022911162523087114, 0.00023670843802392483, 0.00024592329282313585, 0.0002424707927275449, 0.0002625339257065207, 0.0002464228600729257, 0.0002410775632597506, 0.0002566696493886411, 0.0002826726995408535, 0.00028568701236508787, 0.00024106921046040952, 0.00025386991910636425, 0.00021154839487280697, 0.00026487797731533647, 0.0003060832095798105, 0.0002224627969553694, 0.00021524258772842586, 0.00023944981512613595, 0.00021979573648422956, 0.00025553550221957266, 0.00028333519003354013, 0.000229509940254502, 0.0002598800929263234, 0.00025770653155632317, 0.00026170563069172204, 0.0002835555060300976, 0.00025294299121014774, 0.0002452968037687242, 0.00025746942264959216, 0.0002269343676744029, 0.00023256301938090473, 0.00026046234415844083, 0.0002380167570663616, 0.00023906538262963295, 0.0002312737487955019, 0.00025067193200811744, 0.00021462868608068675, 0.0002649682865012437, 0.00026132797938771546, 0.00019577439525164664, 0.00027237983886152506, 0.00026001836522482336, 0.00021902176376897842, 0.00022329237253870815, 0.00022452227130997926, 0.0002222142938990146, 0.0002495474473107606, 0.00023585365852341056, 0.0002197747235186398, 0.00026825437089428306, 0.00025148503482341766, 0.00026521668769419193, 0.0002372174058109522, 0.00030221749329939485, 0.0002525228192098439, 0.0002484163851477206, 0.00024103741452563554, 0.0002434824127703905, 0.0002448849845677614, 0.00022393059043679386, 0.0002502243150956929, 0.0002535504463594407, 0.00023311555560212582, 0.0002317361650057137, 0.00027723974199034274, 0.00025708897737786174, 0.00024051243963185698, 0.00022919544426258653, 0.00024355280038435012, 0.00021443945297505707, 0.0002326833491679281, 0.00023193210654426366, 0.00023627333575859666, 0.0002443080593366176, 0.00026199244894087315, 0.00026280261226929724, 0.00024203106295317411, 0.00025038604508154094, 0.00023802966461516917, 0.0002286409289808944, 0.0002341142389923334, 0.00024912896333262324, 0.00025657127844169736, 0.000274786027148366, 0.0002915071090683341, 0.00023935896751936525, 0.00020214829419273883, 0.00025381750310771167, 0.00023818436602596194, 0.0002581393637228757, 0.00021572479454334825, 0.00026403518859297037, 0.0002620222221594304, 0.00021224204101599753, 0.00025126402033492923, 0.00022479948529507965, 0.00020764951477758586, 0.00023443887766916305, 0.00020830870198551565, 0.00028774116071872413, 0.000238102933508344, 0.00029895195621065795, 0.0002629279042594135, 0.0002511049387976527, 0.00022521757637150586, 0.00022402257309295237, 0.00028396514244377613, 0.00022954824089538306, 0.00022234703646972775, 0.0002681521000340581, 0.00025646758149378, 0.00025987508706748486, 0.00025847929646261036, 0.00024116567510645837, 0.0002769063867162913, 0.0002350566501263529, 0.00022671514307148755, 0.000221383452299051, 0.00023141108977142721, 0.00022337048721965402, 0.00027936440892517567, 0.00021788997401017696, 0.0002779126516543329, 0.00019519217312335968, 0.00029458178323693573, 0.00022441263718064874, 0.00027502307784743607, 0.0002608741633594036, 0.0002805728290695697, 0.00021269831631798297, 0.00026772351702675223, 0.0002811476879287511, 0.0002962379076052457, 0.0002516457752790302, 0.00025655736681073904, 0.00022764192544855177, 0.00023957909434102476, 0.00023028482974041253, 0.00023344744113273919, 0.00022911897394806147, 0.00020779714395757765, 0.0002443868143018335, 0.00021766805730294436, 0.00021548834047280252, 0.00023561161651741713, 0.00021903934248257428, 0.0001982672984013334, 0.00025900971377268434, 0.0002506355522200465, 0.00024136470165103674, 0.00025680853286758065, 0.00020957492233719677, 0.00023491730098612607, 0.00026061871903948486, 0.0002353150921408087, 0.00028143945382907987, 0.0002304702647961676, 0.00024361639225389808, 0.00025022184127010405, 0.0002409371081739664, 0.00023038909421302378, 0.0002869895251933485, 0.0002718330069910735, 0.00025967240799218416, 0.00023405869433190674, 0.0002312986907782033, 0.00024108336947392672, 0.0002653674455359578, 0.00027897514519281685, 0.0002643015468493104, 0.0002575905527919531, 0.00023882150708232075, 0.00022968796838540584, 0.0001940263609867543, 0.00022299037664197385, 0.00025058549363166094, 0.00021546841890085489, 0.00023896867060102522, 0.00022669114696327597, 0.00021995179122313857, 0.00023744787904433906, 0.00025917732273228467, 0.00030323854298330843, 0.0002685021609067917, 0.00026802424690686166, 0.00019854794663842767, 0.00022526610700879246, 0.0002569832431618124, 0.0002584784524515271, 0.000229660450713709, 0.00021906929032411426, 0.00025843281764537096, 0.0002508529869373888, 0.00022632459877058864, 0.00026730122044682503, 0.00023678358411416411, 0.0002383338287472725, 0.00025635751080699265, 0.00026047907886095345, 0.00023284211056306958, 0.00021598540479317307, 0.00022799463476985693, 0.0003165020898450166, 0.00025220084353350103, 0.00025226810248568654, 0.00023359335318673402, 0.00021906876645516604, 0.00022194570919964463, 0.00024255285097751766, 0.00021813045896124095, 0.00026486380374990404, 0.0002345749962842092, 0.000272500969003886, 0.00026525690918788314, 0.000255823164479807, 0.00027007318567484617, 0.00027615699218586087, 0.0002297124156029895, 0.00023511583276558667, 0.0002450857136864215, 0.00022551111760549247, 0.00020980427507311106, 0.0002521779097151011, 0.00027451576897874475, 0.00030569511000066996, 0.0002500670379959047, 0.0002451596665196121, 0.000281965738395229, 0.0002546514733694494, 0.0002646987559273839, 0.00023900730593595654, 0.00025680079124867916, 0.0002207600773544982, 0.00020291520922910422, 0.00023268937366083264, 0.000264524482190609, 0.00023456742928829044, 0.00025926463422365487, 0.0002603524480946362, 0.00020773794676642865, 0.00023480525123886764, 0.00023704307386651635, 0.00023678457364439964, 0.0002691076078917831, 0.0002448260202072561, 0.0002824499970301986, 0.00022441172040998936, 0.0002510297344997525, 0.00022404595802072436, 0.00024053643574006855, 0.0001906991092255339, 0.00023067064466886222, 0.00024096979177556932, 0.00019907171372324228, 0.0002376670017838478, 0.0002346809778828174, 0.00025341229047626257, 0.0002642798062879592, 0.0002354345633648336, 0.0002478148671798408, 0.0002497468958608806, 0.00026214911486022174, 0.00026155394152738154, 0.00026236954727210104, 0.00023863476235419512, 0.0002367641864111647, 0.00020540133118629456, 0.00023713927657809108, 0.00024924660101532936, 0.0001937394408741966, 0.00023627863265573978, 0.00023801758652552962, 0.0002483915304765105, 0.00021831007325090468, 0.0002800729125738144, 0.00024879546253941953, 0.00026326646911911666, 0.00023713796690572053, 0.0002609725634101778, 0.0002522844588384032, 0.0002487991005182266, 0.00022678644745610654, 0.0002724082150962204, 0.00021995512361172587, 0.0002744872763287276, 0.00024174222198780626, 0.0002740032214205712, 0.00024785069399513304, 0.00024337583454325795, 0.0002558059641160071, 0.00023963279090821743, 0.00024398256209678948, 0.00023562794376630336, 0.0002553301746957004, 0.0002580881118774414, 0.0002225197822554037, 0.00022321891447063535, 0.00027218449395149946, 0.00025702634593471885, 0.00021788680169265717, 0.00021255432511679828, 0.0002217876462964341, 0.0002743041259236634, 0.00022654148051515222, 0.0002584262110758573, 0.0002441536053083837, 0.0002481670817360282, 0.00024136035062838346, 0.0002275722217746079, 0.00023592937213834375, 0.000267235649516806, 0.0002439392264932394, 0.00023051456082612276, 0.0002516514796297997, 0.00028024878702126443, 0.00022945375530980527, 0.00022466322116088122, 0.00024835497606545687, 0.0002648582449182868, 0.00024358528025913984, 0.00021450681379064918, 0.0002696118026506156, 0.0002315731398994103, 0.00020941566617693752, 0.00020929881429765373, 0.0002254251594422385, 0.0002217290602857247, 0.0002471989137120545, 0.00023730368411634117, 0.00021431026107165962, 0.00023850236902944744, 0.00022899697069078684, 0.0002290665142936632, 0.0002449169405736029, 0.00019932173017878085, 0.00030659858020953834, 0.00027136504650115967, 0.00021987356012687087, 0.000246942276135087, 0.00023114126815926284, 0.00028019535238854587, 0.0002448070445097983, 0.0002469683822710067, 0.00022186241403687745, 0.0002018282248172909, 0.00026273078401573, 0.00022662633273284882, 0.0002620657905936241, 0.00025438686134293675, 0.00023771502310410142, 0.00023245964257512242, 0.00025166707928292453, 0.0002649687812663615, 0.0002687967789825052, 0.00025675760116428137, 0.0002204591000918299, 0.00023245709598995745, 0.00025434952112846076, 0.0002813702158164233, 0.0002410529414191842, 0.0002682737249415368, 0.0002480138500686735, 0.0002505381708033383, 0.000259412860032171, 0.00026723917108029127, 0.00025007041404023767, 0.000251355639193207, 0.00019361654995009303, 0.00023354600125458091, 0.0002165243640774861, 0.0002496795204933733, 0.00025098957121372223, 0.0002475523215252906, 0.00021921272855252028, 0.00024171463155653328, 0.00020763541397172958, 0.0002470597974024713, 0.00022308681218419224, 0.00023759070609230548, 0.0002616263518575579, 0.0002518767141737044, 0.00019338539277669042, 0.0002243267372250557, 0.0002341175131732598, 0.00021137029398232698, 0.00025127932894974947, 0.00025679863756522536, 0.0002388056745985523, 0.0002512404462322593, 0.00025249863392673433, 0.00025952389114536345, 0.0002306609385414049, 0.00024287129053846002, 0.00023085688007995486, 0.00025422259932383895, 0.00022472304408438504, 0.00026877186610363424, 0.00023963522107806057, 0.0002120543213095516, 0.00021959915466140956, 0.0002442064869683236, 0.00025611810269765556, 0.0002473415224812925, 0.0002458824892528355, 0.00024446877068839967, 0.00023342089843936265, 0.00023279112065210938, 0.00020366536045912653, 0.00024836938246153295, 0.0002395253104623407, 0.00026050713495351374, 0.0002513209474273026, 0.0002605044865049422, 0.00025324517628178, 0.0002519582340028137, 0.0002598500868771225, 0.00022292941866908222, 0.00023427593987435102, 0.0002565405739005655, 0.00024519854923710227, 0.00023513294581789523, 0.0001973059552256018, 0.0002738437324296683, 0.0002212405379395932, 0.000298907543765381, 0.0002366530679864809, 0.00027606828371062875, 0.0002682552149053663, 0.00023454244364984334, 0.0002535851381253451, 0.0002295484155183658, 0.0002973696100525558, 0.00024038857372943312, 0.0002262664056615904, 0.00023456854978576303, 0.00024298929201904684, 0.00022197967336978763, 0.00023991904163267463, 0.0002477002562955022, 0.0002549461496528238, 0.0002232565457234159, 0.000242956230067648, 0.00021702500816900283, 0.0002233724226243794, 0.00025077167083509266, 0.00021182782074902207, 0.0002650354290381074, 0.0002540647401474416, 0.00022964591335039586, 0.00024547564680688083, 0.00020275989663787186, 0.00024078486603684723, 0.00026287586661055684, 0.00023802192299626768, 0.00025213646586053073, 0.00022413581609725952, 0.000256494153290987, 0.00023005898401606828, 0.000278874154901132, 0.00023353791038971394, 0.00022671119950246066, 0.0002770379069261253, 0.0002393988979747519, 0.0002165224723285064, 0.0002297433966305107, 0.00024088466307148337, 0.0002863297413568944, 0.00022024163627065718, 0.00025456506409682333, 0.0002572363300714642, 0.00024426367599517107, 0.00023866959963925183, 0.00023997244716156274, 0.00021729344734922051, 0.0002470439358148724, 0.0002406183775747195, 0.00025402577011846006, 0.00024307501735165715, 0.00022550618450623006, 0.00027228108956478536, 0.00023829263227526098, 0.00024397674133069813, 0.00020002074597869068, 0.00021899475541431457, 0.00025476928567513824, 0.0002427060971967876, 0.00022812896349932998, 0.00024212617427110672, 0.00021225769887678325, 0.0002627021458465606, 0.0002587895723991096, 0.0002406163839623332, 0.00024802639381960034, 0.0002694071445148438, 0.00024129980010911822, 0.00020292199042160064, 0.00024646089877933264, 0.0002467866870574653, 0.00024177308660000563, 0.0002216403081547469, 0.0002693635178729892, 0.00022704592265654355, 0.00024187519738916308, 0.00023574885562993586, 0.00023173147928901017, 0.0002792816376313567, 0.0002161313605029136, 0.00028057870804332197, 0.00021556873980443925, 0.00022923244978301227, 0.00025026436196640134, 0.00026008125860244036, 0.0002617503050714731, 0.00023152791254688054, 0.00022126123076304793, 0.000256342493230477, 0.0002780387585517019, 0.00024387374287471175, 0.00024729338474571705, 0.00025591274606995285, 0.00026391565916128457, 0.00025099102640524507, 0.00026934052584692836, 0.0002554102102294564, 0.00022976849868427962, 0.00025476585142314434, 0.0002235475112684071, 0.00024318700889125466, 0.0002619482111185789, 0.0002442966215312481, 0.0002726639504544437, 0.0002240930189145729, 0.0002588616916909814, 0.00026854092720896006, 0.00025866489158943295, 0.0002304480003658682, 0.00024678983027115464, 0.0002745770034380257, 0.00020526784646790475, 0.00022900491603650153, 0.00024940326693467796, 0.0002511773491278291, 0.00026855189935304224, 0.00026971608167514205, 0.0002490457845851779, 0.0002307341346750036, 0.00024741815286688507, 0.00023908480943646282, 0.0002457680820953101, 0.00025661502149887383, 0.00023739869357086718, 0.00021934274991508573, 0.00023328668612521142, 0.00025440024910494685, 0.00024658438633196056, 0.0002257262822240591, 0.0002607830101624131, 0.0002424568374408409, 0.0002545571478549391, 0.0002597649581730366, 0.00023963666171766818, 0.0002444955753162503, 0.0002286312956130132, 0.00023298716405406594, 0.00023522078117821366, 0.00023720051103737205, 0.00028940284391865134, 0.00021560705499723554, 0.00025140444631688297, 0.00026663573225960135, 0.00026114919455721974, 0.00024439027765765786, 0.0002456893853377551, 0.0002354484749957919, 0.00023922548280097544, 0.0002468667516950518, 0.0002442582044750452, 0.00023732428962830454, 0.0002650587121024728, 0.0002955131931230426, 0.0002380971418460831, 0.00026346786762587726, 0.0002466801379341632, 0.0002399188233539462, 0.00021396907686721534, 0.0002179659641114995, 0.00023039492953103036, 0.00023761080228723586, 0.0002192951214965433, 0.0002501063281670213, 0.0002339465427212417, 0.0002175316185457632, 0.00025493657449260354, 0.0002540054847486317, 0.00024540891172364354, 0.0002629759255796671, 0.000215049265534617, 0.00023922788386698812, 0.00025220931274816394, 0.00024579997989349067, 0.00021884117450099438, 0.0002551256911829114, 0.0002638828009366989, 0.00026155399973504245, 0.00024516417761333287, 0.00024163049238268286, 0.00020174497331026942, 0.00027582707116380334, 0.00023102904378902167, 0.00024644084624014795, 0.0002281293855048716, 0.00026383588556200266, 0.00029270906816236675, 0.00026918677031062543, 0.0002622859028633684, 0.0002382123639108613, 0.00024210341507568955, 0.00024266417312901467, 0.0002761056530289352, 0.00023148016771301627, 0.0002610940719023347, 0.0002481607371009886, 0.00023925497953314334, 0.00028412576648406684, 0.0002681624609977007, 0.00025713900686241686, 0.0002656582510098815, 0.0002636210119817406, 0.0002589656214695424, 0.0002459223323967308, 0.0002394307084614411, 0.000261412700638175, 0.00031347310869023204, 0.0002504074655007571, 0.0001881287171272561, 0.00024481117725372314, 0.00029613979859277606, 0.0002565935719758272, 0.00021511802333407104, 0.0002704978396650404, 0.0002470322069711983, 0.0002533783845137805, 0.000237065672990866, 0.00024777677026577294, 0.0002207653596997261, 0.00020207182387821376, 0.0002573160454630852, 0.00019988504936918616, 0.0002193820255342871, 0.0002428384032100439, 0.00025056645972654223, 0.00023832186707295477, 0.0002804236428346485, 0.0002308720286237076, 0.00024875541566871107, 0.0002104011655319482, 0.00022787445050198585, 0.0002351559669477865, 0.00028407451463863254, 0.00022148135758470744, 0.00025421674945391715, 0.00026751114637590945, 0.0002581494627520442, 0.0002819531655404717, 0.00023781989875715226, 0.00025251289480365813, 0.0002507829340174794, 0.0002558673149906099, 0.00024046319595072418, 0.00026713867555372417, 0.0002725822851061821, 0.00024398689856752753, 0.00024356298672500998, 0.00025267721503041685, 0.0002899586979765445, 0.0002352841111132875, 0.0002395385381532833, 0.00022075639571994543, 0.00025174763868562877, 0.00023962132399901748, 0.00022740877466276288, 0.0002380900550633669, 0.00022117182379588485, 0.00026879512006416917, 0.00026982289273291826, 0.000254115933785215, 0.0002549773780629039, 0.0002421010722173378, 0.0002540476561989635, 0.00022529282432515174, 0.00021804473362863064, 0.000270513555733487, 0.00023608081392012537, 0.00024306465638801455, 0.00026475393678992987, 0.00025234397617168725, 0.00020819854398723692, 0.00028053822461515665, 0.0002282716304762289, 0.00020814832532778382, 0.000263622758211568, 0.00023199502902571112, 0.0002501915441825986, 0.0002441426331643015, 0.00024376990040764213, 0.0002469081955496222, 0.000232669961405918, 0.00025412830291315913, 0.0002038108796114102, 0.0002520874550100416, 0.00024205502995755523, 0.00023888907162472606, 0.00024890515487641096, 0.0002555048558861017, 0.0002778381749521941, 0.00020671385573223233, 0.0002565649920143187, 0.00019100062490906566, 0.00023893089382909238, 0.0002683249185793102, 0.0002694212889764458, 0.00021627015667036176, 0.00027202628552913666, 0.00024493085220456123, 0.00022043485660105944, 0.00026371117564849555, 0.00023427896667271852, 0.0002148527419194579, 0.00024258765915874392, 0.0002627022913657129, 0.000257098232395947, 0.000250450917519629, 0.00023567836615256965, 0.0002552329970058054, 0.0002309886331204325, 0.00022943931980989873, 0.0002186578349210322, 0.0002438036462990567, 0.00028230142197571695, 0.00026015102048404515, 0.0002435744827380404, 0.00024539907462894917, 0.00023134287039283663, 0.0002337040496058762, 0.00023295140999834985, 0.0002637715369928628, 0.000244675058638677, 0.0002573078963905573, 0.00024162714544218034, 0.0002635135315358639, 0.0002738438779488206, 0.00024339424271602184, 0.0002481928968336433, 0.0002624811604619026, 0.0002499383990652859, 0.00029285269556567073, 0.00019308453192934394, 0.0002714089641813189, 0.00024925911566242576, 0.00021890818607062101, 0.00028170307632535696, 0.00026481752865947783, 0.0002673179842531681, 0.00028207406285218894, 0.0002575349935796112, 0.0002357985358685255, 0.00023357121972367167, 0.00024475218378938735, 0.000224805815378204, 0.0002874663914553821, 0.0002581470471341163, 0.0002764575183391571, 0.000287312752334401, 0.00025506093516014516, 0.0002464426215738058, 0.00022064997756388038, 0.00024308734282385558, 0.00023679160221945494, 0.0002857159997802228, 0.0002740266209002584, 0.00024539086734876037, 0.0002440169482724741, 0.00022844228078611195, 0.00030733810854144394, 0.0002542455622460693, 0.0002524646115489304, 0.0002277448948007077, 0.0002680926991160959, 0.0002415427443338558, 0.00025939979241229594, 0.0002528585901018232, 0.0002744212979450822, 0.0002631035749800503, 0.0002549353812355548, 0.0002134328824467957, 0.0002706268569454551, 0.00023574614897370338, 0.00024295086041092873, 0.0002647833025548607, 0.0002405608829576522, 0.0002534163650125265, 0.0002589315117802471, 0.0002151841763406992, 0.00021363221458159387, 0.000254326150752604, 0.00024346359714400023, 0.00020042492542415857, 0.00023620530555490404, 0.00025496099260635674, 0.00024294527247548103, 0.0002704327635001391, 0.00021952533279545605, 0.00023954144853632897, 0.0002693761489354074, 0.0002551104116719216, 0.0002598526480142027, 0.00021031255892012268, 0.0002215817803516984, 0.0002690604014787823, 0.00023107283050194383, 0.0002338223421247676, 0.00026445536059327424, 0.00025344063760712743, 0.0002246252988697961, 0.00024244358064606786, 0.00024782124091871083, 0.00024921121075749397, 0.00020605260215234011, 0.00024077804118860513, 0.0002558573614805937, 0.00027345269336365163, 0.00027438317192718387, 0.0002569839416537434, 0.0002570765500422567, 0.00023919048544485122, 0.0002469283645041287, 0.0002606561756692827, 0.00021192437270656228, 0.00022706371964886785, 0.00026322665507905185, 0.0002117169788107276, 0.00026814895682036877, 0.00023747530940454453, 0.0002326301037101075, 0.00027056841645389795, 0.00025772867957130075, 0.000291226664558053, 0.00022573844762519002, 0.00021348577865865082, 0.0002460925607010722, 0.0002317472972208634, 0.00023377695470117033, 0.00026222865562886, 0.00023841630900278687, 0.00026124718715436757, 0.00022943671501707286, 0.0002459239331074059, 0.00024310403387062252, 0.00025428360095247626, 0.00022918118338566273, 0.00022438698215410113, 0.00023241675808094442, 0.0002450105967000127, 0.0002329280396224931, 0.0002467053127475083, 0.0002592609089333564, 0.00021310342708602548, 0.00022812493261881173, 0.00023275161220226437, 0.00020702082838397473, 0.00023792809224687517, 0.00024592544650658965, 0.00026355445152148604, 0.0002511490019969642, 0.0002550192584749311, 0.00022060336777940392, 0.00022790288494434208, 0.00025044349604286253, 0.0002819782239384949, 0.00025130051653832197, 0.00023576158855576068, 0.00027481443248689175, 0.0002936112286988646, 0.000222983377170749, 0.00022236433869693428, 0.00025649042800068855, 0.00021480719442479312, 0.00026586439344100654, 0.00023767678067088127, 0.0002432506444165483, 0.0002464946883264929, 0.00022179560619406402, 0.0002498346730135381, 0.00023319729370996356, 0.00022408846416510642, 0.00023397713084705174, 0.0002312225551577285, 0.0002375673211645335, 0.00022036109294276685, 0.00026281437021680176, 0.00021203840151429176, 0.0002746088139247149, 0.0002395155606791377, 0.00021069431386422366, 0.00022069999249652028, 0.00023625741596333683, 0.00021938257850706577, 0.0002597163838800043, 0.00021620649204123765, 0.00020767426758538932, 0.0002382867387495935, 0.0002336122706765309, 0.00024729390861466527, 0.00029353430727496743, 0.0002051116753136739, 0.00023254619736690074, 0.00023726755171082914, 0.0002762849326245487, 0.000208077413844876, 0.0002648344961926341, 0.0002859338419511914, 0.00023293300182558596, 0.00023755889560561627, 0.0002451299806125462, 0.00023718552256468683, 0.00022164183610584587, 0.0002614730328787118, 0.00023272880935110152, 0.00021255514002405107, 0.00025708600878715515, 0.00027627460076473653, 0.0002098444674629718, 0.00020279029558878392, 0.00024683313677087426, 0.0002342538646189496, 0.00021300173830240965, 0.00028723469586111605, 0.00022486601665150374, 0.0002581874723546207, 0.0002998806594405323, 0.00024090759688988328, 0.00026113595231436193, 0.00028249830938875675, 0.00021364122221712023, 0.0002505520242266357, 0.0002476035733707249, 0.0002424879203317687, 0.0002740895433817059, 0.0002467527810949832, 0.00024203183420468122, 0.0002513188519515097, 0.00025942016509361565, 0.0002371400478295982, 0.00026145606534555554, 0.00022320603602565825, 0.0002337294426979497, 0.00022519360936712474, 0.00024502884480170906, 0.00025508200633339584, 0.00023532987688668072, 0.00023846732801757753, 0.00023977596720214933, 0.00024520393344573677, 0.00021675502648577094, 0.0002480847469996661, 0.00025968989939428866, 0.00024387276789639145, 0.0002105153544107452, 0.00025305929011665285, 0.00022720301058143377, 0.00023855514882598072, 0.00022986438125371933, 0.0002485330624040216, 0.00019741298456210643, 0.00023506952857133, 0.00021472651860676706, 0.00028226361609995365, 0.00024012643552850932, 0.00020729405514430255, 0.0002654721902217716, 0.0002321280335308984, 0.00022064348740968853, 0.0002682127524167299, 0.00024365491117350757, 0.00025113296578638256, 0.00021418668620754033, 0.00025637669023126364, 0.0002630986855365336, 0.00026290942332707345, 0.00022299216652754694, 0.00026225397596135736, 0.00030295338365249336, 0.00024361156101804227, 0.00027132179820910096, 0.0002779357600957155, 0.00025430897949263453, 0.0002429444284643978, 0.0002686988445930183, 0.00023933814372867346, 0.0002610711962915957, 0.00023357546888291836, 0.0002669719106052071, 0.00023790243722032756, 0.00024130962265189737, 0.00023733718262519687, 0.0002270919649163261, 0.000254081969615072, 0.00024598537129350007, 0.0002277129824506119, 0.00025046904920600355, 0.0002672251430340111, 0.00025029428070411086, 0.00023439561482518911, 0.00029662015731446445, 0.00022136677580419928, 0.0002590494405012578, 0.00023010624863673002, 0.0002680097531992942, 0.00021630529954563826, 0.0003076758002862334, 0.0002217656874563545, 0.0002724422956816852, 0.00023512137704528868, 0.00024231457791756839, 0.00023938901722431183, 0.0002288908581249416, 0.0002655638672877103, 0.000336341792717576, 0.00024016837414819747, 0.00022224079293664545, 0.0002017458900809288, 0.000251662713708356, 0.0002727373503148556, 0.0002202260511694476, 0.00026265159249305725, 0.00024254059826489538, 0.00023080753453541547, 0.0002410465822322294, 0.00026250045630149543, 0.00026034165057353675, 0.0002704289508983493, 0.00022599963995162398, 0.0002548636693973094, 0.00021177096641622484, 0.00024933365057222545, 0.00025620730593800545, 0.0002537671534810215, 0.0002302853245055303, 0.0002330824499949813, 0.00021911333897151053, 0.0002324300294276327, 0.00020927292644046247, 0.0002306341630173847, 0.00024912855587899685, 0.00029949701274745166, 0.0002448312588967383, 0.00024598187883384526, 0.0002182994649047032, 0.00026789159164763987, 0.0002632531977724284, 0.00024940920411609113, 0.00022851431276649237, 0.00023366397363133729, 0.0002212509571108967, 0.0001934902829816565, 0.00022236580844037235, 0.0002714064612518996, 0.00023125781444832683, 0.00024240514903794974, 0.00025139792705886066, 0.00021758946240879595, 0.00026896127383224666, 0.00023842263908591121, 0.0002668000233825296, 0.0002713827707339078, 0.0002667322405613959, 0.00028669589664787054, 0.0002478242095094174, 0.0002821116941049695, 0.0002478470851201564, 0.00024721387308090925, 0.00022370200895238668, 0.0002685929648578167, 0.0002282958012074232, 0.00022241756960283965, 0.00028115478926338255, 0.0002314449957339093, 0.00023169457563199103, 0.0002165548939956352, 0.00022540471400134265, 0.0002404895203653723, 0.00027219264302402735, 0.0002162648452213034, 0.00024218401813413948, 0.0002526991884224117, 0.00023503559350501746, 0.00023953458003234118, 0.0002518353285267949, 0.0002280230401083827, 0.00023112291819415987, 0.00023463381512556225, 0.00022949487902224064, 0.000263088324572891, 0.00021286422270350158, 0.00022412567341234535, 0.00024609401589259505, 0.00023352514836005867, 0.00024397007655352354, 0.0002250035759061575, 0.0002575679973233491, 0.000218907356611453, 0.0002552939986344427, 0.0002088279725285247, 0.0002270788827445358, 0.00022622552933171391, 0.00024312708410434425, 0.00023162899015005678, 0.0002499897964298725, 0.00024355805362574756, 0.00025810973602347076, 0.00024551129899919033, 0.0002826997369993478, 0.0002220142778242007, 0.00027935081743635237, 0.00028027224470861256, 0.00022206261928658932, 0.00022558389173354954, 0.0002998983836732805, 0.00022563443053513765, 0.00021198623289819807, 0.00021783955162391067, 0.0002464106073603034, 0.0002452880435157567, 0.0002022534899879247, 0.00024970737285912037, 0.00026415978209115565, 0.0002702003112062812, 0.00025529085542075336, 0.000237040119827725, 0.000257272447925061, 0.00023042345128487796, 0.00025789474602788687, 0.0003135939477942884, 0.00020639017748180777, 0.0002738673356361687, 0.00024238920013885945, 0.00022714896476827562, 0.00028263547574169934, 0.00023577825049869716, 0.0002235616702819243, 0.00026946087018586695, 0.0002630985982250422, 0.00027287821285426617, 0.0002781029907055199, 0.00022861665638629347, 0.0002480330877006054, 0.000252799189183861, 0.00022998069471213967, 0.00022225978318601847, 0.0002700024633668363, 0.0002536236133892089, 0.00028026814106851816, 0.00021984090562909842, 0.0002796939224936068, 0.00023816392058506608, 0.0002392808673903346, 0.0002312864817213267, 0.00026954422355629504, 0.00023729295935481787, 0.00024986884091049433, 0.00019297393737360835, 0.00021219986956566572, 0.00023315075668506324, 0.0002664995554368943, 0.0002541565918363631, 0.0002771969884634018, 0.0002543221053201705, 0.00024539511650800705, 0.0002571944787632674, 0.00024401202972512692, 0.00022858729062136263, 0.00027711447910405695, 0.00023483484983444214, 0.00023630729992873967, 0.0002382501115789637, 0.00022103218361735344, 0.0002748493861872703, 0.0002779708884190768, 0.0002139379212167114, 0.0002167948114220053, 0.00023709620290901512, 0.00019971885194536299, 0.0002484811411704868, 0.00025040129548870027, 0.0002382817183388397, 0.00025491492124274373, 0.0002544643066357821, 0.00022764121240470558, 0.0002398952201474458, 0.00023188730119727552, 0.0002371602167841047, 0.00022322547738440335, 0.000230467485380359, 0.0002499922411516309, 0.00020569446496665478, 0.00025473604910075665, 0.0002540904679335654, 0.0002396373456576839, 0.0002833616454154253, 0.00024281359219457954, 0.00022871828696224838, 0.0002703974023461342, 0.00022157123021315783, 0.00023634402896277606, 0.00022813329997006804, 0.00021321249369066209, 0.0002926554880104959, 0.0002477779344189912, 0.0002391053130850196, 0.00024742932873778045, 0.0002563353336881846, 0.00022888911189511418, 0.00022494533914141357, 0.0002445183927193284, 0.0002703456557355821, 0.000251767341978848, 0.00023950744071044028, 0.00027654142468236387, 0.000252501864451915, 0.00025880380417220294, 0.00024145595671143383, 0.0002842080721165985, 0.00023939112725201994, 0.00022619852097705007, 0.00024466606555506587, 0.00029588487814180553, 0.0002238705346826464, 0.000268401054199785, 0.00028282642597332597, 0.0002691719855647534, 0.00028394462424330413, 0.000246397772571072, 0.00028628381551243365, 0.00024505870533175766, 0.0002359529462410137, 0.0002369158755755052, 0.00022378290304914117, 0.0002751215361058712, 0.00025500080664642155, 0.00024378523812629282, 0.00024232348368968815, 0.0002516528475098312, 0.00023337082529906183, 0.0002579904394224286, 0.00025612712488509715, 0.0002564234018791467, 0.0002636489225551486, 0.00027893856167793274, 0.00023678000434301794, 0.0002363335806876421, 0.00019411694665905088, 0.0002496244269423187, 0.00025234182248823345, 0.00022846361389383674, 0.0002462911361362785, 0.00027546248747967184, 0.0002302347420481965, 0.0002232194965472445, 0.0002709527325350791, 0.00023454762413166463, 0.0002862139663193375, 0.0002613370306789875, 0.00024545236374251544, 0.00024218075850512832, 0.000258809159277007, 0.00022308831103146076, 0.00022771445219404995, 0.00021821088739670813, 0.00023731880355626345, 0.00024283035600092262, 0.00023958964447956532, 0.00024119345471262932, 0.0002531565842218697, 0.00022067816462367773, 0.0002573750098235905, 0.0002222405601060018, 0.0002452629851177335, 0.0002567757910583168, 0.00023445190163329244, 0.0002049512113444507, 0.0002296703023603186, 0.00018972218094859272, 0.0002308581752004102, 0.0002803773386403918, 0.0002261330228066072, 0.00022614629415329546, 0.0002494468353688717, 0.0002133280213456601, 0.0002006122376769781, 0.00021499804279301316, 0.00021464491146616638, 0.00023319601314142346, 0.00025505435769446194, 0.00024514133110642433, 0.00024290703004226089, 0.00024182764173019677, 0.0002438423689454794, 0.00023901417443994433, 0.00026490268646739423, 0.00027119662263430655, 0.00024375642533414066, 0.0002554968523327261, 0.00021588140225503594, 0.0002652392431627959, 0.0002680285251699388, 0.0002559433050919324, 0.0002421627868898213, 0.000238848922890611, 0.00022069006809033453, 0.00021538067085202783, 0.00022133676975499839, 0.00023581201094202697, 0.00020986988965887576, 0.0002849228330887854, 0.000241243455093354, 0.00028335131355561316, 0.00024323548132088035, 0.00024103678879328072, 0.00023672136012464762, 0.00023542920826002955, 0.00021357515652198344, 0.00022547425760421902, 0.0002463831042405218, 0.00024996345746330917, 0.00024351080355700105, 0.0002184421755373478, 0.0002032539196079597, 0.0002717835013754666, 0.00022646757133770734, 0.00026203671586699784, 0.00021614719298668206, 0.00023727027291897684, 0.0002538871194701642, 0.00024352270702365786, 0.00026822229847311974, 0.0002501869748812169, 0.00025356170954182744, 0.00024096935521811247, 0.0002484233118593693, 0.0002171622618334368, 0.00024171041150111705, 0.00021957943681627512, 0.0002277083694934845, 0.00023701692407485098, 0.0002500161062926054, 0.00024376170767936856, 0.00025165872648358345, 0.0002194469270762056, 0.00026799621991813183, 0.0002278251777170226, 0.0002123707818100229, 0.0002839821972884238, 0.0002506446617189795, 0.00022837630240246654, 0.00024355773348361254, 0.00024726014817133546, 0.00024562308681197464, 0.00021649627888109535, 0.00024165230570361018, 0.0002556085819378495, 0.00027800467796623707, 0.00023920890816953033, 0.0002351271250518039, 0.00025824946351349354, 0.0002341179206268862, 0.00023817081819288433, 0.0002471475745551288, 0.00023325119400396943, 0.00020861119264736772, 0.00023776596935931593, 0.000254190934356302, 0.0002653527772054076, 0.00023579230764880776, 0.00024026834580581635, 0.00021956917771603912, 0.0002264232753077522, 0.0002583116292953491, 0.00022786254703532904, 0.00030136946588754654, 0.0003022809396497905, 0.00021651515271514654, 0.00025074262521229684, 0.0002799264038912952, 0.0002452586777508259, 0.0002219000889454037, 0.00023548112949356437, 0.000260833214269951, 0.00024776984355412424, 0.0002467541489750147, 0.00024224533990491182, 0.00024861624115146697, 0.00020165889873169363, 0.000240541688981466, 0.00023234541004057974, 0.00026542937848716974, 0.0003088208322878927, 0.00021775091590825468, 0.00023662425519432873, 0.0002515497035346925, 0.0002723033248912543, 0.0002661011239979416, 0.00024149562523234636, 0.00022192357573658228, 0.0002654705021996051, 0.0002413222100585699, 0.00023044765111990273, 0.00025367815396748483, 0.00026111939223483205, 0.000279914092971012, 0.00022396173153538257, 0.00024569715606048703, 0.0002847560681402683, 0.0002483529970049858, 0.00024832470808178186, 0.00024781996035017073, 0.00025238958187401295, 0.00026178386178798974, 0.0002399295335635543, 0.00023585114104207605, 0.00027069333009421825, 0.00024986168136820197, 0.0002400832890998572, 0.00023829654674045742, 0.0002406631101621315, 0.00024105372722260654, 0.00023280469758901745, 0.00024259135534521192, 0.0002446153375785798, 0.000253696198342368, 0.000218270331970416, 0.0002737240574788302, 0.00021118108998052776, 0.00023718805459793657, 0.00026677967980504036, 0.00023100912221707404, 0.00024041514552664012, 0.0002762501535471529, 0.00023319835599977523, 0.00020761866471730173, 0.00021626328816637397, 0.00024214886070694774, 0.00027449295157566667, 0.0002544436720199883, 0.00022795637778472155, 0.00023869550204835832, 0.000229675744776614, 0.0002915561490226537, 0.0002712646091822535, 0.00025181256933137774, 0.00023575570958200842, 0.0002325097593711689, 0.00020907065481878817, 0.00023831606085877866, 0.0002807746350299567, 0.00021770966122858226, 0.00028139998903498054, 0.00026526872534304857, 0.0002697002491913736, 0.00028032189584337175, 0.000230209348956123, 0.00023694954870734364, 0.00023851454898249358, 0.00021894791279919446, 0.00023614594829268754, 0.00025715891388244927, 0.00018615880981087685, 0.00022985151736065745, 0.00021867708710487932, 0.00023254209372680634, 0.00026260691811330616, 0.00018866959726437926, 0.00021204541553743184, 0.0002682092017494142, 0.00022729116608388722, 0.0003008521453011781, 0.00022846525826025754, 0.000260591390542686, 0.0002473025524523109, 0.000236448337091133, 0.00019716631504707038, 0.0002186545607401058, 0.00021565445058513433, 0.00022774293029215187, 0.00025806413032114506, 0.00023924831475596875, 0.00023714543203823268, 0.0002395441842963919, 0.00026784822694025934, 0.0002508467296138406, 0.00021535615087486804, 0.00023077958030626178, 0.00024643453070893884, 0.00023058090300764889, 0.0002579186111688614, 0.00023741356562823057, 0.00020731185213662684, 0.0002489516045898199, 0.00021808074961882085, 0.00024124585615936667, 0.00024080040748231113, 0.0002436507202219218, 0.00026040824013762176, 0.00024538973229937255, 0.00026295322459191084, 0.0002486283774487674, 0.00024568915250711143, 0.0002623919863253832, 0.00024204602232202888, 0.00023521338880527765, 0.00023350675473921, 0.0002526265452615917, 0.00026705084019340575, 0.000244260038016364, 0.0002467776066623628, 0.00024484749883413315, 0.00021403726714197546, 0.00027004393632523715, 0.0002294702862855047, 0.00018423149595037103, 0.00023331640113610774, 0.000259216787526384, 0.00025813482352532446, 0.00025684156571514904, 0.00022622235701419413, 0.00022860731405671686, 0.00024259756901301444, 0.00022444709611590952, 0.00027629820397123694, 0.0002272994170198217, 0.0002456442452967167, 0.00024223412037827075, 0.00021695674513466656, 0.00025576737243682146, 0.0001992532634176314, 0.00027339550433680415, 0.00022199031081981957, 0.0002655916032381356, 0.0002565014874562621, 0.00025159461074508727, 0.0002480827970430255, 0.00022591637389268726, 0.0002138090640073642, 0.00027100759325549006, 0.000253274425631389, 0.0002392831665929407, 0.00025143875973299146, 0.00024304819817189127, 0.00024541979655623436, 0.00022278129472397268, 0.0002535992534831166, 0.00023740949109196663, 0.00023627174959983677, 0.0002747086400631815, 0.0002408274303888902, 0.0002536911051720381, 0.00025109818670898676, 0.0002351466828258708, 0.00026260552112944424, 0.00024586537620052695, 0.00024756399216130376, 0.00027776751085184515, 0.00024388614110648632, 0.0002510020276531577, 0.00022671122860629112, 0.00021094482508488, 0.00023524228890892118, 0.0002365369873587042, 0.000245713978074491, 0.00021635147277265787, 0.0002664633502718061, 0.00024072607629932463, 0.00022115383762866259, 0.0002553846570663154, 0.00025756051763892174, 0.00024764708359725773, 0.0001951953599927947, 0.00023489868908654898, 0.0002694255381356925, 0.00025122868828475475, 0.0002835954655893147, 0.0002229425881523639, 0.0002384255494689569, 0.00025123762316070497, 0.0002128201158484444, 0.0002796705812215805, 0.00024145674251485616, 0.00024693176965229213, 0.0002193880791310221, 0.00025769410422071815, 0.0002141279837815091, 0.00021869834745302796, 0.000255250051850453, 0.000260464905295521, 0.0002568838535808027, 0.00023429161228705198, 0.0002485856239218265, 0.0002284494403284043, 0.00021483215095940977, 0.00020731495169457048, 0.00024190642579924315, 0.00027607273659668863, 0.00024354377819690853, 0.00025661647669039667, 0.0002843256515916437, 0.00021407865278888494, 0.00027130034868605435, 0.0002643346961122006, 0.0002816784253809601, 0.000280078238574788, 0.0002338724589208141, 0.0002467594458721578, 0.00025106812245212495, 0.0002206978533649817, 0.00021598637977149338, 0.0002302412613062188, 0.0002496030065231025, 0.0002464299905113876, 0.00023845949908718467, 0.000268878968199715, 0.00026789659750647843, 0.00024173638666979969, 0.00027304922696202993, 0.00025270559126511216, 0.0002507931785658002, 0.0002212481340393424, 0.00026768186944536865, 0.00022269386681728065, 0.00023924693232402205, 0.0002576668339315802, 0.0002431076718494296, 0.00022961631475482136, 0.00022603129036724567, 0.0002687934320420027, 0.00024185798247344792, 0.00021728222782257944, 0.0002412772155366838, 0.00021955001284368336, 0.00023772186250425875, 0.00028047244995832443, 0.0002414383925497532, 0.00022845286002848297, 0.00023765252262819558, 0.0002486649318598211, 0.00021846966410521418, 0.00023940479150041938, 0.0002550050849094987, 0.0002484752330929041, 0.00021052527881693095, 0.00020926880824845284, 0.00025427050422877073, 0.00018486316548660398, 0.0002828440337907523, 0.00024190809926949441, 0.00022786470071878284, 0.0002565695031080395, 0.0002517713000997901, 0.00026760768378153443, 0.00024273466260638088, 0.00024258997291326523, 0.0002356995246373117, 0.00024627105449326336, 0.00028228407609276474, 0.0002759041090030223, 0.00025421619648113847, 0.0002783392264973372, 0.000231821890338324, 0.00024411540653090924, 0.0002393784379819408, 0.0002521403366699815, 0.00019136247283313423, 0.00023866145056672394, 0.00025048377574421465, 0.0002627900103107095, 0.0002489170874468982, 0.00024022557772696018, 0.0002480519760865718, 0.00022245896980166435, 0.00023415613395627588, 0.00025214097695425153, 0.00024416029918938875, 0.0002593306708149612, 0.00021391311020124704, 0.00021875742822885513, 0.0002685110957827419, 0.00026842817896977067, 0.0002279748150613159, 0.0002408507716609165, 0.00024655842571519315, 0.00023778423201292753, 0.00021305741393007338, 0.00024409759498666972, 0.00024399958783760667, 0.0002687718952074647, 0.00020364321244414896, 0.00026732651167549193, 0.0001978620421141386, 0.00026307249208912253, 0.00024555064737796783, 0.00020274342386983335, 0.00024415459483861923, 0.00025657928199507296, 0.000257409003097564, 0.00024405871226917952, 0.0002703092759475112, 0.00023807515390217304, 0.00022517924662679434, 0.00024699646746739745, 0.00021285281400196254, 0.00024173517886083573, 0.0002545334864407778, 0.00021578895393759012, 0.00024590635439381003, 0.00023586081806570292, 0.0002441433898638934, 0.00023459132353309542, 0.00024282281810883433, 0.00025072088465094566, 0.0002563288726378232, 0.00028035606374032795, 0.00020524203137028962, 0.00021371606271713972, 0.00023632077500224113, 0.00024390641192439944, 0.00024478198611177504, 0.00022308775805868208, 0.00020807614782825112, 0.00027336759376339614, 0.0002676787553355098, 0.00023765272635500878, 0.00027156819123774767, 0.0002322750078747049, 0.00024688945268280804, 0.00022831675596535206, 0.00026630956563167274, 0.0002335663011763245, 0.0002141513250535354, 0.00021304194524418563, 0.00023891482851468027, 0.0002218285226263106, 0.00023912731558084488, 0.00025017347070388496, 0.0002433336339890957, 0.0002767059195321053, 0.00026126246666535735, 0.0002023307461058721, 0.00024230671988334507, 0.0002516229869797826, 0.00022240003454498947, 0.00024491947260685265, 0.00023137293464969844, 0.00027782400138676167, 0.00023980280093383044, 0.0002535515232011676, 0.00027452968060970306, 0.00027047996991313994, 0.00021249300334602594, 0.00025626400019973516, 0.0002308833645656705, 0.00028158046188764274, 0.00028283335268497467, 0.00026310226530767977, 0.000253862701356411, 0.00023567733296658844, 0.00029382415232248604, 0.0002986010513268411, 0.0002502133429516107, 0.00026310060638934374, 0.00026402753428556025, 0.00021886169270146638, 0.00021589946118183434, 0.0002584977773949504, 0.0002793159510474652, 0.00021752086468040943, 0.0002311842836206779, 0.000261532433796674, 0.0002479584072716534, 0.00022555091709364206, 0.00024037346884142607, 0.00024079922877717763, 0.00025338176055811346, 0.00023926061112433672, 0.00024125045456457883, 0.00020608471822924912, 0.00022712927602697164, 0.00021367585577536374, 0.0002656300493981689, 0.00025439413730055094, 0.0002220583992311731, 0.0002442014520056546, 0.00025713484501466155, 0.00026103094569407403, 0.00024696651962585747, 0.00021106591157149523, 0.0002122618752764538, 0.00021897914120927453, 0.00026636701659299433, 0.0002874723286367953, 0.00024364577257074416, 0.00023236425477080047, 0.0002527178730815649, 0.000295952515443787, 0.00025017428561113775, 0.0002467600570525974, 0.00022768627968616784, 0.00020492081239353865, 0.00020678773580584675, 0.0002764880482573062, 0.00027718563796952367, 0.0002259222383145243, 0.00023579456319566816, 0.0002591535449028015, 0.00024508245405741036, 0.00021582352928817272, 0.00022822975006420165, 0.00023435043112840503, 0.0002571063523646444, 0.00024782668333500624, 0.00026306070503778756, 0.00023847250849939883, 0.0002508684410713613, 0.00022623452241532505, 0.0002731991990003735, 0.00024829022004269063, 0.00022645397984888405, 0.000263986672507599, 0.00024127832148224115, 0.00021856983948964626, 0.0002310162817593664, 0.00022831586829852313, 0.00028604528051801026, 0.0002684973005671054, 0.0002514398074708879, 0.00025484958314336836, 0.0002167207858292386, 0.00025364276370964944, 0.0002485755831003189, 0.0002493465435691178, 0.00026012182934209704, 0.0002412941394140944, 0.00024711459991522133, 0.00024696861510165036, 0.0002627297362778336, 0.0003057047724723816, 0.0002261250774608925, 0.00023341257474385202, 0.0002536853135097772, 0.0002684430219233036, 0.0002418670483166352, 0.0002448908635415137, 0.00026361687923781574, 0.00021288472635205835, 0.0002443375997245312, 0.00022108270786702633, 0.00026532873744145036, 0.0003151361597701907, 0.0002262948255520314, 0.00022161849483381957, 0.00023681556922383606, 0.0002136754774255678, 0.00023162683646660298, 0.00023702277394477278, 0.00022604693367611617, 0.0002602381573524326, 0.00023483656696043909, 0.00024133433180395514, 0.0002569734933786094, 0.00024731800658628345, 0.00020860621589235961, 0.00027664395747706294, 0.00021371392358560115, 0.0002635825367178768, 0.00023192788648884743, 0.00022762789740227163, 0.0002238816232420504, 0.0002823237737175077, 0.0002795895270537585, 0.00026223171153105795, 0.00021606874361168593, 0.0002458383678458631, 0.00023528322344645858, 0.00022654078202322125, 0.00023220013827085495, 0.0002428827283438295, 0.0002524114097468555, 0.00022914147120900452, 0.00026345031801611185, 0.0002443412959109992, 0.00020479322120081633, 0.00024363330157939345, 0.00023495433561038226, 0.0002574870304670185, 0.00022171364980749786, 0.0002327398251509294, 0.0002972443762701005, 0.0002683224156498909, 0.00023737804440315813, 0.0002303842775290832, 0.0002843255642801523, 0.0002268507523695007, 0.00023376187891699374, 0.00027589721139520407, 0.00023221420997288078, 0.00026232004165649414, 0.0002493092615623027, 0.0002824732509907335, 0.0002467515005264431, 0.0002626188797876239, 0.00019450315448921174, 0.000261352164670825, 0.00023182995209936053, 0.00024378448142670095, 0.00021259053028188646, 0.00023874291218817234, 0.0002114749513566494, 0.00022128228738438338, 0.0002209020167356357, 0.00026461490779183805, 0.0002668543311301619, 0.00022684538271278143, 0.00020638975547626615, 0.0002858694060705602, 0.00023738610616419464, 0.00022267833992373198, 0.00027007117751054466, 0.00023732495901640505, 0.0002219663729192689, 0.00023582558787893504, 0.0002582106681074947, 0.00023176072863861918, 0.00022070178238209337, 0.0002597137645352632, 0.0002656163997016847, 0.0002047637535724789, 0.0002555625978857279, 0.00023896228231023997, 0.00020876603957731277, 0.00022957893088459969, 0.0002601287851575762, 0.00023406511172652245, 0.00025091844145208597, 0.00020372789003886282, 0.00022709459881298244, 0.00025894827558659017, 0.0002486841403879225, 0.00023786406381987035, 0.00024170179676730186, 0.00022474878642242402, 0.00024653482250869274, 0.00023943059204611927, 0.0002465927100274712, 0.00023668854555580765, 0.0002603988104965538, 0.0002797359775286168, 0.00023575966770295054, 0.0002881293767131865, 0.00024533888790756464, 0.000216269152588211, 0.00023139007680583745, 0.0002506434393581003, 0.00022194650955498219, 0.00022383451869245619, 0.00022354719112627208, 0.00024371191102545708, 0.00020340255287010223, 0.0002466432342771441, 0.00023247780336532742, 0.0002485942095518112, 0.00024754254263825715, 0.00022919350885786116, 0.00029772717971354723, 0.00024224612570833415, 0.00022975601314101368, 0.000244211609242484, 0.00024597032461315393, 0.0002632024115882814, 0.00026612344663590193, 0.00031627030693925917, 0.0002424539707135409, 0.00024844094878062606, 0.00025791380903683603, 0.0002593167300801724, 0.00024584101629443467, 0.00023949680326040834, 0.00022066252131480724, 0.0002624284534249455, 0.00023662651074118912, 0.000231862926739268, 0.00025129152345471084, 0.0002743316872511059, 0.0002541304857004434, 0.00019951467402279377, 0.0002218346344307065, 0.0002237647131551057, 0.0002304292138433084, 0.0002569603093434125, 0.00025900889886543155, 0.00025911498232744634, 0.0002602220338303596, 0.0002454063796903938, 0.0002467057784087956, 0.00022855459246784449, 0.00025211169850081205, 0.00025820074370130897, 0.00027436489472165704, 0.0002419500524410978, 0.00026363079086877406, 0.0002923430292867124, 0.0002449130406603217, 0.0002599534345790744, 0.0002538624103181064, 0.0002114820817951113, 0.00021995275164954364, 0.00022084945521783084, 0.00024127817596308887, 0.00022067708778195083, 0.00021982246835250407, 0.0002381675731157884, 0.00021694517636206, 0.00026788582908920944, 0.0002629189984872937, 0.0002714468282647431, 0.00024698508786968887, 0.000271685334155336, 0.00021683011436834931, 0.0002393319591647014, 0.00023075309582054615, 0.00023510474420618266, 0.0002687785599846393, 0.00023351289564743638, 0.000264056958258152, 0.00026585187879391015, 0.00026034857728518546, 0.00021933953394182026, 0.00023772020358592272, 0.00022278568940237164, 0.00025148707209154963, 0.0002234991843579337, 0.0002304989902768284, 0.00024570446112193167, 0.00024099642178043723, 0.00021834750077687204, 0.0002546226605772972, 0.0002741858479566872, 0.00024544907500967383, 0.00021149269014131278, 0.0002531584177631885, 0.00026431813603267074, 0.0002713203430175781, 0.0002292002027388662, 0.00021455991372931749, 0.0002345873072044924, 0.0002762643271125853, 0.000244742666836828, 0.00027089257491752505, 0.0002492025960236788, 0.00022063571668695658, 0.0002509099431335926, 0.00023930672614369541, 0.00024270896392408758, 0.00022966523829381913, 0.00023109793255571276, 0.00022281894052866846, 0.00022717102547176182, 0.0002478781098034233, 0.00023422253434546292, 0.0002449768944643438, 0.00021416055096779019, 0.00023294634593185037, 0.00022165346308611333, 0.00020914892957080156, 0.00025579429347999394, 0.0002310108975507319, 0.0002538283879403025, 0.00024322510580532253, 0.00020745866640936583, 0.00023579428670927882, 0.00023788565886206925, 0.00025262904819101095, 0.0002539029810577631, 0.00023510534083470702, 0.0002587616036180407, 0.0002630137314554304, 0.0002822116366587579, 0.0002845261769834906, 0.000265290291281417, 0.00021556715364567935, 0.00024856330128386617, 0.0002870789903681725, 0.00022838528093416244, 0.0002378542849328369, 0.00025849699159152806, 0.00020842067897319794, 0.0002516856766305864, 0.00022971742146182805, 0.0002896336663980037, 0.000241930247284472, 0.0002592695236671716, 0.0002737629401963204, 0.0002354508324060589, 0.0002637433644849807, 0.0002470817125868052, 0.00024405076692346483, 0.0002470195759087801, 0.0002799908397719264, 0.0003016927221324295, 0.0002743972872849554, 0.00021349143935367465, 0.0002761890063993633, 0.00021216925233602524, 0.00022670632461085916, 0.0002503357536625117, 0.0002549795899540186, 0.00025275402003899217, 0.00026300165336579084, 0.00022706156596541405, 0.00027884638984687626, 0.000270173855824396, 0.0002652794064488262, 0.00021765436395071447, 0.00021346409630496055, 0.0002565342583693564, 0.000295161793474108, 0.00022916914895176888, 0.00023622291337233037, 0.00022790305956732482, 0.00021486238983925432, 0.00028331088833510876, 0.00022258890385273844, 0.000295011792331934, 0.0002693043788895011, 0.00020900426898151636, 0.00025075682788155973, 0.00020916369976475835, 0.0002458440139889717, 0.000263258843915537, 0.00023838963534217328, 0.00019359213183633983, 0.00022247934248298407, 0.0002465755969751626, 0.0002431992907077074, 0.00025768514024093747, 0.0002010143070947379, 0.00022116873878985643, 0.00023326977679971606, 0.00022596387134399265, 0.00023441090888809413, 0.0002838598156813532, 0.0002724903461057693, 0.00020960834808647633, 0.00025640762760303915, 0.0002721930213738233, 0.0002072936767945066, 0.0002536625834181905, 0.00019802483438979834, 0.00026319766766391695, 0.00026742619229480624, 0.00020884207333438098, 0.0002226813812740147, 0.0002702047931961715, 0.0002746563695836812, 0.00025173640460707247, 0.0002455624926369637, 0.0002215251006418839, 0.000234688981436193, 0.00023866105766501278, 0.0002634887641761452, 0.00026972012710757554, 0.00022664009884465486, 0.00023510164464823902, 0.00022558734053745866, 0.0002543599985074252, 0.00023872958263382316, 0.00022214984346646816, 0.00021181380725465715, 0.00024275692703668028, 0.00024151972320396453, 0.00024312014284078032, 0.000216628861380741, 0.00026118566165678203, 0.00026345683727413416, 0.00023894001787994057, 0.0002200987801188603, 0.00023104067076928914, 0.00022335075482260436, 0.0002024399145739153, 0.0002505701268091798, 0.00023461981618311256, 0.00025075016310438514, 0.00026818353217095137, 0.00023546465672552586, 0.0002596877748146653, 0.0002577207633294165, 0.00023756150039844215, 0.00022272433852776885, 0.0002692456473596394, 0.00021353685588110238, 0.00023180796415545046, 0.0002778990601655096, 0.00024974162806756794, 0.00026418676134198904, 0.0002458717208355665, 0.0002258182066725567, 0.00024341557582374662, 0.00025711930356919765, 0.0002533847582526505, 0.00024314026813954115, 0.00022560899378731847, 0.00025414719129912555, 0.0002411861642030999, 0.00023811521532479674, 0.00022684434952680022, 0.0002522316644899547, 0.00022045982768759131, 0.00021285474940668792, 0.0002629040682222694, 0.0002165753103327006, 0.00027583446353673935, 0.00025784713216125965, 0.0002502589486539364, 0.00025420449674129486, 0.0002556290419306606, 0.0002105540770571679, 0.00023357657482847571, 0.00023010185395833105, 0.0002562282606959343, 0.00025433380506001413, 0.00024021227727644145, 0.0002495889784768224, 0.0002234697894891724, 0.0002362867526244372, 0.00026016985066235065, 0.0002427566796541214, 0.00021308886061888188, 0.00026480056112632155, 0.00025651263422332704, 0.0002756523317657411, 0.00022626064310315996, 0.00022461159096565098, 0.00025177435600198805, 0.000288744195131585, 0.0002480891125742346, 0.00027785671409219503, 0.0002330258721485734, 0.0002542044676374644, 0.0002152065426344052, 0.00020306016085669398, 0.0002586190530564636, 0.00024118057626765221, 0.00019258761312812567, 0.00023586713359691203, 0.00025164958788082004, 0.0002477770030964166, 0.00025794372777454555, 0.0002483224670868367, 0.0002954484662041068, 0.00024256983306258917, 0.0002747451653704047, 0.00022888611420057714, 0.0002324490196770057, 0.00019515510939527303, 0.0002748777042143047, 0.0002194149128627032, 0.00028952540014870465, 0.0002507773751858622, 0.00024975513224489987, 0.00023465191770810634, 0.00024966467753984034, 0.0002622476895339787, 0.00022429412638302892, 0.00025204732082784176, 0.00022085595992393792, 0.00023020320804789662, 0.0002864229027181864, 0.00023546579177491367, 0.00024320234660990536, 0.00020525635045487434, 0.00026645883917808533, 0.00025929216644726694, 0.00023786458768881857, 0.0002611359814181924, 0.000271731085376814, 0.00024212685821112245, 0.0002430744789307937, 0.00028298364486545324, 0.00024068658240139484, 0.0002570610085967928, 0.00021676183678209782, 0.0002477356174495071, 0.0002261914050905034, 0.0001980846718652174, 0.0002227767399745062, 0.00024415997904725373, 0.00026842253282666206, 0.00027072051307186484, 0.00026738023734651506, 0.00024477957049384713, 0.00024276402837131172, 0.0002209241210948676, 0.0002254423452541232, 0.00028823481989093125, 0.0002453050692565739, 0.0002513611689209938, 0.00029021548107266426, 0.0002326303510926664, 0.00021574074344243854, 0.00021644175285473466, 0.0002038498205365613, 0.0002542066795285791, 0.0002456026850268245, 0.00026280278689228, 0.00023904336558189243, 0.000263099413132295, 0.00025991559959948063, 0.00024931691586971283, 0.0002422035177005455, 0.00025978792109526694, 0.00023810977290850133, 0.00023089638852979988, 0.00022917924798093736, 0.00024010550987441093, 0.00025692919734865427, 0.0002456229995004833, 0.00023325826623477042, 0.000245361850829795, 0.00023968213645275682, 0.00025088360416702926, 0.00020522270642686635, 0.00029440250364132226, 0.0002350577706238255, 0.00022518631885759532, 0.00020940217655152082, 0.0002565268659964204, 0.0002588669303804636, 0.00021190593542996794, 0.00024904985912144184, 0.0002546807227190584, 0.000201679504243657, 0.00024251884315162897, 0.0002489439502824098, 0.0002490399347152561, 0.000255063729127869, 0.00021731415472459048, 0.00024106369528453797, 0.00024087786732707173, 0.000254113256232813, 0.0002716424933169037, 0.00023414508905261755, 0.0002357004559598863, 0.00022606101993005723, 0.00023527706798631698, 0.00020383430819492787, 0.00022489341790787876, 0.0002238199085695669, 0.00020484808192122728, 0.00022574610193260014, 0.00024465660681016743, 0.0002606096095405519, 0.00020939658861607313, 0.0002641394967213273, 0.0002477552625350654, 0.00021336297504603863, 0.0002477490052115172, 0.00026879218057729304, 0.0002599388244561851, 0.00025289924815297127, 0.00022426724899560213, 0.0002364666579524055, 0.00022233559866435826, 0.00022338390408549458, 0.00024657647009007633, 0.00022495129087474197, 0.00024978863075375557, 0.00023611100914422423, 0.00023686661734245718, 0.0002229734236607328, 0.00029691506642848253, 0.0002344773238291964, 0.00026682455791160464, 0.00025496695889160037, 0.00022320753487292677, 0.00024327395658474416, 0.0002500581322237849, 0.0003006574115715921, 0.0002841828973032534, 0.000238668973906897, 0.00023792726278770715, 0.00022393012477550656, 0.00022684843861497939, 0.00024870000197552145, 0.00030173486447893083, 0.00025497577735222876, 0.0002467027516104281, 0.0002601187152322382, 0.00022952930885367095, 0.00025563855888321996, 0.0002687987871468067, 0.00024429854238405824, 0.0002721982600633055, 0.0002499000693205744, 0.0003026394988410175, 0.0002307665126863867, 0.0002522129798308015, 0.00023324377252720296, 0.00025901262415573, 0.000268285337369889, 0.00026183651061728597, 0.00023179811250884086, 0.00023211538791656494, 0.00019757986592594534, 0.0002563859161455184, 0.0002457053342368454, 0.0002745133824646473, 0.00023823486117180437, 0.0002298825274920091, 0.0002250638062832877, 0.00024505917099304497, 0.0002589744399301708, 0.00025415426352992654, 0.0002809761790558696, 0.00023578373657073826, 0.00025000350433401763, 0.0002388225548202172, 0.00022562257072422653, 0.0002338831836823374, 0.00027641686028800905, 0.00020683910406660289, 0.0002711361739784479, 0.00022948354308027774, 0.00024030683562159538, 0.00023394697927869856, 0.0002303507353644818, 0.00028221451793797314, 0.00027075703837908804, 0.000226405740249902, 0.00025654680212028325, 0.0001773594703990966, 0.0002459640381857753, 0.0002324774832231924, 0.00025662663392722607, 0.00022873842681292444, 0.00023521450930275023, 0.0002052141644526273, 0.0002472249325364828, 0.0002569001808296889, 0.00024202918575610965, 0.00026107969461008906, 0.00024527794448658824, 0.000286722497548908, 0.0002340811479371041, 0.00022521849314216524, 0.0002619509759824723, 0.00023353077995125204, 0.00023006183619145304, 0.00022229204478207976, 0.00022305957099888474, 0.00020792745635844767, 0.0002322165237274021, 0.000233544094953686, 0.0002322730579180643, 0.00024494784884154797, 0.0002365586842643097, 0.0002501521958038211, 0.00022202606487553567, 0.00024518321151845157, 0.00022529471607413143, 0.00025467725936323404, 0.00021492467203643173, 0.00024133766419254243, 0.00024554820265620947, 0.00023885372502263635, 0.0002583221939858049, 0.00022191612515598536, 0.0002360033249715343, 0.0002537498658057302, 0.000260098313447088, 0.00025647730217315257, 0.00023510903702117503, 0.00024878510157577693, 0.00022532857838086784, 0.00024859330733306706, 0.00023389715352095664, 0.00026318643358536065, 0.0002301480999449268, 0.00023836920445319265, 0.000262505462160334, 0.00024237837351392955, 0.0002331540745217353, 0.00024919299175962806, 0.00024530923110432923, 0.00022047714446671307, 0.00023732286354061216, 0.00022977222397457808, 0.0002730331907514483, 0.0002262728667119518, 0.00024789877352304757, 0.00021196241141296923, 0.00021474595996551216, 0.00023964863794390112, 0.00024603019119240344, 0.00023768511891830713, 0.0002491260238457471, 0.0002275474980706349, 0.00021405413281172514, 0.00021918998390901834, 0.000264608534052968, 0.00022715261729899794, 0.00029864348471164703, 0.0002192607062170282, 0.0002652313851285726, 0.0002375115145696327, 0.000266985414782539, 0.000264861824689433, 0.00023138760298024863, 0.0002353185846004635, 0.0002557254920247942, 0.00024693357408978045, 0.0002094610099447891, 0.00024061955627985299, 0.00022954627638682723, 0.0002510675694793463, 0.00018680775247048587, 0.0002568484051153064, 0.00023424244136549532, 0.0002727432583924383, 0.0002279464970342815, 0.00023895858612377197, 0.0002291439741384238, 0.00027397932717576623, 0.00024458119878545403, 0.00025670838658697903, 0.00023178542323876172, 0.0002835516643244773, 0.00024980169837363064, 0.0002353927120566368, 0.0002502988209016621, 0.0002786527620628476, 0.0002577368577476591, 0.0002492585626896471, 0.0002422930847387761, 0.000203107570996508, 0.00028241879772394896, 0.00019838676962535828, 0.00023528905876446515, 0.00021969524095766246, 0.0002464609278831631, 0.00023838219931349158, 0.00025634129997342825, 0.000206883589271456, 0.0002465521392878145, 0.0002492516068741679, 0.00020522276463452727, 0.00024352228501811624, 0.00025488281971774995, 0.0002514246152713895, 0.0002305171510670334, 0.0002359304780839011, 0.0001972042373381555, 0.00022850748791825026, 0.0002502191928215325, 0.00023156337556429207, 0.00021026827744208276, 0.0002613990509416908, 0.0002484869328327477, 0.00021381053375080228, 0.0002566077164374292, 0.0002909144095610827, 0.00023094669450074434, 0.00028170619043521583, 0.00028753248625434935, 0.0002498175308573991, 0.00026645572506822646, 0.00025461037876084447, 0.00023701363534200937, 0.00027045016759075224, 0.0002558261912781745, 0.00024403681163676083, 0.00025053363060578704, 0.00027378948288969696, 0.00023981381673365831, 0.0002159545401809737, 0.0002356014447286725, 0.00021662695507984608, 0.0002262657944811508, 0.0002161194570362568, 0.0002592939999885857, 0.00026386106037534773, 0.00028972284053452313, 0.00021529561490751803, 0.00024220225168392062, 0.00025274144718423486, 0.00024505844339728355, 0.0002518999681342393, 0.0003045401244889945, 0.0002580053696874529, 0.00021135857969056815, 0.00022965690004639328, 0.0002766689576674253, 0.00021895837562624365, 0.00028948512044735253, 0.0002920679107774049, 0.00022666426957584918, 0.00027420243714004755, 0.0002069882320938632, 0.00023650411458220333, 0.00025181524688377976, 0.00022637056827079505, 0.000245818606344983, 0.00022369134239852428, 0.0002609777438919991, 0.0002792166196741164, 0.0002837578649632633, 0.0002303724322700873, 0.00027813465567305684, 0.00022086568060331047, 0.00023548254102934152, 0.0002579943393357098, 0.00022133562015369534, 0.0002413777110632509, 0.00021716710762120783, 0.00028437405126169324, 0.00021887414914090186, 0.0002762311778496951, 0.0002747130929492414, 0.0002343505620956421, 0.0002494955260772258, 0.00022419364540837705, 0.00022785202600061893, 0.0002959416015073657, 0.00026659495779313147, 0.00022399421141017228, 0.00022058343165554106, 0.00023789906117599458, 0.00021117205324117094, 0.00025442198966629803, 0.00031681882683187723, 0.00023712808615528047, 0.00024107677745632827, 0.00022126681869849563, 0.0002469401224516332, 0.0002378633216721937, 0.0002545237075537443, 0.0002831532037816942, 0.00023585867893416435, 0.000253246515057981, 0.00023373813019134104, 0.0002084767329506576, 0.000255145481787622, 0.000260140688624233, 0.00025043130153790116, 0.000242076552240178, 0.00019995566981378943, 0.0002515863161534071, 0.0002652080438565463, 0.00021654002193827182, 0.00023478268121834844, 0.00025772559456527233, 0.0002108280168613419, 0.00023587439500261098, 0.0002447125152684748, 0.00020959126413799822, 0.00026566305314190686, 0.0002744323282968253, 0.00021926293266005814, 0.0002477261296007782, 0.0002561242727097124, 0.0002483967982698232, 0.00023639117716811597, 0.000257499807048589, 0.00021650243434123695, 0.00021022174041718245, 0.00024372099142055959, 0.0002256546140415594, 0.0002741528151091188, 0.00025819893926382065, 0.00024212572316173464, 0.00020084567950107157, 0.00022958761837799102, 0.0002734665758907795, 0.0002830048033501953, 0.00023653145763091743, 0.000245086062932387, 0.0002780846261885017, 0.00026398300542496145, 0.000238968146732077, 0.00027105334447696805, 0.00020654231775552034, 0.00022965627431403846, 0.00022682049893774092, 0.00026575743686407804, 0.00030092793167568743, 0.00023455178597941995, 0.0002518900146242231, 0.0002550565404817462, 0.00024187112285289913, 0.00021943489264231175, 0.000267212773906067, 0.00022926065139472485, 0.0002852460602298379, 0.0002908583264797926, 0.00026381248608231544, 0.00025225160061381757, 0.0002691634581424296, 0.00023738789604976773, 0.00024629186373203993, 0.00025393470423296094, 0.00023241888266056776, 0.00023804490047041327, 0.0002854707126971334, 0.00028038304299116135, 0.00023855284962337464, 0.0002596598642412573, 0.0003077175060752779, 0.00023850746219977736, 0.00027628568932414055, 0.0002934141375590116, 0.00023328422685153782, 0.0002681714831851423, 0.00024083073367364705, 0.00020960521942470223, 0.0002666512446012348, 0.00029808629187755287, 0.00028091183048672974, 0.000262128422036767, 0.00023549594334326684, 0.0002367284323554486, 0.00022299561533145607, 0.00025223055854439735, 0.0002493878419045359, 0.00024904977180995047, 0.00021766767895314842, 0.00025583241949789226, 0.00028161448426544666, 0.0002591037191450596, 0.0002567484334576875, 0.00024306388513650745, 0.00026466959388926625, 0.00027379978564567864, 0.00023027366842143238, 0.00024161233159247786, 0.0002475529909133911, 0.00027361337561160326, 0.00021559606830123812, 0.0002727934916038066, 0.0002721748605836183, 0.00028537464095279574, 0.00027941391454078257, 0.0002219059970229864, 0.00022751843789592385, 0.00028031819965690374, 0.0002235974679933861, 0.0002345731481909752, 0.000288572657154873, 0.0002898987731896341, 0.00022407833603210747, 0.0002558382402639836, 0.00024954500258900225, 0.00021051272051408887, 0.0002260960900457576, 0.0002418489457340911, 0.00020840232900809497, 0.0002388979191891849, 0.0002521575952414423, 0.0002461860713083297, 0.0002610407245811075, 0.0002463732671458274, 0.0002362281666137278, 0.00022854168491903692, 0.00023332673299591988, 0.00024256146571133286, 0.00023470580345019698, 0.000218686560401693, 0.0002732339489739388, 0.00024426673189736903, 0.00022752520453650504, 0.0002457359805703163, 0.00025769724743440747, 0.0002935603552032262, 0.00027224697987549007, 0.0002278075844515115, 0.00024238140031229705, 0.00024854071671143174, 0.00024254586605820805, 0.00022561124933417886, 0.00023250420053955168, 0.00030216629966162145, 0.00024247296096291393, 0.00023073192278388888, 0.0002659493184182793, 0.00026690575759857893, 0.00023376518220175058, 0.00023699874873273075, 0.00024205734371207654, 0.00021190072584431618, 0.00022688106400892138, 0.00024422784917987883, 0.00028972080326639116, 0.0002713756693992764, 0.0002522521244827658, 0.00027365743881091475, 0.00026159812114201486, 0.0002203154581366107, 0.0002483723219484091, 0.0002183134201914072, 0.0002218577719759196, 0.0002336969191674143, 0.00024043014855124056, 0.00023538625100627542, 0.00023676147975493222, 0.00021537883731070906, 0.00024336445494554937, 0.0002684176724869758, 0.0002101389254676178, 0.00025174987968057394, 0.0002492017229087651, 0.0002084031730191782, 0.0002556444669608027, 0.0002345149841858074, 0.00022853781410958618, 0.0002552855294197798, 0.00024800776736810803, 0.0002457275695633143, 0.00023902274551801383, 0.00022595631889998913, 0.0002759236958809197, 0.00021998841839376837, 0.00022995396284386516, 0.0002459567040205002, 0.00022263692517299205, 0.00027437525568529963, 0.0002340200444450602, 0.00022669666213914752, 0.00023320202308241278, 0.00027640286134555936, 0.00023361614148598164, 0.00026745611103251576, 0.00027006547315977514, 0.0002703308709897101, 0.0002571851364336908, 0.0002651963150128722, 0.0002482382988091558, 0.00021975459821987897, 0.0002518348628655076, 0.000265011127339676, 0.00027213242719881237, 0.00022747322509530932, 0.00024266436230391264, 0.0002618991711642593, 0.00025510534760542214, 0.0002606638299766928, 0.0002418501244392246, 0.0002641825412865728, 0.0002462647680658847, 0.00021005388407502323, 0.000250336219323799, 0.0002123545709764585, 0.00021953275427222252, 0.00023129553301259875, 0.00020186950860079378, 0.00025825598277151585, 0.00024289537395816296, 0.00023429412976838648, 0.00021688625565730035, 0.00025878960150294006, 0.00024637000751681626, 0.00024998161825351417, 0.000204831623705104, 0.0002068723551928997, 0.0002537539694458246, 0.00020571544882841408, 0.00022561303921975195, 0.00025206877035088837, 0.00020071069593541324, 0.0002309949923073873, 0.00025073648430407047, 0.00025745813036337495, 0.00027008395409211516, 0.000260549335507676, 0.00023195953690446913, 0.00023291805700864643, 0.00024584002676419914, 0.0002555565151851624, 0.00024149607634171844, 0.00024285013205371797, 0.0002735475718509406, 0.00026531759067438543, 0.0002556437102612108, 0.00023557330132462084, 0.0002622756874188781, 0.00022211065515875816, 0.00024695994216017425, 0.0002337611367693171, 0.00022868627274874598, 0.0002760173811111599, 0.0002227656077593565, 0.0002418263757135719, 0.0002423557307338342, 0.00020976913219783455, 0.00022904879006091505, 0.0002127092593582347, 0.0002454486093483865, 0.0002296051534358412, 0.0002978702250402421, 0.00023599194537382573, 0.00024549828958697617, 0.000244851631578058, 0.0002502789138816297, 0.0002378311037318781, 0.00025053450372070074, 0.00022508545953314751, 0.0002736775204539299, 0.0002530015481170267, 0.0002543244045227766, 0.00028047888190485537, 0.00025283408467657864, 0.00024355736968573183, 0.00024831408518366516, 0.00025656548677943647, 0.00024202765780501068, 0.00025168582214973867, 0.00024297625350300223, 0.00022354043903760612, 0.00024141058383975178, 0.00025380036095157266, 0.00023473230248782784, 0.0002511643397156149, 0.00024235677847173065, 0.0002671718248166144, 0.00025115645257756114, 0.0002430592867312953, 0.00022664875723421574, 0.00025000760797411203, 0.0002408400468993932, 0.0002554590755607933, 0.00025354023091495037, 0.00024774743360467255, 0.00024941295851022005, 0.00023152492940425873, 0.00026134110521525145, 0.00022404278570320457, 0.0002572450612206012, 0.0002897392841987312, 0.00028020795434713364, 0.0002411059249425307, 0.0002249656245112419, 0.00024632143322378397, 0.00026599870761856437, 0.00024242089421022683, 0.0002463690470904112, 0.00023376723402179778, 0.000259352003922686, 0.00022888631792739034, 0.00020606743055395782, 0.00023836796754039824, 0.0002718265459407121, 0.0002553312515374273, 0.0002673599519766867, 0.000254199025221169, 0.00024521967861801386, 0.0002513469953555614, 0.00024649553233757615, 0.00025181827368214726, 0.00019630469614639878, 0.00020439292711671442, 0.00022717671527061611, 0.00027548603247851133, 0.00026896758936345577, 0.0002888468443416059, 0.00022469194664154202, 0.00020741327898576856, 0.0002564360329415649, 0.00024141347967088223, 0.0002451030886732042, 0.00025849774829111993, 0.00022057810565456748, 0.00023698151926510036, 0.0002513138169888407, 0.00024277136253658682, 0.00023562922433484346, 0.0002600150473881513, 0.00023910356685519218, 0.00023632129887118936, 0.00025119661586359143, 0.00023147241154219955, 0.00025203844415955245, 0.0002275348815601319, 0.00025584123795852065, 0.00022928397811483592, 0.00021452759392559528, 0.00022414702107198536, 0.0002662175102159381, 0.000210916405194439, 0.00020652632520068437, 0.000256045488640666, 0.00023818704357836396, 0.00025623536203056574, 0.00020484892593231052, 0.00022246486332733184, 0.00022577749041374773, 0.00024406364536844194, 0.00023465146659873426, 0.00020617358677554876, 0.00025704174186103046, 0.0002745549427345395, 0.00026917975628748536, 0.00021572591504082084, 0.0002840712550096214, 0.0002552721416577697, 0.00024741809465922415, 0.00025353263481520116, 0.00028257229132577777, 0.00025696243392303586, 0.00021733356697950512, 0.0001975790219148621, 0.0002409024309599772, 0.00024097584537230432, 0.00021784358250442892, 0.00027587328804656863, 0.0002599879226181656, 0.0002614782133605331, 0.0002494242216926068, 0.00023841048823669553, 0.00025928375544026494, 0.00023896811762824655, 0.0002292845892952755, 0.0002477580856066197, 0.00027094673714600503, 0.00022195131168700755, 0.0002666065702214837, 0.00021805218420922756, 0.00021768939041066915, 0.0002548430347815156, 0.00024031811335589737, 0.00025500336778350174, 0.00021215947344899178, 0.0002543895097915083, 0.00027583548217080534, 0.0002512022037990391, 0.0002231643593404442, 0.00024277251213788986, 0.0002330514253117144, 0.000236630454310216, 0.0002626063651405275, 0.00023964696447364986, 0.0002140577998943627, 0.00023176148533821106, 0.00019673557835631073, 0.0002771411673165858, 0.00023898344079498202, 0.00021533759718295187, 0.00022761436412110925, 0.00022891313710715622, 0.00023700612655375153, 0.00023049638548400253, 0.00023204588796943426, 0.00023438203788828105, 0.00021026399917900562, 0.0002498895628377795, 0.00023386489192489535, 0.00029265755438245833, 0.0002563372254371643, 0.0002369407593505457, 0.0002456064394209534, 0.00022777642880100757, 0.00025727669708430767, 0.00024359881354030222, 0.00029988831374794245, 0.00022817876015324146, 0.00022868732048664242, 0.0002216766297351569, 0.00022578664356842637, 0.0002724041696637869, 0.00024387268058490008, 0.00023069586313795298, 0.00024936554837040603, 0.00025264598662033677, 0.00019513705046847463, 0.000262857269262895, 0.0002782836963888258, 0.000231321100727655, 0.0002503191935829818, 0.00027766296989284456, 0.0002713444409891963, 0.00025550168356858194, 0.0002399497025180608, 0.00022859175805933774, 0.00022897205781191587, 0.00029460308724083006, 0.00022597849601879716, 0.0002333364827791229, 0.00026319659082219005, 0.0002592014498077333, 0.00023101689293980598, 0.00025187619030475616, 0.0002416551433270797, 0.00022867786174174398, 0.00023488527222070843, 0.00024012188077904284, 0.00027331055025570095, 0.000283485627733171, 0.00021482720330823213, 0.00029860061476938426, 0.00024127471260726452, 0.0002539103734306991, 0.00027519394643604755, 0.00025562572409398854, 0.00022646372963208705, 0.00021665706299245358, 0.0002609184302855283, 0.00024945809855125844, 0.00023219184367917478, 0.00029403105145320296, 0.0002481125120539218, 0.0002598000573925674, 0.0002105778403347358, 0.0002662086335476488, 0.00021296962222550064, 0.00022295984672382474, 0.0002243408525828272, 0.00022169465955812484, 0.00022037354938220233, 0.00024009397020563483, 0.00021798744273837656, 0.0002949541958514601, 0.00020069914171472192, 0.0002654175041243434, 0.00023544336727354676, 0.00023950084869284183, 0.00022736449318472296, 0.0002094461233355105, 0.00023129016335587949, 0.00024690950522199273, 0.000255189515883103, 0.0002846927964128554, 0.0002381867088843137, 0.00021580186148639768, 0.0002120094432029873, 0.00023923181288409978, 0.00020813886658288538, 0.0002477563393767923, 0.00027334332116879523, 0.00022402431932277977, 0.0002696094452403486, 0.00025563707458786666, 0.00021966322674416006, 0.00026534576318226755, 0.0], [0.0005077657988294959, 0.0004999543307349086, 0.00043363592703826725, 0.000533141428604722, 0.0003636453184299171, 0.0005662570474669337, 0.0004974453477188945, 0.00047632979112677276, 0.0005292837740853429, 0.00047607990563847125, 0.0004988768487237394, 0.00047908694250509143, 0.0004895939491689205, 0.00045088856131769717, 0.0004694162053056061, 0.00046002963790670037, 0.0004043962399009615, 0.0004666458407882601, 0.0005581958685070276, 0.0004834065621253103, 0.0004365714848972857, 0.00047646299935877323, 0.00046862781164236367, 0.0004902014043182135, 0.0004898910992778838, 0.00045631261309608817, 0.0005440862150862813, 0.0004447278624866158, 0.0005531539209187031, 0.0004653841897379607, 0.00045111082727089524, 0.00043717699008993804, 0.00052703294204548, 0.00042922215652652085, 0.00043493081466294825, 0.00046710704918950796, 0.0004786008794326335, 0.0005648096557706594, 0.000530941179022193, 0.0004493019951041788, 0.0004669124027714133, 0.0005106439930386841, 0.0004977904609404504, 0.0004187926242593676, 0.00046964266221038997, 0.0004238526744302362, 0.00048459621029905975, 0.0005011326284147799, 0.00048658697050996125, 0.0004956690245307982, 0.0004957943456247449, 0.0004797881410922855, 0.0005222055478952825, 0.0004669820482376963, 0.0004767061909660697, 0.0004263392766006291, 0.0005129025084897876, 0.00044871476711705327, 0.00046951245167292655, 0.0004816808504983783, 0.0005527830799110234, 0.0004904047818854451, 0.000441901502199471, 0.00047258270205929875, 0.0004176503571216017, 0.000458926719147712, 0.0005189337534829974, 0.0005261282785795629, 0.0005187941715121269, 0.00044064424582757056, 0.0004388348897919059, 0.0005097177345305681, 0.0004968616412952542, 0.0004877749888692051, 0.0004505384713411331, 0.0005307699902914464, 0.0005278885364532471, 0.0005629502702504396, 0.0004760135489050299, 0.0004432048590388149, 0.0004940078943036497, 0.0005318907205946743, 0.0003858651907648891, 0.000509372039232403, 0.00048087589675560594, 0.0005414672777988017, 0.0005688706296496093, 0.00044400407932698727, 0.00047067299601621926, 0.0004884860245510936, 0.0004802041221410036, 0.0005611103842966259, 0.0004304767062421888, 0.0005421859095804393, 0.0005206638597883284, 0.0005395343760028481, 0.00046634970931336284, 0.00048226365470327437, 0.0005231995601207018, 0.000448244740255177, 0.0003870472719427198, 0.0004496395122259855, 0.0005363987875171006, 0.00046840240247547626, 0.0004762539465446025, 0.00048570422222837806, 0.00047684527817182243, 0.0005020873504690826, 0.0005139833665452898, 0.0005697829183191061, 0.0005164709291420877, 0.0005390054429881275, 0.0005445898277685046, 0.00047203333815559745, 0.0005527361645363271, 0.0005506113520823419, 0.0005539422272704542, 0.0005315874004736543, 0.0004824634233955294, 0.00044198299292474985, 0.0004848591343034059, 0.0004382722545415163, 0.0004275579412933439, 0.0005635121488012373, 0.00047628823085688055, 0.0004381063918117434, 0.0005126497708261013, 0.00045955352834425867, 0.0004982250393368304, 0.0005642587202601135, 0.000558838015422225, 0.0004874077276326716, 0.0005290493718348444, 0.00042792101157829165, 0.0005662481999024749, 0.00048120826249942183, 0.00045226881047710776, 0.000606793153565377, 0.00046933567500673234, 0.0004175823414698243, 0.0004585997376125306, 0.0005015510832890868, 0.0005422134418040514, 0.00046131465933285654, 0.00047910009743645787, 0.0004551792226266116, 0.0005542676080949605, 0.0004468708939384669, 0.0004517452325671911, 0.0004504234530031681, 0.0004867597308475524, 0.00047613598871976137, 0.0004618400416802615, 0.0005229679518379271, 0.0005908945458941162, 0.0004921354702673852, 0.00045096001122146845, 0.0005540566053241491, 0.0004931323928758502, 0.0004144948616158217, 0.00050447165267542, 0.00047495675971731544, 0.0005118541303090751, 0.0005599409923888743, 0.00044817585148848593, 0.0004756509733851999, 0.0004229466721881181, 0.0005418403306975961, 0.000524552131537348, 0.000535793078597635, 0.0005098916008137167, 0.0004160728130955249, 0.0004764883778989315, 0.0004835096769966185, 0.0004741132433991879, 0.000482104136608541, 0.0005434337654151022, 0.00043149341945536435, 0.00046025848132558167, 0.00048730827984400094, 0.0005249291425570846, 0.00047430951963178813, 0.0004644236178137362, 0.0005558013217523694, 0.0005567669286392629, 0.00047038361662998796, 0.0005196505226194859, 0.00048464201972819865, 0.00040829338831827044, 0.0004739888245239854, 0.0005631268140859902, 0.00045927424798719585, 0.0004949350841343403, 0.00046436936827376485, 0.00046111823758110404, 0.00048296217573806643, 0.0004940382204949856, 0.0005497396923601627, 0.0005669804522767663, 0.0006224655080586672, 0.0004323160392232239, 0.00047278538113459945, 0.0004834937280975282, 0.0005314045702107251, 0.0004937644116580486, 0.00044383175554685295, 0.0005163830355741084, 0.0005393509636633098, 0.0005910928011871874, 0.00045570143265649676, 0.000631264818366617, 0.0004156187060289085, 0.0005038981325924397, 0.0005258292658254504, 0.00048512188368476927, 0.00045754239545203745, 0.0005402586539275944, 0.0004310664953663945, 0.0004981120000593364, 0.0004235939122736454, 0.00045336721814237535, 0.000461924442788586, 0.0004496451001614332, 0.0004816681903321296, 0.0004701202269643545, 0.0005222828476689756, 0.0005112391663715243, 0.00046475723502226174, 0.00048179886653088033, 0.0004560760280583054, 0.000501008122228086, 0.00040875811828300357, 0.0005278286407701671, 0.00043374908273108304, 0.0004649149486795068, 0.000490462698508054, 0.0004536523774731904, 0.0005056738154962659, 0.00041207592585124075, 0.0005373020539991558, 0.0004505882679950446, 0.0004957318305969238, 0.0005136507679708302, 0.0004979663644917309, 0.00042372214375063777, 0.0004362282052170485, 0.0005786273395642638, 0.0005111134378239512, 0.0005526411114260554, 0.00042938912520185113, 0.0004763512406498194, 0.0004900306230410933, 0.00046183328959159553, 0.0004998987424187362, 0.00048212939873337746, 0.0004504070966504514, 0.00043045729398727417, 0.0005006093415431678, 0.0004977238713763654, 0.0004154094203840941, 0.0004624214780051261, 0.0003882508317474276, 0.0004470390558708459, 0.0005203652544878423, 0.00045496117672882974, 0.0004871006531175226, 0.00041139277163892984, 0.00046662267413921654, 0.0005199967417865992, 0.0005176174454391003, 0.0004727358464151621, 0.0005022584809921682, 0.0005552128423005342, 0.00046013868995942175, 0.0005550492787733674, 0.0004423813952598721, 0.0004397663287818432, 0.00046791034401394427, 0.000453898508567363, 0.0004765113117173314, 0.00047420221380889416, 0.0004535975749604404, 0.0005273899878375232, 0.0003878443385474384, 0.000548838812392205, 0.0005075312219560146, 0.0005574759561568499, 0.00046483048936352134, 0.0004643262072931975, 0.0004932010779157281, 0.0004180468386039138, 0.0004870641860179603, 0.0005409861914813519, 0.0005408363649621606, 0.0004195734509266913, 0.0005194139084778726, 0.0004904729430563748, 0.0004645980952773243, 0.0004999845987185836, 0.0004180206742603332, 0.0005697223241440952, 0.0003856236580759287, 0.00046366616152226925, 0.0004335666017141193, 0.0005669098463840783, 0.0004955707699991763, 0.0005577187403105199, 0.00043841375736519694, 0.0005859344964846969, 0.00047921916120685637, 0.0004645430308301002, 0.000529073819052428, 0.0005868793232366443, 0.0005198311991989613, 0.00046767451567575336, 0.0004834810388274491, 0.0005257565644569695, 0.000497126835398376, 0.00040753534995019436, 0.00043270725291222334, 0.0005564837483689189, 0.0005520516424439847, 0.0004609064490068704, 0.00044740206794813275, 0.00042953959200531244, 0.0004597177030518651, 0.0005773786106146872, 0.00045911830966360867, 0.0005144733004271984, 0.0005259498138912022, 0.0005014381022192538, 0.000524125003721565, 0.0004289706703275442, 0.0005573404487222433, 0.00041395617881789804, 0.0005071285995654762, 0.0004977636854164302, 0.0004993294132873416, 0.000517817388754338, 0.0005167035851627588, 0.0004824567004106939, 0.0004975870251655579, 0.0004937510238960385, 0.0005036388174630702, 0.000514502520672977, 0.000508838624227792, 0.0004168620507698506, 0.0005062801064923406, 0.0005042862030677497, 0.000407253741286695, 0.0005525409360416234, 0.0004970662412233651, 0.00041018472984433174, 0.0004509651625994593, 0.0004586607392411679, 0.0004453007422853261, 0.0004699760756921023, 0.0004810009559150785, 0.0005419309600256383, 0.0005189214134588838, 0.0005131280631758273, 0.0004452500434126705, 0.000550535973161459, 0.00043728211312554777, 0.00046566780656576157, 0.0005416467320173979, 0.0004799145390279591, 0.0005096918903291225, 0.0004426608793437481, 0.0005173460813239217, 0.0004561311798170209, 0.0004628530004993081, 0.0004938544007018209, 0.0005591069930233061, 0.00042365549597889185, 0.000505061645526439, 0.00051374442409724, 0.0004671590868383646, 0.0005328764673322439, 0.0004911450087092817, 0.00043959543108940125, 0.0004305915208533406, 0.00045533300726674497, 0.0005148029886186123, 0.00041471701115369797, 0.0005002269754186273, 0.0005102073773741722, 0.00046706199645996094, 0.00045470939949154854, 0.0005066961166448891, 0.0005423035472631454, 0.0004588440351653844, 0.00045799367944709957, 0.0005562318256124854, 0.0005121012218296528, 0.0005050595500506461, 0.0005564926541410387, 0.0005637936410494149, 0.0004509404534474015, 0.0005494393408298492, 0.0004825277137570083, 0.0005081682465970516, 0.0004813653358723968, 0.0004571226018015295, 0.0004906425019726157, 0.0004228657344356179, 0.0005217226571403444, 0.00041183590656146407, 0.00050171441398561, 0.00042646858491934836, 0.0004807851801160723, 0.0004664682492148131, 0.0004167849838268012, 0.0004862051282543689, 0.0005720776389352977, 0.0005055801593698561, 0.0004401418846100569, 0.0005001856479793787, 0.0004416254232637584, 0.0004757218121085316, 0.00042119238059967756, 0.0004664141160901636, 0.000431954103987664, 0.0004586024151649326, 0.0004884627996943891, 0.0005309082916937768, 0.0005349235143512487, 0.0005818305653519928, 0.0005785449175164104, 0.0004530470760073513, 0.00045006960863247514, 0.0006016579573042691, 0.0005043139099143445, 0.0005025200662203133, 0.0005177180282771587, 0.0005318335606716573, 0.0004683728329837322, 0.00046717372606508434, 0.0005323903169482946, 0.00048617320135235786, 0.00044659775448963046, 0.0005226366920396686, 0.0004627763119060546, 0.00042953513911925256, 0.00048440767568536103, 0.0004700035788118839, 0.0005270862602628767, 0.00043330254266038537, 0.0005091377533972263, 0.0005673120031133294, 0.00047929768334142864, 0.0004644610162358731, 0.0004413784190546721, 0.0005387907731346786, 0.00046632217708975077, 0.0005513826035894454, 0.0004754702968057245, 0.0005133586819283664, 0.0004851787234656513, 0.0004713086236733943, 0.00047418204485438764, 0.0004984016995877028, 0.0005039814859628677, 0.00040931557305157185, 0.00047065832768566906, 0.0005182487075217068, 0.00047816045116633177, 0.0004267021722625941, 0.0004669687186833471, 0.0005010251770727336, 0.0004947689594700933, 0.0004940310609526932, 0.0005744877271354198, 0.0004500111099332571, 0.0005020854878239334, 0.00046794977970421314, 0.00043536798330023885, 0.00052625872194767, 0.000564976071473211, 0.0005256993463262916, 0.0005035073845647275, 0.00042827229481190443, 0.0004762460011988878, 0.0004534037143457681, 0.0005024021957069635, 0.0004019488987978548, 0.0004525786207523197, 0.0003978288732469082, 0.0005055976216681302, 0.0005185672198422253, 0.00046986719826236367, 0.00047779062879271805, 0.0004243578587193042, 0.0005364313838072121, 0.00045342184603214264, 0.0004536345659289509, 0.0004889185656793416, 0.0004199146933387965, 0.00047175170038826764, 0.0005149017088115215, 0.0004976657219231129, 0.00047165839350782335, 0.0004971845773980021, 0.00046868567005731165, 0.0005372091545723379, 0.00047591456677764654, 0.0005472878110595047, 0.0004921387298963964, 0.0005102443392388523, 0.0004896638565696776, 0.0005182797904126346, 0.0004105951520614326, 0.0005792506271973252, 0.0004898611805401742, 0.00043622867087833583, 0.0004914891324006021, 0.0004240759590175003, 0.000470121594844386, 0.0004841503396164626, 0.0004840591282118112, 0.0004306532209739089, 0.00042452526395209134, 0.0004726267361547798, 0.0005309569532983005, 0.0005066238809376955, 0.000528202741406858, 0.0004984641564078629, 0.0004989772569388151, 0.0004920601495541632, 0.00043706930591724813, 0.0004683034785557538, 0.000573811586946249, 0.0005181281012482941, 0.0004995730123482645, 0.0004874618898611516, 0.0006250285659916699, 0.0005151716177351773, 0.0004912284784950316, 0.000419940595747903, 0.000606497167609632, 0.0004276741237845272, 0.0005078892572782934, 0.00042458868119865656, 0.0004342154134064913, 0.0005450623575598001, 0.0003945280332118273, 0.0005008105072192848, 0.0005422932445071638, 0.000518942135386169, 0.000457100075436756, 0.000517603475600481, 0.0004682996077463031, 0.0004633179632946849, 0.0005231721443124115, 0.0005254719289951026, 0.000433650886407122, 0.00047450821148231626, 0.0005119879497215152, 0.00048301173956133425, 0.000621472136117518, 0.0005645948695018888, 0.0005555144743993878, 0.00047298817662522197, 0.000466706434963271, 0.00047109558363445103, 0.0005615249974653125, 0.0005333871813490987, 0.0005696502048522234, 0.0004695890820585191, 0.0004858613247051835, 0.00044959751539863646, 0.00045225326903164387, 0.000412046822020784, 0.0004423425125423819, 0.0005528986221179366, 0.0004897208418697119, 0.0005370298749767244, 0.0005158216808922589, 0.0004885340458713472, 0.0005630856612697244, 0.0005447454168461263, 0.00041596408118493855, 0.0005165372276678681, 0.0004764045006595552, 0.0004471670836210251, 0.0005245991051197052, 0.000490454665850848, 0.00040323889697901905, 0.0004490235587581992, 0.000475148408440873, 0.00041822754428721964, 0.0004635663644876331, 0.0005302677163854241, 0.0005133989616297185, 0.000462644558865577, 0.0005556997493840754, 0.0004977520438842475, 0.00048322929069399834, 0.0004959054058417678, 0.0004752479726448655, 0.0004910725983791053, 0.0004590228199958801, 0.00046928401570767164, 0.00040372286457568407, 0.00042180740274488926, 0.0005171051598154008, 0.00043377705151215196, 0.0005481928237713873, 0.0005089062615297735, 0.000513003789819777, 0.00047279419959522784, 0.0005330447456799448, 0.000464405253296718, 0.0004784068441949785, 0.0005359015776775777, 0.0005005047423765063, 0.0004339293809607625, 0.0003780755796469748, 0.0004101844970136881, 0.0005733699072152376, 0.0005847165593877435, 0.0004737975832540542, 0.00044898048508912325, 0.0005320830969139934, 0.000446318241301924, 0.0005813745665363967, 0.0005175801343284547, 0.0005086916498839855, 0.00044563465053215623, 0.000425031001213938, 0.0004711917426902801, 0.0005251190741546452, 0.0005010394961573184, 0.0004322827735450119, 0.0004705666215158999, 0.0004967764834873378, 0.0004290191864129156, 0.0005913384957239032, 0.000507672200910747, 0.000547583622392267, 0.00042268235119991004, 0.0004962661187164485, 0.0005302791832946241, 0.00047936345799826086, 0.00044320241431705654, 0.000551755481865257, 0.0004629952600225806, 0.0005602850578725338, 0.0004759266448672861, 0.0005519738188013434, 0.00043388057383708656, 0.0004905998357571661, 0.000465691409772262, 0.00039593817200511694, 0.0005206129280850291, 0.0005808582063764334, 0.0004605147987604141, 0.0005077600944787264, 0.0004604732384905219, 0.00047808984527364373, 0.0003956292348448187, 0.000466169003630057, 0.00040302309207618237, 0.00047732077655382454, 0.0004738595162052661, 0.00045628412044607103, 0.0005185751360841095, 0.0005048957536928356, 0.000465649354737252, 0.00043536085286177695, 0.0005466177826747298, 0.00045786943519487977, 0.0004758871509693563, 0.0004669056797865778, 0.0005134296952746809, 0.0005425116396509111, 0.0005348504637368023, 0.00048076114035211504, 0.00044919579522684216, 0.0004327000060584396, 0.00045564613537862897, 0.0005232251132838428, 0.00041840437916107476, 0.00039638031739741564, 0.0005389463622123003, 0.0005207622307352722, 0.0004563838883768767, 0.000453595828730613, 0.0004964185645803809, 0.0004436417657416314, 0.0004422672791406512, 0.00044599248212762177, 0.00047518714563921094, 0.0004081169608980417, 0.0005585533799603581, 0.00036988549982197583, 0.00048389265430159867, 0.00048487112508155406, 0.0005585765466094017, 0.0004101572558283806, 0.0005257506854832172, 0.0004991010064259171, 0.0004439815238583833, 0.0004685316525865346, 0.0005071427440270782, 0.0005108785699121654, 0.00048309730482287705, 0.0004749791696667671, 0.0004744101024698466, 0.0005273580900393426, 0.0004960914375260472, 0.0005476886872202158, 0.0004732038942165673, 0.0005190028459765017, 0.00047387328231707215, 0.00046358024701476097, 0.000446965335868299, 0.0004608313029166311, 0.0004950849106535316, 0.000420028722146526, 0.000449743471108377, 0.00046245698467828333, 0.0005222833715379238, 0.00046592013677582145, 0.00045665533980354667, 0.0004906872054561973, 0.0005070458864793181, 0.0004862989590037614, 0.0005360377253964543, 0.0005023616831749678, 0.0005100939888507128, 0.0004247793403919786, 0.0004922451335005462, 0.00045625216444022954, 0.0004283582093194127, 0.0004837594460695982, 0.0004750329826492816, 0.0004899329505860806, 0.000590574461966753, 0.00046318714157678187, 0.0004734729300253093, 0.0004808047378901392, 0.00046436491538770497, 0.00041307604988105595, 0.000604538363404572, 0.0004095213662367314, 0.0004402365011628717, 0.00048471527406945825, 0.00044463257654570043, 0.0005064371507614851, 0.0005117335822433233, 0.0005104420124553144, 0.0005136386607773602, 0.000497749715577811, 0.0005410145386122167, 0.0004608253948390484, 0.0005075193475931883, 0.00048034547944553196, 0.0005092965438961983, 0.00043347495375201106, 0.0004661636194214225, 0.00047967315185815096, 0.00046993306023068726, 0.0005148801137693226, 0.0005428311415016651, 0.00045630946988239884, 0.000489907106384635, 0.0005491168121807277, 0.00048812772729434073, 0.00047711480874568224, 0.0004554888291750103, 0.0005693061393685639, 0.0005414905026555061, 0.00045308994594961405, 0.0005520075792446733, 0.0004899943014606833, 0.0004343954205978662, 0.00041471776785328984, 0.0004964629770256579, 0.0004600314423441887, 0.00046131995622999966, 0.0005333222215995193, 0.000514669984113425, 0.0005373580497689545, 0.0004364538181107491, 0.00045937689719721675, 0.0004140654345974326, 0.0005891147302463651, 0.00043397105764597654, 0.0005218241713009775, 0.0004980265512131155, 0.000446136953542009, 0.0004916855250485241, 0.0005865194834768772, 0.0005001084064133465, 0.0005004385020583868, 0.0004979746881872416, 0.00050711928633973, 0.0005031076143495739, 0.0004742235178127885, 0.000522823364008218, 0.00046337032108567655, 0.00046209446736611426, 0.00048134493408724666, 0.00046236187336035073, 0.00046367739560082555, 0.0004885619273409247, 0.0004974080948159099, 0.0005550057394430041, 0.00040918969898484647, 0.0005196027341298759, 0.0004637118836399168, 0.0004966038977727294, 0.0004707217449322343, 0.0005958943511359394, 0.00046364570152945817, 0.0004994545015506446, 0.0004309173964429647, 0.00045397799112834036, 0.0006041274173185229, 0.00046688629663549364, 0.0005329390405677259, 0.000550248718354851, 0.0005231804680079222, 0.00047174477367661893, 0.0005583392921835184, 0.0005361387738958001, 0.0005054642679169774, 0.0004153141053393483, 0.0004889483097940683, 0.0005187682691030204, 0.00045884543214924634, 0.000491082202643156, 0.00043079687748104334, 0.0004739735450129956, 0.0004731208609882742, 0.0005490572657436132, 0.0005270369001664221, 0.000522160844411701, 0.0004963952233083546, 0.00046421901788562536, 0.0004082796222064644, 0.00046260180533863604, 0.0005450319731608033, 0.0004829604004044086, 0.00043573431321419775, 0.0005105507443659008, 0.0005161177250556648, 0.0004471198481041938, 0.0005096110398881137, 0.0005334354937076569, 0.0005084467120468616, 0.00041934236651286483, 0.000519441207870841, 0.00046380783896893263, 0.0005071453051641583, 0.000516341591719538, 0.0005067867459729314, 0.0004884990630671382, 0.0005132677615620196, 0.0005454182391986251, 0.00046995707089081407, 0.000436500326031819, 0.00039825422572903335, 0.0004398728196974844, 0.0004560764937195927, 0.0004771208041347563, 0.00044791947584599257, 0.00045598275028169155, 0.0004405851068440825, 0.00046208358253352344, 0.0005596907576546073, 0.0005081051494926214, 0.00043309922330081463, 0.00045825858251191676, 0.0004971411544829607, 0.0004485287645366043, 0.0005028962041251361, 0.00042340424261055887, 0.00046260328963398933, 0.0004964640829712152, 0.0004252689250279218, 0.000508100085426122, 0.0005319697083905339, 0.0005213948315940797, 0.0004884825902990997, 0.000492710736580193, 0.00047843772335909307, 0.00048593487008474767, 0.00045176822459325194, 0.0004757969290949404, 0.0005048270686529577, 0.0004784204647876322, 0.0004399728204589337, 0.0005022829864174128, 0.00045998109271749854, 0.0005509476759470999, 0.0004695504903793335, 0.000508224475197494, 0.0005192895187065005, 0.0005341582000255585, 0.0004370051028672606, 0.0005456312210299075, 0.0005113448132760823, 0.00048499327385798097, 0.0004375040007289499, 0.0005759576451964676, 0.0005093894433230162, 0.0005370270810090005, 0.0005248154629953206, 0.0005544712767004967, 0.0004965217667631805, 0.0005223823827691376, 0.0004271815705578774, 0.00048037830856628716, 0.0005262921331450343, 0.000502420705743134, 0.0005216776626184583, 0.0004371142422314733, 0.0005333455628715456, 0.0004451684362720698, 0.0004901443608105183, 0.0005570610519498587, 0.0005937032983638346, 0.0004604374698828906, 0.0005270502879284322, 0.0004994940827600658, 0.0005024017882533371, 0.0005015493952669203, 0.00043853025999851525, 0.0005055943038314581, 0.00047518988139927387, 0.0004519264039117843, 0.0005062638665549457, 0.00043058025767095387, 0.0005000657984055579, 0.0005255412543192506, 0.00044146881555207074, 0.00043316424125805497, 0.00047773472033441067, 0.0004515398759394884, 0.0005654917913489044, 0.0004513293388299644, 0.0004579783126246184, 0.0005084389704279602, 0.0005229474627412856, 0.00048585262265987694, 0.0005545402527786791, 0.0004598087980411947, 0.0005605708574876189, 0.00041985182906500995, 0.0005541013670153916, 0.0005324509111233056, 0.00048265012446790934, 0.0004988969885744154, 0.0005065970472060144, 0.00032676427508704364, 0.00044984265696257353, 0.00045010907342657447, 0.0004877963219769299, 0.0005007812869735062, 0.0004977703792974353, 0.0005235855933278799, 0.0004747894417960197, 0.00047687397454865277, 0.0004535217594821006, 0.00040901266038417816, 0.000539031287189573, 0.00048443765263073146, 0.0004943628446199, 0.00053973967442289, 0.0004442621720954776, 0.0005189260118640959, 0.000460705254226923, 0.0005132233491167426, 0.0005331200663931668, 0.0005344933597370982, 0.00048306092503480613, 0.00048623394104652107, 0.0005448074080049992, 0.0004815042775589973, 0.0005328489351086318, 0.0004911014111712575, 0.0005573483649641275, 0.000481952796690166, 0.000507089716847986, 0.0004204096912872046, 0.0004683196311816573, 0.0005485250148922205, 0.0004385716456454247, 0.00047733908286318183, 0.000539355562068522, 0.0004178939270786941, 0.0006441503064706922, 0.0005203194450587034, 0.0004881843051407486, 0.0005789173883385956, 0.0005041464464738965, 0.0004531155282165855, 0.0004645406443160027, 0.0005627991049550474, 0.0005088396719656885, 0.00048499080003239214, 0.0005140727153047919, 0.00041283926111645997, 0.0004978357465006411, 0.00044009010889567435, 0.00044241599971428514, 0.0004931417061015964, 0.00048618047730997205, 0.0005245918873697519, 0.0004481715150177479, 0.00048354684258811176, 0.00046398097765631974, 0.000534321996383369, 0.0005101028364151716, 0.00042843964183703065, 0.0005062707350589335, 0.0005242679035291076, 0.0004949470749124885, 0.0004969787551090121, 0.0005588039639405906, 0.0005256374133750796, 0.0004632488708011806, 0.00047449543490074575, 0.0004571761528495699, 0.0004437767493072897, 0.00045516432146541774, 0.0005199240986257792, 0.0005053330096416175, 0.0004971472662873566, 0.00041217493708245456, 0.00046027175267226994, 0.0005436206120066345, 0.00043138014734722674, 0.0004912394215352833, 0.0003989461692981422, 0.0004928899579681456, 0.0005568721098825336, 0.0004944745451211929, 0.0005482551641762257, 0.0005327951512299478, 0.0005015977076254785, 0.000507103861309588, 0.0004378189623821527, 0.0004804678028449416, 0.00048457205411978066, 0.000534279621206224, 0.0005188476643525064, 0.0004909708513878286, 0.00047184768482111394, 0.00043212974560447037, 0.0004935602191835642, 0.0004779179289471358, 0.0004558594082482159, 0.0004581824759952724, 0.00043301002006046474, 0.0005480414838530123, 0.000445745448814705, 0.00043748231837525964, 0.0005079349502921104, 0.0005712443380616605, 0.0005020120879635215, 0.00048607680946588516, 0.00046245739213190973, 0.00047389985411427915, 0.0004406812076922506, 0.00043912066030316055, 0.0005347057012841105, 0.0004617037484422326, 0.00042882384150289, 0.00042868053424172103, 0.0005191080854274333, 0.0005347439437173307, 0.0005494131473824382, 0.0003957085427828133, 0.0005017377552576363, 0.0004903592634946108, 0.0005631044623441994, 0.0004778810543939471, 0.0004289340286049992, 0.0005313578294590116, 0.0005617412971332669, 0.00046786112943664193, 0.00047012887080200016, 0.0005378890200518072, 0.00048188757500611246, 0.0003970246762037277, 0.0005120806745253503, 0.0005741002387367189, 0.0005652605323120952, 0.00046157228644005954, 0.0004311544762458652, 0.0004901877837255597, 0.0005100014386698604, 0.0005380299990065396, 0.0005490764742717147, 0.0005057157832197845, 0.00042101921280846, 0.0004933105665259063, 0.00045214081183075905, 0.00045168143697082996, 0.0005309867556206882, 0.000466611614683643, 0.0005054518114775419, 0.00048563192831352353, 0.0004477398470044136, 0.0005378244677558541, 0.0004917882615700364, 0.0004856497689615935, 0.00047794857528060675, 0.00048525980673730373, 0.00040978059405460954, 0.00047080262447707355, 0.000449820829089731, 0.0003892709210049361, 0.0005211986135691404, 0.0004931259900331497, 0.0004921245854347944, 0.0004294556856621057, 0.000501893344335258, 0.000493981409817934, 0.0005260124453343451, 0.0005619622534140944, 0.0004536570340860635, 0.0005062176496721804, 0.0004383640189189464, 0.0005035049980506301, 0.0004930675495415926, 0.0005223772022873163, 0.00047501642256975174, 0.0005732897552661598, 0.0005137150874361396, 0.0004509651043917984, 0.00045885148574598134, 0.0004997594514861703, 0.0005115687381476164, 0.00043332669883966446, 0.0004533088649623096, 0.0005179492873139679, 0.00043006279156543314, 0.00044148016604594886, 0.0004926486290059984, 0.0004823421186301857, 0.0005440893000923097, 0.0004912259173579514, 0.0004045086388941854, 0.00044731085654348135, 0.00046028484939597547, 0.0004635802179109305, 0.00045055142254568636, 0.0004508756101131439, 0.0004915547324344516, 0.0005433423793874681, 0.00039220662438310683, 0.000569047755561769, 0.0005209182272665203, 0.00044042171793989837, 0.00047742482274770737, 0.0004508216225076467, 0.0005078262183815241, 0.0004531049053184688, 0.00043103774078190327, 0.0004115139599889517, 0.0005609974032267928, 0.0005989386118017137, 0.0004306068003643304, 0.0004971266607753932, 0.0004634004144463688, 0.0005042433040216565, 0.0005197127466090024, 0.0004889905685558915, 0.0004804045020136982, 0.0004908302216790617, 0.0004971562302671373, 0.0005235248827375472, 0.0004970257286913693, 0.0004699746205005795, 0.00043538707541301847, 0.0005123034352436662, 0.0005406619748100638, 0.00042467613820917904, 0.0005082240677438676, 0.0005489680333994329, 0.0004645379667636007, 0.0005015628412365913, 0.00045378535287454724, 0.0004983989638276398, 0.0005369361606426537, 0.0004823000344913453, 0.00043789728078991175, 0.0004875015001744032, 0.00043028275831602514, 0.0004905628156848252, 0.0005289990804158151, 0.00048750912537798285, 0.0004608396557159722, 0.0004966538399457932, 0.00046406936598941684, 0.00047681087744422257, 0.00042051501804962754, 0.00045960399438627064, 0.0006512951222248375, 0.0005247777444310486, 0.00042882064008153975, 0.0004970247391611338, 0.0004414931172505021, 0.00046653521712869406, 0.0004930851282551885, 0.0004993620095774531, 0.00045761483488604426, 0.0004383958294056356, 0.00046910179662518203, 0.00043674258631654084, 0.0005129860364831984, 0.0004952896269969642, 0.000473670574137941, 0.0004609761817846447, 0.0005073077627457678, 0.0004330894735176116, 0.0005829675355926156, 0.00048215806600637734, 0.0004923430969938636, 0.0004893640289083123, 0.0005254754214547575, 0.00046423522871918976, 0.000510192709043622, 0.0004991471651010215, 0.0005258399760350585, 0.00047576462384313345, 0.000563389272429049, 0.00048679846804589033, 0.0005242246552370489, 0.00046631909208372235, 0.0004911911673843861, 0.000531013822183013, 0.0004667533212341368, 0.00045493608922697604, 0.0004788025689776987, 0.0005247836816124618, 0.0005157922278158367, 0.0005417067441157997, 0.0005715186125598848, 0.0004017629544250667, 0.0005366363911889493, 0.00048668243107385933, 0.0004987153806723654, 0.0004873461148235947, 0.000444031844381243, 0.00045852328184992075, 0.0004974253824912012, 0.00045060497359372675, 0.0004890318377874792, 0.000485134863993153, 0.0005245708161965013, 0.00044961320236325264, 0.0004898880724795163, 0.0004817206063307822, 0.0004658928082790226, 0.0005524135194718838, 0.0004564645641949028, 0.0004825600190088153, 0.00048154650721699, 0.0004928073612973094, 0.00054165069013834, 0.00055426120525226, 0.0004556055355351418, 0.000543035683222115, 0.0004980405792593956, 0.0004173465713392943, 0.0005259835161268711, 0.000471730949357152, 0.0005446597933769226, 0.0004572735051624477, 0.0005300429183989763, 0.0005561557481996715, 0.000559195235837251, 0.0005011609755456448, 0.0004772497632075101, 0.000424628728069365, 0.0005006551509723067, 0.0005092144128866494, 0.000514249608386308, 0.000466494559077546, 0.0004821841139346361, 0.0005889812018722296, 0.0004928639391437173, 0.0005953499348834157, 0.0005360969807952642, 0.0005065178847871721, 0.000501512608025223, 0.0004907195107080042, 0.000519131834153086, 0.0004834055434912443, 0.0004829264944419265, 0.0004687068285420537, 0.0005139635759405792, 0.0003926675708498806, 0.00043920372263528407, 0.000515524297952652, 0.0005593392997980118, 0.0004925356479361653, 0.00046057230792939663, 0.0004761131713166833, 0.00046890220255590975, 0.0005207581561990082, 0.0004656526434700936, 0.00041238401900045574, 0.0004767684731632471, 0.0004559032677207142, 0.000471273175207898, 0.0005165591137483716, 0.0005043083801865578, 0.00046484440099447966, 0.0004318627470638603, 0.00047860571066848934, 0.0005049968021921813, 0.00047568624722771347, 0.0004749258514493704, 0.0004845734219998121, 0.000443890254246071, 0.0004916852922178805, 0.000470862869406119, 0.00040804914897307754, 0.0004499811620917171, 0.000404939433792606, 0.00044889230048283935, 0.00044663515291176736, 0.0004604117712005973, 0.00045519188279286027, 0.0004598824889399111, 0.0005112686776556075, 0.0005309272091835737, 0.0005328487022779882, 0.0004482343792915344, 0.0004697312833741307, 0.0005150840152055025, 0.0005666824872605503, 0.00048517456161789596, 0.0005602327873930335, 0.0005511678755283356, 0.000467974750790745, 0.0004589320451486856, 0.00047330802772194147, 0.000460355106042698, 0.0004883487708866596, 0.0005274465074762702, 0.0005215276614762843, 0.000508480123244226, 0.0005113144288770854, 0.0004976927884854376, 0.0005346097168512642, 0.0005342485383152962, 0.0005099764675833285, 0.0005016394425183535, 0.0004603070265147835, 0.0005323796067386866, 0.0005318300100043416, 0.0005493225762620568, 0.00048063602298498154, 0.0005188158247619867, 0.0004582150431815535, 0.00048703490756452084, 0.0004788377264048904, 0.0003803037398029119, 0.0005217515281401575, 0.0005344723467715085, 0.0005050949985161424, 0.0004372938710730523, 0.00038947194116190076, 0.0005551464855670929, 0.00047250656643882394, 0.0004661768034566194, 0.000570307020097971, 0.0004779375740326941, 0.0004949806607328355, 0.0005226311623118818, 0.0004755095287691802, 0.0005403815885074437, 0.0004988178261555731, 0.00047600039397366345, 0.0005496923695318401, 0.0005124007002450526, 0.0004233982181176543, 0.000527706986758858, 0.00046919984742999077, 0.0004727788036689162, 0.0005498410318978131, 0.00043023619218729436, 0.000461956748040393, 0.000530701654497534, 0.0005075297667644918, 0.0005539424018934369, 0.00042131621739827096, 0.0005355451139621437, 0.0004809305246453732, 0.0004549959267023951, 0.0004595261998474598, 0.0004943440435454249, 0.00047950484440661967, 0.00046044960618019104, 0.000504219438880682, 0.00044004982919432223, 0.0005041941185481846, 0.0004140756500419229, 0.0005310251144692302, 0.0004661021230276674, 0.0005138233536854386, 0.0004668603069148958, 0.0004928532289341092, 0.0004902274813503027, 0.0004576507199089974, 0.0005139966960996389, 0.0005857035284861922, 0.00047295293188653886, 0.000417630304582417, 0.0005713053396902978, 0.0005200491286814213, 0.0004397557058837265, 0.0005307739484123886, 0.0005183910834603012, 0.0004737509589176625, 0.00048020452959463, 0.0004878750187344849, 0.0004932563751935959, 0.0005125278839841485, 0.000522403686773032, 0.0005092900828458369, 0.00044265788164921105, 0.0004611279000528157, 0.0005678364541381598, 0.0004696525866165757, 0.0005081709241494536, 0.0005258017918094993, 0.00047426889068447053, 0.0004567533324006945, 0.0004458932380657643, 0.000502134149428457, 0.0005350655410438776, 0.0005281370249576867, 0.0004807607620023191, 0.0005028139567002654, 0.0005132247461006045, 0.0005534033989533782, 0.0005415574996732175, 0.0004735838156193495, 0.00049899413716048, 0.00038175436202436686, 0.0005076517118141055, 0.0003934816922992468, 0.0006024733302183449, 0.0004918029299005866, 0.00046581754577346146, 0.0005592358065769076, 0.00047655313392169774, 0.0005302613135427237, 0.0005796414916403592, 0.00037358238478191197, 0.0004386677173897624, 0.000435938622104004, 0.0005038620438426733, 0.0005011848988942802, 0.00040517220622859895, 0.0004961749655194581, 0.0004981867386959493, 0.00047452645958401263, 0.0004827925586141646, 0.0004667968023568392, 0.0004859890614170581, 0.0004886541864834726, 0.0004973737522959709, 0.00047380555770359933, 0.0004905749810859561, 0.0004407369706314057, 0.0005044059944339097, 0.0005404438707046211, 0.00047673247172497213, 0.00043895968701690435, 0.0004684512095991522, 0.0004343972250353545, 0.00038922304520383477, 0.0005222129402682185, 0.00048775176401250064, 0.0005413967883214355, 0.0005158474086783826, 0.0005300398916006088, 0.000454763910965994, 0.0005025669233873487, 0.0004977146163582802, 0.00048346008406952024, 0.0004881853237748146, 0.0005039220559410751, 0.0005189196090213954, 0.00043022376485168934, 0.0005160981672815979, 0.0004629935137927532, 0.0004986888379789889, 0.0005392752937041223, 0.00043005787301808596, 0.0004325154877733439, 0.0005247546941973269, 0.00048085564048960805, 0.0005371504812501371, 0.0004985120613127947, 0.0004496012406889349, 0.00043450307566672564, 0.000525549694430083, 0.0005339882336556911, 0.00047317807911895216, 0.0004621152766048908, 0.0005215829005464911, 0.00047868743422441185, 0.0005371685838326812, 0.000510354177094996, 0.0004510707804001868, 0.00044741888996213675, 0.00048614005208946764, 0.00042504066368564963, 0.0004838725726585835, 0.000516117608640343, 0.00045435651554726064, 0.0005174889229238033, 0.000496949884109199, 0.0004853954305872321, 0.0005303589859977365, 0.0004847091040574014, 0.0005140369175933301, 0.00045174284605309367, 0.0005489482427947223, 0.00044967280700802803, 0.000484146032249555, 0.0004765912890434265, 0.00057826261036098, 0.0004981808015145361, 0.0004823708441108465, 0.0006276897620409727, 0.00046166873653419316, 0.0004636676167137921, 0.0004882767971139401, 0.0004815462161786854, 0.0004374330455902964, 0.000555536535102874, 0.0005009726737625897, 0.0004964162944816053, 0.0004588013398461044, 0.0005279720644466579, 0.0005071996129117906, 0.0005385339609347284, 0.0005021492252126336, 0.0005264994688332081, 0.000459243543446064, 0.0004406595544423908, 0.0005393028259277344, 0.0004445314116310328, 0.0004924025852233171, 0.000478510424727574, 0.00042336125625297427, 0.0004986501880921423, 0.000428461586125195, 0.0005436547799035907, 0.0004868975083809346, 0.0005095265223644674, 0.00045649096136912704, 0.0004751335072796792, 0.00038997794035822153, 0.0004951239679940045, 0.0005842035170644522, 0.0005082334391772747, 0.0005114421946927905, 0.0005109329940751195, 0.0005320446216501296, 0.0005292975693009794, 0.00047843478387221694, 0.0005129398196004331, 0.00047605641884729266, 0.0004785419732797891, 0.0005761407082900405, 0.0005003017722629011, 0.00044106077984906733, 0.0005526655004359782, 0.00047439735499210656, 0.00044410370173864067, 0.000468755722977221, 0.0005085858283564448, 0.0004592950572259724, 0.0005117579712532461, 0.0004833499842789024, 0.0005032085464335978, 0.00046617098269052804, 0.0005121413851156831, 0.000433338776929304, 0.0005724627990275621, 0.0005216054269112647, 0.000436623755376786, 0.0004591302713379264, 0.0005695213912986219, 0.00044971483293920755, 0.0004562927642837167, 0.0005413505132310092, 0.00048029006575234234, 0.00045778739149682224, 0.0005146070034243166, 0.0004369716625660658, 0.0005237869336269796, 0.0004505543038249016, 0.0005007826257497072, 0.00043965724762529135, 0.0004449400876183063, 0.0004572965553961694, 0.0004961933009326458, 0.0004852665588259697, 0.0003835540555883199, 0.00047893793089315295, 0.0005349669954739511, 0.0004928133566863835, 0.0004539165529422462, 0.0005362522788345814, 0.0004521663358900696, 0.000506846234202385, 0.0005604861071333289, 0.0004498977796174586, 0.00041036601760424674, 0.0005500404513441026, 0.0005046452861279249, 0.0005620315787382424, 0.0005132470396347344, 0.0004729103238787502, 0.00045948298065923154, 0.0004962749080732465, 0.000501861039083451, 0.0005657629808411002, 0.0004931257572025061, 0.0005189147195778787, 0.00047350709792226553, 0.0004323307366576046, 0.0004335249832365662, 0.0004519911017268896, 0.0005115704261697829, 0.0005156129482202232, 0.000477132125524804, 0.0004717680858448148, 0.0005742636858485639, 0.00046349153853952885, 0.0004798184090759605, 0.00045838404912501574, 0.000490729056764394, 0.0004998971126042306, 0.0005040563992224634, 0.0006000401917845011, 0.0005086812889203429, 0.0005433854530565441, 0.00044538205838762224, 0.0005974546656943858, 0.0004495919856708497, 0.0005413945182226598, 0.0004566829593386501, 0.0005112726939842105, 0.0003939938032999635, 0.0004696403630077839, 0.0004230653285048902, 0.00058042659657076, 0.0004906100803054869, 0.00046400807332247496, 0.0004809417587239295, 0.0005058967508375645, 0.00045056274393573403, 0.00048655946739017963, 0.0004901397624053061, 0.0004952451563440263, 0.000531145662534982, 0.0004936253535561264, 0.0005352716543711722, 0.0005082649295218289, 0.0004892709548585117, 0.000506101525388658, 0.0005382572417147458, 0.00044931110460311174, 0.000499491230584681, 0.00047960338997654617, 0.00044836572487838566, 0.0004585594870150089, 0.0004941070801578462, 0.0004786904901266098, 0.00047171732876449823, 0.0005244850763119757, 0.0004526736738625914, 0.0005114856758154929, 0.0004660396371036768, 0.0004431776178535074, 0.0005429330049082637, 0.00048743237857706845, 0.00046126038068905473, 0.0004973293398506939, 0.0005376244662329555, 0.0005808457499369979, 0.0005176698323339224, 0.0005074289510957897, 0.000548185664229095, 0.00045503387809731066, 0.0005516565288417041, 0.0004406458465382457, 0.0005237986915744841, 0.0004937131307087839, 0.00042252836283296347, 0.0004711216897703707, 0.0005658940062858164, 0.00046632500016130507, 0.0005059718387201428, 0.0005183551693335176, 0.0004837486776523292, 0.0005168231436982751, 0.0004908453556708992, 0.0006155841983854771, 0.0005519814440049231, 0.00042612195829860866, 0.00046594353625550866, 0.0004891844582743943, 0.00044596154475584626, 0.0005195671692490578, 0.0004044364904984832, 0.0005351470899768174, 0.00048578635323792696, 0.0004215878143440932, 0.000505918578710407, 0.0005096639506518841, 0.00042208691593259573, 0.0005450663738884032, 0.000449358019977808, 0.0005400567315518856, 0.0005370953003875911, 0.0005478899111039937, 0.0004570750461425632, 0.000432527216617018, 0.0004967956338077784, 0.00045662856427952647, 0.00043595427996478975, 0.0005344902165234089, 0.0005605625337921083, 0.0004691501962952316, 0.0004336400597821921, 0.0004472019209060818, 0.00043206207919865847, 0.0004362564068287611, 0.0005282111233100295, 0.0004792375839315355, 0.0004790087987203151, 0.00045320496428757906, 0.0004428582906257361, 0.0004559649678412825, 0.0005169654614292085, 0.0004546957788988948, 0.0005333438748493791, 0.0005031012697145343, 0.0005416401545517147, 0.0005112869548611343, 0.0005041999975219369, 0.0006299151573330164, 0.0004978683427907526, 0.0005580539582297206, 0.00045817712089046836, 0.0005389151046983898, 0.000495548767503351, 0.00041180438711307943, 0.00048538667033426464, 0.0004466203972697258, 0.00043058639857918024, 0.000489975675009191, 0.0004690371861215681, 0.0004828657547477633, 0.00046805135207250714, 0.0004863814392592758, 0.0005286718951538205, 0.000419295160099864, 0.0004490161081776023, 0.00044768067891709507, 0.00042996220872737467, 0.0005439127562567592, 0.0004562858084682375, 0.00046198946074582636, 0.0004935708711855114, 0.000470249739009887, 0.00047171805636025965, 0.00046720696263946593, 0.0004922763910144567, 0.0004842283087782562, 0.0005017476505599916, 0.0004491303116083145, 0.0005299951299093664, 0.00043820595601573586, 0.0004585683345794678, 0.0005808986607007682, 0.0005516909877769649, 0.00045822595711797476, 0.0004923712112940848, 0.0004698093980550766, 0.00047820236068218946, 0.000581823056563735, 0.00043836378608830273, 0.0004300978034734726, 0.00047829424147494137, 0.0005116566317155957, 0.00045028835302218795, 0.0004836521693505347, 0.00048786477418616414, 0.0004243847506586462, 0.00047173514030873775, 0.0005067427409812808, 0.00048424076521769166, 0.0006320216343738139, 0.0005055853398516774, 0.000473085034172982, 0.0005418180953711271, 0.0004550793382804841, 0.0004917093319818377, 0.0005354477907530963, 0.00046906564966775477, 0.0004361251194495708, 0.0004996893112547696, 0.0004471682186704129, 0.0004521553055383265, 0.0005020308890379965, 0.0004611443146131933, 0.0005490590701811016, 0.0004929286078549922, 0.0005067932652309537, 0.0005611194646917284, 0.00047106639249250293, 0.0004956409684382379, 0.0004677104589063674, 0.00044945842819288373, 0.0004893930163234472, 0.00043548006215132773, 0.00044326152419671416, 0.000487539975438267, 0.0004612906195688993, 0.00046766072046011686, 0.0005096436943858862, 0.000403676851419732, 0.0005316540482454002, 0.0004692005168180913, 0.00046778793330304325, 0.0006066193454898894, 0.0004811942926608026, 0.00050634442595765, 0.0004707752668764442, 0.0005273218266665936, 0.0005260713514871895, 0.0005285646766424179, 0.0004506288096308708, 0.0004956001648679376, 0.0005057121743448079, 0.0004226034216117114, 0.0005247683147899806, 0.0005346823018044233, 0.000488724559545517, 0.0004942417726852, 0.0006013049278408289, 0.0004613840428646654, 0.0004417377058416605, 0.0004747459606733173, 0.00041292188689112663, 0.0005218839505687356, 0.0005137193365953863, 0.0005187203059904277, 0.0006091328687034547, 0.0004463068617042154, 0.0005171652301214635, 0.0004879070329479873, 0.0004733219975605607, 0.0005354998866096139, 0.00037816158146597445, 0.0005452755140140653, 0.0005341593641787767, 0.00047177777742035687, 0.00040837834239937365, 0.0005080658593215048, 0.0005075164954178035, 0.0005215344717726111, 0.0004939454374834895, 0.0004598707309924066, 0.0005142982699908316, 0.0005070344195701182, 0.000484721502289176, 0.00045824641711078584, 0.00046700859093107283, 0.0004529351135715842, 0.0005046342848800123, 0.00047925885883159935, 0.00043902985635213554, 0.0005196003476157784, 0.0004799836897291243, 0.00046771796769462526, 0.00047816321603022516, 0.0004208028258290142, 0.0005044767167419195, 0.0005039942916482687, 0.0005626188940368593, 0.0005608591018244624, 0.00043365111923776567, 0.0004906592075712979, 0.0004560099623631686, 0.0005316325114108622, 0.0005390098667703569, 0.0004592291952576488, 0.0005264522042125463, 0.0004586150753311813, 0.0004142107500229031, 0.0004631359479390085, 0.0004916103207506239, 0.0005187841015867889, 0.0004755544359795749, 0.0005306314560584724, 0.0004564046103041619, 0.00048327705007977784, 0.0004935288452543318, 0.0005092351930215955, 0.00047743666800670326, 0.0005388740100897849, 0.0005276708398014307, 0.000498797046020627, 0.0003979481116402894, 0.00047765401541255414, 0.00043984316289424896, 0.00048587608034722507, 0.0004460967902559787, 0.0004693195514846593, 0.0005145989125594497, 0.0004557580396067351, 0.0004969377187080681, 0.0005600836011581123, 0.0005616390844807029, 0.0005408079596236348, 0.00038507863064296544, 0.00045773445162922144, 0.0004557143256533891, 0.00046562953502871096, 0.0004809863166883588, 0.0004720930883195251, 0.00043752146302722394, 0.00040579610504209995, 0.0005091861239634454, 0.0004339540028013289, 0.0005267223459668458, 0.0004988248692825437, 0.00046825295430608094, 0.00046803330769762397, 0.00047005637316033244, 0.00045936356764286757, 0.0005628134240396321, 0.00046404238673858345, 0.0004670174093917012, 0.00046783656580373645, 0.0004983014077879488, 0.0005189955118112266, 0.0004919779603369534, 0.0004944300162605941, 0.0004855933366343379, 0.00043889714288525283, 0.000504055293276906, 0.00047155821812339127, 0.00045919662807136774, 0.000452319101896137, 0.0004970956942997873, 0.0005636839778162539, 0.0005412875325419009, 0.0004520927323028445, 0.0005072923377156258, 0.00052405777387321, 0.00044626419548876584, 0.000458654947578907, 0.0005013634799979627, 0.000452943000709638, 0.0005260237376205623, 0.000410526932682842, 0.000453727669082582, 0.00047411961713805795, 0.0004871714045293629, 0.00048617064021527767, 0.0004713972157333046, 0.00047864916268736124, 0.00042945233872160316, 0.00041788016096688807, 0.0004613065393641591, 0.00043866291525773704, 0.0004422024358063936, 0.0005941332201473415, 0.00046468398068100214, 0.00046272858162410557, 0.0004858161846641451, 0.0005235056742094457, 0.00047631870256736875, 0.00047853204887360334, 0.00046848892816342413, 0.0004855476145166904, 0.00045269139809533954, 0.00040423215250484645, 0.0004522797535173595, 0.0005005523562431335, 0.00041470114956609905, 0.0004088201967533678, 0.0004593023913912475, 0.0005216829013079405, 0.0005527667817659676, 0.0005148442578501999, 0.0005266563966870308, 0.0005218167789280415, 0.0004972397000528872, 0.0005072494968771935, 0.0005002996767871082, 0.00048536324175074697, 0.0004166954313404858, 0.00047305223415605724, 0.0004080511862412095, 0.0004832337726838887, 0.0005094034131616354, 0.0004968071589246392, 0.000480740302009508, 0.0004567443684209138, 0.00042600627057254314, 0.0004911806900054216, 0.00046770769404247403, 0.0004558570799417794, 0.00043774693040177226, 0.0005364649696275592, 0.0005026051076129079, 0.00044646961032412946, 0.0005317170289345086, 0.000483310897834599, 0.00047658555558882654, 0.0004587110597640276, 0.0005404122639447451, 0.0005341764772310853, 0.0005081892595626414, 0.0005429866723716259, 0.000473306339699775, 0.0005119628622196615, 0.0004472190048545599, 0.00044804217759519815, 0.00045889176544733346, 0.0005468728486448526, 0.0005081251147203147, 0.0005120229907333851, 0.0005263999337330461, 0.0004824993957299739, 0.00044966780114918947, 0.0004279793647583574, 0.0004338567960076034, 0.00046464172191917896, 0.0004749821964651346, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]], "dtype": "float32"}}, {"a": [{"__nd__": [[2.0]], "dtype": "float32"}, {"__nd__": [1], "dtype": "int64"}, 3.0], "e": {"__nd__": [[1.0]], "dtype": "float32"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(masked_softmax, json.loads(_TESTS), 'triton', *[1e-05, 1e-06])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")